In [ ]:
# InGame Score Prediction Model Notebook 5: Feature Analysis, Validation & Visualization

In [ ]:
# Cell 4A: Feature Engineering Analysis and Validation

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

print("=" * 80)
print("PART 1: FEATURE CORRELATION ANALYSIS")
print("=" * 80)

def analyze_feature_correlations(features_df, threshold=0.8, figsize=(20, 16)):
    """
    Analyze and visualize feature correlations with advanced metrics including
    non-linear relationships and statistical significance testing.
    
    Args:
        features_df: DataFrame with calculated features
        threshold: Correlation threshold to highlight
        figsize: Figure size for the heatmap
        
    Returns:
        Dictionary with correlation results and feature relationships
    """
    from scipy import stats
    import seaborn as sns
    
    # Select only numeric columns for correlation analysis
    numeric_features = features_df.select_dtypes(include=['int64', 'float64'])
    
    print(f"Analyzing correlations between {len(numeric_features.columns)} numeric features")
    
    # Calculate Pearson (linear) correlation matrix
    pearson_corr = numeric_features.corr(method='pearson')
    
    # Calculate Spearman (rank) correlation matrix for non-linear relationships
    spearman_corr = numeric_features.corr(method='spearman')
    
    # Identify differences between Pearson and Spearman
    # Large differences indicate non-linear relationships
    correlation_diff = np.abs(pearson_corr - spearman_corr)
    
    # Create a mask for the upper triangle of the correlation matrix
    mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
    
    # Set up the matplotlib figure for Pearson correlations
    plt.figure(figsize=figsize)
    
    # Create a custom diverging colormap
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    
    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(pearson_corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
                square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")
    
    plt.title('Feature Correlation Matrix (Pearson - Linear)', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Find highly correlated features
    high_corr_pairs = {}
    for i in range(len(pearson_corr.columns)):
        for j in range(i):
            if abs(pearson_corr.iloc[i, j]) > threshold:
                feat_i = pearson_corr.columns[i]
                feat_j = pearson_corr.columns[j]
                high_corr_pairs[(feat_i, feat_j)] = pearson_corr.iloc[i, j]
    
    # Print the results
    if high_corr_pairs:
        print(f"\nHighly correlated features (|r| > {threshold}):")
        for (feat1, feat2), corr in sorted(high_corr_pairs.items(), key=lambda x: abs(x[1]), reverse=True):
            print(f"  • {feat1} and {feat2}: r = {corr:.3f}")
        
        # Group correlations by feature to identify redundant sets
        feature_groups = {}
        for (feat1, feat2), _ in high_corr_pairs.items():
            if feat1 not in feature_groups:
                feature_groups[feat1] = set()
            if feat2 not in feature_groups:
                feature_groups[feat2] = set()
            feature_groups[feat1].add(feat2)
            feature_groups[feat2].add(feat1)
        
        # Find groups with multiple high correlations
        print("\nPotential feature groups to simplify:")
        for feature, correlated in feature_groups.items():
            if len(correlated) > 1:
                print(f"  • {feature} is highly correlated with: {', '.join(correlated)}")
    else:
        print(f"No feature pairs with correlation above {threshold} found.")
    
    # Now also display the non-linear analysis if differences are significant
    if (correlation_diff > 0.2).any().any():
        print("\nDetected potential non-linear relationships. Displaying Spearman correlation matrix...")
        
        plt.figure(figsize=figsize)
        sns.heatmap(spearman_corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
                    square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")
        
        plt.title('Non-linear Correlation Matrix (Spearman)', fontsize=16)
        plt.tight_layout()
        plt.show()
        
        # Find significant non-linear relationships
        non_linear_relationships = []
        for i in range(len(correlation_diff.columns)):
            for j in range(i):
                if correlation_diff.iloc[i, j] > 0.2:  # Significant difference
                    feat_i = correlation_diff.columns[i]
                    feat_j = correlation_diff.columns[j]
                    non_linear_relationships.append((
                        feat_i, feat_j, 
                        pearson_corr.iloc[i, j],
                        spearman_corr.iloc[i, j],
                        correlation_diff.iloc[i, j]
                    ))
        
        if non_linear_relationships:
            print("\nPotential non-linear relationships found:")
            for feat1, feat2, pearson, spearman, diff in sorted(non_linear_relationships, key=lambda x: x[4], reverse=True):
                print(f"  • {feat1} and {feat2}: Pearson={pearson:.3f}, Spearman={spearman:.3f}, Diff={diff:.3f}")
    
    return high_corr_pairs
# =============================================================================
print("\n" + "=" * 80)
print("PART 2: EDGE CASE VALIDATION")
print("=" * 80)

def create_edge_case_tests():
    """
    Create a set of edge case scenarios to test feature generation robustness.
    
    Returns:
        DataFrame with various edge cases
    """
    print("Creating edge case test scenarios...")
    
    # Base test case (normal game)
    normal_case = {
        'game_id': 10001,
        'home_team': 'Boston Celtics',
        'away_team': 'Los Angeles Lakers',
        'game_date': pd.Timestamp('2023-01-15'),
        'home_q1': 28, 'home_q2': 26, 'home_q3': 30, 'home_q4': 29,
        'away_q1': 25, 'away_q2': 28, 'away_q3': 24, 'away_q4': 26,
        'home_score': 113,
        'away_score': 103,
        'current_quarter': 4,
        'description': 'Normal complete game'
    }
    
    # Edge case 1: Missing quarter data
    missing_quarters = normal_case.copy()
    missing_quarters.update({
        'game_id': 10002,
        'home_q3': None, 'home_q4': None,
        'away_q3': None, 'away_q4': None,
        'home_score': 54,
        'away_score': 53,
        'current_quarter': 2,
        'description': 'Missing Q3 and Q4 (game in progress)'
    })
    
    # Edge case 2: All quarters missing
    all_quarters_missing = normal_case.copy()
    all_quarters_missing.update({
        'game_id': 10003,
        'home_q1': None, 'home_q2': None, 'home_q3': None, 'home_q4': None,
        'away_q1': None, 'away_q2': None, 'away_q3': None, 'away_q4': None,
        'home_score': 0,
        'away_score': 0,
        'current_quarter': 0,
        'description': 'Pre-game (all quarters missing)'
    })
    
    # Edge case 3: Extreme momentum shift
    extreme_momentum = normal_case.copy()
    extreme_momentum.update({
        'game_id': 10004,
        'home_q1': 10, 'home_q2': 40, 'home_q3': 10, 'home_q4': 40,
        'away_q1': 40, 'away_q2': 10, 'away_q3': 40, 'away_q4': 10,
        'home_score': 100,
        'away_score': 100,
        'description': 'Extreme momentum shifts'
    })
    
    # Edge case 4: Game with no history (new matchup)
    new_matchup = normal_case.copy()
    new_matchup.update({
        'game_id': 10005,
        'home_team': 'Expansion Team A',
        'away_team': 'Expansion Team B',
        'description': 'New teams with no history'
    })
    
    # Edge case 5: Zero scoring quarter
    zero_quarter = normal_case.copy()
    zero_quarter.update({
        'game_id': 10006,
        'home_q3': 0,
        'away_q3': 0,
        'description': 'Quarter with zero points (Q3)'
    })
    
    # Edge case 6: Missing game date
    missing_date = normal_case.copy()
    missing_date.update({
        'game_id': 10007,
        'game_date': None,
        'description': 'Missing game date'
    })
    
    # Combine all test cases
    test_cases = [
        normal_case,
        missing_quarters,
        all_quarters_missing,
        extreme_momentum,
        new_matchup,
        zero_quarter,
        missing_date
    ]
    
    # Create DataFrame
    test_df = pd.DataFrame(test_cases)
    
    # Print test case summary
    print(f"Created {len(test_df)} test cases:")
    for i, case in enumerate(test_cases):
        print(f"  {i+1}. {case['description']}")
    
    return test_df

def validate_edge_cases(feature_generator, test_df):
    """
    Process edge cases through the feature generator and validate the output.
    
    Args:
        feature_generator: Instance of NBAFeatureGenerator
        test_df: DataFrame with test cases
        
    Returns:
        DataFrame with processed features
    """
    print("\nValidating feature generation for edge cases...")
    
    # Process the test cases
    processed_df = feature_generator.generate_all_features(test_df, skip_rest_calc=True)
    
    # Check results
    validation_results = []
    
    for idx, row in processed_df.iterrows():
        game_id = row['game_id']
        desc = row['description']
        
        # Get original test case
        orig = test_df[test_df['game_id'] == game_id].iloc[0]
        
        # Define checks for each case
        if 'Missing Q3 and Q4' in desc:
            momentum_valid = pd.notna(row['q1_to_q2_momentum']) and pd.isna(row['q2_to_q3_momentum'])
            current_q_valid = row['current_quarter'] == 2
            
        elif 'Pre-game' in desc:
            momentum_valid = all(row[[
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum'
            ]].isna() | (row[[
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum'
            ]] == 0))
            current_q_valid = row['current_quarter'] == 0
            
        elif 'Extreme momentum' in desc:
            q1_to_q2 = row['q1_to_q2_momentum']
            q2_to_q3 = row['q2_to_q3_momentum']
            momentum_valid = (q1_to_q2 > 0 and q2_to_q3 < 0) or (q1_to_q2 < 0 and q2_to_q3 > 0)
            current_q_valid = row['current_quarter'] == 4
            
        elif 'new history' in desc:
            history_valid = row['prev_matchup_diff'] == 0
            momentum_valid = True
            current_q_valid = True
            
        elif 'zero points' in desc:
            momentum_valid = row['q2_to_q3_momentum'] != 0  # Should reflect the shift
            current_q_valid = row['current_quarter'] == 4
            
        elif 'Missing game date' in desc:
            # Rest features should be defaulted
            rest_valid = (row['rest_days_home'] == 2 and row['rest_days_away'] == 2)
            momentum_valid = True
            current_q_valid = True
            
        else:  # Normal case
            momentum_valid = True
            current_q_valid = row['current_quarter'] == 4
        
        # Common validations for all cases
        quarter_sums_valid = abs((row['home_q1'] + row['home_q2'] + row['home_q3'] + row['home_q4']) - 
                                 row['home_score']) < 0.01
        first_half_valid = abs(row['first_half_diff'] - 
                              ((row['home_q1'] + row['home_q2']) - (row['away_q1'] + row['away_q2']))) < 0.01
        
        # Store validation result
        validation_results.append({
            'game_id': game_id,
            'description': desc,
            'momentum_features_valid': momentum_valid,
            'current_quarter_valid': current_q_valid,
            'quarter_sums_valid': quarter_sums_valid,
            'first_half_diff_valid': first_half_valid
        })
    
    # Create validation summary DataFrame
    validation_df = pd.DataFrame(validation_results)
    
    # Print validation summary
    print("\nValidation Results:")
    for idx, result in validation_df.iterrows():
        all_valid = all([
            result['momentum_features_valid'],
            result['current_quarter_valid'],
            result['quarter_sums_valid'],
            result['first_half_diff_valid']
        ])
        status = "✅ PASSED" if all_valid else "❌ FAILED"
        print(f"Test Case {idx+1}: {result['description']} - {status}")
        
        if not all_valid:
            print(f"  Failed checks:")
            if not result['momentum_features_valid']:
                print("  - Momentum features invalid")
            if not result['current_quarter_valid']:
                print("  - Current quarter invalid")
            if not result['quarter_sums_valid']:
                print("  - Quarter sums don't match game score")
            if not result['first_half_diff_valid']:
                print("  - First half differential incorrect")
    
    # Return both raw features and validation results
    return processed_df, validation_df

# Create and run edge case tests
edge_case_df = create_edge_case_tests()
test_generator = NBAFeatureGenerator(debug=True)
processed_edge_cases, validation_results = validate_edge_cases(test_generator, edge_case_df)

# Display detailed edge case results for the most interesting cases
print("\nDetailed Feature Values for Selected Edge Cases:")
interesting_cases = ['Missing Q3 and Q4', 'Extreme momentum', 'Missing game date']
for idx, row in processed_edge_cases.iterrows():
    if any(case in row['description'] for case in interesting_cases):
        print(f"\nTest Case: {row['description']}")
        print(f"  Current Quarter: {row['current_quarter']}")
        print(f"  Score: {row['home_team']} {row['home_score']} - {row['away_score']} {row['away_team']}")
        print(f"  Quarter Scores: {row['home_q1']}-{row['home_q2']}-{row['home_q3']}-{row['home_q4']} vs "
              f"{row['away_q1']}-{row['away_q2']}-{row['away_q3']}-{row['away_q4']}")
        print(f"  Momentum Features:")
        print(f"    Q1→Q2: {row.get('q1_to_q2_momentum', 'N/A')}, "
              f"Q2→Q3: {row.get('q2_to_q3_momentum', 'N/A')}, "
              f"Q3→Q4: {row.get('q3_to_q4_momentum', 'N/A')}")
        print(f"    Cumulative: {row.get('cumulative_momentum', 'N/A')}")
        print(f"  Critical Features:")
        print(f"    First Half Diff: {row.get('first_half_diff', 'N/A')}")
        print(f"    Pre-Q4 Diff: {row.get('pre_q4_diff', 'N/A')}")

print("\nFeature engineering validation complete!")

In [ ]:
# Cell 7: Early Quarter XGBoost Model Implementation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Early Quarter XGBoost Model Enhancement\n")

# Check for XGBoost availability
try:
    import xgboost as xgb
    print("XGBoost is available for use.")
    xgboost_available = True
except ImportError:
    print("XGBoost is not installed. Some functionality will be limited.")
    print("Install XGBoost with: pip install xgboost")
    xgboost_available = False

from models.features import EarlyQuarterModelOptimizer, NBAFeatureGenerator

# Load historical data for training
from sqlalchemy import create_engine
import config

print("Loading historical data for early quarter model training...")
engine = create_engine(config.DATABASE_URL)
query = """
SELECT * FROM nba_historical_game_stats 
WHERE game_date >= CURRENT_DATE - INTERVAL '365 days'
ORDER BY game_date
"""
historical_df = pd.read_sql(query, engine)
print(f"Loaded {len(historical_df)} historical games")

if len(historical_df) < 100:
    print("Not enough historical data for effective training.")
else:
    # Generate features
    feature_generator = NBAFeatureGenerator(debug=True)
    features_df = feature_generator.generate_all_features(historical_df)
    print(f"Generated features for {len(features_df)} games")
    
    # Split data into training and testing sets
    train_df, test_df = train_test_split(features_df, test_size=0.2, random_state=42)
    print(f"Training set: {len(train_df)} games, Test set: {len(test_df)} games")
    
    # Initialize the early quarter model optimizer
    optimizer = EarlyQuarterModelOptimizer(feature_generator=feature_generator)
    
    # Initialize XGBoost models (if available)
    if xgboost_available:
        optimizer.initialize_xgboost_models()
        
        # Train models
        print("\nTraining early quarter models...")
        training_results = optimizer.train_models(train_df, target_col='home_score')
        
        # Evaluate on test data
        print("\nEvaluating models on test data...")
        eval_results = optimizer.evaluate_models(test_df, target_col='home_score')
        
        if not eval_results.empty:
            # Display evaluation results
            print("\nEarly Quarter Model Performance:")
            display(eval_results)
            
            # Compare to baseline
            baseline_rmse = {
                "1": 7.0,  # Baseline RMSE for Q1
                "2": 6.2   # Baseline RMSE for Q2
            }
            comparison = optimizer.compare_to_baseline(eval_results, baseline_rmse)
            
            if not comparison.empty:
                print("\nImprovement over baseline:")
                display(comparison[['quarter', 'model', 'feature_set', 'rmse', 'baseline_rmse', 'pct_improvement']])
                
                # Visualize comparison
                plt.figure(figsize=(12, 6))
                
                # Plot Q1 models
                q1_data = comparison[comparison['quarter'] == 'Q1']
                if not q1_data.empty:
                    plt.subplot(1, 2, 1)
                    plt.bar(q1_data['model'] + '_' + q1_data['feature_set'], q1_data['rmse'], color='cornflowerblue')
                    plt.axhline(y=baseline_rmse["1"], color='r', linestyle='--', 
                               label=f'Baseline: {baseline_rmse["1"]:.2f}')
                    plt.title('Q1 Model Performance')
                    plt.ylabel('RMSE')
                    plt.xticks(rotation=45, ha='right')
                    plt.legend()
                    plt.grid(axis='y', alpha=0.3)
                
                # Plot Q2 models
                q2_data = comparison[comparison['quarter'] == 'Q2']
                if not q2_data.empty:
                    plt.subplot(1, 2, 2)
                    plt.bar(q2_data['model'] + '_' + q2_data['feature_set'], q2_data['rmse'], color='lightcoral')
                    plt.axhline(y=baseline_rmse["2"], color='r', linestyle='--', 
                               label=f'Baseline: {baseline_rmse["2"]:.2f}')
                    plt.title('Q2 Model Performance')
                    plt.ylabel('RMSE')
                    plt.xticks(rotation=45, ha='right')
                    plt.legend()
                    plt.grid(axis='y', alpha=0.3)
                
                plt.tight_layout()
                plt.show()
                
                # Visualize feature importance for best model
                print("\nFeature Importance Analysis:")
                
                # Best Q1 model feature importance
                best_q1 = q1_data.sort_values('rmse').iloc[0]
                print(f"Best Q1 Model: {best_q1['model']}_{best_q1['feature_set']} (RMSE: {best_q1['rmse']:.2f})")
                q1_importance_fig = optimizer.visualize_feature_importance(1)
                if q1_importance_fig:
                    plt.figure(q1_importance_fig.number)
                    plt.show()
                
                # Best Q2 model feature importance
                best_q2 = q2_data.sort_values('rmse').iloc[0]
                print(f"Best Q2 Model: {best_q2['model']}_{best_q2['feature_set']} (RMSE: {best_q2['rmse']:.2f})")
                q2_importance_fig = optimizer.visualize_feature_importance(2)
                if q2_importance_fig:
                    plt.figure(q2_importance_fig.number)
                    plt.show()
                
                # Test the best models on a sample game
                print("\nTesting best models on sample game data...")
                sample_game = test_df.iloc[0].copy()
                
                # Q1 prediction
                q1_model = best_q1['model']
                q1_feature_set = best_q1['feature_set']
                q1_pred = optimizer.predict_quarter(sample_game, 1, model_name=q1_model, feature_set=q1_feature_set)
                
                # Q2 prediction
                q2_model = best_q2['model']
                q2_feature_set = best_q2['feature_set']
                q2_pred = optimizer.predict_quarter(sample_game, 2, model_name=q2_model, feature_set=q2_feature_set)
                
                print(f"Sample Game: {sample_game['home_team']} vs {sample_game['away_team']}")
                print(f"Actual Q1 score: {sample_game['home_q1']}, Predicted: {q1_pred:.2f}")
                print(f"Actual Q2 score: {sample_game['home_q2']}, Predicted: {q2_pred:.2f}")
                
                # Save the best models
                print("\nSaving best early quarter models...")
                optimizer.save_models()
    else:
        print("Cannot train XGBoost models. Please install XGBoost first.")

def create_chronological_split(df, test_fraction=0.2, date_column='game_date'):
    """
    Create chronologically ordered train-test split to prevent data leakage.
    
    Args:
        df: DataFrame to split
        test_fraction: Fraction of data to use for testing
        date_column: Column name containing dates
        
    Returns:
        tuple: (train_df, test_df)
    """
    # Ensure DataFrame is sorted by date
    if date_column in df.columns:
        sorted_df = df.sort_values(date_column).copy()
    else:
        print(f"Warning: {date_column} not found, using existing order")
        sorted_df = df.copy()
    
    # Determine split point
    split_idx = int(len(sorted_df) * (1 - test_fraction))
    
    # Split data
    train_df = sorted_df.iloc[:split_idx].copy()
    test_df = sorted_df.iloc[split_idx:].copy()
    
    return train_df, test_df

def ensure_no_data_leakage(train_df, test_df, date_column='game_date'):
    """
    Verify that no data leakage exists between train and test sets.
    
    Args:
        train_df: Training DataFrame
        test_df: Testing DataFrame
        date_column: Column name containing dates
        
    Returns:
        bool: True if no leakage, False otherwise
    """
    if date_column not in train_df.columns or date_column not in test_df.columns:
        print(f"Warning: Cannot verify data leakage without {date_column} column")
        return False
    
    train_max_date = train_df[date_column].max()
    test_min_date = test_df[date_column].min()
    
    if train_max_date >= test_min_date:
        print(f"Data leakage detected: Train max date ({train_max_date}) >= Test min date ({test_min_date})")
        return False
    
    print(f"No data leakage: Train dates end at {train_max_date}, Test dates begin at {test_min_date}")
    return True

In [ ]:
# Cell 7B: Enhanced Live Prediction Function with Momentum and Win Probability

from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import joblib
import math
import traceback

def calculate_win_probability(home_score: float, away_score: float, quarter: int, time_remaining: float = None) -> float:
    """
    Calculate win probability for the home team using a logistic function.
    """
    score_diff = home_score - away_score
    if time_remaining is not None:
        total_time = 48.0
        elapsed = total_time - time_remaining
        progress = elapsed / total_time
    else:
        progress = min(quarter / 4.0, 1.0)
    k = 0.05 + (progress * 0.15)
    return 1.0 / (1.0 + math.exp(-k * score_diff))

def calculate_momentum(home_q1, home_q2, home_q3, home_q4, away_q1, away_q2, away_q3, away_q4, current_quarter: int) -> float:
    """Calculate normalized cumulative momentum."""
    h = [float(x or 0) for x in [home_q1, home_q2, home_q3, home_q4]]
    a = [float(x or 0) for x in [away_q1, away_q2, away_q3, away_q4]]
    momentum = 0
    if current_quarter >= 2:
        m1 = (h[1] - h[0]) - (a[1] - a[0])
    else:
        m1 = 0
    if current_quarter >= 3:
        m2 = (h[2] - h[1]) - (a[2] - a[1])
    else:
        m2 = 0
    if current_quarter >= 4:
        m3 = (h[3] - h[2]) - (a[3] - a[2])
    else:
        m3 = 0
    if current_quarter == 2:
        momentum = m1
    elif current_quarter == 3:
        momentum = (m1*0.2 + m2*0.3) / (0.2+0.3)
    elif current_quarter >= 4:
        momentum = (m1*0.2 + m2*0.3 + m3*0.5) / (0.2+0.3+0.5)
    return np.clip(momentum/15.0, -1, 1)

# Example usage:
home_prob = calculate_win_probability(110, 105, 3)
momentum_val = calculate_momentum(28, 27, 22, 0, 24, 29, 26, 0, 3)
print(f"Win Probability: {home_prob:.2f}, Normalized Momentum: {momentum_val:.2f}")

def initialize_quarter_specific_models(features_df, safe_init=True):
    """
    Initialize quarter-specific models with safe handling of features.
    
    Args:
        features_df: DataFrame with features
        safe_init: Whether to use safe initialization
        
    Returns:
        dict: Quarter-specific models
    """
    models = {}
    
    # Define feature sets for each quarter with fallbacks
    quarter_features = {
        'q1': ['rest_days_home', 'rest_days_away', 'prev_matchup_diff', 
              'rolling_home_score', 'rolling_away_score'],
        'q2': ['home_q1', 'away_q1', 'prev_matchup_diff', 'rest_advantage'],
        'q3': ['home_q1', 'home_q2', 'away_q1', 'away_q2', 'first_half_diff'],
        'q4': ['home_q1', 'home_q2', 'home_q3', 'away_q1', 'away_q2', 'away_q3', 'pre_q4_diff']
    }
    
    for quarter, features in quarter_features.items():
        # Skip if DataFrame is empty
        if features_df.empty:
            continue
            
        # Safe init checks that all features exist
        if safe_init:
            # Get current existing features
            existing_features = [f for f in features if f in features_df.columns]
            
            # If missing critical features, use fallbacks
            if len(existing_features) < len(features) * 0.6:  # Missing >40% of features
                from sklearn.linear_model import Ridge
                print(f"Warning: Missing critical features for {quarter}. Using fallback model.")
                models[f'{quarter}_model'] = Ridge(alpha=1.0, random_state=42)
                continue
            
            # Use what we have, but warn
            if len(existing_features) < len(features):
                print(f"Warning: Using {len(existing_features)}/{len(features)} features for {quarter} model")
                features = existing_features
        
        # Get target and training data
        q_num = quarter[1]  # Extract quarter number from 'q1', 'q2', etc.
        target_col = f'home_q{q_num}'
        
        if target_col not in features_df.columns:
            print(f"Warning: Target column {target_col} not found for {quarter}. Using fallback model.")
            from sklearn.linear_model import Ridge
            models[f'{quarter}_model'] = Ridge(alpha=1.0, random_state=42)
            continue
        
        # Prepare data
        X = ensure_features_exist(features_df, features)
        X = X[features].copy()
        y = features_df[target_col]
        
        # Handle missing values
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        # Create and train model
        if quarter == 'q4':
            # Ridge regression for Q4 (more stable)
            from sklearn.linear_model import Ridge
            model = Ridge(alpha=1.0, random_state=42)
        else:
            # GradientBoosting for other quarters
            from sklearn.ensemble import GradientBoostingRegressor
            model = GradientBoostingRegressor(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.1,
                random_state=42
            )
        
        # Train model
        model.fit(X, y)
        models[f'{quarter}_model'] = model
    
    return models


In [ ]:
#Cell 7B-2 : Enhanced Momentum Integration
import pandas as pd
import numpy as np
import time

def calculate_momentum_features(df, vectorized=True):
    """
    Calculate momentum features for game data.
    
    Args:
        df: DataFrame with game data
        vectorized: Whether to use vectorized implementation (default: True)
        
    Returns:
        DataFrame with momentum features added
    """
    if vectorized:
        return _calculate_momentum_features_vectorized(df)
    else:
        return _calculate_momentum_features_fixed(df)

def _calculate_momentum_features_fixed(df):
    """
    Calculate momentum features with proper type handling (non-vectorized).
    Use for debugging or when vectorized operations cause issues.
    
    Args:
        df: DataFrame with game data
        
    Returns:
        DataFrame with momentum features added
    """
    features_df = df.copy()
    
    # Initialize momentum columns with explicit float type
    features_df['q1_to_q2_momentum'] = 0.0
    features_df['q2_to_q3_momentum'] = 0.0
    features_df['q3_to_q4_momentum'] = 0.0
    features_df['cumulative_momentum'] = 0.0
    
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        
        # Calculate quarter-to-quarter momentum with explicit float casting
        if current_quarter >= 2:
            home_q1 = float(row.get('home_q1', 0) or 0)
            home_q2 = float(row.get('home_q2', 0) or 0)
            away_q1 = float(row.get('away_q1', 0) or 0)
            away_q2 = float(row.get('away_q2', 0) or 0)
            
            q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
            features_df.at[idx, 'q1_to_q2_momentum'] = float(q1_to_q2)
        
        if current_quarter >= 3:
            home_q2 = float(row.get('home_q2', 0) or 0)
            home_q3 = float(row.get('home_q3', 0) or 0)
            away_q2 = float(row.get('away_q2', 0) or 0)
            away_q3 = float(row.get('away_q3', 0) or 0)
            
            q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
            features_df.at[idx, 'q2_to_q3_momentum'] = float(q2_to_q3)
        
        if current_quarter >= 4:
            home_q3 = float(row.get('home_q3', 0) or 0)
            home_q4 = float(row.get('home_q4', 0) or 0)
            away_q3 = float(row.get('away_q3', 0) or 0)
            away_q4 = float(row.get('away_q4', 0) or 0)
            
            q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
            features_df.at[idx, 'q3_to_q4_momentum'] = float(q3_to_q4)
        
        # Calculate weighted momentum with explicit float handling
        weights = [0.2, 0.3, 0.5]  # Weights for each quarter transition
        
        if current_quarter == 2:
            momentum = float(features_df.at[idx, 'q1_to_q2_momentum'])
            features_df.at[idx, 'cumulative_momentum'] = float(momentum / 15.0)
        elif current_quarter == 3:
            momentum = float(
                (features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                 features_df.at[idx, 'q2_to_q3_momentum'] * weights[1]) / 
                (weights[0] + weights[1])
            )
            features_df.at[idx, 'cumulative_momentum'] = float(momentum / 15.0)
        elif current_quarter >= 4:
            momentum = float(
                (features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                 features_df.at[idx, 'q2_to_q3_momentum'] * weights[1] + 
                 features_df.at[idx, 'q3_to_q4_momentum'] * weights[2]) / 
                sum(weights)
            )
            features_df.at[idx, 'cumulative_momentum'] = float(momentum / 15.0)
        
        # Normalize to [-1, 1] with explicit float handling
        features_df.at[idx, 'cumulative_momentum'] = float(max(min(float(features_df.at[idx, 'cumulative_momentum']), 1.0), -1.0))
    
    return features_df

def _calculate_momentum_features_vectorized(df):
    """
    Calculate momentum features using vectorized operations with proper type handling.
    Significantly improves performance while eliminating dtype warnings.
    
    Args:
        df: DataFrame with game data
        
    Returns:
        DataFrame with momentum features added
    """
    # Create copy to avoid modifying original
    features_df = df.copy()
    
    # Ensure quarters are numeric with proper handling for None/NaN values
    for team in ['home', 'away']:
        for q in range(1, 5):
            col = f'{team}_q{q}'
            if col in features_df.columns:
                # Convert to float and handle None/NaN values
                features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0.0)
    
    # Ensure current_quarter is numeric
    if 'current_quarter' in features_df.columns:
        features_df['current_quarter'] = pd.to_numeric(features_df['current_quarter'], errors='coerce').fillna(0).astype(int)
    
    # Initialize momentum columns with explicit float dtype
    momentum_cols = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']
    for col in momentum_cols:
        features_df[col] = np.zeros(len(features_df), dtype=np.float64)
    
    # Calculate quarter-to-quarter momentum with vectorized operations
    # Q1 to Q2 momentum
    q2_mask = features_df['current_quarter'] >= 2
    if q2_mask.any():
        features_df.loc[q2_mask, 'q1_to_q2_momentum'] = (
            (features_df.loc[q2_mask, 'home_q2'] - features_df.loc[q2_mask, 'home_q1']) - 
            (features_df.loc[q2_mask, 'away_q2'] - features_df.loc[q2_mask, 'away_q1'])
        )
    
    # Q2 to Q3 momentum
    q3_mask = features_df['current_quarter'] >= 3
    if q3_mask.any():
        features_df.loc[q3_mask, 'q2_to_q3_momentum'] = (
            (features_df.loc[q3_mask, 'home_q3'] - features_df.loc[q3_mask, 'home_q2']) - 
            (features_df.loc[q3_mask, 'away_q3'] - features_df.loc[q3_mask, 'away_q2'])
        )
    
    # Q3 to Q4 momentum
    q4_mask = features_df['current_quarter'] >= 4
    if q4_mask.any():
        features_df.loc[q4_mask, 'q3_to_q4_momentum'] = (
            (features_df.loc[q4_mask, 'home_q4'] - features_df.loc[q4_mask, 'home_q3']) - 
            (features_df.loc[q4_mask, 'away_q4'] - features_df.loc[q4_mask, 'away_q3'])
        )
    
    # Calculate cumulative momentum using vectorized operations with weights
    weights = np.array([0.2, 0.3, 0.5])  # Weights for each quarter transition
    
    # Calculate for Q2
    q2_mask = features_df['current_quarter'] == 2
    if q2_mask.any():
        features_df.loc[q2_mask, 'cumulative_momentum'] = features_df.loc[q2_mask, 'q1_to_q2_momentum'] / 15.0
    
    # Calculate for Q3
    q3_mask = features_df['current_quarter'] == 3
    if q3_mask.any():
        # Calculate weighted average manually to ensure proper dtype handling
        weighted_sum = (
            features_df.loc[q3_mask, 'q1_to_q2_momentum'] * weights[0] +
            features_df.loc[q3_mask, 'q2_to_q3_momentum'] * weights[1]
        )
        weight_sum = weights[0] + weights[1]
        features_df.loc[q3_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
    
    # Calculate for Q4
    q4_mask = features_df['current_quarter'] >= 4
    if q4_mask.any():
        # Calculate weighted average manually to ensure proper dtype handling
        weighted_sum = (
            features_df.loc[q4_mask, 'q1_to_q2_momentum'] * weights[0] +
            features_df.loc[q4_mask, 'q2_to_q3_momentum'] * weights[1] +
            features_df.loc[q4_mask, 'q3_to_q4_momentum'] * weights[2]
        )
        weight_sum = weights.sum()
        features_df.loc[q4_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
    
    # Clip to [-1, 1] range
    for col in momentum_cols:
        # Using .clip() is faster than max/min in a loop
        features_df[col] = features_df[col].clip(-1.0, 1.0)
    
    return features_df

def calculate_dynamic_momentum_impact(momentum, score_differential, vectorized=True):
    """
    Calculate momentum impact that decreases in blowout games.
    
    Args:
        momentum: Raw momentum value(s) (-1 to 1), Series or scalar
        score_differential: Absolute score difference(s), Series or scalar
        vectorized: Whether to use vectorized implementation (default: True)
        
    Returns:
        Adjusted momentum value(s) (Series or scalar)
    """
    # Check if inputs are pandas Series or numpy arrays
    is_series = isinstance(momentum, (pd.Series, np.ndarray)) or hasattr(momentum, '__iter__')
    
    if is_series and vectorized:
        return _calculate_dynamic_momentum_impact_vectorized(momentum, score_differential)
    elif is_series:
        # Apply scalar function to each value using pandas apply
        return pd.Series([_calculate_dynamic_momentum_impact_scalar(m, s) 
                         for m, s in zip(momentum, score_differential)], 
                         index=getattr(momentum, 'index', None))
    else:
        # Single scalar values
        return _calculate_dynamic_momentum_impact_scalar(momentum, score_differential)

def _calculate_dynamic_momentum_impact_scalar(momentum, score_differential):
    """
    Calculate momentum impact for a single pair of values.
    
    Args:
        momentum: Raw momentum value (-1 to 1)
        score_differential: Absolute score difference
        
    Returns:
        float: Adjusted momentum value
    """
    # Calculate blowout factor: 1.0 for close games, approaches 0 for blowouts
    blowout_threshold = 20  # Point differential considered a complete blowout
    blowout_factor = max(0.0, 1.0 - (abs(score_differential) / blowout_threshold))
    
    # Apply to momentum - full impact in close games, reduced in blowouts
    return momentum * blowout_factor

def _calculate_dynamic_momentum_impact_vectorized(momentum_series, score_differential_series):
    """
    Vectorized version of momentum impact calculation.
    
    Args:
        momentum_series: Series of momentum values
        score_differential_series: Series of score differentials
        
    Returns:
        Series: Adjusted momentum values
    """
    # Convert inputs to numpy arrays for faster operations
    momentum = np.asarray(momentum_series, dtype=np.float64)
    score_diff = np.asarray(score_differential_series, dtype=np.float64)
    
    # Calculate blowout factor: 1.0 for close games, approaches 0 for blowouts
    blowout_threshold = 20.0  # Point differential considered a complete blowout
    blowout_factor = np.maximum(0.0, 1.0 - (np.abs(score_diff) / blowout_threshold))
    
    # Apply to momentum - full impact in close games, reduced in blowouts
    adjusted_momentum = momentum * blowout_factor
    
    return pd.Series(adjusted_momentum, index=getattr(momentum_series, 'index', None))

# Example usage section - only run this code if executing the cell directly
if __name__ == "__main__":
    # Test the momentum calculation with a simple example
    test_games = pd.DataFrame([
        {
            'game_id': 1001,
            'home_team': 'Boston Celtics',
            'away_team': 'Los Angeles Lakers',
            'current_quarter': 3,
            'home_q1': 28, 'home_q2': 30, 'home_q3': 25, 'home_q4': 0,
            'away_q1': 25, 'away_q2': 27, 'away_q3': 30, 'away_q4': 0,
            'home_score': 83,
            'away_score': 82
        }
    ])

    # Test with both implementations
    fixed_features = calculate_momentum_features(test_games, vectorized=False)
    vectorized_features = calculate_momentum_features(test_games, vectorized=True)
    
    print("Momentum features comparison:")
    momentum_cols = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']
    display(fixed_features[momentum_cols])
    
    # Test dynamic momentum impact
    score_diffs = [0, 5, 10, 15, 20, 25]
    momentum_val = 0.8
    print("\nDynamic momentum impact by score differential:")
    for diff in score_diffs:
        adjusted = calculate_dynamic_momentum_impact(momentum_val, diff, vectorized=False)
        print(f"Score diff: {diff:2d} | Momentum: {momentum_val:.1f} → Adjusted: {adjusted:.3f}")
    
    # Performance comparison
    def create_test_dataset(size=1000):
        """Create a test dataset for momentum calculation benchmarking"""
        np.random.seed(42)
        
        test_data = []
        for i in range(size):
            # Random quarter (0-4)
            quarter = np.random.randint(0, 5)
            
            # Generate random quarter scores
            home_scores = np.random.randint(15, 35, 4)
            away_scores = np.random.randint(15, 35, 4)
            
            # For quarters that haven't been played, set to 0
            if quarter < 4:
                home_scores[quarter:] = 0
                away_scores[quarter:] = 0
            
            # Create game dict
            game = {
                'game_id': 1000 + i,
                'current_quarter': quarter,
                'home_team': 'Team A',
                'away_team': 'Team B',
                'home_score': home_scores.sum(),
                'away_score': away_scores.sum()
            }
            
            # Add quarter scores
            for q in range(4):
                game[f'home_q{q+1}'] = home_scores[q]
                game[f'away_q{q+1}'] = away_scores[q]
            
            test_data.append(game)
        
        return pd.DataFrame(test_data)

    # Performance test with larger dataset
    def run_performance_test(dataset_size=5000):
        print(f"\nPerformance test with {dataset_size} records:")
        test_df = create_test_dataset(dataset_size)
        
        # Test non-vectorized implementation
        start_time = time.time()
        fixed_result = calculate_momentum_features(test_df, vectorized=False)
        fixed_time = time.time() - start_time
        print(f"Non-vectorized time: {fixed_time:.4f} seconds")
        
        # Test vectorized implementation
        start_time = time.time()
        vectorized_result = calculate_momentum_features(test_df, vectorized=True)
        vectorized_time = time.time() - start_time
        print(f"Vectorized time: {vectorized_time:.4f} seconds")
        
        # Calculate speedup
        speedup = fixed_time / vectorized_time
        print(f"Speedup factor: {speedup:.2f}x")
        
        # Verify results match
        for col in momentum_cols:
            if col in fixed_result.columns and col in vectorized_result.columns:
                max_diff = (fixed_result[col] - vectorized_result[col]).abs().max()
                print(f"Maximum difference in {col}: {max_diff:.6f}")
    
    # Uncomment to run performance tests
    # run_performance_test(5000)
    
    # Test dynamic momentum impact on vectors
    sample_momentum = pd.Series(np.random.uniform(-1, 1, 5))
    sample_score_diff = pd.Series(np.random.uniform(0, 30, 5))
    impact = calculate_dynamic_momentum_impact(sample_momentum, sample_score_diff)
    
    print("\nDynamic Momentum Impact Example:")
    for i in range(len(sample_momentum)):
        print(f"Momentum: {sample_momentum.iloc[i]:.2f}, Score Diff: {sample_score_diff.iloc[i]:.1f}, Impact: {impact.iloc[i]:.3f}")

In [ ]:
# Cell 7C: Comprehensive Monitoring System with Automatic Scheduling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from datetime import datetime, timedelta
from IPython.display import clear_output
import traceback
import threading
import json
import os

class NBAGameMonitor:
    """
    System for monitoring NBA games with automated updates and prediction tracking.
    Builds on the NBAGamePredictor to provide continuous monitoring.
    """
    
    def __init__(self, update_interval=60, auto_save=True, debug=True):
        """
        Initialize the monitor with configurable settings.
        
        Args:
            update_interval: Seconds between updates (default: 60)
            auto_save: Whether to auto-save prediction history (default: True)
            debug: Enable detailed debug logging (default: True)
        """
        self.predictor = NBAGamePredictor()  # Assuming NBAGamePredictor is defined elsewhere
        self.update_interval = update_interval
        self.auto_save = auto_save
        self.debug = debug
        self.running = False
        self.update_thread = None
        self.prediction_history = {}
        self.update_count = 0
        self.last_update_time = None
        self.validation_results = None
        
        # File paths for saving data
        self.history_file = "prediction_history.json"
        self.validation_file = "model_validation.json"
        
        # Create directories if needed
        os.makedirs("data", exist_ok=True)
        
        # Load previous history if it exists
        self.load_history()
    
    def log(self, message, level="INFO"):
        """Log message with timestamp."""
        if not self.debug and level == "DEBUG":
            return
            
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{timestamp}] {level}: {message}")
    
    def load_history(self):
        """Load prediction history from file if available."""
        history_path = os.path.join("data", self.history_file)
        
        if os.path.exists(history_path):
            try:
                with open(history_path, 'r') as f:
                    raw_history = json.load(f)
                
                # Convert string timestamps back to datetime objects
                for game_id, predictions in raw_history.items():
                    for pred in predictions:
                        if 'timestamp' in pred and isinstance(pred['timestamp'], str):
                            pred['timestamp'] = datetime.fromisoformat(pred['timestamp'])
                
                self.prediction_history = raw_history
                self.log(f"Loaded prediction history for {len(self.prediction_history)} games")
            except Exception as e:
                self.log(f"Error loading prediction history: {e}", "ERROR")
    
    def save_history(self):
        """Save prediction history to file."""
        if not self.prediction_history:
            return
            
        history_path = os.path.join("data", self.history_file)
        
        try:
            # Convert datetime objects to ISO format strings for JSON serialization
            serializable_history = {}
            
            for game_id, predictions in self.prediction_history.items():
                serializable_predictions = []
                for pred in predictions:
                    pred_copy = pred.copy()
                    if 'timestamp' in pred_copy and isinstance(pred_copy['timestamp'], datetime):
                        pred_copy['timestamp'] = pred_copy['timestamp'].isoformat()
                    serializable_predictions.append(pred_copy)
                
                serializable_history[str(game_id)] = serializable_predictions
            
            with open(history_path, 'w') as f:
                json.dump(serializable_history, f)
            
            self.log(f"Saved prediction history to {history_path}")
        except Exception as e:
            self.log(f"Error saving prediction history: {e}", "ERROR")
    
    def validate_model(self, num_test_games=20):
        """Run validation on historical games to assess model performance."""
        self.log("Running model validation...")
        
        try:
            # Validate that the model is loaded
            if self.predictor.model is None:
                self.predictor.load_model()
            
            # Fetch historical games
            current_date = datetime.now().strftime('%Y-%m-%d')
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .lt("game_date", current_date)\
                .order('game_date', desc=True)\
                .limit(num_test_games).execute()
            
            if not response.data:
                self.log("No historical games found for validation", "WARNING")
                return
            
            # Process historical games
            historical_games = response.data
            validation_results = []
            
            self.log(f"Validating model on {len(historical_games)} historical games")
            
            for game in historical_games:
                # Get actual final scores
                actual_home_score = game['home_score']
                actual_away_score = game['away_score']
                game_id = game['game_id']
                
                # Test prediction from each quarter
                quarter_results = []
                
                for test_quarter in range(1, 5):
                    # Create simulated in-progress game
                    sim_game = {
                        'game_id': game_id,
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': test_quarter
                    }
                    
                    # Add quarter scores up to test_quarter
                    for q in range(1, 5):
                        q_key_home = f'home_q{q}'
                        q_key_away = f'away_q{q}'
                        
                        if q <= test_quarter:
                            sim_game[q_key_home] = game.get(q_key_home, 0)
                            sim_game[q_key_away] = game.get(q_key_away, 0)
                        else:
                            sim_game[q_key_home] = 0
                            sim_game[q_key_away] = 0
                    
                    # Calculate current score
                    sim_game['home_score'] = sum([sim_game.get(f'home_q{q}', 0) or 0 for q in range(1, test_quarter+1)])
                    sim_game['away_score'] = sum([sim_game.get(f'away_q{q}', 0) or 0 for q in range(1, test_quarter+1)])
                    
                    # Make prediction
                    features_df = self.predictor.prepare_features(pd.DataFrame([sim_game]))
                    if features_df.empty:
                        continue
                        
                    prediction_result = self.predictor.predict_game_scores(features_df)
                    if prediction_result.empty:
                        continue
                    
                    pred_row = prediction_result.iloc[0]
                    predicted_home = pred_row['predicted_home_final']
                    predicted_away = pred_row['predicted_away_final']
                    
                    # Calculate errors
                    home_error = predicted_home - actual_home_score
                    away_error = predicted_away - actual_away_score
                    
                    quarter_results.append({
                        'quarter': test_quarter,
                        'actual_home': actual_home_score,
                        'actual_away': actual_away_score,
                        'predicted_home': predicted_home,
                        'predicted_away': predicted_away,
                        'home_error': home_error,
                        'away_error': away_error,
                        'absolute_error': (abs(home_error) + abs(away_error)) / 2
                    })
                
                validation_results.append({
                    'game_id': game_id,
                    'home_team': game['home_team'],
                    'away_team': game['away_team'],
                    'quarter_results': quarter_results
                })
            
            # Save validation results
            self.validation_results = validation_results
            
            # Calculate and display summary metrics
            self.display_validation_results()
            
            # Save to file
            validation_path = os.path.join("data", self.validation_file)
            with open(validation_path, 'w') as f:
                json.dump(validation_results, f)
            
            self.log(f"Validation results saved to {validation_path}")
            
            return validation_results
            
        except Exception as e:
            self.log(f"Error during model validation: {e}", "ERROR")
            traceback.print_exc()
            return None
    
    def display_validation_results(self):
        """Display and visualize validation results."""
        if not self.validation_results:
            self.log("No validation results to display", "WARNING")
            return
        
        # Extract metrics by quarter
        quarter_errors = {1: [], 2: [], 3: [], 4: []}
        
        for game in self.validation_results:
            for qr in game['quarter_results']:
                quarter = qr['quarter']
                quarter_errors[quarter].append(qr['absolute_error'])
        
        # Calculate average errors by quarter
        avg_errors = {}
        for quarter, errors in quarter_errors.items():
            if errors:
                avg_errors[quarter] = sum(errors) / len(errors)
            else:
                avg_errors[quarter] = None
        
        # Display results
        print("\nModel Validation Results:")
        print("=" * 60)
        print("Average Prediction Error by Quarter:")
        for quarter, avg_error in avg_errors.items():
            if avg_error is not None:
                print(f"  Quarter {quarter}: {avg_error:.2f} points")
        
        # Visualization
        plt.figure(figsize=(10, 6))
        quarters = list(avg_errors.keys())
        errors = [avg_errors[q] for q in quarters if avg_errors[q] is not None]
        
        bars = plt.bar(quarters, errors, color='salmon')
        plt.xlabel('Quarter')
        plt.ylabel('Average Absolute Error (points)')
        plt.title('Prediction Error by Quarter')
        plt.xticks(quarters)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.2f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
    
    def start_monitoring(self, max_updates=None, run_validation=True):
        """Start the monitoring process."""
        self.running = True
        self.update_count = 0
        
        # Run model validation if requested
        if run_validation:
            self.validate_model()
        
        # Ensure the model is loaded
        if self.predictor.model is None:
            self.predictor.load_model()
        
        self.log(f"Starting monitoring with {self.update_interval} second updates")
        
        # Begin update loop
        while self.running:
            try:
                self.update_count += 1
                self.last_update_time = datetime.now()
                
                self.log(f"Update #{self.update_count} at {self.last_update_time.strftime('%H:%M:%S')}")
                
                # Clear previous output
                clear_output(wait=True)
                
                # Run full prediction pipeline
                prediction_results = self.predictor.run_full_prediction_pipeline()
                
                # Update prediction history
                if prediction_results is not None and not prediction_results.empty:
                    for _, pred in prediction_results.iterrows():
                        game_id = pred['game_id']
                        if game_id not in self.prediction_history:
                            self.prediction_history[game_id] = []
                        
                        # Add timestamp if missing
                        if 'timestamp' not in pred:
                            pred['timestamp'] = self.last_update_time
                        
                        # Store prediction
                        self.prediction_history[game_id].append(pred.to_dict())
                
                # Save history if auto-save is enabled
                if self.auto_save:
                    self.save_history()
                
                # Check if we've reached max updates
                if max_updates and self.update_count >= max_updates:
                    self.log(f"Reached maximum updates ({max_updates})")
                    break
                
                # Wait for next update
                if self.running and (max_updates is None or self.update_count < max_updates):
                    self.log(f"Waiting {self.update_interval} seconds until next update...")
                    time.sleep(self.update_interval)
                
            except KeyboardInterrupt:
                self.log("Monitoring stopped by user")
                self.running = False
                break
            except Exception as e:
                self.log(f"Error during monitoring update: {e}", "ERROR")
                traceback.print_exc()
                
                # Don't stop on errors, just wait for next update
                time.sleep(self.update_interval)
        
        self.running = False
        self.log("Monitoring stopped")
    
    def start_async_monitoring(self, max_updates=None, run_validation=True):
        """Start monitoring in a background thread."""
        if self.update_thread is not None and self.update_thread.is_alive():
            self.log("Monitoring already running")
            return
        
        self.update_thread = threading.Thread(
            target=self.start_monitoring,
            args=(max_updates, run_validation)
        )
        self.update_thread.daemon = True
        self.update_thread.start()
        self.log("Monitoring started in background thread")
    
    def stop_monitoring(self):
        """Stop the monitoring process."""
        self.running = False
        self.log("Stopping monitoring...")
        
        if self.update_thread is not None and self.update_thread.is_alive():
            self.update_thread.join(timeout=5)
            if self.update_thread.is_alive():
                self.log("Monitoring thread is still running", "WARNING")
        
        self.save_history()
        self.log("Monitoring stopped and history saved")
    
    def get_prediction_accuracy(self):
        """Calculate prediction accuracy for completed games."""
        if not self.prediction_history:
            self.log("No prediction history available")
            return None
        
        accuracy_results = []
        
        for game_id, predictions in self.prediction_history.items():
            # Check if this is a completed game with actual results
            if predictions and 'actual_home_final' in predictions[-1]:
                final_pred = predictions[-1]
                actual_home = final_pred['actual_home_final']
                actual_away = final_pred['actual_away_final']
                
                # Calculate accuracy for each prediction in the game
                game_accuracy = []
                
                for i, pred in enumerate(predictions):
                    predicted_home = pred['predicted_home_final']
                    predicted_away = pred['predicted_away_final']
                    quarter = pred['current_quarter']
                    
                    # Calculate errors
                    home_error = abs(predicted_home - actual_home)
                    away_error = abs(predicted_away - actual_away)
                    avg_error = (home_error + away_error) / 2
                    
                    # Check if prediction got winner correct
                    actual_winner = 'home' if actual_home > actual_away else 'away'
                    predicted_winner = 'home' if predicted_home > predicted_away else 'away'
                    winner_correct = (actual_winner == predicted_winner)
                    
                    game_accuracy.append({
                        'prediction_number': i + 1,
                        'quarter': quarter,
                        'home_error': home_error,
                        'away_error': away_error,
                        'avg_error': avg_error,
                        'winner_correct': winner_correct
                    })
                
                accuracy_results.append({
                    'game_id': game_id,
                    'home_team': predictions[0]['home_team'],
                    'away_team': predictions[0]['away_team'],
                    'predictions': game_accuracy
                })
        
        # Create summary
        if accuracy_results:
            self.display_accuracy_results(accuracy_results)
        
        return accuracy_results
    
    def display_accuracy_results(self, accuracy_results):
        """Display accuracy results."""
        if not accuracy_results:
            return
        
        print("\nPrediction Accuracy Analysis")
        print("=" * 60)
        
        # Overall stats
        total_predictions = 0
        correct_winner_count = 0
        error_by_quarter = {0: [], 1: [], 2: [], 3: [], 4: []}
        
        for game in accuracy_results:
            for pred in game['predictions']:
                total_predictions += 1
                if pred['winner_correct']:
                    correct_winner_count += 1
                
                quarter = pred['quarter']
                error_by_quarter[quarter].append(pred['avg_error'])
        
        # Calculate winner accuracy
        winner_accuracy = (correct_winner_count / total_predictions) if total_predictions > 0 else 0
        print(f"Overall Winner Prediction Accuracy: {winner_accuracy:.1%}")
        
        # Calculate average error by quarter
        print("\nAverage Error by Quarter:")
        for quarter, errors in error_by_quarter.items():
            if errors:
                avg_error = sum(errors) / len(errors)
                print(f"  Quarter {quarter}: {avg_error:.2f} points")
        
        # Visualize
        plt.figure(figsize=(12, 8))
        
        # First subplot - error by quarter
        plt.subplot(2, 1, 1)
        quarters = []
        avg_errors = []
        
        for quarter, errors in error_by_quarter.items():
            if errors:
                quarters.append(quarter)
                avg_errors.append(sum(errors) / len(errors))
        
        bars = plt.bar(quarters, avg_errors, color='lightblue')
        plt.xlabel('Quarter')
        plt.ylabel('Average Error (points)')
        plt.title('Prediction Error by Quarter')
        plt.xticks(quarters)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.2f}', ha='center', va='bottom')
        
        # Second subplot - winner accuracy by quarter
        plt.subplot(2, 1, 2)
        quarters = []
        accuracies = []
        
        for quarter in range(5):
            correct = 0
            total = 0
            
            for game in accuracy_results:
                for pred in game['predictions']:
                    if pred['quarter'] == quarter:
                        total += 1
                        if pred['winner_correct']:
                            correct += 1
            
            if total > 0:
                quarters.append(quarter)
                accuracies.append(correct / total)
        
        bars = plt.bar(quarters, accuracies, color='lightgreen')
        plt.xlabel('Quarter')
        plt.ylabel('Accuracy')
        plt.title('Winner Prediction Accuracy by Quarter')
        plt.xticks(quarters)
        plt.ylim(0, 1)
        plt.grid(axis='y', alpha=0.3)
        
        # Add values on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.2%}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()

def match_games_to_schedule(live_games_df, schedule_df):
    """Improved function to match games to schedule with better validation"""
    if live_games_df.empty or schedule_df.empty:
        return live_games_df
    
    # Make a copy to avoid modifying the original
    matched_games = []
    
    # Process the schedule first
    for _, scheduled_game in schedule_df.iterrows():
        schedule_game_id = scheduled_game['game_id']
        schedule_home = scheduled_game['home_team']
        schedule_away = scheduled_game['away_team']
        schedule_date = scheduled_game['game_date']
        
        # Try to find this scheduled game in live data
        matching_live = live_games_df[
            (live_games_df['home_team'] == schedule_home) & 
            (live_games_df['away_team'] == schedule_away)
        ]
        
        if not matching_live.empty:
            # Found match - use the scheduled game ID
            live_game = matching_live.iloc[0].to_dict()
            live_game['game_id'] = schedule_game_id
            live_game['verified'] = True
            matched_games.append(live_game)
        else:
            # No live data for this scheduled game - create template
            template = {
                'game_id': schedule_game_id,
                'home_team': schedule_home,
                'away_team': schedule_away,
                'game_date': schedule_date,
                'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
                'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
                'home_score': 0, 'away_score': 0,
                'current_quarter': 0,
                'verified': True,
                'game_status': 'SCHEDULED'
            }
            matched_games.append(template)
    
    return pd.DataFrame(matched_games)

# For testing, we'll need to define a basic NBAGamePredictor class
class NBAGamePredictor:
    def __init__(self):
        self.model = None
        print("[{}] INFO: Initializing NBA Game Predictor".format(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
        
    def load_model(self):
        # Placeholder for model loading
        self.model = "mock_model"
        
    def prepare_features(self, df):
        # Placeholder returning the input
        return df
        
    def predict_game_scores(self, features_df):
        # Mock prediction
        if features_df.empty:
            return pd.DataFrame()
            
        result = features_df.copy()
        result['predicted_home_final'] = 100
        result['predicted_away_final'] = 95
        return result
        
    def run_full_prediction_pipeline(self):
        # Mock prediction results
        return pd.DataFrame({
            'game_id': [1001, 1002],
            'home_team': ['Boston', 'LA Lakers'],
            'away_team': ['Philadelphia', 'Phoenix'],
            'current_quarter': [2, 3],
            'predicted_home_final': [105.5, 110.3],
            'predicted_away_final': [98.2, 103.7],
            'current_quarter': [2, 3]
        })

# Start with just testing the class initialization
monitor = NBAGameMonitor(update_interval=30, auto_save=False)
print("Monitor initialized successfully")

# Instead of running actual monitoring, let's just verify the logger works
monitor.log("Test log message")

In [ ]:
# Cell 7C-2: Win Prediction Validation


def analyze_win_prediction_accuracy(predictions_history, actual_results=None):
    """
    Analyze the accuracy of win probability predictions across quarters.
    
    Args:
        predictions_history: Dict of game_id -> list of prediction points
        actual_results: DataFrame with actual game results (optional)
        
    Returns:
        tuple: (accuracy_df, visualization_fig)
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Prepare results container
    results = []
    
    # Process each game's prediction history
    for game_id, history in predictions_history.items():
        if not history:
            continue
            
        # Get the actual winner if available
        actual_home_win = None
        final_prediction = history[-1]  # Last prediction in history
        
        if actual_results is not None and game_id in actual_results.index:
            actual = actual_results.loc[game_id]
            actual_home_score = actual.get('home_score', 0)
            actual_away_score = actual.get('away_score', 0)
            actual_home_win = actual_home_score > actual_away_score
        
        # Process each prediction point in this game's history
        for i, pred in enumerate(history):
            quarter = pred.get('current_quarter', 0)
            win_prob = pred.get('win_probability', 0.5)
            
            # Predicted winner (home team if win_prob > 0.5)
            predicted_home_win = win_prob > 0.5
            
            # Check if prediction was correct (if we have actual results)
            correct_prediction = None
            if actual_home_win is not None:
                correct_prediction = (predicted_home_win == actual_home_win)
            
            # For later predictions in the same game, calculate change
            win_prob_change = None
            if i > 0:
                prev_win_prob = history[i-1].get('win_probability', 0.5)
                win_prob_change = win_prob - prev_win_prob
            
            # Add result
            results.append({
                'game_id': game_id,
                'quarter': quarter,
                'win_probability': win_prob, 
                'predicted_home_win': predicted_home_win,
                'correct_prediction': correct_prediction,
                'win_prob_change': win_prob_change,
                'prediction_number': i+1
            })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Create visualization
    if not results_df.empty:
        fig, axs = plt.subplots(2, 1, figsize=(12, 10))
        
        # Plot 1: Win prediction accuracy by quarter
        if 'correct_prediction' in results_df.columns and not results_df['correct_prediction'].isna().all():
            # Group by quarter and calculate accuracy
            quarter_accuracy = results_df.groupby('quarter')['correct_prediction'].mean()
            
            # Plot
            axs[0].bar(quarter_accuracy.index, quarter_accuracy.values, color='lightgreen')
            axs[0].set_xlabel('Quarter')
            axs[0].set_ylabel('Prediction Accuracy')
            axs[0].set_title('Win Prediction Accuracy by Quarter')
            axs[0].set_ylim(0, 1)
            axs[0].set_xticks(quarter_accuracy.index)
            axs[0].grid(axis='y', alpha=0.3)
            
            # Add percentage labels
            for i, v in enumerate(quarter_accuracy):
                axs[0].text(i, v + 0.02, f'{v:.1%}', ha='center')
        else:
            axs[0].text(0.5, 0.5, "No accuracy data available", 
                     ha='center', va='center', transform=axs[0].transAxes)
        
        # Plot 2: Win probability confidence distribution
        axs[1].hist(results_df['win_probability'], bins=10, range=(0, 1), alpha=0.7)
        axs[1].set_xlabel('Win Probability')
        axs[1].set_ylabel('Frequency')
        axs[1].set_title('Distribution of Win Probability Predictions')
        axs[1].set_xlim(0, 1)
        axs[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
    else:
        fig = plt.figure()
        plt.text(0.5, 0.5, "No prediction data available", ha='center', va='center')
    
    return results_df, fig

# Create sample prediction history for testing
sample_history = {
    1001: [
        {'current_quarter': 1, 'win_probability': 0.55, 'home_team': 'Lakers', 'away_team': 'Celtics'},
        {'current_quarter': 2, 'win_probability': 0.62, 'home_team': 'Lakers', 'away_team': 'Celtics'},
        {'current_quarter': 3, 'win_probability': 0.45, 'home_team': 'Lakers', 'away_team': 'Celtics'},
        {'current_quarter': 4, 'win_probability': 0.58, 'home_team': 'Lakers', 'away_team': 'Celtics'}
    ],
    1002: [
        {'current_quarter': 1, 'win_probability': 0.48, 'home_team': 'Warriors', 'away_team': 'Bucks'},
        {'current_quarter': 2, 'win_probability': 0.72, 'home_team': 'Warriors', 'away_team': 'Bucks'},
        {'current_quarter': 3, 'win_probability': 0.85, 'home_team': 'Warriors', 'away_team': 'Bucks'},
        {'current_quarter': 4, 'win_probability': 0.91, 'home_team': 'Warriors', 'away_team': 'Bucks'}
    ]
}

# Sample actual results for validation
sample_results = pd.DataFrame([
    {'game_id': 1001, 'home_score': 105, 'away_score': 108},  # Home team lost
    {'game_id': 1002, 'home_score': 120, 'away_score': 105}   # Home team won
]).set_index('game_id')

# Test the win prediction accuracy analysis
win_predictions_df, win_predictions_fig = analyze_win_prediction_accuracy(sample_history, sample_results)
plt.figure(win_predictions_fig.number)
plt.show()

print("Win Prediction Metrics:")
display(win_predictions_df)

In [ ]:
# Cell 7D: Enhanced NBAGameMonitor with Integrated Pipeline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
from IPython.display import clear_output
import json
import os
import traceback
import threading

class EnhancedNBAGameMonitor:
    """
    Enhanced monitoring system with improved data flow pipeline and state management.
    Connects live game fetching, prediction, and visualization in a continuous loop.
    """
    
    def __init__(self, update_intervals=None, auto_save=True, debug=True, visualization_mode='dashboard'):
        """
        Initialize the enhanced monitor with configurable settings.
        
        Args:
            update_intervals: Dict with update frequencies in seconds for different game states
            auto_save: Whether to auto-save prediction history (default: True)
            debug: Enable detailed debug logging (default: True)
            visualization_mode: Display mode - 'dashboard' or 'individual' (default: dashboard)
        """
        self.predictor = NBAGamePredictor()
        
        # Configure dynamic update intervals based on game state
        self.update_intervals = update_intervals or {
            'pregame': 300,      # 5 mins for pregame
            'live': 60,          # 1 min for live games
            'close_game': 30,    # 30 seconds for close games (margin < 5)
            'finished': 600,     # 10 mins for finished games
            'default': 120       # 2 mins default
        }
        
        self.auto_save = auto_save
        self.debug = debug
        self.visualization_mode = visualization_mode
        self.running = False
        self.update_thread = None
        self.prediction_history = {}
        self.update_count = 0
        self.last_update_time = None
        self.game_states = {}  # Track state of each game
        self.alerts = []       # Store system alerts
        
        # File paths for saving data
        self.data_dir = "data"
        self.history_file = os.path.join(self.data_dir, "prediction_history.json")
        self.state_file = os.path.join(self.data_dir, "game_states.json")
        
        # Create directories if needed
        os.makedirs(self.data_dir, exist_ok=True)
        
        # Load previous history if it exists
        self.load_history()
        self.load_game_states()
    
    def log(self, message, level="INFO"):
        """Log message with timestamp."""
        if not self.debug and level == "DEBUG":
            return
            
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{timestamp}] {level}: {message}")
    
    def load_history(self):
        """Load prediction history from file if available."""
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    raw_history = json.load(f)
                
                # Convert string timestamps back to datetime objects
                for game_id, predictions in raw_history.items():
                    for pred in predictions:
                        if 'timestamp' in pred and isinstance(pred['timestamp'], str):
                            pred['timestamp'] = datetime.fromisoformat(pred['timestamp'])
                
                self.prediction_history = raw_history
                self.log(f"Loaded prediction history for {len(self.prediction_history)} games")
            except Exception as e:
                self.log(f"Error loading prediction history: {e}", "ERROR")
    
    def load_game_states(self):
        """Load game states from file if available."""
        if os.path.exists(self.state_file):
            try:
                with open(self.state_file, 'r') as f:
                    self.game_states = json.load(f)
                self.log(f"Loaded states for {len(self.game_states)} games")
            except Exception as e:
                self.log(f"Error loading game states: {e}", "ERROR")
    
    def save_history(self):
        """Save prediction history to file."""
        if not self.prediction_history:
            return
            
        try:
            # Convert datetime objects to ISO format strings for JSON serialization
            serializable_history = {}
            
            for game_id, predictions in self.prediction_history.items():
                serializable_predictions = []
                for pred in predictions:
                    pred_copy = pred.copy()
                    if 'timestamp' in pred_copy and isinstance(pred_copy['timestamp'], datetime):
                        pred_copy['timestamp'] = pred_copy['timestamp'].isoformat()
                    serializable_predictions.append(pred_copy)
                
                serializable_history[str(game_id)] = serializable_predictions
            
            with open(self.history_file, 'w') as f:
                json.dump(serializable_history, f)
            
            self.log(f"Saved prediction history to {self.history_file}")
        except Exception as e:
            self.log(f"Error saving prediction history: {e}", "ERROR")
    
    def save_game_states(self):
        """Save game states to file."""
        if not self.game_states:
            return
            
        try:
            with open(self.state_file, 'w') as f:
                json.dump(self.game_states, f)
            self.log(f"Saved game states to {self.state_file}")
        except Exception as e:
            self.log(f"Error saving game states: {e}", "ERROR")
    
    def prune_old_data(self, days_to_keep=7):
        """Remove old data to prevent unlimited growth."""
        if not self.prediction_history:
            return
        
        cutoff_date = datetime.now() - timedelta(days=days_to_keep)
        pruned_history = {}
        
        for game_id, predictions in self.prediction_history.items():
            # Check the most recent prediction timestamp
            if predictions and 'timestamp' in predictions[-1]:
                last_timestamp = predictions[-1]['timestamp']
                if isinstance(last_timestamp, str):
                    last_timestamp = datetime.fromisoformat(last_timestamp)
                
                if last_timestamp > cutoff_date:
                    pruned_history[game_id] = predictions
        
        pruned_count = len(self.prediction_history) - len(pruned_history)
        if pruned_count > 0:
            self.prediction_history = pruned_history
            self.log(f"Pruned {pruned_count} old games from history")
    
    def determine_update_interval(self, game_data):
        """
        Determine appropriate update interval based on game state.
        
        Args:
            game_data: Dict or DataFrame row with game information
            
        Returns:
            int: Update interval in seconds
        """
        game_status = game_data.get('game_status', '').lower()
        
        if game_status == 'finished':
            return self.update_intervals['finished']
        
        if game_status == 'pregame' or game_data.get('current_quarter', 0) == 0:
            return self.update_intervals['pregame']
        
        # For live games, check if it's a close game
        if game_status == 'live':
            home_score = float(game_data.get('home_score', 0))
            away_score = float(game_data.get('away_score', 0))
            margin = abs(home_score - away_score)
            
            if margin < 5 and game_data.get('current_quarter', 0) >= 3:
                return self.update_intervals['close_game']
            
            return self.update_intervals['live']
        
        return self.update_intervals['default']
    
    def update_game_state(self, game_id, game_data):
        """
        Update tracked state for a specific game.
        
        Args:
            game_id: Unique identifier for the game
            game_data: Dict with current game state
        """
        str_game_id = str(game_id)
        
        if str_game_id not in self.game_states:
            self.game_states[str_game_id] = {
                'first_seen': datetime.now().isoformat(),
                'updates': 0,
                'status_history': []
            }
        
        # Update the state
        current_state = {
            'timestamp': datetime.now().isoformat(),
            'current_quarter': game_data.get('current_quarter', 0),
            'home_score': game_data.get('home_score', 0),
            'away_score': game_data.get('away_score', 0),
            'status': game_data.get('game_status', 'unknown')
        }
        
        # Check for state changes
        if self.game_states[str_game_id]['status_history']:
            last_state = self.game_states[str_game_id]['status_history'][-1]
            
            # Detect quarter changes
            if last_state['current_quarter'] != current_state['current_quarter']:
                self.log(f"Game {game_id} quarter changed: {last_state['current_quarter']} → {current_state['current_quarter']}")
            
            # Detect status changes
            if last_state['status'] != current_state['status']:
                self.log(f"Game {game_id} status changed: {last_state['status']} → {current_state['status']}")
        
        # Update the history
        self.game_states[str_game_id]['status_history'].append(current_state)
        self.game_states[str_game_id]['updates'] += 1
        self.game_states[str_game_id]['last_update'] = datetime.now().isoformat()
    
    def detect_anomalies(self, game_id, current_prediction, previous_predictions):
        """
        Detect anomalies in predictions and raise alerts if necessary.
        
        Args:
            game_id: Game identifier
            current_prediction: Current prediction data
            previous_predictions: List of previous predictions
            
        Returns:
            list: Detected anomalies (if any)
        """
        anomalies = []
        
        if not previous_predictions:
            return anomalies
        
        # Get the most recent previous prediction
        prev_prediction = previous_predictions[-1]
        
        # Check for large score prediction swings
        prev_home = float(prev_prediction.get('predicted_home_score', 0))
        prev_away = float(prev_prediction.get('predicted_away_score', 0))
        
        curr_home = float(current_prediction.get('predicted_home_score', 0))
        curr_away = float(current_prediction.get('predicted_away_score', 0))
        
        home_change = abs(curr_home - prev_home)
        away_change = abs(curr_away - prev_away)
        
        # Threshold for large prediction change (points)
        threshold = 8
        
        if home_change > threshold or away_change > threshold:
            anomaly = {
                'type': 'large_prediction_swing',
                'game_id': game_id,
                'timestamp': datetime.now().isoformat(),
                'details': {
                    'home_change': home_change,
                    'away_change': away_change,
                    'previous': {'home': prev_home, 'away': prev_away},
                    'current': {'home': curr_home, 'away': curr_away}
                }
            }
            anomalies.append(anomaly)
            
            self.log(f"Anomaly detected in game {game_id}: Large prediction swing " +
                    f"(Home: {prev_home:.1f}→{curr_home:.1f}, Away: {prev_away:.1f}→{curr_away:.1f})", 
                    "WARNING")
            
            # Add to global alerts
            self.alerts.append(anomaly)
        
        return anomalies
    
    def integrated_pipeline_step(self):
        """Execute a single step of the integrated data flow pipeline."""
        try:
            self.update_count += 1
            self.last_update_time = datetime.now()
            
            self.log(f"Pipeline update #{self.update_count} at {self.last_update_time.strftime('%H:%M:%S')}")
            
            # 1. Fetch live games
            live_games_df = self.predictor.fetch_live_games_pacific()
            
            if live_games_df.empty:
                self.log("No games available for processing")
                return self.update_intervals['default']
            
            # 2. Generate predictions for all games
            prediction_results = self.predictor.run_full_prediction_pipeline(live_games_df)
            
            # Default update interval
            next_update_interval = self.update_intervals['default']
            
            if prediction_results is not None and not prediction_results.empty:
                # Process each game prediction
                for _, game_pred in prediction_results.iterrows():
                    game_id = game_pred['game_id']
                    str_game_id = str(game_id)
                    
                    # Determine update interval
                    game_interval = self.determine_update_interval(game_pred)
                    next_update_interval = min(next_update_interval, game_interval)
                    
                    # Update game state
                    self.update_game_state(game_id, game_pred)
                    
                    # Update prediction history
                    if str_game_id not in self.prediction_history:
                        self.prediction_history[str_game_id] = []
                    
                    # Check for anomalies
                    anomalies = self.detect_anomalies(
                        game_id, 
                        game_pred.to_dict(), 
                        self.prediction_history[str_game_id]
                    )
                    
                    # Add prediction to history
                    prediction_data = game_pred.to_dict()
                    prediction_data['timestamp'] = self.last_update_time
                    if anomalies:
                        prediction_data['anomalies'] = anomalies
                    
                    self.prediction_history[str_game_id].append(prediction_data)
                
                # 3. Visualize the results
                if self.visualization_mode == 'dashboard':
                    self.display_dashboard(prediction_results)
                else:
                    self.predictor.visualize_predictions(prediction_results)
                
                # Save data if auto-save is enabled
                if self.auto_save:
                    self.save_history()
                    self.save_game_states()
            
            return next_update_interval
            
        except Exception as e:
            self.log(f"Error in pipeline step: {e}", "ERROR")
            traceback.print_exc()
            return self.update_intervals['default']
    
    def display_dashboard(self, prediction_results):
        """
        Display an integrated dashboard with all visualizations.
        
        Args:
            prediction_results: DataFrame with prediction results
        """
        # Clear previous output
        clear_output(wait=True)
        
        # Display header
        print(f"NBA Score Prediction Dashboard - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)
        
        # Display update info
        print(f"Update #{self.update_count} | Next update in: varies by game state")
        
        # Group games by status
        status_groups = {}
        for _, game in prediction_results.iterrows():
            status = game.get('game_status', 'unknown')
            if status not in status_groups:
                status_groups[status] = []
            status_groups[status].append(game)
        
        # Display game summary by status
        for status, games in status_groups.items():
            print(f"\n{status.upper()} GAMES ({len(games)}):")
            print("-" * 80)
            
            for game in games:
                game_id = game['game_id']
                home_team = game['home_team']
                away_team = game['away_team']
                current_quarter = game.get('current_quarter', 0)
                
                # Format based on game state
                if status.lower() == 'pregame':
                    # Display predicted outcome
                    print(f"{home_team} vs {away_team}")
                    if 'predicted_home_score' in game:
                        home_pred = game['predicted_home_score']
                        away_pred = game['predicted_away_score']
                        print(f"  Prediction: {home_team} {home_pred:.1f} - {away_pred:.1f} {away_team}")
                        
                        # Add win probability if available
                        if 'win_probability' in game:
                            win_prob = game['win_probability'] * 100
                            fav_team = home_team if win_prob > 50 else away_team
                            fav_prob = win_prob if win_prob > 50 else 100 - win_prob
                            print(f"  Favorite: {fav_team} ({fav_prob:.1f}%)")
                
                elif status.lower() in ['live', 'in progress', 'ongoing']:
                    # Display current score and prediction
                    home_score = game.get('home_score', 0)
                    away_score = game.get('away_score', 0)
                    quarter_text = f"Q{current_quarter}" if current_quarter > 0 else "Pre-game"
                    
                    print(f"{home_team} vs {away_team} - {quarter_text}")
                    print(f"  Current: {home_team} {home_score} - {away_score} {away_team}")
                    
                    if 'predicted_home_score' in game:
                        home_pred = game['predicted_home_score']
                        away_pred = game['predicted_away_score']
                        print(f"  Prediction: {home_team} {home_pred:.1f} - {away_pred:.1f} {away_team}")
                        
                        # Add win probability
                        if 'win_probability' in game:
                            win_prob = game['win_probability'] * 100
                            print(f"  Win Probability: {home_team} {win_prob:.1f}%")
                    
                    # Determine update frequency
                    update_interval = self.determine_update_interval(game)
                    print(f"  Update Frequency: Every {update_interval} seconds")
                
                else:  # Finished or other states
                    # Display final score
                    home_score = game.get('home_score', 0)
                    away_score = game.get('away_score', 0)
                    
                    print(f"{home_team} vs {away_team} - FINAL")
                    print(f"  Final Score: {home_team} {home_score} - {away_score} {away_team}")
                
                # Add separator between games
                print("-" * 40)
        
        # Display any alerts
        if self.alerts:
            print("\nALERTS:")
            print("-" * 80)
            
            # Show most recent 5 alerts
            for alert in self.alerts[-5:]:
                alert_type = alert.get('type', 'unknown')
                game_id = alert.get('game_id', 'unknown')
                timestamp = alert.get('timestamp', '')
                
                if isinstance(timestamp, str):
                    try:
                        timestamp = datetime.fromisoformat(timestamp)
                        timestamp_str = timestamp.strftime('%H:%M:%S')
                    except:
                        timestamp_str = timestamp
                else:
                    timestamp_str = str(timestamp)
                
                print(f"[{timestamp_str}] {alert_type.upper()} in game {game_id}")
                
                # Add details based on alert type
                if alert_type == 'large_prediction_swing' and 'details' in alert:
                    details = alert['details']
                    prev = details.get('previous', {})
                    curr = details.get('current', {})
                    print(f"  Prediction changed significantly:")
                    print(f"  Home: {prev.get('home', 0):.1f} → {curr.get('home', 0):.1f}")
                    print(f"  Away: {prev.get('away', 0):.1f} → {curr.get('away', 0):.1f}")
                
                print("-" * 40)
        
        # Create visualizations
        # We could add additional visualizations here
        # For now, just use the standard visualization function
        self.predictor.visualize_predictions(prediction_results)
    
    def start_monitoring(self, max_updates=None):
        """Start the monitoring process with dynamic update intervals."""
        self.running = True
        self.update_count = 0
        
        self.log(f"Starting enhanced monitoring with dynamic update intervals")
        
        # Begin update loop
        while self.running:
            try:
                # Execute pipeline step and get next update interval
                next_interval = self.integrated_pipeline_step()
                
                # Check if we've reached max updates
                if max_updates and self.update_count >= max_updates:
                    self.log(f"Reached maximum updates ({max_updates})")
                    break
                
                # Wait for next update
                if self.running and (max_updates is None or self.update_count < max_updates):
                    self.log(f"Waiting {next_interval} seconds until next update...")
                    time.sleep(next_interval)
                
            except KeyboardInterrupt:
                self.log("Monitoring stopped by user")
                self.running = False
                break
            except Exception as e:
                self.log(f"Error during monitoring: {e}", "ERROR")
                traceback.print_exc()
                time.sleep(self.update_intervals['default'])
        
        self.running = False
        self.save_history()
        self.save_game_states()
        self.log("Monitoring stopped")
    
    def start_async_monitoring(self, max_updates=None):
        """Start monitoring in a background thread."""
        if self.update_thread is not None and self.update_thread.is_alive():
            self.log("Monitoring already running")
            return
        
        self.update_thread = threading.Thread(
            target=self.start_monitoring,
            args=(max_updates,)
        )
        self.update_thread.daemon = True
        self.update_thread.start()
        self.log("Monitoring started in background thread")
    
    def stop_monitoring(self):
        """Stop the monitoring process."""
        self.running = False
        self.log("Stopping monitoring...")
        
        if self.update_thread is not None and self.update_thread.is_alive():
            self.update_thread.join(timeout=5)
            if self.update_thread.is_alive():
                self.log("Monitoring thread is still running", "WARNING")
        
        self.save_history()
        self.save_game_states()
        self.log("Monitoring stopped and data saved")

In [ ]:
# Cell 7E: Comprehensive Performance Metrics Logging and Visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import json
import os
import traceback

class PredictionPerformanceTracker:
    """
    System for tracking, analyzing, and visualizing prediction performance over time.
    """
    
    def __init__(self, data_dir="data", metrics_file="prediction_metrics.json"):
        """
        Initialize the performance tracker.
        
        Args:
            data_dir: Directory for storing performance data
            metrics_file: Filename for the metrics JSON file
        """
        self.data_dir = data_dir
        self.metrics_file = os.path.join(data_dir, metrics_file)
        self.metrics_history = {}
        
        # Create data directory if it doesn't exist
        os.makedirs(data_dir, exist_ok=True)
        
        # Load existing metrics if available
        self.load_metrics()
    
    def load_metrics(self):
        """Load performance metrics from file if available."""
        if os.path.exists(self.metrics_file):
            try:
                with open(self.metrics_file, 'r') as f:
                    self.metrics_history = json.load(f)
                print(f"Loaded performance metrics from {self.metrics_file}")
            except Exception as e:
                print(f"Error loading performance metrics: {e}")
    
    def save_metrics(self):
        """Save performance metrics to file."""
        try:
            with open(self.metrics_file, 'w') as f:
                json.dump(self.metrics_history, f)
            print(f"Saved performance metrics to {self.metrics_file}")
        except Exception as e:
            print(f"Error saving performance metrics: {e}")
    
    def evaluate_prediction(self, game_id, prediction_data, actual_results):
        """
        Evaluate prediction accuracy and record metrics.
        
        Args:
            game_id: Unique identifier for the game
            prediction_data: Dict or Series with prediction information
            actual_results: Dict or Series with actual final results
            
        Returns:
            dict: Calculated performance metrics
        """
        # Convert Series to dict if needed
        if isinstance(prediction_data, pd.Series):
            prediction_data = prediction_data.to_dict()
        
        if isinstance(actual_results, pd.Series):
            actual_results = actual_results.to_dict()
        
        # Extract prediction values
        predicted_home = float(prediction_data.get('predicted_home_score', 0))
        predicted_away = float(prediction_data.get('predicted_away_score', 0))
        
        # Extract actual final scores
        actual_home = float(actual_results.get('home_score', 0))
        actual_away = float(actual_results.get('away_score', 0))
        
        # Calculate error metrics
        home_error = predicted_home - actual_home
        away_error = predicted_away - actual_away
        
        home_abs_error = abs(home_error)
        away_abs_error = abs(away_error)
        
        total_abs_error = home_abs_error + away_abs_error
        avg_abs_error = total_abs_error / 2
        
        home_squared_error = home_error ** 2
        away_squared_error = away_error ** 2
        
        total_squared_error = home_squared_error + away_squared_error
        avg_squared_error = total_squared_error / 2
        
        # Calculate margin error
        predicted_margin = predicted_home - predicted_away
        actual_margin = actual_home - actual_away
        margin_error = predicted_margin - actual_margin
        margin_abs_error = abs(margin_error)
        
        # Check winner prediction accuracy
        predicted_winner = 'home' if predicted_home > predicted_away else 'away'
        actual_winner = 'home' if actual_home > actual_away else 'away'
        winner_correct = predicted_winner == actual_winner
        
        # Check if within confidence interval if available
        within_confidence = None
        if ('home_lower_bound' in prediction_data and 
            'home_upper_bound' in prediction_data and
            'away_lower_bound' in prediction_data and
            'away_upper_bound' in prediction_data):
            
            home_lower = float(prediction_data.get('home_lower_bound', 0))
            home_upper = float(prediction_data.get('home_upper_bound', 0))
            away_lower = float(prediction_data.get('away_lower_bound', 0))
            away_upper = float(prediction_data.get('away_upper_bound', 0))
            
            home_within = home_lower <= actual_home <= home_upper
            away_within = away_lower <= actual_away <= away_upper
            
            within_confidence = home_within and away_within
        
        # Extract contextual data
        game_state = {
            'quarter': prediction_data.get('current_quarter', 0),
            'timestamp': prediction_data.get('timestamp', datetime.now().isoformat()),
            'time_remaining': prediction_data.get('time_remaining_mins', 0),
            'score_diff': prediction_data.get('score_differential', 0),
            'home_team': prediction_data.get('home_team', 'Home'),
            'away_team': prediction_data.get('away_team', 'Away')
        }
        
        # Compile metrics
        metrics = {
            'game_id': game_id,
            'game_state': game_state,
            'predictions': {
                'home': predicted_home,
                'away': predicted_away,
                'margin': predicted_margin
            },
            'actuals': {
                'home': actual_home,
                'away': actual_away,
                'margin': actual_margin
            },
            'errors': {
                'home_error': home_error,
                'away_error': away_error,
                'home_abs_error': home_abs_error,
                'away_abs_error': away_abs_error,
                'total_abs_error': total_abs_error,
                'avg_abs_error': avg_abs_error,
                'margin_error': margin_error,
                'margin_abs_error': margin_abs_error
            },
            'squared_errors': {
                'home_squared_error': home_squared_error,
                'away_squared_error': away_squared_error,
                'total_squared_error': total_squared_error,
                'avg_squared_error': avg_squared_error
            },
            'accuracy': {
                'winner_correct': winner_correct,
                'within_confidence': within_confidence
            }
        }
        
        # Store metrics in history
        game_id_str = str(game_id)
        if game_id_str not in self.metrics_history:
            self.metrics_history[game_id_str] = []
        
        self.metrics_history[game_id_str].append(metrics)
        
        # Save updated metrics
        self.save_metrics()
        
        return metrics
    
    def evaluate_game_history(self, game_id, prediction_history, actual_results):
        """
        Evaluate all predictions in a game's history.
        
        Args:
            game_id: Unique identifier for the game
            prediction_history: List of prediction data points
            actual_results: Dict or Series with actual final results
            
        Returns:
            list: Performance metrics for each prediction
        """
        metrics_list = []
        
        for pred in prediction_history:
            metrics = self.evaluate_prediction(game_id, pred, actual_results)
            metrics_list.append(metrics)
        
        return metrics_list
    
    def get_aggregated_metrics(self, filter_days=None):
        """
        Calculate aggregated performance metrics.
        
        Args:
            filter_days: Only include metrics from the last N days (optional)
            
        Returns:
            dict: Aggregated performance metrics
        """
        all_metrics = []
        
        # Flatten metrics from all games
        for game_id, metrics_list in self.metrics_history.items():
            for metrics in metrics_list:
                # Apply date filter if specified
                if filter_days is not None:
                    timestamp = metrics['game_state'].get('timestamp')
                    if timestamp:
                        if isinstance(timestamp, str):
                            try:
                                timestamp = datetime.fromisoformat(timestamp)
                            except:
                                # Skip if can't parse timestamp
                                continue
                        
                        days_ago = (datetime.now() - timestamp).days
                        if days_ago > filter_days:
                            continue
                
                all_metrics.append(metrics)
        
        if not all_metrics:
            return {}
        
        # Initialize aggregation containers
        agg_metrics = {
            'count': len(all_metrics),
            'mae': {},
            'rmse': {},
            'winner_accuracy': {},
            'confidence_accuracy': {}
        }
        
        # Separate metrics by quarter for more detailed analysis
        metrics_by_quarter = {}
        for metrics in all_metrics:
            quarter = metrics['game_state'].get('quarter', 0)
            if quarter not in metrics_by_quarter:
                metrics_by_quarter[quarter] = []
            metrics_by_quarter[quarter].append(metrics)
        
        # Overall metrics (all quarters)
        avg_abs_errors = [m['errors']['avg_abs_error'] for m in all_metrics]
        avg_squared_errors = [m['squared_errors']['avg_squared_error'] for m in all_metrics]
        winner_correct = [m['accuracy']['winner_correct'] for m in all_metrics if m['accuracy']['winner_correct'] is not None]
        within_confidence = [m['accuracy']['within_confidence'] for m in all_metrics if m['accuracy']['within_confidence'] is not None]
        
        agg_metrics['mae']['overall'] = sum(avg_abs_errors) / len(avg_abs_errors) if avg_abs_errors else None
        agg_metrics['rmse']['overall'] = np.sqrt(sum(avg_squared_errors) / len(avg_squared_errors)) if avg_squared_errors else None
        agg_metrics['winner_accuracy']['overall'] = sum(winner_correct) / len(winner_correct) if winner_correct else None
        agg_metrics['confidence_accuracy']['overall'] = sum(within_confidence) / len(within_confidence) if within_confidence else None
        
        # Metrics by quarter
        for quarter, quarter_metrics in metrics_by_quarter.items():
            q_avg_abs_errors = [m['errors']['avg_abs_error'] for m in quarter_metrics]
            q_avg_squared_errors = [m['squared_errors']['avg_squared_error'] for m in quarter_metrics]
            q_winner_correct = [m['accuracy']['winner_correct'] for m in quarter_metrics if m['accuracy']['winner_correct'] is not None]
            q_within_confidence = [m['accuracy']['within_confidence'] for m in quarter_metrics if m['accuracy']['within_confidence'] is not None]
            
            agg_metrics['mae'][f'q{quarter}'] = sum(q_avg_abs_errors) / len(q_avg_abs_errors) if q_avg_abs_errors else None
            agg_metrics['rmse'][f'q{quarter}'] = np.sqrt(sum(q_avg_squared_errors) / len(q_avg_squared_errors)) if q_avg_squared_errors else None
            agg_metrics['winner_accuracy'][f'q{quarter}'] = sum(q_winner_correct) / len(q_winner_correct) if q_winner_correct else None
            agg_metrics['confidence_accuracy'][f'q{quarter}'] = sum(q_within_confidence) / len(q_within_confidence) if q_within_confidence else None
            
            agg_metrics[f'count_q{quarter}'] = len(quarter_metrics)
        
        return agg_metrics
    
    def visualize_performance(self, filter_days=None):
        """
        Create visualizations of prediction performance.
        
        Args:
            filter_days: Only include metrics from the last N days (optional)
            
        Returns:
            matplotlib Figure with performance visualizations
        """
        aggregated = self.get_aggregated_metrics(filter_days)
        
        if not aggregated:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, "No performance data available", 
                   ha='center', va='center', fontsize=14)
            return fig
        
        # Create visualization
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Error by Quarter (MAE and RMSE)
        quarters = sorted([k for k in aggregated['mae'].keys() if k.startswith('q')])
        
        if quarters:
            mae_values = [aggregated['mae'][q] for q in quarters]
            rmse_values = [aggregated['rmse'][q] for q in quarters]
            
            ax1 = axs[0, 0]
            x = range(len(quarters))
            
            ax1.bar(x, mae_values, width=0.4, label='MAE', color='skyblue')
            ax1.bar([i + 0.4 for i in x], rmse_values, width=0.4, label='RMSE', color='salmon')
            
            ax1.set_xticks([i + 0.2 for i in x])
            ax1.set_xticklabels(quarters)
            ax1.set_ylabel('Points')
            ax1.set_title('Prediction Error by Quarter')
            ax1.legend()
            ax1.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for i, v in enumerate(mae_values):
                ax1.text(i, v + 0.5, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
            
            for i, v in enumerate(rmse_values):
                ax1.text(i + 0.4, v + 0.5, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
        
        # 2. Winner Prediction Accuracy by Quarter
        quarters = sorted([k for k in aggregated['winner_accuracy'].keys() if k.startswith('q')])
        
        if quarters:
            accuracy_values = [aggregated['winner_accuracy'][q] * 100 for q in quarters]
            
            ax2 = axs[0, 1]
            x = range(len(quarters))
            
            ax2.bar(x, accuracy_values, width=0.6, color='lightgreen')
            
            ax2.set_xticks(x)
            ax2.set_xticklabels(quarters)
            ax2.set_ylabel('Accuracy (%)')
            ax2.set_title('Winner Prediction Accuracy by Quarter')
            ax2.set_ylim(0, 100)
            ax2.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for i, v in enumerate(accuracy_values):
                ax2.text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # 3. Confidence Interval Accuracy
        quarters = sorted([k for k in aggregated['confidence_accuracy'].keys() if k.startswith('q') and aggregated['confidence_accuracy'][k] is not None])
        
        if quarters:
            conf_values = [aggregated['confidence_accuracy'][q] * 100 for q in quarters]
            
            ax3 = axs[1, 0]
            x = range(len(quarters))
            
            ax3.bar(x, conf_values, width=0.6, color='plum')
            
            ax3.set_xticks(x)
            ax3.set_xticklabels(quarters)
            ax3.set_ylabel('Within Interval (%)')
            ax3.set_title('Confidence Interval Accuracy by Quarter')
            ax3.set_ylim(0, 100)
            ax3.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for i, v in enumerate(conf_values):
                ax3.text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
        
        # 4. Sample Size by Quarter
        quarters = [f'q{i}' for i in range(5)]
        counts = [aggregated.get(f'count_{q}', 0) for q in quarters]
        
        ax4 = axs[1, 1]
        x = range(len(quarters))
        
        ax4.bar(x, counts, width=0.6, color='lightblue')
        
        ax4.set_xticks(x)
        ax4.set_xticklabels(quarters)
        ax4.set_ylabel('Count')
        ax4.set_title('Sample Size by Quarter')
        ax4.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(counts):
            ax4.text(i, v + 0.5, str(v), ha='center', va='bottom', fontsize=9)
        
        # Add overall title with filter information
        if filter_days is not None:
            title = f"Prediction Performance Metrics (Last {filter_days} Days)"
        else:
            title = "Prediction Performance Metrics (All Time)"
        
        title += f" - {aggregated['count']} Total Predictions"
        fig.suptitle(title, fontsize=16)
        
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        return fig
    
    def generate_performance_report(self, filter_days=None):
        """
        Generate a text-based performance report.
        
        Args:
            filter_days: Only include metrics from the last N days (optional)
            
        Returns:
            str: Formatted performance report
        """
        aggregated = self.get_aggregated_metrics(filter_days)
        
        if not aggregated:
            return "No performance data available for reporting."
        
        # Create report
        if filter_days is not None:
            report = f"Performance Report (Last {filter_days} Days)\n"
        else:
            report = "Performance Report (All Time)\n"
        
        report += "=" * 60 + "\n\n"
        
        # Overall metrics
        report += f"Total Predictions: {aggregated['count']}\n\n"
        
        # Error metrics
        report += "Error Metrics by Quarter:\n"
        report += "-" * 40 + "\n"
        report += f"{'Quarter':<10}{'MAE':>10}{'RMSE':>10}{'Count':>10}\n"
        report += "-" * 40 + "\n"
        
        # Overall row
        overall_mae = aggregated['mae'].get('overall')
        overall_rmse = aggregated['rmse'].get('overall')
        overall_mae_str = f"{overall_mae:.2f}" if overall_mae is not None else "N/A"
        overall_rmse_str = f"{overall_rmse:.2f}" if overall_rmse is not None else "N/A"
        report += f"{'Overall':<10}{overall_mae_str:>10}{overall_rmse_str:>10}{aggregated['count']:>10}\n"
        
        # Quarter-specific rows
        for i in range(5):
            quarter_key = f'q{i}'
            mae = aggregated['mae'].get(quarter_key)
            rmse = aggregated['rmse'].get(quarter_key)
            count = aggregated.get(f'count_{quarter_key}', 0)
            
            mae_str = f"{mae:.2f}" if mae is not None else "N/A"
            rmse_str = f"{rmse:.2f}" if rmse is not None else "N/A"
            
            report += f"{quarter_key:<10}{mae_str:>10}{rmse_str:>10}{count:>10}\n"
        
        report += "\n"
        
        # Accuracy metrics
        report += "Prediction Accuracy by Quarter:\n"
        report += "-" * 50 + "\n"
        report += f"{'Quarter':<10}{'Winner %':>12}{'Confidence %':>15}{'Count':>10}\n"
        report += "-" * 50 + "\n"
        
        # Overall row
        overall_winner = aggregated['winner_accuracy'].get('overall')
        overall_conf = aggregated['confidence_accuracy'].get('overall')
        
        winner_str = f"{overall_winner*100:.1f}%" if overall_winner is not None else "N/A"
        conf_str = f"{overall_conf*100:.1f}%" if overall_conf is not None else "N/A"
        
        report += f"{'Overall':<10}{winner_str:>12}{conf_str:>15}{aggregated['count']:>10}\n"
        
        # Quarter-specific rows
        for i in range(5):
            quarter_key = f'q{i}'
            winner_acc = aggregated['winner_accuracy'].get(quarter_key)
            conf_acc = aggregated['confidence_accuracy'].get(quarter_key)
            count = aggregated.get(f'count_{quarter_key}', 0)
            
            winner_str = f"{winner_acc*100:.1f}%" if winner_acc is not None else "N/A"
            conf_str = f"{conf_acc*100:.1f}%" if conf_acc is not None else "N/A"
            
            report += f"{quarter_key:<10}{winner_str:>12}{conf_str:>15}{count:>10}\n"
        
        report += "\n"
        report += "=" * 60 + "\n"
        return report

In [ ]:
# Cell 7F: Comprehensive Validation Framework

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import traceback

class NBAModelValidator:
    """
    Comprehensive validation framework for NBA prediction models
    using historical game data to simulate live prediction scenarios.
    """
    
    def __init__(self, model_path=None, feature_list=None):
        """
        Initialize the validator.
        
        Args:
            model_path: Path to the model file (optional)
            feature_list: List of features expected by the model (optional)
        """
        self.model = None
        self.feature_list = feature_list
        
        # Load model if path provided
        if model_path:
            self.load_model(model_path)
    
    def load_model(self, model_path):
        """Load model from file."""
        try:
            self.model = joblib.load(model_path)
            print(f"Model loaded from {model_path}")
            
            # Try to determine feature list if not provided
            if self.feature_list is None and hasattr(self.model, 'feature_importances_'):
                n_features = len(self.model.feature_importances_)
                
                if n_features > 8:
                    self.feature_list = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'prev_matchup_diff',
                        'rest_days_home', 'rest_days_away', 'rest_advantage',
                        'is_back_to_back_home', 'is_back_to_back_away',
                        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                    ]
                else:
                    self.feature_list = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                    ]
                
                print(f"Using detected feature list with {n_features} features")
        except Exception as e:
            print(f"Error loading model: {e}")
            traceback.print_exc()
    
    def fetch_validation_games(self, db_engine, sample_size=50, min_date=None, max_date=None, specific_teams=None):
        """
        Fetch historical games for validation.
        
        Args:
            db_engine: SQLAlchemy database engine
            sample_size: Maximum number of games to fetch
            min_date: Minimum game date (optional)
            max_date: Maximum game date (optional)
            specific_teams: List of teams to include (optional)
            
        Returns:
            DataFrame with historical games
        """
        try:
            # Build query conditionally
            query = "SELECT * FROM nba_historical_game_stats WHERE 1=1"
            
            if min_date:
                query += f" AND game_date >= '{min_date}'"
            
            if max_date:
                query += f" AND game_date <= '{max_date}'"
            
            if specific_teams:
                team_list = "', '".join(specific_teams)
                query += f" AND (home_team IN ('{team_list}') OR away_team IN ('{team_list}'))"
            
            query += f" ORDER BY game_date DESC LIMIT {sample_size}"
            
            # Fetch data
            games_df = pd.read_sql(query, db_engine)
            print(f"Fetched {len(games_df)} games for validation")
            
            return games_df
        except Exception as e:
            print(f"Error fetching validation games: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def prepare_game_features(self, game, current_quarter, feature_generator=None):
        """
        Prepare features for a game at a specific quarter.
        
        Args:
            game: Game data (Series or dict)
            current_quarter: Quarter to simulate (0-4)
            feature_generator: Optional NBAFeatureGenerator instance
            
        Returns:
            DataFrame with prepared features
        """
        # Convert to DataFrame if not already
        if isinstance(game, pd.Series):
            game_df = pd.DataFrame([game.to_dict()])
        elif isinstance(game, dict):
            game_df = pd.DataFrame([game])
        else:
            game_df = pd.DataFrame([game])
        
        # Simulate in-progress game by zeroing out future quarters
        for q in range(1, 5):
            if q > current_quarter:
                game_df[f'home_q{q}'] = 0
                game_df[f'away_q{q}'] = 0
        
        # Update current score based on quarters that have been played
        game_df['home_score'] = sum(float(game_df.iloc[0].get(f'home_q{q}', 0) or 0) for q in range(1, current_quarter+1))
        game_df['away_score'] = sum(float(game_df.iloc[0].get(f'away_q{q}', 0) or 0) for q in range(1, current_quarter+1))
        
        # Set current quarter
        game_df['current_quarter'] = current_quarter
        
        # If we have a feature generator, use it
        if feature_generator:
            # Generate all features
            features_df = feature_generator.generate_all_features(game_df)
            return features_df
        
        # Otherwise do basic feature preparation
        # Calculate score ratio
        home_score = float(game_df['home_score'])
        away_score = float(game_df['away_score'])
        total_score = home_score + away_score
        game_df['score_ratio'] = (home_score / total_score) if total_score > 0 else 0.5
        
        # Set dummy values for any missing features
        if self.feature_list:
            for feature in self.feature_list:
                if feature not in game_df.columns:
                    game_df[feature] = 0.0
        
        return game_df
    
    def simulate_game_predictions(self, game, ensemble=None):
        """
        Simulate predictions for a game as it progresses quarter by quarter.
        
        Args:
            game: Game data (Series or dict)
            ensemble: Optional AdaptiveEnsemble instance for weighted predictions
            
        Returns:
            dict: Simulated predictions for each quarter
        """
        if self.model is None:
            print("No model loaded for prediction")
            return {}
        
        # Initialize results
        results = {'game_id': game.get('game_id', 'unknown')}
        
        # Get actual final scores for comparison
        actual_home_score = float(game.get('home_score', 0))
        actual_away_score = float(game.get('away_score', 0))
        results['actual'] = {'home': actual_home_score, 'away': actual_away_score}
        
        # Get team names
        results['home_team'] = game.get('home_team', 'Home')
        results['away_team'] = game.get('away_team', 'Away')
        
        # Simulate predictions for each quarter
        quarter_predictions = {}
        
        for quarter in range(5):  # 0-4 (0 = pre-game)
            # Prepare features for this quarter
            features_df = self.prepare_game_features(game, quarter)
            
            if features_df.empty:
                continue
            
            try:
                # Extract features for prediction
                if self.feature_list:
                    X = features_df[self.feature_list]
                else:
                    # Fallback - use all numeric columns
                    X = features_df.select_dtypes(include=['number'])
                
                # Make prediction
                home_pred = self.model.predict(X)[0]
                
                # Estimate away score
                # Simple heuristic: assume similar home advantage as actual
                home_advantage = actual_home_score - actual_away_score
                away_pred = home_pred - (home_advantage * 0.8)  # Discount the advantage slightly
                
                # Calculate confidence (higher in later quarters)
                confidence = 0.5 + (quarter * 0.1)
                
                # Store prediction
                quarter_predictions[f'q{quarter}'] = {
                    'home': home_pred,
                    'away': away_pred,
                    'confidence': confidence
                }
                
                # Apply ensemble weighting if available
                if ensemble:
                    # Get quarter-specific prediction if applicable
                    if quarter > 0 and f'q{quarter}_model' in ensemble.models:
                        quarter_model = ensemble.models[f'q{quarter}_model']
                        quarter_pred = quarter_model.predict(X)[0]
                    else:
                        # Fallback - use similar prediction with slight adjustment
                        quarter_pred = home_pred * (1 + (random.random() - 0.5) * 0.1)
                    
                    # Get ensemble weights and prediction
                    ensemble_pred, weights, _ = ensemble.predict(
                        home_pred, quarter_pred, quarter, 
                        {
                            'score_differential': features_df['score_differential'].iloc[0],
                            'momentum': features_df['cumulative_momentum'].iloc[0] if 'cumulative_momentum' in features_df.columns else 0
                        }
                    )
                    
                    # Add ensemble prediction
                    quarter_predictions[f'q{quarter}']['ensemble'] = ensemble_pred
                    quarter_predictions[f'q{quarter}']['weights'] = weights
            
            except Exception as e:
                print(f"Error simulating quarter {quarter}: {e}")
                traceback.print_exc()
        
        results['predictions'] = quarter_predictions
        return results
    
    def batch_validate(self, games_df, feature_generator=None, ensemble=None):
        """
        Run batch validation on multiple games.
        
        Args:
            games_df: DataFrame with games to validate
            feature_generator: Optional NBAFeatureGenerator instance
            ensemble: Optional AdaptiveEnsemble instance
            
        Returns:
            list: Validation results for each game
        """
        if self.model is None:
            print("No model loaded for validation")
            return []
        
        validation_results = []
        
        for _, game in games_df.iterrows():
            result = self.simulate_game_predictions(game, ensemble)
            validation_results.append(result)
        
        return validation_results
    
    def calculate_metrics(self, validation_results):
        """
        Calculate accuracy metrics from validation results.
        
        Args:
            validation_results: List of validation result dicts
            
        Returns:
            DataFrame with accuracy metrics by quarter
        """
        metrics = []
        
        for result in validation_results:
            actual_home = result['actual']['home']
            actual_away = result['actual']['away']
            
            for quarter, preds in result['predictions'].items():
                pred_home = preds['home']
                pred_away = preds['away']
                
                # Calculate errors
                home_error = pred_home - actual_home
                away_error = pred_away - actual_away
                home_abs_error = abs(home_error)
                away_abs_error = abs(away_error)
                
                # Calculate margin error
                actual_margin = actual_home - actual_away
                pred_margin = pred_home - pred_away
                margin_error = pred_margin - actual_margin
                
                # Check winner prediction
                actual_winner = 'home' if actual_home > actual_away else 'away'
                pred_winner = 'home' if pred_home > pred_away else 'away'
                correct_winner = actual_winner == pred_winner
                
                # Store metrics
                metrics.append({
                    'game_id': result['game_id'],
                    'quarter': quarter,
                    'home_team': result['home_team'],
                    'away_team': result['away_team'],
                    'actual_home': actual_home,
                    'actual_away': actual_away,
                    'pred_home': pred_home,
                    'pred_away': pred_away,
                    'home_error': home_error,
                    'away_error': away_error,
                    'home_abs_error': home_abs_error,
                    'away_abs_error': away_abs_error,
                    'avg_abs_error': (home_abs_error + away_abs_error) / 2,
                    'margin_error': margin_error,
                    'actual_winner': actual_winner,
                    'pred_winner': pred_winner,
                    'correct_winner': correct_winner
                })
        
        return pd.DataFrame(metrics)
    
    def visualize_validation_results(self, metrics_df):
        """
        Create visualizations of validation results.
        
        Args:
            metrics_df: DataFrame with validation metrics
            
        Returns:
            matplotlib Figure with visualizations
        """
        if metrics_df.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, "No validation data available", 
                   ha='center', va='center', fontsize=14)
            return fig
        
        # Create figure
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. Error by Quarter
        quarter_metrics = metrics_df.groupby('quarter').agg({
            'home_abs_error': 'mean',
            'away_abs_error': 'mean',
            'avg_abs_error': 'mean',
            'correct_winner': 'mean',
            'game_id': 'count'
        }).reset_index()
        
        quarter_metrics = quarter_metrics.sort_values('quarter')
        
        ax1 = axs[0, 0]
        x = range(len(quarter_metrics))
        
        ax1.bar(x, quarter_metrics['avg_abs_error'], color='lightblue')
        ax1.set_xticks(x)
        ax1.set_xticklabels(quarter_metrics['quarter'])
        ax1.set_ylabel('Average Absolute Error (points)')
        ax1.set_title('Prediction Error by Quarter')
        ax1.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(quarter_metrics['avg_abs_error']):
            ax1.text(i, v + 0.5, f'{v:.2f}', ha='center')
        
        # 2. Winner Prediction Accuracy
        ax2 = axs[0, 1]
        accuracy = quarter_metrics['correct_winner'] * 100
        
        ax2.bar(x, accuracy, color='lightgreen')
        ax2.set_xticks(x)
        ax2.set_xticklabels(quarter_metrics['quarter'])
        ax2.set_ylabel('Accuracy (%)')
        ax2.set_title('Winner Prediction Accuracy by Quarter')
        ax2.set_ylim(0, 100)
        ax2.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(accuracy):
            ax2.text(i, v + 2, f'{v:.1f}%', ha='center')
        
        # 3. Error Distribution
        ax3 = axs[1, 0]
        ax3.hist(metrics_df['avg_abs_error'], bins=20, color='skyblue', alpha=0.7)
        ax3.set_xlabel('Average Absolute Error')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Error Distribution')
        ax3.grid(axis='y', alpha=0.3)
        
        # 4. Sample Size
        ax4 = axs[1, 1]
        samples = quarter_metrics['game_id']
        
        ax4.bar(x, samples, color='salmon')
        ax4.set_xticks(x)
        ax4.set_xticklabels(quarter_metrics['quarter'])
        ax4.set_ylabel('Count')
        ax4.set_title('Sample Size by Quarter')
        ax4.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(samples):
            ax4.text(i, v + 1, str(int(v)), ha='center')
        
        # Add overall title
        fig.suptitle('Model Validation Results', fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        
        return fig

In [ ]:
# Cell 7G: Continuous Learning System

import pandas as pd
import numpy as np
import joblib
import os
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import traceback

class NBAModelTrainer:
    """
    Continuous learning system for NBA prediction models.
    Automatically retrains models with new game data and 
    implements model performance monitoring.
    """
    
    def __init__(self, models_dir="models", data_dir="data"):
        """
        Initialize the model trainer.
        
        Args:
            models_dir: Directory for storing trained models
            data_dir: Directory for storing training data and metrics
        """
        self.models_dir = models_dir
        self.data_dir = data_dir
        self.model_metrics = {}
        self.feature_sets = {
            'standard': [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ],
            'enhanced': [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
        }
        
        # Create directories if they don't exist
        os.makedirs(models_dir, exist_ok=True)
        os.makedirs(data_dir, exist_ok=True)
        
        # Load existing metrics if available
        self.metrics_file = os.path.join(data_dir, "model_metrics.joblib")
        if os.path.exists(self.metrics_file):
            try:
                self.model_metrics = joblib.load(self.metrics_file)
                print(f"Loaded model metrics from {self.metrics_file}")
            except Exception as e:
                print(f"Error loading model metrics: {e}")
    
    def save_metrics(self):
        """Save model metrics to file."""
        try:
            joblib.dump(self.model_metrics, self.metrics_file)
            print(f"Saved model metrics to {self.metrics_file}")
        except Exception as e:
            print(f"Error saving model metrics: {e}")
    
    def fetch_training_data(self, db_engine, days_lookback=180, min_games=500):
        """
        Fetch historical game data for training.
        
        Args:
            db_engine: SQLAlchemy database engine
            days_lookback: Number of days to look back for data
            min_games: Minimum number of games to fetch
            
        Returns:
            DataFrame with training data
        """
        try:
            # Start with recent games
            lookback_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
            
            query = f"""
            SELECT * FROM nba_historical_game_stats 
            WHERE game_date >= '{lookback_date}'
            ORDER BY game_date DESC
            """
            
            df = pd.read_sql(query, db_engine)
            print(f"Fetched {len(df)} games since {lookback_date}")
            
            # If we don't have enough games, fetch more
            if len(df) < min_games:
                additional_needed = min_games - len(df)
                
                additional_query = f"""
                SELECT * FROM nba_historical_game_stats 
                WHERE game_date < '{lookback_date}'
                ORDER BY game_date DESC
                LIMIT {additional_needed}
                """
                
                additional_df = pd.read_sql(additional_query, db_engine)
                print(f"Fetched {len(additional_df)} additional games")
                
                # Combine datasets
                df = pd.concat([df, additional_df], ignore_index=True)
            
            return df
        except Exception as e:
            print(f"Error fetching training data: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def prepare_features(self, df, feature_generator=None, feature_set='enhanced'):
        """
        Prepare features for training.
        
        Args:
            df: DataFrame with raw game data
            feature_generator: Optional NBAFeatureGenerator instance
            feature_set: Which feature set to use ('standard' or 'enhanced')
            
        Returns:
            DataFrame with prepared features
        """
        # If we have a feature generator, use it
        if feature_generator:
            features_df = feature_generator.generate_all_features(df)
            print(f"Generated features using feature generator: {len(features_df)} rows")
            return features_df
        
        # Otherwise do basic feature preparation
        print("No feature generator provided. Using basic feature preparation.")
        features_df = df.copy()
        
        # Ensure all required columns exist with default values
        for col in self.feature_sets[feature_set]:
            if col not in features_df.columns:
                features_df[col] = 0.0
        
        return features_df
    
    def train_model(self, features_df, target_col='home_score', feature_set='enhanced', model_params=None):
        """
        Train a new prediction model.
        
        Args:
            features_df: DataFrame with prepared features
            target_col: Target column to predict
            feature_set: Which feature set to use ('standard' or 'enhanced')
            model_params: Dictionary with model parameters (or None for defaults)
            
        Returns:
            Trained model object
        """
        # Check if we have data
        if features_df.empty:
            print("No data for training.")
            return None
        
        # Get feature columns
        feature_cols = self.feature_sets.get(feature_set, self.feature_sets['standard'])
        
        # Make sure all feature columns exist
        missing_cols = [col for col in feature_cols if col not in features_df.columns]
        if missing_cols:
            print(f"Warning: Missing feature columns: {missing_cols}")
            feature_cols = [col for col in feature_cols if col in features_df.columns]
        
        # Check if we have the target column
        if target_col not in features_df.columns:
            print(f"Error: Target column '{target_col}' not found.")
            return None
        
        # Prepare X and y
        X = features_df[feature_cols]
        y = features_df[target_col]
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples")
        
        # Set up model params
        if model_params is None:
            model_params = {
                'n_estimators': 200,
                'max_depth': 5,
                'learning_rate': 0.1,
                'subsample': 0.8,
                'random_state': 42
            }
        
        # Create and train model
        model = GradientBoostingRegressor(**model_params)
        
        try:
            # Train the model
            start_time = datetime.now()
            model.fit(X_train, y_train)
            train_time = (datetime.now() - start_time).total_seconds()
            
            # Evaluate the model
            train_mse = mean_squared_error(y_train, model.predict(X_train))
            test_mse = mean_squared_error(y_test, model.predict(X_test))
            r2 = r2_score(y_test, model.predict(X_test))
            
            print(f"Training completed in {train_time:.2f} seconds")
            print(f"Training MSE: {train_mse:.2f}")
            print(f"Test MSE: {test_mse:.2f}")
            print(f"R² Score: {r2:.4f}")
            
            # Store metrics
            metrics = {
                'feature_set': feature_set,
                'train_mse': train_mse,
                'test_mse': test_mse,
                'train_rmse': np.sqrt(train_mse),
                'test_rmse': np.sqrt(test_mse),
                'r2': r2,
                'train_size': len(X_train),
                'test_size': len(X_test),
                'train_time': train_time,
                'timestamp': datetime.now().isoformat()
            }
            
            return model, metrics
            
        except Exception as e:
            print(f"Error training model: {e}")
            traceback.print_exc()
            return None, None
    
    def save_model(self, model, metrics, model_name=None):
        """
        Save trained model and its metrics.
        
        Args:
            model: Trained model object
            metrics: Dict with model metrics
            model_name: Name for the model file (or None for auto-generated)
            
        Returns:
            str: Path to the saved model file
        """
        if model is None:
            return None
        
        # Generate model name if not provided
        if model_name is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            feature_set = metrics.get('feature_set', 'unknown')
            test_rmse = metrics.get('test_rmse', 0)
            model_name = f"score_prediction_{feature_set}_{test_rmse:.2f}_{timestamp}.pkl"
        
        # Save the model
        model_path = os.path.join(self.models_dir, model_name)
        
        try:
            joblib.dump(model, model_path)
            print(f"Model saved to {model_path}")
            
            # Store metrics
            self.model_metrics[model_name] = metrics
            self.save_metrics()
            
            return model_path
        except Exception as e:
            print(f"Error saving model: {e}")
            return None
    
    def check_performance_drift(self, current_metrics, drift_threshold=0.1):
        """
        Check if model performance has drifted significantly.
        
        Args:
            current_metrics: Metrics from current model
            drift_threshold: Threshold for significant drift
            
        Returns:
            bool: True if significant drift detected
        """
        if not self.model_metrics:
            # No previous metrics to compare
            return False
        
        # Find the most recent model with the same feature set
        feature_set = current_metrics.get('feature_set')
        comparable_models = [
            (name, metrics) for name, metrics in self.model_metrics.items()
            if metrics.get('feature_set') == feature_set
        ]
        
        if not comparable_models:
            return False
        
        # Sort by timestamp
        comparable_models.sort(key=lambda x: x[1].get('timestamp', ''), reverse=True)
        
        # Get the most recent model
        recent_model_name, recent_metrics = comparable_models[0]
        
        # Compare RMSE
        recent_rmse = recent_metrics.get('test_rmse', float('inf'))
        current_rmse = current_metrics.get('test_rmse', float('inf'))
        
        # Calculate relative change
        if recent_rmse > 0:
            relative_change = abs(current_rmse - recent_rmse) / recent_rmse
            
            print(f"Performance drift check: Current RMSE = {current_rmse:.2f}, Previous = {recent_rmse:.2f}")
            print(f"Relative change: {relative_change:.2%}")
            
            return relative_change > drift_threshold
        
        return False
    
    def train_and_save_model(self, features_df, target_col='home_score', feature_set='enhanced'):
        """
        Train a new model and save it if it performs well.
        
        Args:
            features_df: DataFrame with prepared features
            target_col: Target column to predict
            feature_set: Which feature set to use
            
        Returns:
            tuple: (model, model_path)
        """
        # Train the model
        model, metrics = self.train_model(features_df, target_col, feature_set)
        
        if model is None or metrics is None:
            return None, None
        
        # Check if model performance has drifted
        drift_detected = self.check_performance_drift(metrics)
        
        # Save the model
        if drift_detected:
            print("Significant performance drift detected! Saving as a new version.")
            model_path = self.save_model(model, metrics)
        else:
            # Use a more generic name for incremental updates
            timestamp = datetime.now().strftime("%Y%m%d")
            model_name = f"score_prediction_{feature_set}_{timestamp}.pkl"
            model_path = self.save_model(model, metrics, model_name)
        
        return model, model_path
    
    def visualize_model_history(self):
        """
        Visualize the history of model performance.
        
        Returns:
            matplotlib Figure with performance history
        """
        if not self.model_metrics:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.text(0.5, 0.5, "No model history available", 
                   ha='center', va='center', fontsize=14)
            return fig
        
        # Convert metrics to DataFrame
        metrics_data = []
        
        for model_name, metrics in self.model_metrics.items():
            metrics_data.append({
                'model_name': model_name,
                'feature_set': metrics.get('feature_set', 'unknown'),
                'train_rmse': metrics.get('train_rmse', None),
                'test_rmse': metrics.get('test_rmse', None),
                'r2': metrics.get('r2', None),
                'timestamp': metrics.get('timestamp', None),
                'train_size': metrics.get('train_size', 0)
            })
        
        metrics_df = pd.DataFrame(metrics_data)
        
        # Convert timestamp to datetime
        metrics_df['timestamp'] = pd.to_datetime(metrics_df['timestamp'], errors='coerce')
        
        # Sort by timestamp
        metrics_df = metrics_df.sort_values('timestamp')
        
        # Create visualization
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Plot RMSE over time
        for feature_set in metrics_df['feature_set'].unique():
            subset = metrics_df[metrics_df['feature_set'] == feature_set]
            
            ax1.plot(subset['timestamp'], subset['test_rmse'], 'o-', 
                    label=f"{feature_set} (Test)")
            ax1.plot(subset['timestamp'], subset['train_rmse'], '--', alpha=0.5,
                    label=f"{feature_set} (Train)")
        
        ax1.set_ylabel('RMSE')
        ax1.set_title('Model Performance Over Time')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Format x-axis with dates
        ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
        
        # Plot R² over time
        for feature_set in metrics_df['feature_set'].unique():
            subset = metrics_df[metrics_df['feature_set'] == feature_set]
            
            ax2.plot(subset['timestamp'], subset['r2'], 'o-', 
                    label=feature_set)
        
        ax2.set_ylabel('R² Score')
        ax2.set_xlabel('Training Date')
        ax2.set_title('Model R² Score Over Time')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Format x-axis with dates
        ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))
        plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        return fig

In [ ]:
# Cell 8: Improved Score Prediction Model Evaluation with Time-Based Cross-Validation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
from datetime import datetime
import time
import traceback
import os
import json

def evaluate_model_with_time_cv(model, X, y, date_column, n_splits=5, visualize=True):
    """
    Evaluate a model using time-based cross-validation.
    
    Args:
        model: The trained model to evaluate
        X: Feature DataFrame (should NOT include datetime columns used for sorting in numeric features)
        y: Target Series
        date_column: Name of the date column for time-based splitting
        n_splits: Number of time-based CV splits
        visualize: Whether to visualize results
        
    Returns:
        Dictionary with evaluation metrics
    """
    # Verify the date column exists
    if date_column not in X.index.names and date_column not in X.columns:
        raise ValueError(f"Date column '{date_column}' not found in data")
    
    # Sort data by date, then remove the date column from X so it doesn't break numeric fitting
    if date_column in X.columns:
        sorted_indices = X[date_column].argsort()
        X_sorted = X.iloc[sorted_indices].copy()
        y_sorted = y.iloc[sorted_indices].copy()
        # Remove the date column from numeric features
        X_sorted.drop(columns=[date_column], inplace=True)
    else:
        # Already indexed by date
        X_sorted = X.copy()
        y_sorted = y.copy()
    
    # Prepare time-based cross-validation
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    # Track metrics across folds
    train_scores = []
    test_scores = []
    train_rmse = []
    test_rmse = []
    train_mae = []
    test_mae = []
    r2_scores = []
    fold_sizes = []
    
    print(f"\nPerforming {n_splits}-fold time-based cross-validation")
    for i, (train_idx, test_idx) in enumerate(tscv.split(X_sorted)):
        X_train_fold = X_sorted.iloc[train_idx]
        X_test_fold = X_sorted.iloc[test_idx]
        y_train_fold = y_sorted.iloc[train_idx]
        y_test_fold = y_sorted.iloc[test_idx]
        
        fold_sizes.append(len(test_idx))
        
        try:
            # Train model on this fold (if model is not already trained)
            if hasattr(model, 'fit'):
                model.fit(X_train_fold, y_train_fold)
            
            # Predict
            y_train_pred = model.predict(X_train_fold)
            y_test_pred = model.predict(X_test_fold)
            
            # Calculate metrics
            train_mse = mean_squared_error(y_train_fold, y_train_pred)
            test_mse = mean_squared_error(y_test_fold, y_test_pred)
            train_scores.append(train_mse)
            test_scores.append(test_mse)
            
            train_rmse.append(np.sqrt(train_mse))
            test_rmse.append(np.sqrt(test_mse))
            
            train_mae.append(mean_absolute_error(y_train_fold, y_train_pred))
            test_mae.append(mean_absolute_error(y_test_fold, y_test_pred))
            
            r2 = r2_score(y_test_fold, y_test_pred)
            r2_scores.append(r2)
            
            print(f"Fold {i+1}: Train MSE = {train_mse:.2f}, Test MSE = {test_mse:.2f}, "
                  f"Test RMSE = {np.sqrt(test_mse):.2f}, R² = {r2:.4f}")
            
        except Exception as e:
            print(f"Error in fold {i+1}: {e}")
            traceback.print_exc()
    
    if len(train_scores) == 0:
        print("\nAll folds encountered errors. No valid metrics to report.")
        return {}
    
    # Calculate aggregate metrics
    mean_train_mse = np.mean(train_scores)
    mean_test_mse = np.mean(test_scores)
    mean_train_rmse = np.mean(train_rmse)
    mean_test_rmse = np.mean(test_rmse)
    mean_train_mae = np.mean(train_mae)
    mean_test_mae = np.mean(test_mae)
    mean_r2 = np.mean(r2_scores)
    
    # Check for overfitting
    mse_ratio = mean_test_mse / mean_train_mse if mean_train_mse > 0 else float('inf')
    overfitting_detected = mse_ratio > 2.0
    
    print("\nCross-Validation Summary:")
    print(f"Mean Train MSE: {mean_train_mse:.2f}, Mean Test MSE: {mean_test_mse:.2f}")
    print(f"Mean Train RMSE: {mean_train_rmse:.2f}, Mean Test RMSE: {mean_test_rmse:.2f}")
    print(f"Mean Train MAE: {mean_train_mae:.2f}, Mean Test MAE: {mean_test_mae:.2f}")
    print(f"Mean R² Score: {mean_r2:.4f}")
    
    if overfitting_detected:
        print(f"\nWarning: Potential overfitting detected. Test/Train MSE ratio: {mse_ratio:.2f}")
        print("Consider adjusting regularization or model complexity.")
    
    # Visualize results if requested and we have valid folds
    if visualize and len(train_scores) > 0:
        plt.figure(figsize=(14, 7))
        
        plt.subplot(1, 2, 1)
        plt.plot(range(1, len(train_scores) + 1), train_scores, 'o-', label='Training MSE')
        plt.plot(range(1, len(test_scores) + 1), test_scores, 'o-', label='Validation MSE')
        plt.title('Learning Curve: MSE by Time Period')
        plt.xlabel('Fold (Earlier to Later)')
        plt.ylabel('Mean Squared Error')
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        plt.subplot(1, 2, 2)
        try:
            full_preds = model.predict(X_sorted)
            errors = y_sorted - full_preds
            plt.hist(errors, bins=30, alpha=0.7, color='skyblue')
            plt.axvline(x=0, color='red', linestyle='--')
            plt.title('Prediction Error Distribution')
            plt.xlabel('Error (Actual - Predicted)')
            plt.ylabel('Frequency')
            plt.grid(True, alpha=0.3)
        except Exception as e:
            print(f"Error generating full prediction error distribution: {e}")
        
        plt.tight_layout()
        plt.show()
        
        if hasattr(model, 'feature_importances_'):
            feature_names = X_sorted.columns
            importances = model.feature_importances_
            indices = np.argsort(importances)[::-1]
            
            plt.figure(figsize=(10, 6))
            plt.title('Feature Importance')
            plt.barh(range(len(indices)), importances[indices], color='skyblue')
            plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
            plt.xlabel('Relative Importance')
            plt.tight_layout()
            plt.show()
    
    # Return metrics dictionary
    return {
        'train_mse': mean_train_mse,
        'test_mse': mean_test_mse,
        'train_rmse': mean_train_rmse,
        'test_rmse': mean_test_rmse,
        'train_mae': mean_train_mae,
        'test_mae': mean_test_mae,
        'r2': mean_r2,
        'fold_metrics': {
            'train_mse': train_scores,
            'test_mse': test_scores,
            'train_rmse': train_rmse,
            'test_rmse': test_rmse,
            'train_mae': train_mae,
            'test_mae': test_mae,
            'r2': r2_scores,
            'fold_sizes': fold_sizes
        },
        'overfitting_detected': overfitting_detected,
        'mse_ratio': mse_ratio
    }

# Check if we have a model to evaluate
if 'score_model' in globals() and globals()['score_model'] is not None:
    model_to_evaluate = globals()['score_model']
    print("Using existing 'score_model' for evaluation")
elif 'model' in globals() and globals()['model'] is not None:
    model_to_evaluate = globals()['model']
    print("Using existing 'model' for evaluation")
else:
    try:
        model_to_evaluate = joblib.load(config.MODEL_PATH)
        print(f"Loaded model from {config.MODEL_PATH} for evaluation")
    except Exception as e:
        print(f"Error loading model: {e}")
        model_to_evaluate = None

if model_to_evaluate is not None:
    print("Loading historical data for model evaluation...")
    
    try:
        from sqlalchemy import create_engine
        import config
        
        engine = create_engine(config.DATABASE_URL)
        
        query = """
        SELECT * FROM nba_historical_game_stats 
        WHERE game_date >= CURRENT_DATE - INTERVAL '365 days'
        ORDER BY game_date
        """
        
        historical_df = pd.read_sql(query, engine)
        print(f"Loaded {len(historical_df)} historical games for evaluation")
        
        if len(historical_df) >= 100:
            historical_df['game_date'] = pd.to_datetime(historical_df['game_date'])
            
            try:
                from models.features import NBAFeatureGenerator
                feature_generator = NBAFeatureGenerator(debug=False)
                features_df = feature_generator.generate_all_features(historical_df)
                print(f"Generated features for {len(features_df)} games")
            except Exception as e:
                print(f"Error generating features: {e}")
                features_df = historical_df.copy()
                
                basic_features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                    'away_q1', 'away_q2', 'away_q3', 'away_q4'
                ]
                
                features_df['score_ratio'] = features_df['home_score'] / (features_df['home_score'] + features_df['away_score'])
                features_df['score_ratio'] = features_df['score_ratio'].fillna(0.5)
                
                X_cols = basic_features + ['score_ratio', 'game_date']
                features_df = features_df[X_cols]
            
            if hasattr(model_to_evaluate, 'feature_importances_'):
                n_features = len(model_to_evaluate.feature_importances_)
                print(f"Model expects {n_features} features")
                
                available_features = features_df.columns.tolist()
                expected_features = None
                
                if n_features <= 8:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                    ]
                else:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4',
                        'score_ratio', 'prev_matchup_diff',
                        'rest_days_home', 'rest_days_away', 'rest_advantage',
                        'is_back_to_back_home', 'is_back_to_back_away',
                        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                    ]
                
                missing_features = [f for f in expected_features if f not in available_features]
                if missing_features:
                    print(f"Warning: Missing features: {missing_features}")
                    for feature in missing_features:
                        features_df[feature] = 0.0
                
                features_df['game_date'] = pd.to_datetime(historical_df['game_date'])
                
                X = features_df[expected_features + ['game_date']]
                y = historical_df['home_score']
                
                eval_metrics = evaluate_model_with_time_cv(
                    model_to_evaluate, 
                    X, 
                    y, 
                    date_column='game_date', 
                    n_splits=5, 
                    visualize=True
                )
                
                globals()['eval_metrics'] = eval_metrics
                
                try:
                    metrics_dir = "metrics"
                    os.makedirs(metrics_dir, exist_ok=True)
                    
                    # Prepare a version of metrics with only JSON-serializable types
                    save_metrics = {k: v for k, v in eval_metrics.items() if k != 'fold_metrics'}
                    save_metrics['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    
                    metrics_file = os.path.join(metrics_dir, "model_evaluation.json")
                    with open(metrics_file, 'w') as f:
                        json.dump(save_metrics, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
                    
                    print(f"Saved evaluation metrics to {metrics_file}")
                except Exception as e:
                    print(f"Error saving metrics: {e}")
            else:
                print("Model doesn't have feature_importances_ attribute. Cannot determine expected features.")
        else:
            print("Not enough historical data for effective evaluation.")
    except Exception as e:
        print(f"Error during model evaluation: {e}")
        traceback.print_exc()
else:
    print("No model available for evaluation.")


In [ ]:
# Cell 11: Refactored NBA Score Prediction with Clean Architecture and Ensemble Model

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
import traceback
import logging
from typing import Dict, List, Tuple, Optional, Any, Union
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
import joblib

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('nba_prediction')

# ===== VALIDATION FUNCTIONS =====

def validate_game_data(games_df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate and clean game data to ensure it's ready for prediction.
    
    Args:
        games_df: DataFrame with raw game data
        
    Returns:
        Clean, validated DataFrame
    """
    if games_df is None or games_df.empty:
        logger.warning("No game data provided for validation")
        return pd.DataFrame()
    
    # Make a copy to avoid modifying the original
    df = games_df.copy()
    
    # Convert date fields to datetime
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
    
    # Ensure quarter fields are numeric
    for q in range(1, 5):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in df.columns:
            df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
        else:
            df[home_q] = 0
            
        if away_q in df.columns:
            df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
        else:
            df[away_q] = 0
    
    # Calculate current quarter based on non-zero quarter scores
    df['current_quarter'] = 0
    for idx, row in df.iterrows():
        max_quarter = 0
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if ((home_q in row and pd.notnull(row[home_q]) and row[home_q] > 0) or 
                (away_q in row and pd.notnull(row[away_q]) and row[away_q] > 0)):
                max_quarter = q
        
        df.at[idx, 'current_quarter'] = max_quarter
    
    # Ensure team names are strings
    if 'home_team' in df.columns:
        df['home_team'] = df['home_team'].astype(str)
    if 'away_team' in df.columns:
        df['away_team'] = df['away_team'].astype(str)
    
    # Calculate current score as sum of quarters
    if 'home_score' not in df.columns:
        df['home_score'] = 0
    if 'away_score' not in df.columns:
        df['away_score'] = 0
        
    for idx, row in df.iterrows():
        current_q = int(row.get('current_quarter', 0) or 0)
        
        home_score = 0
        away_score = 0
        for q in range(1, current_q + 1):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if home_q in row and away_q in row:
                home_score += float(row[home_q] or 0)
                away_score += float(row[away_q] or 0)
        
        df.at[idx, 'home_score'] = home_score
        df.at[idx, 'away_score'] = away_score
    
    # Calculate score ratio
    df['score_ratio'] = 0.5  # Default to even
    for idx, row in df.iterrows():
        home_score = float(row.get('home_score', 0) or 0)
        away_score = float(row.get('away_score', 0) or 0)
        total = home_score + away_score
        
        if total > 0:
            df.at[idx, 'score_ratio'] = home_score / total
    
    # Initialize current_home_score and current_away_score if not present
    if 'current_home_score' not in df.columns:
        df['current_home_score'] = df['home_score']
    if 'current_away_score' not in df.columns:
        df['current_away_score'] = df['away_score']
    
    logger.info(f"Validated {len(df)} games")
    return df

def match_games_to_schedule(games_df: pd.DataFrame, schedule_df: pd.DataFrame) -> pd.DataFrame:
    """
    Match games to the official schedule for consistent IDs and metadata.
    
    Args:
        games_df: DataFrame with game data to match
        schedule_df: DataFrame with official schedule
        
    Returns:
        DataFrame with games matched to schedule
    """
    if games_df.empty or schedule_df.empty:
        return games_df
    
    # Make a copy to avoid modifying original
    result_df = games_df.copy()
    result_df['matched_to_schedule'] = False
    
    # Process each game
    for idx, game in result_df.iterrows():
        home_team = game['home_team']
        away_team = game['away_team']
        
        # Look for exact match in schedule
        schedule_match = schedule_df[
            (schedule_df['home_team'] == home_team) & 
            (schedule_df['away_team'] == away_team)
        ]
        
        if not schedule_match.empty:
            # Found a match
            match_row = schedule_match.iloc[0]
            result_df.at[idx, 'matched_to_schedule'] = True
            
            # Copy over official game_id if different
            if 'game_id' in match_row and match_row['game_id'] != game['game_id']:
                result_df.at[idx, 'original_game_id'] = game['game_id']
                result_df.at[idx, 'game_id'] = match_row['game_id']
        else:
            # Try reverse order (sometimes teams are swapped)
            reverse_match = schedule_df[
                (schedule_df['home_team'] == away_team) &
                (schedule_df['away_team'] == home_team)
            ]
            
            if not reverse_match.empty:
                match_row = reverse_match.iloc[0]
                result_df.at[idx, 'matched_to_schedule'] = True
                result_df.at[idx, 'teams_reversed'] = True
                
                # Copy over official game_id if different
                if 'game_id' in match_row and match_row['game_id'] != game['game_id']:
                    result_df.at[idx, 'original_game_id'] = game['game_id']
                    result_df.at[idx, 'game_id'] = match_row['game_id']
    
    matches_count = result_df['matched_to_schedule'].sum()
    logger.info(f"Matched {matches_count} of {len(result_df)} games to official schedule")
    return result_df

# ===== FEATURE ENGINEERING MODULE =====

class FeatureEngineering:
    """Unified feature engineering module for consistent feature calculation."""
    
    @staticmethod
    def get_team_rolling_averages(days_lookback: int = 60) -> Dict[str, Dict[str, float]]:
        """
        Retrieve rolling statistics for each team.
        
        Args:
            days_lookback: Number of days to look back for calculating averages
            
        Returns:
            Dictionary mapping team names to their scoring statistics
        """
        try:
            from caching.supabase_client import supabase
            
            # Calculate threshold date
            threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
            
            # Query recent game data
            response = supabase.table("nba_historical_game_stats").select("*")\
                .gte("game_date", threshold_date).execute()
            
            if not response.data:
                logger.warning(f"No historical game data found for the past {days_lookback} days")
                return {}
            
            # Process data
            df = pd.DataFrame(response.data)
            df['game_date'] = pd.to_datetime(df['game_date'])
            df = df.sort_values('game_date')
            
            # Initialize dictionary for team statistics
            team_stats = {}
            
            # Get unique teams
            all_teams = set(df['home_team'].unique()).union(set(df['away_team'].unique()))
            
            for team in all_teams:
                # Get home games where team is home
                home_games = df[df['home_team'] == team]
                
                # Get away games where team is away
                away_games = df[df['away_team'] == team]
                
                # Collect scoring data
                home_scores = home_games['home_score'].values
                away_scores = away_games['away_score'].values
                
                # Collect quarter data
                q1_scores = np.concatenate([
                    home_games['home_q1'].values if 'home_q1' in home_games else [],
                    away_games['away_q1'].values if 'away_q1' in away_games else []
                ])
                
                q2_scores = np.concatenate([
                    home_games['home_q2'].values if 'home_q2' in home_games else [],
                    away_games['away_q2'].values if 'away_q2' in away_games else []
                ])
                
                q3_scores = np.concatenate([
                    home_games['home_q3'].values if 'home_q3' in home_games else [],
                    away_games['away_q3'].values if 'away_q3' in away_games else []
                ])
                
                q4_scores = np.concatenate([
                    home_games['home_q4'].values if 'home_q4' in home_games else [],
                    away_games['away_q4'].values if 'away_q4' in away_games else []
                ])
                
                # Combine all scores
                all_scores = np.concatenate([home_scores, away_scores])
                
                # Calculate advanced stats
                team_stats[team] = {
                    'avg_score': np.mean(all_scores) if len(all_scores) > 0 else 105.0,
                    'avg_q1': np.mean(q1_scores) if len(q1_scores) > 0 else 26.0,
                    'avg_q2': np.mean(q2_scores) if len(q2_scores) > 0 else 26.0,
                    'avg_q3': np.mean(q3_scores) if len(q3_scores) > 0 else 26.0,
                    'avg_q4': np.mean(q4_scores) if len(q4_scores) > 0 else 27.0,
                    'std_score': np.std(all_scores) if len(all_scores) > 0 else 12.0,
                    'recent_avg': np.mean(all_scores[-10:]) if len(all_scores) >= 10 else
                                np.mean(all_scores) if len(all_scores) > 0 else 105.0
                }
            
            logger.info(f"Calculated rolling statistics for {len(team_stats)} teams")
            return team_stats
        except Exception as e:
            logger.error(f"Error getting team rolling statistics: {e}")
            traceback.print_exc()
            return {}

    @staticmethod
    def get_previous_matchup_diff(home_team: str, away_team: str, max_lookback: int = 5) -> Dict[str, float]:
        """
        Get detailed statistics from previous matchups between two teams.
        
        Args:
            home_team: Home team name
            away_team: Away team name
            max_lookback: Maximum number of previous matchups to consider
            
        Returns:
            Dictionary with matchup differentials and statistics
        """
        try:
            from caching.supabase_client import supabase
            
            # Query for games where home team hosted away team
            home_response = supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", home_team)\
                .eq("away_team", away_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
            
            # Query for games where away team hosted home team
            away_response = supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", away_team)\
                .eq("away_team", home_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
            
            # Combine results
            home_matchups = home_response.data or []
            away_matchups = away_response.data or []
            matchups = home_matchups + away_matchups
            
            if not matchups:
                # No previous matchups found
                return {
                    'avg_diff': 0.0,
                    'first_half_diff': 0.0,
                    'second_half_diff': 0.0,
                    'q1_diff': 0.0,
                    'q2_diff': 0.0,
                    'q3_diff': 0.0,
                    'q4_diff': 0.0,
                    'games_played': 0
                }
            
            # Sort by date (most recent first)
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]  # Limit to max_lookback games
            
            # Calculate differentials from home team's perspective
            diffs = []
            q1_diffs = []
            q2_diffs = []
            q3_diffs = []
            q4_diffs = []
            first_half_diffs = []
            second_half_diffs = []
            
            for game in matchups:
                if game['home_team'] == home_team and game['away_team'] == away_team:
                    # Home team was home in this matchup
                    diff = game['home_score'] - game['away_score']
                    
                    if 'home_q1' in game and 'away_q1' in game:
                        q1_diffs.append(game['home_q1'] - game['away_q1'])
                    
                    if 'home_q2' in game and 'away_q2' in game:
                        q2_diffs.append(game['home_q2'] - game['away_q2'])
                    
                    if 'home_q3' in game and 'away_q3' in game:
                        q3_diffs.append(game['home_q3'] - game['away_q3'])
                    
                    if 'home_q4' in game and 'away_q4' in game:
                        q4_diffs.append(game['home_q4'] - game['away_q4'])
                    
                elif game['home_team'] == away_team and game['away_team'] == home_team:
                    # Home team was away in this matchup
                    diff = game['away_score'] - game['home_score']
                    
                    if 'home_q1' in game and 'away_q1' in game:
                        q1_diffs.append(game['away_q1'] - game['home_q1'])
                    
                    if 'home_q2' in game and 'away_q2' in game:
                        q2_diffs.append(game['away_q2'] - game['home_q2'])
                    
                    if 'home_q3' in game and 'away_q3' in game:
                        q3_diffs.append(game['away_q3'] - game['home_q3'])
                    
                    if 'home_q4' in game and 'away_q4' in game:
                        q4_diffs.append(game['away_q4'] - game['home_q4'])
                else:
                    continue
                
                diffs.append(diff)
                
                # Calculate half differentials
                if len(q1_diffs) > 0 and len(q2_diffs) > 0:
                    first_half_diffs.append(q1_diffs[-1] + q2_diffs[-1])
                
                if len(q3_diffs) > 0 and len(q4_diffs) > 0:
                    second_half_diffs.append(q3_diffs[-1] + q4_diffs[-1])
            
            # Calculate averages and cap extreme values
            result = {
                'avg_diff': sum(diffs) / len(diffs) if diffs else 0.0,
                'first_half_diff': sum(first_half_diffs) / len(first_half_diffs) if first_half_diffs else 0.0,
                'second_half_diff': sum(second_half_diffs) / len(second_half_diffs) if second_half_diffs else 0.0,
                'q1_diff': sum(q1_diffs) / len(q1_diffs) if q1_diffs else 0.0,
                'q2_diff': sum(q2_diffs) / len(q2_diffs) if q2_diffs else 0.0,
                'q3_diff': sum(q3_diffs) / len(q3_diffs) if q3_diffs else 0.0,
                'q4_diff': sum(q4_diffs) / len(q4_diffs) if q4_diffs else 0.0,
                'games_played': len(diffs)
            }
            
            # Cap extreme values at +/- 15 points
            for key in ['avg_diff', 'first_half_diff', 'second_half_diff', 
                       'q1_diff', 'q2_diff', 'q3_diff', 'q4_diff']:
                result[key] = max(min(result[key], 15.0), -15.0)
                
            return result
        except Exception as e:
            logger.error(f"Error getting previous matchups: {e}")
            return {
                'avg_diff': 0.0,
                'first_half_diff': 0.0,
                'second_half_diff': 0.0,
                'q1_diff': 0.0,
                'q2_diff': 0.0,
                'q3_diff': 0.0,
                'q4_diff': 0.0,
                'games_played': 0
            }

    @staticmethod
    def calculate_momentum_features(games_df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate momentum-related features for all games.
        
        Args:
            games_df: DataFrame with game data
            
        Returns:
            DataFrame with added momentum features
        """
        if games_df.empty:
            return games_df
        
        # Make a copy to avoid modifying original
        features_df = games_df.copy()
        
        # Ensure all quarter fields are present and numeric
        for q in range(1, 5):
            for team in ['home', 'away']:
                col = f'{team}_q{q}'
                if col not in features_df:
                    features_df[col] = 0.0
                else:
                    features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0.0)
        
        # Initialize momentum columns with explicit float type
        momentum_cols = [
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 
            'cumulative_momentum', 'home_momentum', 'away_momentum',
            'momentum_differential', 'scoring_trend'
        ]
        for col in momentum_cols:
            features_df[col] = np.zeros(len(features_df), dtype=np.float64)
        
        # Calculate quarter-to-quarter momentum with vectorized operations
        # Q1 to Q2 momentum
        q2_mask = features_df['current_quarter'] >= 2
        if q2_mask.any():
            features_df.loc[q2_mask, 'q1_to_q2_momentum'] = (
                (features_df.loc[q2_mask, 'home_q2'] - features_df.loc[q2_mask, 'home_q1']) - 
                (features_df.loc[q2_mask, 'away_q2'] - features_df.loc[q2_mask, 'away_q1'])
            )
        
        # Q2 to Q3 momentum
        q3_mask = features_df['current_quarter'] >= 3
        if q3_mask.any():
            features_df.loc[q3_mask, 'q2_to_q3_momentum'] = (
                (features_df.loc[q3_mask, 'home_q3'] - features_df.loc[q3_mask, 'home_q2']) - 
                (features_df.loc[q3_mask, 'away_q3'] - features_df.loc[q3_mask, 'away_q2'])
            )
        
        # Q3 to Q4 momentum
        q4_mask = features_df['current_quarter'] >= 4
        if q4_mask.any():
            features_df.loc[q4_mask, 'q3_to_q4_momentum'] = (
                (features_df.loc[q4_mask, 'home_q4'] - features_df.loc[q4_mask, 'home_q3']) - 
                (features_df.loc[q4_mask, 'away_q4'] - features_df.loc[q4_mask, 'away_q3'])
            )
        
        # Calculate home and away team momentum separately
        for q_idx in range(1, 4):
            curr_q = q_idx + 1
            prev_q = q_idx
            
            mask = features_df['current_quarter'] >= curr_q
            if mask.any():
                # Home team momentum
                curr_q_col = f'home_q{curr_q}'
                prev_q_col = f'home_q{prev_q}'
                if curr_q_col in features_df.columns and prev_q_col in features_df.columns:
                    momentum_col = f'home_q{prev_q}_to_q{curr_q}_momentum'
                    features_df[momentum_col] = 0.0
                    features_df.loc[mask, momentum_col] = (
                        features_df.loc[mask, curr_q_col] - features_df.loc[mask, prev_q_col]
                    )
                
                # Away team momentum
                curr_q_col = f'away_q{curr_q}'
                prev_q_col = f'away_q{prev_q}'
                if curr_q_col in features_df.columns and prev_q_col in features_df.columns:
                    momentum_col = f'away_q{prev_q}_to_q{curr_q}_momentum'
                    features_df[momentum_col] = 0.0
                    features_df.loc[mask, momentum_col] = (
                        features_df.loc[mask, curr_q_col] - features_df.loc[mask, prev_q_col]
                    )
        
        # Calculate cumulative momentum using vectorized operations with weights
        weights = np.array([0.2, 0.3, 0.5])  # Weights for each quarter transition
        
        # Calculate for Q2
        q2_mask = features_df['current_quarter'] == 2
        if q2_mask.any():
            features_df.loc[q2_mask, 'cumulative_momentum'] = features_df.loc[q2_mask, 'q1_to_q2_momentum'] / 15.0
            
            # Calculate home and away momentum
            features_df.loc[q2_mask, 'home_momentum'] = (
                features_df.loc[q2_mask, 'home_q2'] - features_df.loc[q2_mask, 'home_q1']
            ) / features_df.loc[q2_mask, 'home_q1'].replace(0, 1)
            
            features_df.loc[q2_mask, 'away_momentum'] = (
                features_df.loc[q2_mask, 'away_q2'] - features_df.loc[q2_mask, 'away_q1']
            ) / features_df.loc[q2_mask, 'away_q1'].replace(0, 1)
        
        # Calculate for Q3
        q3_mask = features_df['current_quarter'] == 3
        if q3_mask.any():
            weighted_sum = (
                features_df.loc[q3_mask, 'q1_to_q2_momentum'] * weights[0] +
                features_df.loc[q3_mask, 'q2_to_q3_momentum'] * weights[1]
            )
            weight_sum = weights[0] + weights[1]
            features_df.loc[q3_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
            
            # Calculate home and away momentum (Q1 to Q3 change)
            features_df.loc[q3_mask, 'home_momentum'] = (
                (features_df.loc[q3_mask, 'home_q2'] + features_df.loc[q3_mask, 'home_q3']) - 
                (features_df.loc[q3_mask, 'home_q1'] * 2)
            ) / (features_df.loc[q3_mask, 'home_q1'] * 2).replace(0, 2)
            
            features_df.loc[q3_mask, 'away_momentum'] = (
                (features_df.loc[q3_mask, 'away_q2'] + features_df.loc[q3_mask, 'away_q3']) - 
                (features_df.loc[q3_mask, 'away_q1'] * 2)
            ) / (features_df.loc[q3_mask, 'away_q1'] * 2).replace(0, 2)
        
        # Calculate for Q4
        q4_mask = features_df['current_quarter'] >= 4
        if q4_mask.any():
            weighted_sum = (
                features_df.loc[q4_mask, 'q1_to_q2_momentum'] * weights[0] +
                features_df.loc[q4_mask, 'q2_to_q3_momentum'] * weights[1] +
                features_df.loc[q4_mask, 'q3_to_q4_momentum'] * weights[2]
            )
            weight_sum = weights.sum()
            features_df.loc[q4_mask, 'cumulative_momentum'] = weighted_sum / (weight_sum * 15.0)
            
            # Calculate home and away momentum (across all quarters)
            features_df.loc[q4_mask, 'home_momentum'] = (
                features_df.loc[q4_mask, 'home_q4'] - features_df.loc[q4_mask, 'home_q1']
            ) / features_df.loc[q4_mask, 'home_q1'].replace(0, 1)
            
            features_df.loc[q4_mask, 'away_momentum'] = (
                features_df.loc[q4_mask, 'away_q4'] - features_df.loc[q4_mask, 'away_q1']
            ) / features_df.loc[q4_mask, 'away_q1'].replace(0, 1)
        
        # Calculate momentum differential
        features_df['momentum_differential'] = features_df['home_momentum'] - features_df['away_momentum']
        
        # Calculate scoring trend (increasing/decreasing scoring through quarters)
        for idx, row in features_df.iterrows():
            current_q = int(row['current_quarter'])
            if current_q <= 1:
                features_df.at[idx, 'scoring_trend'] = 0.0
                continue
                
            quarters = []
            for q in range(1, current_q + 1):
                quarters.append((float(row[f'home_q{q}']) + float(row[f'away_q{q}'])))
            
            # Calculate trend using linear regression slope
            x = np.arange(len(quarters))
            if len(x) >= 2:  # Need at least 2 points for a slope
                slope = np.polyfit(x, quarters, 1)[0]
                features_df.at[idx, 'scoring_trend'] = slope / 10.0  # Normalize
            else:
                features_df.at[idx, 'scoring_trend'] = 0.0
        
        # Clip to [-1, 1] range
        for col in momentum_cols:
            features_df[col] = features_df[col].clip(-1.0, 1.0)
        
        logger.info(f"Added enhanced momentum features to {len(features_df)} games")
        return features_df

@staticmethod
def calculate_rest_features(games_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate rest-related features for all games with improved handling of back-to-back games.
    
    Args:
        games_df: DataFrame with game data
        
    Returns:
        DataFrame with added rest features
    """
    if games_df.empty:
        return games_df
    
    # Make a copy to avoid modifying original
    features_df = games_df.copy()
    
    # Initialize rest features with defaults
    features_df['rest_days_home'] = 2.0  # Default to 2 days rest
    features_df['rest_days_away'] = 2.0
    features_df['is_back_to_back_home'] = 0
    features_df['is_back_to_back_away'] = 0
    features_df['rest_advantage'] = 0.0
    features_df['home_games_in_5_days'] = 0
    features_df['away_games_in_5_days'] = 0

    try:
        # Process each game
        for idx, row in features_df.iterrows():
            home_team = row['home_team']
            away_team = row['away_team']
            game_date = row.get('game_date')

            if pd.isna(game_date):
                continue

            # Convert game_date to string format if needed
            if isinstance(game_date, pd.Timestamp):
                game_date_str = game_date.strftime('%Y-%m-%d')
            else:
                game_date_str = str(game_date)

            five_days_ago = (pd.to_datetime(game_date) - timedelta(days=5)).strftime('%Y-%m-%d')

            # Get previous games for home team
            home_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + home_team + ",away_team.eq." + home_team)
                .lt("game_date", game_date_str)
                .order("game_date", desc=True)
                .limit(1)
                .execute()
            )

            # Get all games in last 5 days for home team
            home_recent_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + home_team + ",away_team.eq." + home_team)
                .lt("game_date", game_date_str)
                .gte("game_date", five_days_ago)
                .execute()
            )

            # Get previous games for away team
            away_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + away_team + ",away_team.eq." + away_team)
                .lt("game_date", game_date_str)
                .order("game_date", desc=True)
                .limit(1)
                .execute()
            )

            # Get all games in last 5 days for away team
            away_recent_response = (
                supabase
                .table("nba_historical_game_stats")
                .select("game_date")
                .or_("home_team.eq." + away_team + ",away_team.eq." + away_team)
                .lt("game_date", game_date_str)
                .gte("game_date", five_days_ago)
                .execute()
            )

            # Calculate rest days for home team
            if home_response.data:
                prev_game = home_response.data[0]
                prev_date = pd.to_datetime(prev_game['game_date'])
                rest_days = (pd.to_datetime(game_date) - prev_date).days
                features_df.at[idx, 'rest_days_home'] = min(max(rest_days, 0), 10)  # Cap 0–10
                features_df.at[idx, 'is_back_to_back_home'] = 1 if rest_days <= 1 else 0

            # Calculate games in last 5 days for home team
            if home_recent_response.data:
                features_df.at[idx, 'home_games_in_5_days'] = len(home_recent_response.data)

            # Calculate rest days for away team
            if away_response.data:
                prev_game = away_response.data[0]
                prev_date = pd.to_datetime(prev_game['game_date'])
                rest_days = (pd.to_datetime(game_date) - prev_date).days
                features_df.at[idx, 'rest_days_away'] = min(max(rest_days, 0), 10)  # Cap 0–10
                features_df.at[idx, 'is_back_to_back_away'] = 1 if rest_days <= 1 else 0

            # Calculate games in last 5 days for away team
            if away_recent_response.data:
                features_df.at[idx, 'away_games_in_5_days'] = len(away_recent_response.data)

            # Calculate rest advantage
            home_rest = features_df.at[idx, 'rest_days_home']
            away_rest = features_df.at[idx, 'rest_days_away']
            features_df.at[idx, 'rest_advantage'] = home_rest - away_rest

            # Calculate overall fatigue factor
            home_b2b = features_df.at[idx, 'is_back_to_back_home']
            away_b2b = features_df.at[idx, 'is_back_to_back_away']
            home_games_5d = features_df.at[idx, 'home_games_in_5_days']
            away_games_5d = features_df.at[idx, 'away_games_in_5_days']

            features_df.at[idx, 'home_fatigue'] = home_b2b * 1.0 + (home_games_5d * 0.25)
            features_df.at[idx, 'away_fatigue'] = away_b2b * 1.0 + (away_games_5d * 0.25)
            features_df.at[idx, 'fatigue_differential'] = (
                features_df.at[idx, 'away_fatigue'] - features_df.at[idx, 'home_fatigue']
            )

    except Exception as e:
        logger.error(f"Error calculating rest features: {e}")
        traceback.print_exc()

    return features_df


@staticmethod
def prepare_features(
        games_df: pd.DataFrame, 
        team_stats: Optional[Dict[str, Dict[str, float]]] = None,
        include_momentum: bool = True,
        include_rest: bool = True,
        quarter_specific: bool = True
    ) -> pd.DataFrame:
        """
        Prepare all features needed for prediction with comprehensive feature set.
        
        Args:
            games_df: DataFrame with validated game data
            team_stats: Dictionary with team scoring statistics (optional)
            include_momentum: Whether to include momentum features
            include_rest: Whether to include rest-related features
            quarter_specific: Whether to include quarter-specific features
            
        Returns:
            DataFrame with all features needed for prediction
        """
        if games_df.empty:
            return pd.DataFrame()
        
        # Make a copy to avoid modifying original
        features_df = games_df.copy()
        
        # Get team rolling averages if not provided
        if team_stats is None:
            team_stats = FeatureEngineering.get_team_rolling_averages()
        
        # Add rolling scores based on team averages
        for idx, row in features_df.iterrows():
            home_team = row['home_team']
            away_team = row['away_team']
            
            # Add team average scores
            if home_team in team_stats:
                features_df.at[idx, 'rolling_home_score'] = team_stats[home_team]['avg_score']
                if quarter_specific:
                    features_df.at[idx, 'avg_home_q1'] = team_stats[home_team]['avg_q1']
                    features_df.at[idx, 'avg_home_q2'] = team_stats[home_team]['avg_q2']
                    features_df.at[idx, 'avg_home_q3'] = team_stats[home_team]['avg_q3']
                    features_df.at[idx, 'avg_home_q4'] = team_stats[home_team]['avg_q4']
            else:
                features_df.at[idx, 'rolling_home_score'] = 105.0
                if quarter_specific:
                    features_df.at[idx, 'avg_home_q1'] = 26.0
                    features_df.at[idx, 'avg_home_q2'] = 26.0
                    features_df.at[idx, 'avg_home_q3'] = 26.0
                    features_df.at[idx, 'avg_home_q4'] = 27.0
            
            if away_team in team_stats:
                features_df.at[idx, 'rolling_away_score'] = team_stats[away_team]['avg_score']
                if quarter_specific:
                    features_df.at[idx, 'avg_away_q1'] = team_stats[away_team]['avg_q1']
                    features_df.at[idx, 'avg_away_q2'] = team_stats[away_team]['avg_q2']
                    features_df.at[idx, 'avg_away_q3'] = team_stats[away_team]['avg_q3']
                    features_df.at[idx, 'avg_away_q4'] = team_stats[away_team]['avg_q4']
            else:
                features_df.at[idx, 'rolling_away_score'] = 105.0
                if quarter_specific:
                    features_df.at[idx, 'avg_away_q1'] = 26.0
                    features_df.at[idx, 'avg_away_q2'] = 26.0
                    features_df.at[idx, 'avg_away_q3'] = 26.0
                    features_df.at[idx, 'avg_away_q4'] = 27.0
            
            # Add previous matchup differential
            matchup_stats = FeatureEngineering.get_previous_matchup_diff(home_team, away_team)
            features_df.at[idx, 'prev_matchup_diff'] = matchup_stats['avg_diff']
            
            # Add detailed matchup statistics
            if quarter_specific:
                features_df.at[idx, 'prev_first_half_diff'] = matchup_stats['first_half_diff']
                features_df.at[idx, 'prev_second_half_diff'] = matchup_stats['second_half_diff']
                features_df.at[idx, 'prev_q1_diff'] = matchup_stats['q1_diff']
                features_df.at[idx, 'prev_q2_diff'] = matchup_stats['q2_diff']
                features_df.at[idx, 'prev_q3_diff'] = matchup_stats['q3_diff']
                features_df.at[idx, 'prev_q4_diff'] = matchup_stats['q4_diff']
                features_df.at[idx, 'prev_matchups_count'] = matchup_stats['games_played']
        
        # Add momentum features if requested
        if include_momentum:
            features_df = FeatureEngineering.calculate_momentum_features(features_df)
        
        # Add rest features if requested
        if include_rest:
            features_df = FeatureEngineering.calculate_rest_features(features_df)
        
        # Add quarter-specific features
        if quarter_specific:
            current_quarter = features_df['current_quarter'].fillna(0).astype(int)
            
            # Pre-game (Q0) and Q1 features
            features_df['is_q0_or_q1'] = (current_quarter <= 1).astype(int)
            
            # Q2 features
            features_df['is_q2'] = (current_quarter == 2).astype(int)
            
            # Q3 features
            features_df['is_q3'] = (current_quarter == 3).astype(int)
            
            # Q4 features
            features_df['is_q4'] = (current_quarter == 4).astype(int)
            
            # Create time remaining feature (approximation)
            features_df['time_remaining_pct'] = (4 - current_quarter) / 4
            features_df.loc[current_quarter == 0, 'time_remaining_pct'] = 1.0
            
            # Enhanced scoring trends by quarter
            for idx, row in features_df.iterrows():
                q = int(row['current_quarter'])
                if q >= 1:
                    features_df.at[idx, 'q1_score_total'] = float(row['home_q1']) + float(row['away_q1'])
                if q >= 2:
                    features_df.at[idx, 'q2_score_total'] = float(row['home_q2']) + float(row['away_q2'])
                    features_df.at[idx, 'q2_vs_q1_trend'] = features_df.at[idx, 'q2_score_total'] - features_df.at[idx, 'q1_score_total']
                if q >= 3:
                    features_df.at[idx, 'q3_score_total'] = float(row['home_q3']) + float(row['away_q3'])
                    features_df.at[idx, 'q3_vs_q2_trend'] = features_df.at[idx, 'q3_score_total'] - features_df.at[idx, 'q2_score_total']
                    features_df.at[idx, 'second_half_predicted'] = 1
                else:
                    features_df.at[idx, 'second_half_predicted'] = 0
        
        # Add score differential features
        for idx, row in features_df.iterrows():
            current_q = int(row['current_quarter'])
            if current_q > 0:
                home_score = float(row.get('home_score', 0) or 0)
                away_score = float(row.get('away_score', 0) or 0)
                features_df.at[idx, 'score_differential'] = home_score - away_score
                features_df.at[idx, 'abs_score_differential'] = abs(home_score - away_score)
                features_df.at[idx, 'close_game'] = 1 if abs(home_score - away_score) <= 10 else 0
                features_df.at[idx, 'blowout_game'] = 1 if abs(home_score - away_score) >= 20 else 0
        
        logger.info(f"Prepared comprehensive features for {len(features_df)} games")
        return features_df

# ===== QUARTER-SPECIFIC MODELS =====

class QuarterModels:
    """Container for quarter-specific prediction models."""
    
    def __init__(self):
        self.pre_game_model = None
        self.q1_model = None
        self.q2_model = None
        self.q3_model = None
        self.q4_model = None
        self.ensemble_weights = {
            0: {'main': 0.80, 'quarter': 0.20},
            1: {'main': 0.70, 'quarter': 0.30},
            2: {'main': 0.60, 'quarter': 0.40},
            3: {'main': 0.40, 'quarter': 0.60},
            4: {'main': 0.30, 'quarter': 0.70}
        }
        self.feature_lists = {}
        self.is_initialized = False
    
    def initialize_models(self):
        """Initialize all quarter-specific models."""
        try:
            # Pre-game model
            self.pre_game_model = GradientBoostingRegressor(
                n_estimators=120,
                learning_rate=0.1,
                max_depth=4,
                random_state=42,
                subsample=0.85
            )
            self.feature_lists[0] = [
                'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away'
            ]
            
            # Q1 model
            self.q1_model = GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=3,
                random_state=42,
                subsample=0.85
            )
            self.feature_lists[1] = [
                'home_q1', 'away_q1', 'score_ratio',
                'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage'
            ]
            
            # Q2 model
            self.q2_model = GradientBoostingRegressor(
                n_estimators=120,
                learning_rate=0.1,
                max_depth=4,
                random_state=42,
                subsample=0.85
            )
            self.feature_lists[2] = [
                'home_q1', 'home_q2', 'away_q1', 'away_q2',
                'score_ratio', 'q1_to_q2_momentum', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'prev_first_half_diff'
            ]
            
            # Q3 model
            self.q3_model = GradientBoostingRegressor(
                n_estimators=150,
                learning_rate=0.1,
                max_depth=4,
                random_state=42,
                subsample=0.8
            )
            self.feature_lists[3] = [
                'home_q1', 'home_q2', 'home_q3', 
                'away_q1', 'away_q2', 'away_q3',
                'score_ratio', 'q1_to_q2_momentum', 'q2_to_q3_momentum',
                'cumulative_momentum', 'prev_matchup_diff',
                'score_differential', 'prev_second_half_diff'
            ]
            
            # Q4 model - Use Ridge regression for better stability
            self.q4_model = Ridge(
                alpha=1.0,
                fit_intercept=True,
                normalize=False,
                random_state=42
            )
            self.feature_lists[4] = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'away_q1', 'away_q2', 'away_q3', 'away_q4',
                'score_ratio', 'score_differential',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum',
                'cumulative_momentum', 'prev_q4_diff'
            ]
            
            self.is_initialized = True
            logger.info("Quarter-specific models initialized successfully")
            return True
        except Exception as e:
            logger.error(f"Error initializing quarter models: {e}")
            traceback.print_exc()
            return False
    
    def load_models(self, model_dir: str = None):
        """
        Load pre-trained quarter-specific models from disk.
        
        Args:
            model_dir: Directory containing saved models (optional)
        
        Returns:
            True if successful, False otherwise
        """
        try:
            import config
            if model_dir is None:
                model_dir = config.MODEL_DIR
                
            # Load pre-game model
            self.pre_game_model = joblib.load(f"{model_dir}/pre_game_model.joblib")
            
            # Load quarter models
            self.q1_model = joblib.load(f"{model_dir}/q1_model.joblib")
            self.q2_model = joblib.load(f"{model_dir}/q2_model.joblib")
            self.q3_model = joblib.load(f"{model_dir}/q3_model.joblib")
            self.q4_model = joblib.load(f"{model_dir}/q4_model.joblib")
            
            # Load feature lists 
            feature_lists_path = f"{model_dir}/feature_lists.joblib"
            if os.path.exists(feature_lists_path):
                self.feature_lists = joblib.load(feature_lists_path)
            
            self.is_initialized = True
            logger.info("Quarter-specific models loaded successfully")
            return True
        except Exception as e:
            logger.error(f"Error loading quarter models: {e}")
            logger.info("Initializing new models instead")
            return self.initialize_models()
    
    def get_model_for_quarter(self, quarter: int):
        """
        Get the appropriate model for a specific quarter.
        
        Args:
            quarter: Current quarter (0-4)
            
        Returns:
            Tuple of (model, feature_list)
        """
        if not self.is_initialized:
            self.initialize_models()
            
        if quarter == 0:
            return self.pre_game_model, self.feature_lists.get(0, [])
        elif quarter == 1:
            return self.q1_model, self.feature_lists.get(1, [])
        elif quarter == 2:
            return self.q2_model, self.feature_lists.get(2, [])
        elif quarter == 3:
            return self.q3_model, self.feature_lists.get(3, [])
        elif quarter >= 4:
            return self.q4_model, self.feature_lists.get(4, [])
        else:
            logger.warning(f"Invalid quarter: {quarter}, using pre-game model")
            return self.pre_game_model, self.feature_lists.get(0, [])
    
    def get_ensemble_weights(self, quarter: int) -> Dict[str, float]:
        """
        Get the appropriate ensemble weights for a specific quarter.
        
        Args:
            quarter: Current quarter (0-4)
            
        Returns:
            Dictionary of weight values
        """
        return self.ensemble_weights.get(quarter, self.ensemble_weights[0])

# ===== MODEL MANAGEMENT =====

class ModelManager:
    """Class to manage prediction models and ensembles."""
    
    def __init__(self):
        self.main_model = None
        self.quarter_models = QuarterModels()
        self.team_stats = None
        self.feature_list = None
        self.initialized = False
    
    def initialize(self):
        """Initialize the model manager and load all models."""
        if self.initialized:
            return True
            
        try:
            # Initialize quarter-specific models
            self.quarter_models.initialize_models()
            
            # Get main model from global or load from file
            self.main_model, self.feature_list = self.get_main_prediction_model()
            
            # Get team statistics
            self.team_stats = FeatureEngineering.get_team_rolling_averages()
            
            self.initialized = True
            logger.info("Model manager initialized successfully")
            return True
        except Exception as e:
            logger.error(f"Error initializing model manager: {e}")
            traceback.print_exc()
            return False
    
    def get_main_prediction_model(self) -> Tuple[Any, List[str]]:
        """
        Get the main prediction model and its expected features.
        
        Returns:
            Tuple of (model, feature_list)
        """
        model = None
        
        # Try to get model from global variables first
        if 'model' in globals() and globals()['model'] is not None:
            model = globals()['model']
            logger.info("Using model from global 'model' variable")
        elif 'score_model' in globals() and globals()['score_model'] is not None:
            model = globals()['score_model']
            logger.info("Using model from global 'score_model' variable")
        
        # If no model in globals, try to load from file
        if model is None:
            try:
                import config
                model = joblib.load(config.MODEL_PATH)
                logger.info(f"Loaded model from {config.MODEL_PATH}")
            except Exception as e:
                logger.error(f"Error loading model: {e}")
                logger.warning("Proceeding without a main prediction model")
                model = GradientBoostingRegressor(
                    n_estimators=150,
                    learning_rate=0.1,
                    max_depth=4,
                    random_state=42,
                    subsample=0.8
                )
        
        # Determine feature list based on model type
        feature_list = []
        
        if model is not None and hasattr(model, 'feature_importances_'):
            # Enhanced model with 15 features
            if len(model.feature_importances_) > 12:
                feature_list = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
                logger.info("Using enhanced feature set with momentum and rest features")
            # Medium model with 9 features
            elif len(model.feature_importances_) > 8:
                feature_list = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage'
                ]
                logger.info("Using medium feature set with rest features")
            # Original model with 8 features
            else:
                feature_list = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
                logger.info("Using original feature set")
        else:
            # Default to enhanced feature set
            feature_list = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
            logger.warning("Could not determine feature list from model, using enhanced features")
        
        return model, feature_list
    
    def predict_with_ensemble(self, features_df: pd.DataFrame) -> pd.DataFrame:
        """
        Make predictions using the ensemble of models.
        
        Args:
            features_df: DataFrame with features
            
        Returns:
            DataFrame with predictions
        """
        if not self.initialized:
            self.initialize()
            
        result_df = features_df.copy()
        
        # Add prediction columns
        result_df['predicted_home_score'] = 0.0
        result_df['predicted_away_score'] = 0.0
        result_df['main_model_home_pred'] = 0.0
        result_df['quarter_model_home_pred'] = 0.0
        result_df['prediction_confidence'] = 0.0
        result_df['ensemble_weight_main'] = 0.0
        result_df['ensemble_weight_quarter'] = 0.0
        
        # Make predictions for each game
        for idx, row in features_df.iterrows():
            current_q = int(row.get('current_quarter', 0) or 0)
            
            # Get current scores
            current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
            current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
            
            # Get ensemble weights for this quarter
            weights = self.quarter_models.get_ensemble_weights(current_q)
            result_df.at[idx, 'ensemble_weight_main'] = weights['main']
            result_df.at[idx, 'ensemble_weight_quarter'] = weights['quarter']
            
            # Make prediction with main model if available
            main_pred_home = None
            if self.main_model is not None:
                try:
                    # Extract features for main model
                    main_features = [f for f in self.feature_list if f in features_df.columns]
                    if all(f in features_df.columns for f in main_features):
                        X_main = features_df.loc[[idx], main_features]
                        main_pred_home = float(self.main_model.predict(X_main)[0])
                        result_df.at[idx, 'main_model_home_pred'] = main_pred_home
                    else:
                        missing = [f for f in self.feature_list if f not in features_df.columns]
                        logger.warning(f"Missing features for main model: {missing}")
                except Exception as e:
                    logger.error(f"Error predicting with main model: {e}")
            
            # Make prediction with quarter-specific model
            quarter_pred_home = None
            try:
                # Get appropriate quarter model
                quarter_model, quarter_features = self.quarter_models.get_model_for_quarter(current_q)
                
                if quarter_model is not None:
                    # Extract features for quarter model
                    available_features = [f for f in quarter_features if f in features_df.columns]
                    
                    # Only predict if we have enough features
                    if len(available_features) >= len(quarter_features) * 0.75:
                        X_quarter = features_df.loc[[idx], available_features]
                        quarter_pred_home = float(quarter_model.predict(X_quarter)[0])
                        result_df.at[idx, 'quarter_model_home_pred'] = quarter_pred_home
                    else:
                        missing = [f for f in quarter_features if f not in features_df.columns]
                        logger.warning(f"Missing too many features for Q{current_q} model: {missing}")
            except Exception as e:
                logger.error(f"Error predicting with quarter-specific model: {e}")
            
            # Combine predictions using ensemble weights
            if main_pred_home is not None and quarter_pred_home is not None:
                # Check if predictions are reasonably close
                pred_diff = abs(main_pred_home - quarter_pred_home)
                if pred_diff > 15:  # Large disagreement between models
                    # Adjust weights to favor the model that's closer to current score
                    main_diff = abs(main_pred_home - current_home)
                    quarter_diff = abs(quarter_pred_home - current_home)
                    
                    if main_diff < quarter_diff:
                        # Main model seems more reasonable
                        weights = {'main': 0.8, 'quarter': 0.2}
                    else:
                        # Quarter model seems more reasonable
                        weights = {'main': 0.2, 'quarter': 0.8}
                    
                    logger.info(f"Large model disagreement ({pred_diff:.1f} pts), adjusting weights")
                
                # Calculate ensemble prediction
                ensemble_home = (main_pred_home * weights['main']) + (quarter_pred_home * weights['quarter'])
                result_df.at[idx, 'predicted_home_score'] = max(ensemble_home, current_home)
            elif main_pred_home is not None:
                # Only main model available
                result_df.at[idx, 'predicted_home_score'] = max(main_pred_home, current_home)
            elif quarter_pred_home is not None:
                # Only quarter model available
                result_df.at[idx, 'predicted_home_score'] = max(quarter_pred_home, current_home)
            else:
                # No models available, use basic prediction
                self._predict_baseline(result_df, idx, row)
                continue
            
            # Calculate away score prediction
            self._calculate_away_score(result_df, idx, row)
            
            # Calculate confidence based on quarter
            result_df.at[idx, 'prediction_confidence'] = calculate_prediction_confidence(current_q)
            
            # Calculate win probability
            home_pred = result_df.at[idx, 'predicted_home_score']
            away_pred = result_df.at[idx, 'predicted_away_score']
            result_df.at[idx, 'win_probability'] = calculate_win_probability(home_pred, away_pred, current_q)
            
            # Calculate remaining points
            result_df.at[idx, 'remaining_home_points'] = max(0, home_pred - current_home)
            result_df.at[idx, 'remaining_away_points'] = max(0, away_pred - current_away)
            
            # Calculate margin and total
            result_df.at[idx, 'projected_margin'] = home_pred - away_pred
            result_df.at[idx, 'total_projected_score'] = home_pred + away_pred
        
        return result_df
    
    def _predict_baseline(self, result_df: pd.DataFrame, idx: int, row: Dict):
        """Make baseline prediction when models are unavailable."""
        home_team = row['home_team']
        away_team = row['away_team']
        current_q = int(row.get('current_quarter', 0) or 0)
        
        # Get current scores
        current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
        current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
        
        # Get team averages
        if self.team_stats and home_team in self.team_stats:
            home_avg = self.team_stats[home_team]['avg_score']
        else:
            home_avg = 105.0
            
        if self.team_stats and away_team in self.team_stats:
            away_avg = self.team_stats[away_team]['avg_score']
        else:
            away_avg = 105.0
        
        # Calculate remaining portion of game
        if current_q == 0:
            # Pre-game prediction
            result_df.at[idx, 'predicted_home_score'] = home_avg
            result_df.at[idx, 'predicted_away_score'] = away_avg
        else:
            # In-game prediction
            remaining_pct = 1.0 - (0.25 * current_q)
            
            result_df.at[idx, 'predicted_home_score'] = current_home + (home_avg * remaining_pct)
            result_df.at[idx, 'predicted_away_score'] = current_away + (away_avg * remaining_pct)
        
        # Add confidence metrics
        result_df.at[idx, 'prediction_confidence'] = calculate_prediction_confidence(current_q)
        
        # Calculate win probability
        home_pred = result_df.at[idx, 'predicted_home_score']
        away_pred = result_df.at[idx, 'predicted_away_score']
        result_df.at[idx, 'win_probability'] = calculate_win_probability(home_pred, away_pred, current_q)
    
    def _calculate_away_score(self, result_df: pd.DataFrame, idx: int, row: Dict):
        """Calculate away score based on predicted home score and context."""
        home_team = row['home_team']
        away_team = row['away_team']
        current_q = int(row.get('current_quarter', 0) or 0)
        
        # Get current scores
        current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
        current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
        
        # Get predicted home score
        predicted_home = result_df.at[idx, 'predicted_home_score']
        
        # Calculate away score prediction
        # Using matchup differential, home advantage, and momentum
        matchup_diff = float(row.get('prev_matchup_diff', 0) or 0)
        home_adv = 3.5  # Standard home court advantage
        
        # Get momentum adjustment if available
        momentum_adj = 0.0
        if 'cumulative_momentum' in row:
            momentum = float(row.get('cumulative_momentum', 0) or 0)
            momentum_adj = momentum * 3.0  # Scale to points impact
        
        # For later quarters, preserve the current point differential more
        current_diff = current_home - current_away
        
        if current_q >= 3:
            # For later quarters, the current differential is more important
            quarter_weight = 0.7
            predicted_diff = (current_diff * quarter_weight) + (matchup_diff + home_adv + momentum_adj) * (1 - quarter_weight)
        else:
            # For earlier quarters, use more of the historical factors
            quarter_weight = 0.3 * current_q
            predicted_diff = (current_diff * quarter_weight) + (matchup_diff + home_adv + momentum_adj) * (1 - quarter_weight)
        
        # Calculate predicted away score
        predicted_away = predicted_home - predicted_diff
        
        # CRITICAL: Ensure predicted scores are never less than current scores
        predicted_away = max(predicted_away, current_away)
        
        # Store prediction
        result_df.at[idx, 'predicted_away_score'] = predicted_away

def calculate_prediction_confidence(quarter: int) -> float:
    """
    Calculate confidence percentage based on game quarter.
    
    Args:
        quarter: Current quarter (0-4)
        
    Returns:
        Confidence value between 0-1
    """
    confidence_map = {
        0: 0.3,  # Pre-game
        1: 0.45, # First quarter
        2: 0.65, # Second quarter
        3: 0.8,  # Third quarter
        4: 0.95  # Fourth quarter
    }
    return confidence_map.get(quarter, 0.3)

def calculate_win_probability(home_score: float, away_score: float, quarter: int, time_remaining: float = None) -> float:
    """
    Calculate win probability for the home team.
    
    Args:
        home_score: Current or predicted home score
        away_score: Current or predicted away score
        quarter: Current quarter (0-4)
        time_remaining: Minutes remaining in the game (optional)
        
    Returns:
        Win probability between 0-1
    """
    import math
    
    score_diff = home_score - away_score
    
    # Calculate game progress
    if time_remaining is not None:
        total_time = 48.0
        elapsed = total_time - time_remaining
        progress = elapsed / total_time
    else:
        progress = min(quarter / 4.0, 1.0)
    
    # Steepness parameter increases with game progress
    k = 0.05 + (progress * 0.15)
    
    # Logistic function for win probability
    return 1.0 / (1.0 + math.exp(-k * score_diff))

# ===== MAIN PREDICTION FUNCTION =====

def predict_final_scores(
    games_df: pd.DataFrame, 
    team_stats: Dict[str, Dict[str, float]] = None,
    prediction_history: Dict[str, List[Dict]] = None,
    official_schedule: pd.DataFrame = None
) -> pd.DataFrame:
    """
    Predict final scores for NBA games with clean architecture and ensemble model.
    
    Args:
        games_df: DataFrame with game data
        team_stats: Dictionary of team scoring averages (optional)
        prediction_history: Dictionary to store prediction history (optional)
        official_schedule: DataFrame with official game schedule (optional)
        
    Returns:
        DataFrame with predictions and confidence metrics
    """
    start_time = time.time()
    logger.info(f"Starting prediction for {len(games_df)} games")
    
    # Validate and clean input data
    validated_df = validate_game_data(games_df)
    if validated_df.empty:
        logger.warning("No valid game data available for prediction")
        return pd.DataFrame()
    
    # Match against official schedule if provided
    if official_schedule is not None and not official_schedule.empty:
        validated_df = match_games_to_schedule(validated_df, official_schedule)
    
    # Initialize model manager if needed
    if 'model_manager' not in globals() or globals()['model_manager'] is None:
        globals()['model_manager'] = ModelManager()
    
    model_manager = globals()['model_manager']
    if not model_manager.initialized:
        model_manager.initialize()
    
    # Get team statistics if not provided
    if team_stats is None:
        team_stats = model_manager.team_stats or FeatureEngineering.get_team_rolling_averages()
    
    # Prepare comprehensive features
    features_df = FeatureEngineering.prepare_features(
        validated_df, 
        team_stats=team_stats,
        include_momentum=True,
        include_rest=True,
        quarter_specific=True
    )
    
    # Make ensemble predictions
    try:
        result_df = model_manager.predict_with_ensemble(features_df)
    except Exception as e:
        logger.error(f"Error making ensemble predictions: {e}")
        traceback.print_exc()
        
        # Fallback to simple predictions
        result_df = validated_df.copy()
        
        # Add baseline predictions based on team averages
        for idx, row in result_df.iterrows():
            home_team = row['home_team']
            away_team = row['away_team']
            current_q = int(row.get('current_quarter', 0) or 0)
            
            # Get current scores
            current_home = float(row.get('current_home_score', row.get('home_score', 0)) or 0)
            current_away = float(row.get('current_away_score', row.get('away_score', 0)) or 0)
            
            # Get team averages
            home_avg = 105.0
            away_avg = 105.0
            
            if team_stats:
                if home_team in team_stats:
                    home_avg = team_stats[home_team]['avg_score']
                if away_team in team_stats:
                    away_avg = team_stats[away_team]['avg_score']
            
            # Calculate remaining portion of game
            remaining_pct = 1.0 - (0.25 * current_q)
            
            # Make fallback predictions
            result_df.at[idx, 'predicted_home_score'] = current_home + (home_avg * remaining_pct)
            result_df.at[idx, 'predicted_away_score'] = current_away + (away_avg * remaining_pct)
            
            # Add metrics
            result_df.at[idx, 'prediction_confidence'] = calculate_prediction_confidence(current_q)
            
            home_pred = result_df.at[idx, 'predicted_home_score']
            away_pred = result_df.at[idx, 'predicted_away_score']
            result_df.at[idx, 'win_probability'] = calculate_win_probability(home_pred, away_pred, current_q)
    
    # Add timestamp
    result_df['prediction_timestamp'] = datetime.now()
    
    # Update prediction history if provided
    if prediction_history is not None:
        for _, game in result_df.iterrows():
            game_id = str(game['game_id'])
            
            if game_id not in prediction_history:
                prediction_history[game_id] = []
            
            # Add current prediction to history
            prediction_record = {
                'timestamp': datetime.now(),
                'game_id': game_id,
                'home_team': game['home_team'],
                'away_team': game['away_team'],
                'current_quarter': int(game.get('current_quarter', 0) or 0),
                'current_home_score': float(game.get('current_home_score', game.get('home_score', 0)) or 0),
                'current_away_score': float(game.get('current_away_score', game.get('away_score', 0)) or 0),
                'predicted_home_score': float(game['predicted_home_score']),
                'predicted_away_score': float(game['predicted_away_score']),
                'win_probability': float(game['win_probability']),
                'prediction_confidence': float(game['prediction_confidence'])
            }
            
            # Add momentum if available
            if 'cumulative_momentum' in features_df.columns:
                prediction_record['momentum'] = float(features_df.at[game.name, 'cumulative_momentum'])
            
            # Add ensemble weights if available
            if 'ensemble_weight_main' in game:
                prediction_record['ensemble_weight_main'] = float(game['ensemble_weight_main'])
                prediction_record['ensemble_weight_quarter'] = float(game['ensemble_weight_quarter'])
            
            prediction_history[game_id].append(prediction_record)
        
        logger.info(f"Updated prediction history for {len(result_df)} games")
    
    elapsed_time = time.time() - start_time
    logger.info(f"Completed predictions in {elapsed_time:.2f} seconds")
    
    return result_df

# ===== VISUALIZATION FUNCTIONS =====

def visualize_predictions(predictions_df: pd.DataFrame, prediction_history: Dict = None) -> plt.Figure:
    """
    Create visualizations for score predictions.
    
    Args:
        predictions_df: DataFrame with predictions
        prediction_history: Optional prediction history dictionary
        
    Returns:
        matplotlib Figure object
    """
    if predictions_df.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, "No predictions available", 
                ha='center', va='center', fontsize=14)
        return fig
    
    # Determine layout based on number of games
    n_games = len(predictions_df)
    rows = min(n_games, 3)  # Max 3 rows
    fig, axs = plt.subplots(rows, 2, figsize=(16, 5 * rows))
    
    # Handle case with single game
    if n_games == 1:
        axs = np.array([axs])
    
    # Process each game
    for i, (_, game) in enumerate(predictions_df.iterrows()):
        row = i % rows
        
        # Left plot: Current vs Predicted scores
        ax1 = axs[row, 0]
        
        # Extract data
        teams = [game['home_team'], game['away_team']]
        current_scores = [
            float(game.get('current_home_score', game.get('home_score', 0)) or 0),
            float(game.get('current_away_score', game.get('away_score', 0)) or 0)
        ]
        predicted_scores = [
            float(game['predicted_home_score']),
            float(game['predicted_away_score'])
        ]
        
        # Create bar chart
        x = np.arange(len(teams))
        width = 0.35
        
        current_bars = ax1.bar(x - width/2, current_scores, width, label='Current')
        predicted_bars = ax1.bar(x + width/2, predicted_scores, width, label='Predicted Final')
        
        # Add bar labels
        for bar in current_bars:
            height = bar.get_height()
            ax1.annotate(f'{int(height)}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')
        
        for bar in predicted_bars:
            height = bar.get_height()
            ax1.annotate(f'{height:.1f}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')
        
        # Customize plot
        ax1.set_xticks(x)
        ax1.set_xticklabels(teams)
        ax1.legend()
        
        # Add quarter and confidence info
        quarter = int(game.get('current_quarter', 0) or 0)
        confidence = float(game.get('prediction_confidence', 0.5)) * 100
        quarter_text = "Pre-game" if quarter == 0 else f"Quarter {quarter}"
        
        ax1.set_title(f"{teams[0]} vs {teams[1]} - {quarter_text} (Confidence: {confidence:.0f}%)")
        
        # Add win probability annotation
        if 'win_probability' in game:
            win_prob = float(game['win_probability']) * 100
            win_team = teams[0] if win_prob > 50 else teams[1]
            win_prob_display = win_prob if win_prob > 50 else (100 - win_prob)
            
            ax1.annotate(f"{win_team} Win: {win_prob_display:.1f}%",
                        xy=(0.5, -0.1),
                        xycoords='axes fraction',
                        ha='center',
                        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="orange", alpha=0.8))
        
        # Right plot: Quarter breakdown or prediction history
        ax2 = axs[row, 1]
        
        # If prediction history available, plot the prediction trends
        if prediction_history and str(game['game_id']) in prediction_history:
            history = prediction_history[str(game['game_id'])]
            
            if len(history) > 1:  # Only plot if we have multiple predictions
                timestamps = [h['timestamp'] for h in history]
                home_predictions = [h['predicted_home_score'] for h in history]
                away_predictions = [h['predicted_away_score'] for h in history]
                
                # Convert timestamps to relative time for better display
                start_time = min(timestamps)
                rel_times = [(t - start_time).total_seconds() / 60.0 for t in timestamps]
                
                # Plot prediction trends
                ax2.plot(rel_times, home_predictions, 'b-o', label=f"{teams[0]} (Predicted)")
                ax2.plot(rel_times, away_predictions, 'r-o', label=f"{teams[1]} (Predicted)")
                
                # Add actual scores if available
                if 'current_quarter' in history[0]:
                    home_actuals = [h.get('current_home_score', 0) for h in history]
                    away_actuals = [h.get('current_away_score', 0) for h in history]
                    
                    ax2.plot(rel_times, home_actuals, 'b--', label=f"{teams[0]} (Actual)")
                    ax2.plot(rel_times, away_actuals, 'r--', label=f"{teams[1]} (Actual)")
                
                ax2.set_xlabel('Time (minutes)')
                ax2.set_ylabel('Score')
                ax2.legend(loc='upper left', fontsize=8)
                ax2.set_title('Prediction Trend Over Time')
                
                # Add quarter transitions if available
                quarters = [h.get('current_quarter', 0) for h in history]
                for q in range(1, 5):
                    # Find first instance of this quarter
                    try:
                        q_idx = quarters.index(q)
                        q_time = rel_times[q_idx]
                        ax2.axvline(x=q_time, color='gray', linestyle='--', alpha=0.5)
                        ax2.text(q_time, ax2.get_ylim()[1] * 0.95, f"Q{q}", 
                                 rotation=90, va='top', ha='right')
                    except ValueError:
                        pass  # Quarter not in history
            else:
                # Not enough history for trend
                plot_quarter_breakdown(ax2, game, teams)
        else:
            # No history, show quarter breakdown
            plot_quarter_breakdown(ax2, game, teams)
    
    # Hide unused subplots
    for i in range(n_games, rows * 2):
        row = i // 2
        col = i % 2
        if row < rows and col < 2:  # Check if axis exists in our grid
            axs[row, col].axis('off')
    
    plt.tight_layout()
    return fig

def plot_quarter_breakdown(ax, game, teams):
    """
    Plot quarter-by-quarter breakdown when prediction history isn't available.
    
    Args:
        ax: Matplotlib axis to plot on
        game: Row from predictions DataFrame
        teams: List of team names [home, away]
    """
    quarters = []
    home_scores = []
    away_scores = []
    
    # Get current quarter scores
    current_q = int(game.get('current_quarter', 0) or 0)
    for q in range(1, current_q + 1):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in game and away_q in game:
            quarters.append(f"Q{q}")
            home_scores.append(float(game[home_q] or 0))
            away_scores.append(float(game[away_q] or 0))
    
    # Add predicted remaining quarters if not full game yet
    if current_q < 4:
        # Get current totals
        current_home = float(game.get('current_home_score', game.get('home_score', 0)) or 0)
        current_away = float(game.get('current_away_score', game.get('away_score', 0)) or 0)
        
        # Get predicted finals
        predicted_home = float(game['predicted_home_score'])
        predicted_away = float(game['predicted_away_score'])
        
        # Calculate remaining points
        remaining_home = max(0, predicted_home - current_home)
        remaining_away = max(0, predicted_away - current_away)
        
        # Calculate remaining quarters
        remaining_quarters = 4 - current_q
        
        if remaining_quarters > 0:
            quarters.append(f"Rem. {remaining_quarters}Q")
            home_scores.append(remaining_home)
            away_scores.append(remaining_away)
    
    # If no quarter data or pre-game, show predicted final distribution
    if not quarters:
        quarters = ["Predicted Final"]
        home_scores = [float(game['predicted_home_score'])]
        away_scores = [float(game['predicted_away_score'])]
    
    # Create bar chart
    x = np.arange(len(quarters))
    width = 0.35
    
    ax.bar(x - width/2, home_scores, width, label=teams[0], color='blue', alpha=0.7)
    ax.bar(x + width/2, away_scores, width, label=teams[1], color='red', alpha=0.7)
    
    # Customize chart
    ax.set_xticks(x)
    ax.set_xticklabels(quarters)
    ax.legend()
    
    current_q_text = "Pre-game" if current_q == 0 else f"Through Q{current_q}"
    ax.set_title(f"Score Breakdown ({current_q_text})")
    
    # Add labels to bars
    for i, v in enumerate(home_scores):
        ax.text(i - width/2, v + 0.5, f"{v:.1f}", ha='center')
    
    for i, v in enumerate(away_scores):
        ax.text(i + width/2, v + 0.5, f"{v:.1f}", ha='center')

# ===== USAGE EXAMPLE =====

def demo_prediction_visualization():
    """
    Demonstrate the prediction visualization with sample data.
    """
    # Sample game data (mid-game)
    sample_games = pd.DataFrame([
        {
            'game_id': '1001',
            'home_team': 'Lakers', 
            'away_team': 'Celtics',
            'current_quarter': 2,
            'home_q1': 28, 'away_q1': 26,
            'home_q2': 24, 'away_q2': 30,
            'home_score': 52, 'away_score': 56,
            'predicted_home_score': 108.5, 'predicted_away_score': 104.2,
            'win_probability': 0.62, 'prediction_confidence': 0.65
        },
        {
            'game_id': '1002',
            'home_team': 'Warriors', 
            'away_team': 'Bucks',
            'current_quarter': 3,
            'home_q1': 32, 'away_q1': 22,
            'home_q2': 28, 'away_q2': 26,
            'home_q3': 24, 'away_q3': 31,
            'home_score': 84, 'away_score': 79,
            'predicted_home_score': 120.3, 'predicted_away_score': 115.7,
            'win_probability': 0.73, 'prediction_confidence': 0.80
        },
        {
            'game_id': '1003',
            'home_team': 'Nets', 
            'away_team': 'Heat',
            'current_quarter': 0,  # Pre-game
            'home_score': 0, 'away_score': 0,
            'predicted_home_score': 112.8, 'predicted_away_score': 109.4,
            'win_probability': 0.58, 'prediction_confidence': 0.35
        }
    ])
    
    # Sample prediction history
    sample_history = {
        '1001': [
            {
                'timestamp': datetime.now() - timedelta(minutes=30),
                'current_quarter': 1,
                'current_home_score': 28, 'current_away_score': 26,
                'predicted_home_score': 110.5, 'predicted_away_score': 102.3
            },
            {
                'timestamp': datetime.now() - timedelta(minutes=15),
                'current_quarter': 2,
                'current_home_score': 52, 'current_away_score': 56,
                'predicted_home_score': 108.5, 'predicted_away_score': 104.2
            }
        ]
    }
    
    # Generate visualization
    fig = visualize_predictions(sample_games, sample_history)
    
    # Show plot
    plt.show()
    
    return fig

# Run demo if executed directly
if __name__ == "__main__":
    demo_prediction_visualization()

In [ ]:
# Cell 11A: Enhanced Layered Fallback System (Integrated Version)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import traceback
from typing import Dict, List, Union, Optional, Tuple, Any

class LayeredFallbackPredictor:
    """
    Enhanced layered fallback system for NBA score predictions that gracefully degrades
    when features or model availability changes. This system is designed to be used
    with the main prediction pipeline and supports different levels of prediction
    quality based on available data.
    
    Fallback Levels:
    1. Enhanced Model: Uses full feature set including momentum features (highest accuracy)
    2. Basic Model: Uses core features without momentum (high accuracy)
    3. Quarter Extrapolation: When model is unavailable but quarter data exists (medium accuracy)
    4. Season Averages: When only team identities are known (lower accuracy)
    5. League Averages: Ultimate fallback with minimal information (lowest accuracy)
    """
    
    def __init__(self, model=None, team_avgs=None):
        """
        Initialize the fallback prediction system.
        
        Args:
            model: The primary prediction model (optional)
            team_avgs: Dictionary of team scoring averages (optional)
        """
        self.model = model
        self.team_avgs = team_avgs or {}
        self.league_avg_score = 110.0  # NBA average as default
        self.home_advantage = 3.5      # Standard NBA home court advantage
        
        # Track which fallback level was used for analytics
        self.used_fallback_levels = {
            "enhanced_model": 0,
            "basic_model": 0,
            "quarter_extrapolation": 0,
            "season_averages": 0,
            "league_averages": 0
        }
    
    def predict(self, game_data: pd.DataFrame) -> pd.DataFrame:
        """
        Make predictions using the best available strategy based on data quality.
        
        Args:
            game_data: DataFrame with game information
            
        Returns:
            DataFrame with predictions and metadata about strategy used
        """
        if game_data.empty:
            print("No game data provided to fallback system")
            return pd.DataFrame()
        
        results = []
        for idx, game in game_data.iterrows():
            try:
                result = self._predict_single_game(game)
                results.append(result)
            except Exception as e:
                print(f"Error in fallback prediction for game {game.get('game_id', 'unknown')}: {str(e)}")
                traceback.print_exc()
                # Add emergency fallback result
                results.append(self._create_emergency_fallback(game))
        
        # Create DataFrame with results
        return pd.DataFrame(results)
    
    def _predict_single_game(self, game: pd.Series) -> Dict[str, Any]:
        """
        Predict a single game using the best available strategy.
        
        Args:
            game: Series with single game data
            
        Returns:
            Dict with prediction results and metadata
        """
        # Extract basic game information
        game_id = game.get('game_id')
        home_team = game.get('home_team')
        away_team = game.get('away_team')
        current_quarter = int(game.get('current_quarter', 0))
        
        # Initialize result dict with game info
        result = {
            'game_id': game_id,
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter
        }
        
        # Calculate current score
        current_home_score = 0
        current_away_score = 0
        if current_quarter > 0:
            for q in range(1, current_quarter+1):
                current_home_score += float(game.get(f'home_q{q}', 0) or 0)
                current_away_score += float(game.get(f'away_q{q}', 0) or 0)
        
        result['current_home_score'] = current_home_score
        result['current_away_score'] = current_away_score
        
        # Determine data availability
        has_quarter_data = all(f'home_q{q}' in game for q in range(1, current_quarter+1))
        has_team_data = home_team in self.team_avgs and away_team in self.team_avgs
        has_enhanced_features = ('cumulative_momentum' in game) or ('q1_to_q2_momentum' in game)
        has_model = self.model is not None
        
        # LEVEL 1: Full enhanced model prediction
        if has_model and has_quarter_data and has_enhanced_features and hasattr(self.model, 'feature_importances_') and len(self.model.feature_importances_) > 8:
            print(f"Using enhanced model for {home_team} vs {away_team}")
            self.used_fallback_levels["enhanced_model"] += 1
            
            # Prepare features for enhanced model
            features = self._prepare_enhanced_features(game)
            
            try:
                # Make prediction using enhanced features
                predicted_home_score = self._predict_with_model(features)
            except ValueError as e:
                # If feature name mismatch error is encountered, log and try basic model
                print(f"Enhanced model prediction failed due to feature mismatch: {e}")
                if has_quarter_data:
                    print(f"Falling back to basic model for {home_team} vs {away_team}")
                    self.used_fallback_levels["basic_model"] += 1
                    features = self._prepare_basic_features(game)
                    predicted_home_score = self._predict_with_model(features)
                    prev_matchup_diff = float(game.get('prev_matchup_diff', 0))
                    predicted_away_score = predicted_home_score - prev_matchup_diff - self.home_advantage
                    result['fallback_model_type'] = "basic_model"
                    result['prediction_confidence'] = 0.8 * (1 + current_quarter / 4) / 2
                else:
                    raise e
            else:
                # Calculate away score based on differential factors for enhanced model
                diff_weight = min(0.3 + (0.1 * current_quarter), 0.6)
                momentum_adj = float(game.get('cumulative_momentum', 0)) * 3.0
                prev_matchup_diff = float(game.get('prev_matchup_diff', 0))
                predicted_away_score = predicted_home_score - (prev_matchup_diff * diff_weight) - momentum_adj
                result['fallback_model_type'] = "enhanced_model"
                result['prediction_confidence'] = 0.9 * (1 + current_quarter / 4) / 2
            
        # LEVEL 2: Basic model prediction
        elif has_model and has_quarter_data:
            print(f"Using basic model for {home_team} vs {away_team}")
            self.used_fallback_levels["basic_model"] += 1
            
            # Prepare features for basic model
            features = self._prepare_basic_features(game)
            predicted_home_score = self._predict_with_model(features)
            prev_matchup_diff = float(game.get('prev_matchup_diff', 0))
            predicted_away_score = predicted_home_score - prev_matchup_diff - self.home_advantage
            result['fallback_model_type'] = "basic_model"
            result['prediction_confidence'] = 0.8 * (1 + current_quarter / 4) / 2
            
        # LEVEL 3: Quarter extrapolation
        elif current_quarter > 0 and has_quarter_data:
            print(f"Using quarter extrapolation for {home_team} vs {away_team}")
            self.used_fallback_levels["quarter_extrapolation"] += 1
            
            home_ppq = current_home_score / current_quarter
            away_ppq = current_away_score / current_quarter
            
            remaining_quarters = 4 - current_quarter
            predicted_home_score = current_home_score + (home_ppq * remaining_quarters)
            predicted_away_score = current_away_score + (away_ppq * remaining_quarters)
            result['fallback_model_type'] = "quarter_extrapolation"
            result['prediction_confidence'] = 0.7 * (1 + current_quarter / 4) / 2
            
        # LEVEL 4: Season averages
        elif has_team_data:
            print(f"Using season averages for {home_team} vs {away_team}")
            self.used_fallback_levels["season_averages"] += 1
            
            home_avg = self.team_avgs[home_team]
            away_avg = self.team_avgs[away_team]
            predicted_home_score = home_avg + (self.home_advantage / 2)
            predicted_away_score = away_avg - (self.home_advantage / 2)
            result['fallback_model_type'] = "season_averages"
            result['prediction_confidence'] = 0.5
            
        # LEVEL 5: League averages (ultimate fallback)
        else:
            print(f"Using league averages for {home_team} vs {away_team}")
            self.used_fallback_levels["league_averages"] += 1
            
            predicted_home_score = self.league_avg_score + (self.home_advantage / 2)
            predicted_away_score = self.league_avg_score - (self.home_advantage / 2)
            result['fallback_model_type'] = "league_averages"
            result['prediction_confidence'] = 0.3
        
        # Ensure predictions are at least current scores
        if current_quarter > 0:
            predicted_home_score = max(predicted_home_score, current_home_score)
            predicted_away_score = max(predicted_away_score, current_away_score)
        
        # Calculate win probability
        score_diff = predicted_home_score - predicted_away_score
        result['predicted_home_score'] = predicted_home_score
        result['predicted_away_score'] = predicted_away_score
        result['win_probability'] = 1.0 / (1.0 + np.exp(-0.1 * score_diff))
        
        result['remaining_home_points'] = max(0, predicted_home_score - current_home_score)
        result['remaining_away_points'] = max(0, predicted_away_score - current_away_score)
        
        return result
    
    def _prepare_enhanced_features(self, game: pd.Series) -> pd.DataFrame:
        """
        Prepare enhanced feature set for prediction.
        
        Args:
            game: Series with game data
            
        Returns:
            DataFrame with prepared features (NaN values are filled with 0)
        """
        features = pd.DataFrame({
            'home_q1': [float(game.get('home_q1', 0) or 0)],
            'home_q2': [float(game.get('home_q2', 0) or 0)],
            'home_q3': [float(game.get('home_q3', 0) or 0)],
            'home_q4': [float(game.get('home_q4', 0) or 0)],
            'score_ratio': [float(game.get('score_ratio', 0.5) or 0.5)],
            'prev_matchup_diff': [float(game.get('prev_matchup_diff', 0) or 0)],
            'rest_days_home': [float(game.get('rest_days_home', 2) or 2)],
            'rest_days_away': [float(game.get('rest_days_away', 2) or 2)],
            'rest_advantage': [float(game.get('rest_advantage', 0) or 0)],
            # Use the naming expected by the model:
            'is_back_to_back_home': [float(game.get('is_back_to_back_home', 0) or 0)],
            'is_back_to_back_away': [float(game.get('is_back_to_back_away', 0) or 0)],
            'q1_to_q2_momentum': [float(game.get('q1_to_q2_momentum', 0) or 0)],
            'q2_to_q3_momentum': [float(game.get('q2_to_q3_momentum', 0) or 0)],
            'q3_to_q4_momentum': [float(game.get('q3_to_q4_momentum', 0) or 0)],
            'cumulative_momentum': [float(game.get('cumulative_momentum', 0) or 0)]
        })
        # Attempt to add missing feature "momentum_rest_interaction" if expected by the model
        if self.model is not None and hasattr(self.model, "feature_names_in_"):
            if "momentum_rest_interaction" in self.model.feature_names_in_ and "momentum_rest_interaction" not in features.columns:
                # Define as product of cumulative_momentum and rest_advantage
                features["momentum_rest_interaction"] = features["cumulative_momentum"] * features["rest_advantage"]
        features.fillna(0, inplace=True)
        return features
    
    def _prepare_basic_features(self, game: pd.Series) -> pd.DataFrame:
        """
        Prepare basic feature set for prediction.
        
        Args:
            game: Series with game data
            
        Returns:
            DataFrame with prepared features (NaN values are filled with 0)
        """
        home_team = game.get('home_team')
        away_team = game.get('away_team')
        
        rolling_home = self.team_avgs.get(home_team, 105.0)
        rolling_away = self.team_avgs.get(away_team, 105.0)
        
        features = pd.DataFrame({
            'home_q1': [float(game.get('home_q1', 0) or 0)],
            'home_q2': [float(game.get('home_q2', 0) or 0)],
            'home_q3': [float(game.get('home_q3', 0) or 0)],
            'home_q4': [float(game.get('home_q4', 0) or 0)],
            'score_ratio': [float(game.get('score_ratio', 0.5) or 0.5)],
            'rolling_home_score': [rolling_home],
            'rolling_away_score': [rolling_away],
            'prev_matchup_diff': [float(game.get('prev_matchup_diff', 0) or 0)]
        })
        features.fillna(0, inplace=True)
        return features
    
    def _predict_with_model(self, features: pd.DataFrame) -> float:
        """
        Make prediction using the model with error handling for feature name mismatches.
        
        Args:
            features: DataFrame with features
            
        Returns:
            float: Predicted home score
        """
        if self.model is None:
            raise ValueError("Model not available for prediction")
        
        features.fillna(0, inplace=True)
        
        try:
            prediction = float(self.model.predict(features)[0])
            return prediction
        except ValueError as e:
            if "feature names" in str(e) and hasattr(self.model, "feature_names_in_"):
                print(f"Feature name mismatch encountered: {e}")
                expected = set(self.model.feature_names_in_)
                current = set(features.columns)
                
                rename_map = {}
                # Map our naming to expected naming if possible
                if "is_home_b2b" in expected and "is_back_to_back_home" in current:
                    rename_map["is_back_to_back_home"] = "is_home_b2b"
                if "is_away_b2b" in expected and "is_back_to_back_away" in current:
                    rename_map["is_back_to_back_away"] = "is_away_b2b"
                
                if rename_map:
                    features = features.rename(columns=rename_map)
                
                # Add any missing expected columns with default 0
                for col in expected:
                    if col not in features.columns:
                        features[col] = 0.0
                # Reorder columns to match the fitted order
                features = features[list(self.model.feature_names_in_)]
                
                # Retry prediction after adjustments
                prediction = float(self.model.predict(features)[0])
                return prediction
            else:
                raise e
    
    def _create_emergency_fallback(self, game: pd.Series) -> Dict[str, Any]:
        """
        Create emergency fallback prediction when all else fails.
        
        Args:
            game: Game data
            
        Returns:
            Dict with minimal prediction
        """
        return {
            'game_id': game.get('game_id', 'unknown'),
            'home_team': game.get('home_team', 'Unknown'),
            'away_team': game.get('away_team', 'Unknown'),
            'current_quarter': int(game.get('current_quarter', 0)),
            'current_home_score': 0,
            'current_away_score': 0,
            'predicted_home_score': 105,
            'predicted_away_score': 102,
            'win_probability': 0.55,
            'fallback_model_type': "emergency_fallback",
            'prediction_confidence': 0.1
        }
    
    def get_fallback_usage_stats(self) -> Dict[str, int]:
        """
        Get statistics on which fallback levels were used.
        
        Returns:
            Dict with counts of each fallback level used
        """
        return self.used_fallback_levels
    
    def integration_test(self, test_data: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        Run an integration test of the fallback system with test data.
        
        Args:
            test_data: Optional DataFrame with test data, generates sample if None
            
        Returns:
            DataFrame with test results
        """
        if test_data is None:
            test_data = pd.DataFrame([
                {
                    'game_id': 1001,
                    'home_team': 'Boston Celtics',
                    'away_team': 'Miami Heat',
                    'current_quarter': 3,
                    'home_q1': 28, 'home_q2': 30, 'home_q3': 25,
                    'away_q1': 25, 'away_q2': 27, 'away_q3': 24,
                    'cumulative_momentum': 0.2,
                    'prev_matchup_diff': 4.5,
                    'rest_days_home': 2,
                    'rest_days_away': 1,
                    'rest_advantage': 1,
                    'is_back_to_back_home': 0,
                    'is_back_to_back_away': 1
                },
                {
                    'game_id': 1002,
                    'home_team': 'Los Angeles Lakers',
                    'away_team': 'Golden State Warriors',
                    'current_quarter': 2,
                    'home_q1': 31, 'home_q2': 28,
                    'away_q1': 29, 'away_q2': 32
                },
                {
                    'game_id': 1003,
                    'home_team': 'Boston Celtics',
                    'away_team': 'Los Angeles Lakers',
                    'current_quarter': 0
                },
                {
                    'game_id': 1004,
                    'home_team': 'Unknown Team',
                    'away_team': 'Mystery Team',
                    'current_quarter': 1
                }
            ])
        
        # Reset fallback usage stats
        self.used_fallback_levels = {
            "enhanced_model": 0,
            "basic_model": 0,
            "quarter_extrapolation": 0,
            "season_averages": 0,
            "league_averages": 0
        }
        
        print("Testing layered fallback system with different scenarios...")
        test_results = self.predict(test_data)
        
        print("\nFallback System Test Results:")
        display_cols = ['home_team', 'away_team', 'current_quarter', 
                        'predicted_home_score', 'predicted_away_score',
                        'fallback_model_type', 'prediction_confidence']
        
        if 'display' in globals():
            from IPython.display import display
            display(test_results[display_cols])
        else:
            print(test_results[display_cols])
        
        print("\nFallback Level Usage:")
        for level, count in self.used_fallback_levels.items():
            print(f"  {level}: {count} times")
        
        return test_results

# Instantiate the fallback system and run a test
if 'model' in globals() and globals()['model'] is not None:
    fallback_predictor = LayeredFallbackPredictor(model=globals()['model'])
else:
    fallback_predictor = LayeredFallbackPredictor()

# Add team averages if available
team_avgs = {}
if 'get_team_rolling_averages' in globals():
    try:
        team_avgs = globals()['get_team_rolling_averages']()
        fallback_predictor.team_avgs = team_avgs
    except Exception as e:
        print(f"Could not load team averages: {e}")

# Run the integration test
test_results = fallback_predictor.integration_test()

# Example of how to use with the main prediction pipeline:
"""
def run_prediction_pipeline(games_df, use_fallback=True):
    try:
        if 'model' in globals() and globals()['model'] is not None:
            predictions = make_predictions(games_df, globals()['model'])
            return predictions
        else:
            print("Main model not available, falling back to layered system")
            use_fallback = True
    except Exception as e:
        print(f"Error in main prediction: {e}")
        use_fallback = True
    
    if use_fallback:
        print("Using layered fallback system")
        return fallback_predictor.predict(games_df)
"""


In [ ]:
# Cell 11B: Refactored Strategy Selection System with Registration Pattern

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import time
import traceback
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
import json

class GameContextAnalyzer:
    """
    Analyzes game state to determine context for strategy selection.
    """
    
    def __init__(self):
        # Priority order for primary context determination
        self.context_priorities = [
            "clutch_time",
            "final_minutes",
            "blowout",
            "close_game",
            "high_momentum",
            "comeback_potential",
            "run",
            "late_game",
            "early_game",
            "pregame"
        ]
    
    def analyze(self, game_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Analyze the game context to determine appropriate prediction strategy.
        
        Args:
            game_data: Dictionary with game information
            
        Returns:
            Dict with context information
        """
        # Extract key game attributes
        current_quarter = int(game_data.get('current_quarter', 0))
        home_score = float(game_data.get('home_score', 0) or 0)
        away_score = float(game_data.get('away_score', 0) or 0)
        score_diff = home_score - away_score
        abs_score_diff = abs(score_diff)
        
        # Extract momentum if available
        momentum = 0.0
        if 'cumulative_momentum' in game_data:
            momentum = float(game_data.get('cumulative_momentum', 0))
        elif 'momentum_shift' in game_data:
            momentum = float(game_data.get('momentum_shift', 0))
        
        # Extract time remaining if available
        time_remaining = float(game_data.get('time_remaining', 12))
        
        # Game phase determination
        game_phase = ""
        if current_quarter == 0:
            game_phase = "pregame"
        elif current_quarter <= 2:
            game_phase = "first_half"
        else:
            game_phase = "second_half"
        
        # Identify specific contexts that apply to this game state
        contexts = [game_phase]
        
        # Quarter-specific context
        if current_quarter == 0:
            contexts.append("pregame")
        elif current_quarter == 1:
            contexts.append("early_game")
        elif current_quarter == 4:
            contexts.append("late_game")
            
            # Final minutes
            if time_remaining <= 5:
                contexts.append("final_minutes")
                
                # Clutch time (close & final minutes)
                if abs_score_diff <= 8:
                    contexts.append("clutch_time")
        
        # Score differential contexts
        if abs_score_diff <= 5:
            contexts.append("close_game")
        elif abs_score_diff >= 20:
            contexts.append("blowout")
        elif abs_score_diff >= 12:
            contexts.append("large_lead")
        
        # Momentum-based contexts
        if abs(momentum) >= 0.6:
            contexts.append("high_momentum")
            
            # Direction of momentum
            if momentum > 0:
                contexts.append("home_momentum")
            else:
                contexts.append("away_momentum")
        
        # Potential comeback context
        if abs_score_diff >= 10 and ((score_diff > 0 and momentum < -0.3) or 
                                     (score_diff < 0 and momentum > 0.3)):
            contexts.append("comeback_potential")
        
        # Recent run context
        run_detected = False
        for q in range(1, 5):
            if q > current_quarter:
                break
                
            if q == current_quarter:
                # Check most recent quarter
                home_q = game_data.get(f'home_q{q}', 0) or 0
                away_q = game_data.get(f'away_q{q}', 0) or 0
                
                # If one team outscored the other by 8+ points
                if abs(home_q - away_q) >= 8:
                    run_detected = True
                    if home_q > away_q:
                        contexts.append("home_run")
                    else:
                        contexts.append("away_run")
        
        if run_detected:
            contexts.append("run")
        
        # Scoring pace context
        total_score = home_score + away_score
        expected_score_by_quarter = [0, 50, 105, 160, 215]  # Rough estimates
        
        if current_quarter > 0 and current_quarter < 5:
            expected_score = expected_score_by_quarter[current_quarter]
            if total_score > expected_score * 1.15:
                contexts.append("high_scoring")
            elif total_score < expected_score * 0.85:
                contexts.append("low_scoring")
        
        # Calculate a primary context (most specific/important one)
        primary_context = self._determine_primary_context(contexts)
        
        # Return the full context analysis
        return {
            'current_quarter': current_quarter,
            'score_differential': score_diff,
            'abs_score_differential': abs_score_diff,
            'momentum': momentum,
            'time_remaining': time_remaining,
            'contexts': contexts,
            'primary_context': primary_context,
            'game_phase': game_phase
        }
    
    def _determine_primary_context(self, contexts: List[str]) -> str:
        """Determine the most important context for strategy selection."""
        # Return the highest priority context that applies
        for context in self.context_priorities:
            if context in contexts:
                return context
        
        # Default to first context if no priority match
        return contexts[0] if contexts else "unknown"


class PredictionStrategy:
    """
    Base class for prediction strategies.
    """
    
    def __init__(self, name: str, description: str, preferred_contexts: List[str], model_key: Optional[str] = None):
        self.name = name
        self.description = description
        self.preferred_contexts = preferred_contexts
        self.model_key = model_key
        self.fallback_strategy = None
        self.success_rate = 0.0
        self.usage_count = 0
        self.avg_error = None
    
    def set_fallback(self, fallback_strategy: Optional['PredictionStrategy'] = None):
        """Set fallback strategy to use if this one fails."""
        self.fallback_strategy = fallback_strategy
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        """
        Execute prediction for this strategy.
        
        Args:
            game_data: Game data dictionary
            model: Model to use for prediction (if needed)
            team_avgs: Team scoring averages
            
        Returns:
            Dictionary with prediction results
        """
        raise NotImplementedError("Subclasses must implement predict method")


class EarlyQuartersStrategy(PredictionStrategy):
    """Strategy specialized for early quarters (Q0-Q2)."""
    
    def __init__(self):
        super().__init__(
            name="Early Quarters XGBoost",
            description="Specialized XGBoost model for Q0-Q2",
            preferred_contexts=["pregame", "early_game", "first_half"],
            model_key="early_quarters_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        # In a real implementation, this would use the model
        # For now, we'll implement a simple prediction
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get team averages if available
        home_avg = team_avgs.get(home_team, 105.0) if team_avgs else 105.0
        away_avg = team_avgs.get(away_team, 105.0) if team_avgs else 105.0
        
        # For pregame, adjust with home court advantage
        if current_quarter == 0:
            home_advantage = 3.5
            predicted_home = home_avg + (home_advantage / 2)
            predicted_away = away_avg - (home_advantage / 2)
        else:
            # Get current score
            home_score = float(game_data.get('home_score', 0))
            away_score = float(game_data.get('away_score', 0))
            
            # Calculate remaining portion of game
            remaining_factor = 1.0 - (0.25 * current_quarter)
            
            # Calculate predicted final scores
            predicted_home = home_score + (home_avg * remaining_factor)
            predicted_away = away_score + (away_avg * remaining_factor)
        
        # Calculate win probability
        score_diff = predicted_home - predicted_away
        win_prob = 1.0 / (1.0 + np.exp(-0.08 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.7 + (0.1 * current_quarter)  # Increases with quarter
        }


class MomentumStrategy(PredictionStrategy):
    """Strategy that weights predictions based on momentum factors."""
    
    def __init__(self):
        super().__init__(
            name="Momentum-Weighted Strategy",
            description="Strategy that weights models based on momentum factors",
            preferred_contexts=["high_momentum", "comeback", "run"],
            model_key="momentum_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        # Extract momentum
        momentum = 0.0
        if 'cumulative_momentum' in game_data:
            momentum = float(game_data.get('cumulative_momentum', 0))
        elif 'momentum_shift' in game_data:
            momentum = float(game_data.get('momentum_shift', 0))
        
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get team averages if available
        home_avg = team_avgs.get(home_team, 105.0) if team_avgs else 105.0
        away_avg = team_avgs.get(away_team, 105.0) if team_avgs else 105.0
        
        # Calculate remaining portion of game
        remaining_factor = 1.0 - (0.25 * current_quarter)
        
        # Apply momentum adjustment
        momentum_factor = 5.0 * momentum  # Points impact
        
        # Calculate predicted final scores
        predicted_home = home_score + (home_avg * remaining_factor) + (momentum_factor if momentum > 0 else 0)
        predicted_away = away_score + (away_avg * remaining_factor) + (-momentum_factor if momentum < 0 else 0)
        
        # Calculate win probability with momentum influence
        momentum_adjusted_diff = (predicted_home - predicted_away) + (momentum * 8.0)
        win_prob = 1.0 / (1.0 + np.exp(-0.1 * momentum_adjusted_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.6 + (0.1 * current_quarter),  # Slightly lower confidence due to momentum volatility
            'momentum_adjustment': momentum_factor
        }


class LateQuartersStrategy(PredictionStrategy):
    """Strategy specialized for late quarters (Q3-Q4)."""
    
    def __init__(self):
        super().__init__(
            name="Late Quarters Strategy",
            description="Specialized model for Q3-Q4",
            preferred_contexts=["late_game", "second_half", "close_game_late"],
            model_key="late_quarters_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get time remaining if available
        time_remaining = float(game_data.get('time_remaining', 12))
        
        # For late quarters, current score carries more weight
        remaining_minutes = (4 - current_quarter) * 12 + time_remaining
        total_minutes = 48.0
        remaining_factor = remaining_minutes / total_minutes
        
        # Get team scoring rates
        home_scoring_rate = home_score / (1 - remaining_factor) if remaining_factor < 1 else team_avgs.get(home_team, 105.0)
        away_scoring_rate = away_score / (1 - remaining_factor) if remaining_factor < 1 else team_avgs.get(away_team, 105.0)
        
        # Calculate predicted final scores
        predicted_home = home_score + (home_scoring_rate * remaining_factor)
        predicted_away = away_score + (away_scoring_rate * remaining_factor)
        
        # Calculate win probability
        score_diff = predicted_home - predicted_away
        # Late quarter predictions should have more confidence in current leader
        win_prob = 1.0 / (1.0 + np.exp(-0.12 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.8 + (0.05 * current_quarter)  # Higher confidence in late quarters
        }


class BlowoutStrategy(PredictionStrategy):
    """Strategy specialized for blowout games."""
    
    def __init__(self):
        super().__init__(
            name="Blowout Specialist",
            description="Model specialized for predicting blowout games",
            preferred_contexts=["blowout", "large_lead"],
            model_key="blowout_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Calculate current differential
        score_diff = home_score - away_score
        abs_diff = abs(score_diff)
        
        # Calculate scoring rate for teams in blowout situations
        remaining_factor = 1.0 - (0.25 * current_quarter)
        
        # In blowouts, the leading team usually slows down and trailing team may catch up slightly
        regression_factor = 0.7  # How much the blowout continues at same pace
        
        # Points per quarter so far
        ppq_leader = home_score / current_quarter if score_diff > 0 else away_score / current_quarter
        ppq_trailer = away_score / current_quarter if score_diff > 0 else home_score / current_quarter
        
        # Adjust rates for blowout pattern
        adjusted_ppq_leader = ppq_leader * regression_factor
        adjusted_ppq_trailer = ppq_trailer * (1 + (1 - regression_factor))  # Trailing team gets slight boost
        
        # Calculate remaining points
        if score_diff > 0:  # Home team leading
            remaining_home_points = adjusted_ppq_leader * 4 * remaining_factor
            remaining_away_points = adjusted_ppq_trailer * 4 * remaining_factor
        else:  # Away team leading
            remaining_home_points = adjusted_ppq_trailer * 4 * remaining_factor
            remaining_away_points = adjusted_ppq_leader * 4 * remaining_factor
        
        # Calculate predicted final scores
        predicted_home = home_score + remaining_home_points
        predicted_away = away_score + remaining_away_points
        
        # Blowouts have very high win probability for the leader
        final_diff = predicted_home - predicted_away
        win_prob = 1.0 / (1.0 + np.exp(-0.15 * final_diff))  # Steeper curve for higher certainty
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.85  # High confidence in blowout outcome
        }


class CloseGameStrategy(PredictionStrategy):
    """Strategy specialized for close games in critical moments."""
    
    def __init__(self):
        super().__init__(
            name="Close Game Specialist",
            description="Model focused on close games in critical moments",
            preferred_contexts=["close_game", "clutch_time", "final_minutes"],
            model_key="close_game_model"
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get time remaining
        time_remaining = float(game_data.get('time_remaining', 12))
        minutes_played = 48 - ((4 - current_quarter) * 12 + time_remaining)
        
        # Calculate current pace
        total_points = home_score + away_score
        points_per_minute = total_points / minutes_played if minutes_played > 0 else 4.4  # ~105 per team full game
        
        # In close games, especially late, current lead is very significant
        lead_weight = min(0.7, 0.4 + (current_quarter * 0.1))  # Increases with quarter, max 0.7
        
        # Remaining points based on pace
        remaining_minutes = (4 - current_quarter) * 12 + time_remaining
        remaining_points = points_per_minute * remaining_minutes
        
        # Team strength from averages
        home_strength = team_avgs.get(home_team, 105.0) / (team_avgs.get(home_team, 105.0) + team_avgs.get(away_team, 105.0))
        away_strength = 1 - home_strength
        
        # Distribute remaining points by team strength
        remaining_home_points = remaining_points * home_strength
        remaining_away_points = remaining_points * away_strength
        
        # Add slight bonus for home team in close games
        if abs(home_score - away_score) <= 5:
            home_clutch_advantage = 2.0  # Small home advantage in clutch
            remaining_home_points += home_clutch_advantage
            remaining_away_points -= home_clutch_advantage
        
        # Calculate predicted final scores
        predicted_home = home_score + remaining_home_points
        predicted_away = away_score + remaining_away_points
        
        # Win probability calculation - close games have higher uncertainty
        score_diff = predicted_home - predicted_away
        # Use lower coefficient for higher uncertainty
        win_prob = 1.0 / (1.0 + np.exp(-0.07 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.6  # Lower confidence in close games
        }


class BasicExtrapolationStrategy(PredictionStrategy):
    """Simple fallback strategy that uses basic extrapolation."""
    
    def __init__(self):
        super().__init__(
            name="Basic Score Extrapolation",
            description="Simple extrapolation based on current score and averages",
            preferred_contexts=[],  # Fallback strategy, works in any context
            model_key=None  # Doesn't require a specific model
        )
    
    def predict(self, game_data: Dict[str, Any], model: Any = None, team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        current_quarter = game_data.get('current_quarter', 0)
        home_team = game_data.get('home_team', '')
        away_team = game_data.get('away_team', '')
        
        # Get current score
        home_score = float(game_data.get('home_score', 0))
        away_score = float(game_data.get('away_score', 0))
        
        # Get team averages
        home_avg = team_avgs.get(home_team, 105.0) if team_avgs else 105.0
        away_avg = team_avgs.get(away_team, 105.0) if team_avgs else 105.0
        
        # Simple extrapolation based on current quarter
        if current_quarter == 0:
            # Pregame
            predicted_home = home_avg
            predicted_away = away_avg
        else:
            # In-game extrapolation
            remaining_factor = 1.0 - (0.25 * current_quarter)
            predicted_home = home_score + (home_avg * remaining_factor)
            predicted_away = away_score + (away_avg * remaining_factor)
        
        # Calculate win probability
        score_diff = predicted_home - predicted_away
        win_prob = 1.0 / (1.0 + np.exp(-0.08 * score_diff))
        
        return {
            'game_id': game_data.get('game_id'),
            'home_team': home_team,
            'away_team': away_team,
            'current_quarter': current_quarter,
            'predicted_home_score': predicted_home,
            'predicted_away_score': predicted_away,
            'win_probability': win_prob,
            'prediction_confidence': 0.5  # Moderate confidence for basic extrapolation
        }


class StrategyRegistry:
    """
    Registry for prediction strategies with dynamic registration.
    """
    
    def __init__(self):
        self.strategies = {}
        self.fallback_strategy = None
    
    def register(self, strategy_id: str, strategy: PredictionStrategy, fallback_id: Optional[str] = None):
        """
        Register a new prediction strategy.
        
        Args:
            strategy_id: Unique identifier for the strategy
            strategy: Strategy object to register
            fallback_id: ID of fallback strategy to use if this one fails
        """
        self.strategies[strategy_id] = strategy
        
        # Set fallback strategy if specified and exists
        if fallback_id and fallback_id in self.strategies:
            strategy.set_fallback(self.strategies[fallback_id])
        
        # If this is the first strategy, make it the default fallback
        if len(self.strategies) == 1:
            self.fallback_strategy = strategy
    
    def get_strategy(self, strategy_id: str) -> Optional[PredictionStrategy]:
        """Get a strategy by ID."""
        return self.strategies.get(strategy_id)
    
    def get_fallback_strategy(self) -> Optional[PredictionStrategy]:
        """Get the fallback strategy."""
        return self.fallback_strategy
    
    def all_strategies(self) -> Dict[str, PredictionStrategy]:
        """Get all registered strategies."""
        return self.strategies


class ErrorMetricsCollector:
    """
    Collects and analyzes prediction error metrics.
    """
    
    def __init__(self):
        self.strategy_errors = {}
        self.context_errors = {}
        self.quarter_errors = {}
        self.combined_errors = {}
    
    def add_error_record(self, 
                        prediction: Dict[str, Any],
                        actual: Dict[str, Any]):
        """
        Add an error record for a prediction.
        
        Args:
            prediction: Prediction dictionary
            actual: Actual results dictionary
        """
        # Ensure we have the required fields
        if 'strategy_id' not in prediction or 'context' not in prediction:
            return
        
        strategy_id = prediction['strategy_id']
        context = prediction['context']
        
        # Calculate error metrics
        try:
            pred_home_score = float(prediction.get('predicted_home_score', 0))
            pred_away_score = float(prediction.get('predicted_away_score', 0))
            
            actual_home_score = float(actual.get('home_score', 0))
            actual_away_score = float(actual.get('away_score', 0))
            
            # Mean Absolute Error
            home_error = abs(pred_home_score - actual_home_score)
            away_error = abs(pred_away_score - actual_away_score)
            mae = (home_error + away_error) / 2
            
            # Bias (over/under prediction)
            home_bias = pred_home_score - actual_home_score
            away_bias = pred_away_score - actual_away_score
            bias = (home_bias + away_bias) / 2
            
            # Winner prediction accuracy
            actual_winner = 'home' if actual_home_score > actual_away_score else 'away'
            predicted_winner = 'home' if pred_home_score > pred_away_score else 'away'
            winner_correct = int(actual_winner == predicted_winner)
            
            # Create error record
            error_record = {
                'timestamp': datetime.now().isoformat(),
                'game_id': prediction.get('game_id'),
                'strategy_id': strategy_id,
                'primary_context': context.get('primary_context', 'unknown'),
                'quarter': context.get('current_quarter', 0),
                'mae': mae,
                'bias': bias,
                'winner_correct': winner_correct,
                'pred_confidence': prediction.get('prediction_confidence', 0.8)
            }
            
            # Add to strategy errors
            if strategy_id not in self.strategy_errors:
                self.strategy_errors[strategy_id] = []
            self.strategy_errors[strategy_id].append(error_record)
            
            # Add to context errors
            primary_context = context.get('primary_context', 'unknown')
            if primary_context not in self.context_errors:
                self.context_errors[primary_context] = []
            self.context_errors[primary_context].append(error_record)
            
            # Add to quarter errors
            quarter = context.get('current_quarter', 0)
            if quarter not in self.quarter_errors:
                self.quarter_errors[quarter] = []
            self.quarter_errors[quarter].append(error_record)
            
            # Add to combined errors
            combined_key = f"{strategy_id}_{primary_context}"
            if combined_key not in self.combined_errors:
                self.combined_errors[combined_key] = []
            self.combined_errors[combined_key].append(error_record)
            
        except Exception as e:
            print(f"Error adding metrics: {str(e)}")
    
    def update_strategy_metrics(self, registry: StrategyRegistry):
        """
        Update strategy success rates and error metrics.
        
        Args:
            registry: Strategy registry to update
        """
        for strategy_id, records in self.strategy_errors.items():
            if not records:
                continue
                
            strategy = registry.get_strategy(strategy_id)
            if not strategy:
                continue
                
            # Update success rate
            strategy.usage_count = len(records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            strategy.success_rate = correct_predictions / len(records) if records else 0
            
            # Update error rate
            total_error = sum(r['mae'] for r in records)
            strategy.avg_error = total_error / len(records) if records else None
    
    def get_error_summary(self) -> Dict[str, Any]:
        """Get summary of error metrics."""
        summary = {
            'by_strategy': {},
            'by_context': {},
            'by_quarter': {},
            'overall': {
                'records': 0,
                'avg_mae': None,
                'avg_winner_accuracy': None
            }
        }
        
        # Collect all records
        all_records = []
        
        # Strategy metrics
        for strategy_id, records in self.strategy_errors.items():
            if not records:
                continue
                
            total_mae = sum(r['mae'] for r in records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            
            summary['by_strategy'][strategy_id] = {
                'records': len(records),
                'avg_mae': total_mae / len(records),
                'winner_accuracy': correct_predictions / len(records)
            }
            
            all_records.extend(records)
        
        # Context metrics
        for context, records in self.context_errors.items():
            if not records:
                continue
                
            total_mae = sum(r['mae'] for r in records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            
            summary['by_context'][context] = {
                'records': len(records),
                'avg_mae': total_mae / len(records),
                'winner_accuracy': correct_predictions / len(records)
            }
        
        # Quarter metrics
        for quarter, records in self.quarter_errors.items():
            if not records:
                continue
                
            total_mae = sum(r['mae'] for r in records)
            correct_predictions = sum(r['winner_correct'] for r in records)
            
            summary['by_quarter'][quarter] = {
                'records': len(records),
                'avg_mae': total_mae / len(records),
                'winner_accuracy': correct_predictions / len(records)
            }
        
        # Overall metrics
        if all_records:
            total_mae = sum(r['mae'] for r in all_records)
            correct_predictions = sum(r['winner_correct'] for r in all_records)
            
            summary['overall'] = {
                'records': len(all_records),
                'avg_mae': total_mae / len(all_records),
                'avg_winner_accuracy': correct_predictions / len(all_records)
            }
        
        return summary
    
    def visualize_error_metrics(self):
        """
        Visualize error metrics by strategy and quarter.
        """
        error_summary = self.get_error_summary()
        
        # Skip if no data
        if not error_summary['by_strategy']:
            print("No error metrics available for visualization.")
            return
        
        # Create figure with two subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Strategy MAE Comparison
        strategies = list(error_summary['by_strategy'].keys())
        mae_values = [error_summary['by_strategy'][s]['avg_mae'] for s in strategies]
        accuracy_values = [error_summary['by_strategy'][s]['winner_accuracy'] for s in strategies]
        records_count = [error_summary['by_strategy'][s]['records'] for s in strategies]
        
        # Sort by MAE (lower is better)
        sorted_indices = np.argsort(mae_values)
        sorted_strategies = [strategies[i] for i in sorted_indices]
        sorted_mae = [mae_values[i] for i in sorted_indices]
        sorted_accuracy = [accuracy_values[i] for i in sorted_indices]
        sorted_records = [records_count[i] for i in sorted_indices]
        
        # MAE Plot
        bars1 = ax1.barh(sorted_strategies, sorted_mae, color='skyblue')
        ax1.set_xlabel('Mean Absolute Error')
        ax1.set_title('Strategy Prediction Error (lower is better)')
        ax1.grid(axis='x', alpha=0.3)
        
        # Add sample size labels
        for i, bar in enumerate(bars1):
            ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, 
                    f"n={sorted_records[i]}", va='center')
        
        # Accuracy Plot
        bars2 = ax2.barh(sorted_strategies, sorted_accuracy, color='lightgreen')
        ax2.set_xlabel('Winner Prediction Accuracy')
        ax2.set_title('Strategy Winner Accuracy (higher is better)')
        ax2.set_xlim(0, 1)
        ax2.grid(axis='x', alpha=0.3)
        
        # Add percentage labels
        for i, bar in enumerate(bars2):
            ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                    f"{sorted_accuracy[i]:.1%}", va='center')
        
        plt.tight_layout()
        plt.show()
        
        # Plot error by quarter if available
        if error_summary['by_quarter']:
            plt.figure(figsize=(10, 6))
            
            quarters = sorted(error_summary['by_quarter'].keys(), key=lambda x: int(x))
            quarter_mae = [error_summary['by_quarter'][q]['avg_mae'] for q in quarters]
            quarter_acc = [error_summary['by_quarter'][q]['winner_accuracy'] for q in quarters]
            
            plt.plot(quarters, quarter_mae, 'o-', label='MAE', color='red')
            plt.xlabel('Quarter')
            plt.ylabel('Mean Absolute Error')
            plt.grid(alpha=0.3)
            
            # Add second y-axis for accuracy
            ax3 = plt.gca().twinx()
            ax3.plot(quarters, quarter_acc, 's-', label='Accuracy', color='green')
            ax3.set_ylabel('Winner Accuracy')
            ax3.set_ylim(0, 1)
            
            # Add labels
            for i, q in enumerate(quarters):
                plt.text(q, quarter_mae[i] + 0.5, f"{quarter_mae[i]:.1f}", ha='center')
                ax3.text(q, quarter_acc[i] + 0.02, f"{quarter_acc[i]:.1%}", ha='center')
            
            # Combine legends
            lines1, labels1 = plt.gca().get_legend_handles_labels()
            lines2, labels2 = ax3.get_legend_handles_labels()
            ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
            
            plt.title('Prediction Performance by Quarter')
            plt.tight_layout()
            plt.show()


class StrategySelector:
    """
    Selects the most appropriate prediction strategy based on game context.
    """
    
    def __init__(self, registry: StrategyRegistry, model_registry: Dict[str, Any] = None):
        """
        Initialize the strategy selector.
        
        Args:
            registry: Strategy registry containing available strategies
            model_registry: Dictionary mapping model keys to model objects
        """
        self.registry = registry
        self.model_registry = model_registry or {}
        self.context_analyzer = GameContextAnalyzer()
        self.metrics_collector = ErrorMetricsCollector()
        self.selection_history = []
    
    def select_strategy(self, game_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Select the most appropriate prediction strategy based on game context.
        
        Args:
            game_data: Dictionary with game information
            
        Returns:
            Dict with selected strategy and context information
        """
        # Analyze the game context
        context = self.context_analyzer.analyze(game_data)
        
        # Score strategies based on context match and past performance
        strategy_scores = {}
        
        for strategy_id, strategy in self.registry.all_strategies().items():
            # Base score
            score = 1.0
            
            # Context match bonus
            preferred_contexts = strategy.preferred_contexts
            current_contexts = context['contexts']
            context_matches = [c for c in current_contexts if c in preferred_contexts]
            
            context_score = len(context_matches) * 2.0
            
            # Bonus for matching the primary context
            if context['primary_context'] in preferred_contexts:
                context_score += 3.0
            
            # Performance history bonus
            success_bonus = 0.0
            if strategy.usage_count > 0:
                success_rate = strategy.success_rate
                # Scale by usage - more confident with more samples
                usage_scale = min(1.0, strategy.usage_count / 10.0)
                success_bonus = success_rate * 5.0 * usage_scale
            
            # Error metrics penalty
            error_penalty = 0.0
            if strategy.avg_error is not None:
                # Scale error to a 0-5 range (assuming error is in points)
                max_acceptable_error = 15.0
                error_scale = min(strategy.avg_error / max_acceptable_error, 1.0)
                error_penalty = error_scale * 5.0
            
            # Calculate total score
            total_score = score + context_score + success_bonus - error_penalty
            
            strategy_scores[strategy_id] = {
                'total_score': total_score,
                'context_score': context_score,
                'success_bonus': success_bonus,
                'error_penalty': error_penalty
            }
        
        # Select the strategy with the highest score
        if not strategy_scores:
            # No strategies available, use fallback
            fallback = self.registry.get_fallback_strategy()
            selected_strategy_id = "basic_extrapolation"
            selected_strategy = fallback
        else:
            selected_strategy_id = max(strategy_scores, key=lambda k: strategy_scores[k]['total_score'])
            selected_strategy = self.registry.get_strategy(selected_strategy_id)
        
        # Check if model is available for selected strategy
        model_key = selected_strategy.model_key
        fallback_used = False
        fallback_reason = None
        
        if model_key and model_key not in self.model_registry:
            # Model not available, use fallback
            fallback = selected_strategy.fallback_strategy or self.registry.get_fallback_strategy()
            if fallback:
                fallback_used = True
                fallback_reason = f"Model {model_key} not available"
                selected_strategy = fallback
                # Update selected strategy ID
                for sid, strategy in self.registry.all_strategies().items():
                    if strategy == fallback:
                        selected_strategy_id = sid
                        break
        
        # Record the selection
        selection_record = {
            'timestamp': datetime.now().isoformat(),
            'game_id': game_data.get('game_id'),
            'selected_strategy_id': selected_strategy_id,
            'context': context,
            'strategy_scores': strategy_scores,
            'fallback_used': fallback_used,
            'fallback_reason': fallback_reason
        }
        self.selection_history.append(selection_record)
        
        # Increment usage count
        selected_strategy.usage_count += 1
        
        # Return selection result
        return {
            'strategy_id': selected_strategy_id,
            'strategy': selected_strategy,
            'context': context,
            'selection_record': selection_record
        }
    
    def execute_prediction(self, 
                        game_data: Dict[str, Any],
                        team_avgs: Dict[str, float] = None) -> Dict[str, Any]:
        """
        Execute prediction for a game using the selected strategy.
        
        Args:
            game_data: Dictionary with game information
            team_avgs: Dictionary of team scoring averages
            
        Returns:
            Dict with prediction results and metadata
        """
        # Select appropriate strategy
        strategy_selection = self.select_strategy(game_data)
        strategy_id = strategy_selection['strategy_id']
        strategy = strategy_selection['strategy']
        context = strategy_selection['context']
        
        # Track execution time
        start_time = time.time()
        
        try:
            # Get relevant model if needed
            model = None
            if strategy.model_key is not None and self.model_registry is not None:
                model = self.model_registry.get(strategy.model_key)
            
            # Execute prediction
            prediction_result = strategy.predict(game_data, model, team_avgs)
            
            # Track execution time
            execution_time = time.time() - start_time
            
            # Add metadata to result
            prediction_result.update({
                'strategy_id': strategy_id,
                'strategy_name': strategy.name,
                'strategy_description': strategy.description,
                'context': context,
                'execution_time': execution_time,
                'prediction_timestamp': datetime.now().isoformat()
            })
            
            # Return the prediction with all metadata
            return prediction_result
            
        except Exception as e:
            # Error during execution
            error_msg = str(e)
            print(f"Error executing strategy {strategy_id}: {error_msg}")
            
            # Try to use fallback strategy
            fallback = strategy.fallback_strategy or self.registry.get_fallback_strategy()
            
            if fallback:
                try:
                    print(f"Using fallback strategy: {fallback.name}")
                    
                    # Get model for fallback if needed
                    fallback_model = None
                    if fallback.model_key is not None and self.model_registry is not None:
                        fallback_model = self.model_registry.get(fallback.model_key)
                    
                    # Execute fallback prediction
                    fallback_result = fallback.predict(game_data, fallback_model, team_avgs)
                    
                    # Get fallback strategy ID
                    fallback_id = "unknown_fallback"
                    for sid, s in self.registry.all_strategies().items():
                        if s == fallback:
                            fallback_id = sid
                            break
                    
                    # Track execution time
                    execution_time = time.time() - start_time
                    
                    # Add metadata about fallback
                    fallback_result.update({
                        'strategy_id': fallback_id,
                        'strategy_name': fallback.name,
                        'original_strategy_id': strategy_id,
                        'fallback_reason': error_msg,
                        'context': context,
                        'execution_time': execution_time,
                        'prediction_timestamp': datetime.now().isoformat()
                    })
                    
                    # Increment usage count for fallback
                    fallback.usage_count += 1
                    
                    return fallback_result
                    
                except Exception as fallback_error:
                    # Both original and fallback failed
                    print(f"Fallback strategy also failed: {str(fallback_error)}")
                    return {
                        'game_id': game_data.get('game_id'),
                        'error': error_msg,
                        'fallback_error': str(fallback_error),
                        'execution_time': time.time() - start_time
                    }
            
            # No fallback available
            return {
                'game_id': game_data.get('game_id'),
                'error': error_msg,
                'execution_time': time.time() - start_time
            }
    
    def update_strategy_metrics(self, actual_results: Dict[str, Dict[str, Any]]):
        """
        Update strategy metrics based on actual game results.
        
        Args:
            actual_results: Dictionary mapping game_id to actual results
        """
        # Update metrics in the collector
        self.metrics_collector.update_strategy_metrics(self.registry)
    
    def get_error_summary(self) -> Dict[str, Any]:
        """Get summary of prediction errors."""
        return self.metrics_collector.get_error_summary()
    
    def visualize_performance(self):
        """Visualize strategy performance metrics."""
        self.metrics_collector.visualize_error_metrics()


# Setup and demonstration 
def setup_prediction_system():
    """Set up the prediction system with registered strategies."""
    # Create registry and register strategies
    registry = StrategyRegistry()
    
    # Create and register strategies
    registry.register("early_quarters", EarlyQuartersStrategy(), "basic_extrapolation")
    registry.register("momentum", MomentumStrategy(), "late_quarters")
    registry.register("late_quarters", LateQuartersStrategy(), "basic_extrapolation")
    registry.register("blowout", BlowoutStrategy(), "basic_extrapolation")
    registry.register("close_game", CloseGameStrategy(), "late_quarters")
    registry.register("basic_extrapolation", BasicExtrapolationStrategy())
    
    # Create strategy selector
    selector = StrategySelector(registry)
    
    return selector

def demo_prediction_system():
    """Demonstrate the prediction system with sample games."""
    # Set up the system
    selector = setup_prediction_system()
    
    # Create sample game scenarios
    sample_games = [
        # Pregame
        {
            'game_id': 1001,
            'home_team': 'Lakers',
            'away_team': 'Celtics',
            'current_quarter': 0,
            'home_score': 0,
            'away_score': 0
        },
        # Early close game
        {
            'game_id': 1002,
            'home_team': 'Warriors',
            'away_team': 'Bucks',
            'current_quarter': 1,
            'home_q1': 28,
            'away_q1': 26,
            'home_score': 28,
            'away_score': 26,
            'cumulative_momentum': 0.2
        },
        # Blowout in Q3
        {
            'game_id': 1003,
            'home_team': 'Heat',
            'away_team': 'Nuggets',
            'current_quarter': 3,
            'home_q1': 32, 'home_q2': 35, 'home_q3': 30,
            'away_q1': 22, 'away_q2': 20, 'away_q3': 18,
            'home_score': 97,
            'away_score': 60,
            'cumulative_momentum': 0.7
        },
        # Close late game
        {
            'game_id': 1004,
            'home_team': 'Nets',
            'away_team': 'Suns',
            'current_quarter': 4,
            'home_q1': 28, 'home_q2': 25, 'home_q3': 24, 'home_q4': 18,
            'away_q1': 30, 'away_q2': 27, 'away_q3': 22, 'away_q4': 20,
            'home_score': 95,
            'away_score': 99,
            'cumulative_momentum': -0.3,
            'time_remaining': 3.5
        },
        # Comeback potential
        {
            'game_id': 1005,
            'home_team': 'Bulls',
            'away_team': 'Mavs',
            'current_quarter': 3,
            'home_q1': 20, 'home_q2': 22, 'home_q3': 32,
            'away_q1': 30, 'away_q2': 33, 'away_q3': 18,
            'home_score': 74,
            'away_score': 81,
            'cumulative_momentum': 0.6,  # Strong home momentum (comeback potential)
            'time_remaining': 0.5
        }
    ]
    
    # Team averages for demonstration
    team_avgs = {
        'Lakers': 112.5,
        'Celtics': 110.8,
        'Warriors': 116.2,
        'Bucks': 114.5,
        'Heat': 108.3,
        'Nuggets': 115.7,
        'Nets': 112.0,
        'Suns': 113.5,
        'Bulls': 108.0,
        'Mavs': 112.8
    }
    
    # Process each game
    print("Strategy Selection and Prediction Examples\n")
    print("=" * 80)
    
    for game in sample_games:
        # Analyze context
        context = selector.context_analyzer.analyze(game)
        
        print(f"\nGame {game['game_id']}: {game['home_team']} vs {game['away_team']} (Q{game['current_quarter']})")
        print(f"Primary Context: {context['primary_context']}")
        print(f"All Contexts: {', '.join(context['contexts'])}")
        
        # Select and execute strategy
        prediction = selector.execute_prediction(game, team_avgs)
        
        print(f"Selected Strategy: {prediction['strategy_name']} ({prediction['strategy_id']})")
        print(f"Strategy Description: {prediction['strategy_description']}")
        
        print("Prediction Results:")
        print(f"  Home Score: {prediction['predicted_home_score']:.1f}")
        print(f"  Away Score: {prediction['predicted_away_score']:.1f}")
        print(f"  Win Probability: {prediction['win_probability']:.1%}")
        print(f"  Confidence: {prediction['prediction_confidence']:.2f}")
        
        if 'execution_time' in prediction:
            print(f"  Execution Time: {prediction['execution_time']*1000:.2f} ms")
        
        print("-" * 80)
    
    # Simulate some validations for demo purposes
    print("\nSimulating validation with actual results...")
    
    # Mock actual results
    actual_results = {
        1001: {'game_id': 1001, 'home_score': 110, 'away_score': 105},
        1002: {'game_id': 1002, 'home_score': 115, 'away_score': 110},
        1003: {'game_id': 1003, 'home_score': 120, 'away_score': 85},
        1004: {'game_id': 1004, 'home_score': 105, 'away_score': 110},
        1005: {'game_id': 1005, 'home_score': 110, 'away_score': 105}  # Comeback completed
    }
    
    # Process each game again and update metrics
    for game in sample_games:
        # Execute prediction
        prediction = selector.execute_prediction(game, team_avgs)
        
        # Update error metrics
        game_id = game['game_id']
        if game_id in actual_results:
            selector.metrics_collector.add_error_record(prediction, actual_results[game_id])
    
    # Update strategy metrics
    selector.update_strategy_metrics(actual_results)
    
    # Visualize performance
    selector.visualize_performance()
    
    return {
        'selector': selector,
        'sample_games': sample_games,
        'error_summary': selector.get_error_summary()
    }

# Run the demonstration
if __name__ == "__main__":
    demo_results = demo_prediction_system()

In [ ]:
# Cell 12: Refactored Dashboard with Confidence Display and Error Handling

from IPython.display import clear_output, display, HTML
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import pandas as pd
import warnings
import traceback

class PredictionDashboard:
    """
    A dashboard class for visualizing game predictions with confidence metrics.
    Separates data preparation from visualization for better maintainability.
    """
    
    def __init__(self, use_plotly=False):
        """
        Initialize the dashboard.
        
        Args:
            use_plotly: If True, tries to use Plotly for interactive visualizations
        """
        self.use_plotly = use_plotly
        self._check_plotly_available()
        self.default_colors = {
            'home': '#1f77b4',  # blue
            'away': '#d62728',  # red
            'confidence': '#2ca02c',  # green
            'background': '#f8f9fa',
            'grid': '#dddddd'
        }
    
    def _check_plotly_available(self):
        """Check if Plotly is available and set flag accordingly."""
        if self.use_plotly:
            try:
                import plotly.graph_objects as go
                import plotly.express as px
                self.plotly_available = True
            except ImportError:
                warnings.warn("Plotly not available. Using Matplotlib instead.")
                self.plotly_available = False
        else:
            self.plotly_available = False
    
    def prepare_game_data(self, team_predictions, prediction_history=None):
        """
        Prepare game data for visualization.
        
        Args:
            team_predictions: DataFrame with team predictions
            prediction_history: Dictionary mapping game_id to list of predictions
            
        Returns:
            List of prepared game data dictionaries
        """
        if team_predictions is None or team_predictions.empty:
            return []
        
        prepared_games = []
        
        for _, game in team_predictions.iterrows():
            try:
                # Extract basic game information
                game_id = game.get('game_id', None)
                if game_id is None:
                    continue  # Skip games without ID
                
                home_team = game.get('home_team', 'Home')
                away_team = game.get('away_team', 'Away')
                current_quarter = int(game.get('current_quarter', 0))
                
                # Get current scores
                current_home_score = float(game.get('current_home_score', game.get('home_score', 0)) or 0)
                current_away_score = float(game.get('current_away_score', game.get('away_score', 0)) or 0)
                
                # Get predicted scores
                predicted_home_score = float(game.get('predicted_home_score', 0))
                predicted_away_score = float(game.get('predicted_away_score', 0))
                
                # Get confidence metrics if available
                confidence = float(game.get('prediction_confidence', 0.5))
                win_probability = float(game.get('win_probability', 0.5))
                
                # Calculate margins and totals
                current_margin = current_home_score - current_away_score
                predicted_margin = predicted_home_score - predicted_away_score
                current_total = current_home_score + current_away_score
                predicted_total = predicted_home_score + predicted_away_score
                
                # Get prediction history for this game if available
                game_history = None
                if prediction_history is not None and game_id in prediction_history:
                    game_history = prediction_history[game_id]
                
                # Prepare game data
                game_data = {
                    'game_id': game_id,
                    'home_team': home_team,
                    'away_team': away_team,
                    'current_quarter': current_quarter,
                    'current_scores': {
                        'home': current_home_score,
                        'away': current_away_score,
                        'margin': current_margin,
                        'total': current_total
                    },
                    'predicted_scores': {
                        'home': predicted_home_score,
                        'away': predicted_away_score,
                        'margin': predicted_margin,
                        'total': predicted_total
                    },
                    'metrics': {
                        'confidence': confidence,
                        'win_probability': win_probability
                    },
                    'history': game_history
                }
                
                prepared_games.append(game_data)
                
            except Exception as e:
                warnings.warn(f"Error preparing game data: {str(e)}")
                continue
        
        return prepared_games
    
    def create_live_dashboard(self, team_predictions, prediction_history=None, include_text=True):
        """
        Create a dashboard visualization of live game predictions with confidence metrics.
        
        Args:
            team_predictions: DataFrame with team predictions
            prediction_history: Dictionary mapping game_id to list of predictions
            include_text: Whether to include text summary in addition to visualization
            
        Returns:
            None (displays visualization in notebook)
        """
        try:
            # Clear previous output
            clear_output(wait=True)
            
            # Prepare data
            prepared_games = self.prepare_game_data(team_predictions, prediction_history)
            
            if not prepared_games:
                print("No live games to display.")
                return
            
            # Create visualizations
            if self.plotly_available:
                self._create_plotly_dashboard(prepared_games)
            else:
                self._create_matplotlib_dashboard(prepared_games)
            
            # Display text summary if requested
            if include_text:
                self._display_text_summary(prepared_games)
            
        except Exception as e:
            print(f"Error creating dashboard: {str(e)}")
            traceback.print_exc()
    
    def _create_matplotlib_dashboard(self, prepared_games):
        """Create dashboard using Matplotlib."""
        n_games = len(prepared_games)
        fig, axs = plt.subplots(n_games, 2, figsize=(15, 5*n_games))
        fig.suptitle(f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", fontsize=16)
        
        # Handle single game case
        if n_games == 1:
            axs = np.array([axs])
        
        for i, game in enumerate(prepared_games):
            # Game scores visualization
            ax_scores = axs[i, 0]
            self._plot_game_scores(ax_scores, game)
            
            # Prediction history visualization
            ax_history = axs[i, 1]
            self._plot_prediction_history(ax_history, game)
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
    
    def _plot_game_scores(self, ax, game):
        """Plot game scores in Matplotlib ax."""
        teams = [game['home_team'], game['away_team']]
        current_scores = [game['current_scores']['home'], game['current_scores']['away']]
        predicted_scores = [game['predicted_scores']['home'], game['predicted_scores']['away']]
        
        x = np.arange(len(teams))
        width = 0.35
        
        # Plot bars
        ax.bar(x - width/2, current_scores, width, label='Current', color=self.default_colors['home'])
        ax.bar(x + width/2, predicted_scores, width, label='Predicted Final', color=self.default_colors['away'])
        
        # Add labels and styling
        ax.set_xticks(x)
        ax.set_xticklabels(teams)
        ax.legend()
        
        confidence = int(game['metrics']['confidence'] * 100)
        quarter_text = 'PRE-GAME' if game['current_quarter'] == 0 else f"Q{game['current_quarter']}"
        ax.set_title(f"{game['home_team']} vs {game['away_team']} - {quarter_text} (Confidence: {confidence}%)")
        
        # Add value labels
        for j, v in enumerate(current_scores):
            ax.text(j - width/2, v + 1, str(int(v)), ha='center')
        for j, v in enumerate(predicted_scores):
            ax.text(j + width/2, v + 1, f"{v:.1f}", ha='center')
    
    def _plot_prediction_history(self, ax, game):
        """Plot prediction history in Matplotlib ax."""
        history = game.get('history')
        
        if history and len(history) > 1:
            # Extract data
            timestamps = range(len(history))
            home_scores = [h.get('predicted_home_score', 0) for h in history]
            away_scores = [h.get('predicted_away_score', 0) for h in history]
            
            # Draw lines
            ax.plot(timestamps, home_scores, 'o-', label=f"{game['home_team']} Final", 
                   marker='o', color=self.default_colors['home'])
            ax.plot(timestamps, away_scores, 's-', label=f"{game['away_team']} Final", 
                   marker='s', color=self.default_colors['away'])
            
            # Add win probability if available
            if all('win_probability' in h for h in history):
                win_probs = [h.get('win_probability', 0.5) for h in history]
                ax_twin = ax.twinx()
                ax_twin.plot(timestamps, win_probs, 'd-', label='Win Prob', 
                           color=self.default_colors['confidence'])
                ax_twin.set_ylim(0, 1)
                ax_twin.set_ylabel('Win Probability')
                
                # Add legend items from both axes
                lines1, labels1 = ax.get_legend_handles_labels()
                lines2, labels2 = ax_twin.get_legend_handles_labels()
                ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
            else:
                ax.legend()
            
            # Styling
            ax.set_title("Prediction Evolution")
            ax.set_xlabel("Update Index")
            ax.set_ylabel("Predicted Score")
            ax.grid(alpha=0.3)
            
        else:
            ax.text(0.5, 0.5, "Not enough prediction history yet", 
                   ha='center', va='center', transform=ax.transAxes)
    
    def _create_plotly_dashboard(self, prepared_games):
        """Create dashboard using Plotly for interactive visualization."""
        try:
            import plotly.graph_objects as go
            import plotly.subplots as sp
            
            # Create a 2-column layout for each game
            rows = len(prepared_games)
            fig = sp.make_subplots(
                rows=rows, 
                cols=2,
                subplot_titles=[f"{g['home_team']} vs {g['away_team']}" for g in prepared_games for _ in range(2)],
                specs=[[{"type": "bar"}, {"type": "scatter"}] for _ in range(rows)]
            )
            
            for i, game in enumerate(prepared_games):
                # Add 1 to i since plotly is 1-indexed
                row = i + 1
                
                # Game scores chart
                teams = [game['home_team'], game['away_team']]
                current_scores = [game['current_scores']['home'], game['current_scores']['away']]
                predicted_scores = [game['predicted_scores']['home'], game['predicted_scores']['away']]
                
                fig.add_trace(
                    go.Bar(
                        x=teams,
                        y=current_scores,
                        name='Current',
                        marker_color='royalblue',
                        hovertemplate='%{x}: %{y}<br><extra>Current Score</extra>',
                        text=current_scores,
                        textposition='auto'
                    ),
                    row=row, col=1
                )
                
                fig.add_trace(
                    go.Bar(
                        x=teams,
                        y=predicted_scores,
                        name='Predicted Final',
                        marker_color='firebrick',
                        hovertemplate='%{x}: %{y:.1f}<br><extra>Predicted Final</extra>',
                        text=[f"{s:.1f}" for s in predicted_scores],
                        textposition='auto'
                    ),
                    row=row, col=1
                )
                
                # Add confidence indicator
                confidence = game['metrics']['confidence'] * 100
                quarter_text = 'PRE-GAME' if game['current_quarter'] == 0 else f"Q{game['current_quarter']}"
                fig.update_xaxes(title_text=f"{quarter_text} (Confidence: {confidence:.0f}%)", row=row, col=1)
                
                # Prediction history chart
                history = game.get('history')
                if history and len(history) > 1:
                    # Extract data
                    timestamps = list(range(len(history)))
                    home_scores = [h.get('predicted_home_score', 0) for h in history]
                    away_scores = [h.get('predicted_away_score', 0) for h in history]
                    
                    fig.add_trace(
                        go.Scatter(
                            x=timestamps,
                            y=home_scores,
                            mode='lines+markers',
                            name=f"{game['home_team']} Prediction",
                            marker_color='royalblue',
                            hovertemplate='Update %{x}: %{y:.1f}<br><extra>Home Prediction</extra>'
                        ),
                        row=row, col=2
                    )
                    
                    fig.add_trace(
                        go.Scatter(
                            x=timestamps,
                            y=away_scores,
                            mode='lines+markers',
                            name=f"{game['away_team']} Prediction",
                            marker_color='firebrick',
                            hovertemplate='Update %{x}: %{y:.1f}<br><extra>Away Prediction</extra>'
                        ),
                        row=row, col=2
                    )
                    
                    # Add win probability if available
                    if all('win_probability' in h for h in history):
                        win_probs = [h.get('win_probability', 0.5) * 100 for h in history]
                        
                        fig.add_trace(
                            go.Scatter(
                                x=timestamps,
                                y=win_probs,
                                mode='lines+markers',
                                name='Win Probability',
                                marker_color='green',
                                yaxis='y2',
                                hovertemplate='Update %{x}: %{y:.1f}%<br><extra>Win Probability</extra>'
                            ),
                            row=row, col=2
                        )
                        
                        # Add secondary y-axis for win probability
                        fig.update_layout(
                            yaxis2=dict(
                                title='Win Probability (%)',
                                titlefont=dict(color='green'),
                                tickfont=dict(color='green'),
                                anchor='x',
                                overlaying='y',
                                side='right',
                                range=[0, 100]
                            )
                        )
                    
                    fig.update_xaxes(title_text="Update", row=row, col=2)
                    fig.update_yaxes(title_text="Predicted Score", row=row, col=2)
                    
                else:
                    # Add placeholder text for insufficient history
                    fig.add_trace(
                        go.Scatter(
                            x=[0],
                            y=[0],
                            mode='text',
                            text=['Not enough prediction history yet'],
                            textposition='middle center',
                            hoverinfo='none'
                        ),
                        row=row, col=2
                    )
                    
            
            # Update layout
            fig.update_layout(
                title=f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
                showlegend=False,
                height=300 * rows,
                template='plotly_white'
            )
            
            fig.show()
            
        except Exception as e:
            warnings.warn(f"Error creating Plotly dashboard: {str(e)}. Falling back to Matplotlib.")
            traceback.print_exc()
            # Fallback to matplotlib
            self._create_matplotlib_dashboard(prepared_games)
    
    def _display_text_summary(self, prepared_games):
        """Display text summary of game predictions."""
        print(f"Live Game Predictions - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*80)
        
        for game in prepared_games:
            quarter_text = "Pre-game" if game['current_quarter'] == 0 else f"Quarter {game['current_quarter']}"
            print(f"\n{game['home_team']} vs {game['away_team']} - {quarter_text}")
            print(f"Current Score: {game['home_team']} {game['current_scores']['home']} - {game['away_team']} {game['current_scores']['away']}")
            print(f"Predicted Final: {game['home_team']} {game['predicted_scores']['home']:.1f} - {game['away_team']} {game['predicted_scores']['away']:.1f}")
            
            win_prob = game['metrics']['win_probability']
            favorite = game['home_team'] if win_prob > 0.5 else game['away_team']
            win_pct = max(win_prob, 1-win_prob) * 100
            print(f"Win Probability: {favorite} {win_pct:.1f}%")
            
            confidence = game['metrics']['confidence'] * 100
            print(f"Prediction Confidence: {confidence:.0f}%")
            
            print("-"*80)
    
    def get_html_dashboard(self, team_predictions, prediction_history=None):
        """
        Generate HTML for the dashboard (for web display).
        
        Args:
            team_predictions: DataFrame with team predictions
            prediction_history: Dictionary mapping game_id to list of predictions
            
        Returns:
            HTML string for the dashboard
        """
        try:
            # Prepare data
            prepared_games = self.prepare_game_data(team_predictions, prediction_history)
            
            if not prepared_games:
                return "<p>No live games to display.</p>"
            
            # Build HTML
            html = f"""
            <div class="nba-prediction-dashboard">
                <style>
                    .nba-prediction-dashboard {{
                        font-family: Arial, sans-serif;
                        max-width: 1200px;
                        margin: 0 auto;
                        padding: 10px;
                    }}
                    .dashboard-header {{
                        text-align: center;
                        margin-bottom: 20px;
                    }}
                    .games-container {{
                        display: flex;
                        flex-wrap: wrap;
                        gap: 20px;
                        justify-content: center;
                    }}
                    .game-card {{
                        border: 1px solid #ddd;
                        border-radius: 8px;
                        padding: 15px;
                        width: 350px;
                        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
                        background-color: #fff;
                    }}
                    .game-header {{
                        border-bottom: 1px solid #eee;
                        padding-bottom: 10px;
                        margin-bottom: 10px;
                        font-weight: bold;
                        font-size: 1.2em;
                    }}
                    .score-display {{
                        display: flex;
                        justify-content: space-between;
                        margin: 15px 0;
                    }}
                    .team-score {{
                        text-align: center;
                        width: 45%;
                    }}
                    .team-name {{
                        font-weight: bold;
                        margin-bottom: 5px;
                    }}
                    .current-score {{
                        font-size: 1.8em;
                        font-weight: bold;
                    }}
                    .predicted-score {{
                        font-size: 1.4em;
                        color: #666;
                    }}
                    .vs-divider {{
                        display: flex;
                        align-items: center;
                        justify-content: center;
                        width: 10%;
                    }}
                    .confidence-bar {{
                        margin: 15px 0;
                        background-color: #f0f0f0;
                        border-radius: 4px;
                        height: 8px;
                    }}
                    .confidence-level {{
                        background-color: #4CAF50;
                        height: 100%;
                        border-radius: 4px;
                    }}
                    .metrics {{
                        display: flex;
                        justify-content: space-between;
                        margin-top: 15px;
                        font-size: 0.9em;
                        color: #666;
                    }}
                    .refresh-time {{
                        text-align: right;
                        font-size: 0.8em;
                        color: #999;
                        margin-top: 10px;
                    }}
                </style>
                
                <div class="dashboard-header">
                    <h2>NBA Game Predictions</h2>
                    <p>Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                </div>
                
                <div class="games-container">
            """
            
            # Add game cards
            for game in prepared_games:
                # Determine game status text
                if game['current_quarter'] == 0:
                    status_text = "PRE-GAME"
                else:
                    status_text = f"QUARTER {game['current_quarter']}"
                
                # Determine win probability text
                win_prob = game['metrics']['win_probability']
                favorite = game['home_team'] if win_prob > 0.5 else game['away_team']
                win_pct = max(win_prob, 1-win_prob) * 100
                
                # Add game card HTML
                html += f"""
                    <div class="game-card">
                        <div class="game-header">
                            {game['home_team']} vs {game['away_team']}
                            <span style="float: right; background-color: #007bff; color: white; padding: 2px 6px; border-radius: 4px; font-size: 0.8em;">
                                {status_text}
                            </span>
                        </div>
                        
                        <div class="score-display">
                            <div class="team-score">
                                <div class="team-name">{game['home_team']}</div>
                                <div class="current-score">{int(game['current_scores']['home'])}</div>
                                <div class="predicted-score">Predicted: {game['predicted_scores']['home']:.1f}</div>
                            </div>
                            
                            <div class="vs-divider">VS</div>
                            
                            <div class="team-score">
                                <div class="team-name">{game['away_team']}</div>
                                <div class="current-score">{int(game['current_scores']['away'])}</div>
                                <div class="predicted-score">Predicted: {game['predicted_scores']['away']:.1f}</div>
                            </div>
                        </div>
                        
                        <div>
                            <div style="font-size: 0.9em; margin-bottom: 5px;">Win Probability</div>
                            <div class="confidence-bar">
                                <div class="confidence-level" style="width: {win_pct}%;"></div>
                            </div>
                            <div style="text-align: center; margin-top: 5px; font-size: 0.9em;">
                                {favorite}: {win_pct:.1f}%
                            </div>
                        </div>
                        
                        <div class="metrics">
                            <div>Prediction Confidence: {game['metrics']['confidence']*100:.0f}%</div>
                            <div>Margin: {abs(game['predicted_scores']['margin']):.1f}</div>
                        </div>
                        
                        <div class="refresh-time">
                            Last update: {datetime.now().strftime('%H:%M:%S')}
                        </div>
                    </div>
                """
            
            # Close containers
            html += """
                </div>
            </div>
            """
            
            return html
            
        except Exception as e:
            error_html = f"""
            <div style="border: 1px solid #f44336; padding: 10px; border-radius: 4px; background-color: #ffebee;">
                <h3 style="color: #f44336;">Error Generating Dashboard</h3>
                <p>{str(e)}</p>
            </div>
            """
            return error_html


# Example usage:
def demo_prediction_dashboard():
    """Demonstrate the refactored prediction dashboard with sample data."""
    # Create sample prediction data
    sample_games = pd.DataFrame([
        {
            'game_id': 'game1',
            'home_team': 'Lakers',
            'away_team': 'Celtics',
            'current_quarter': 3,
            'home_score': 85,
            'away_score': 82,
            'predicted_home_score': 112.5,
            'predicted_away_score': 108.2,
            'prediction_confidence': 0.85,
            'win_probability': 0.67
        },
        {
            'game_id': 'game2',
            'home_team': 'Warriors',
            'away_team': 'Bucks',
            'current_quarter': 2,
            'home_score': 58,
            'away_score': 61,
            'predicted_home_score': 118.3,
            'predicted_away_score': 121.7,
            'prediction_confidence': 0.72,
            'win_probability': 0.42
        }
    ])
    
    # Create sample prediction history
    sample_history = {
        'game1': [
            {
                'timestamp': '2023-03-13T19:30:00',
                'current_quarter': 1,
                'current_home_score': 28,
                'current_away_score': 25,
                'predicted_home_score': 107.2,
                'predicted_away_score': 102.6,
                'win_probability': 0.62
            },
            {
                'timestamp': '2023-03-13T20:00:00',
                'current_quarter': 2,
                'current_home_score': 56,
                'current_away_score': 52,
                'predicted_home_score': 110.3,
                'predicted_away_score': 106.1,
                'win_probability': 0.64
            },
            {
                'timestamp': '2023-03-13T20:30:00',
                'current_quarter': 3,
                'current_home_score': 85,
                'current_away_score': 82,
                'predicted_home_score': 112.5,
                'predicted_away_score': 108.2,
                'win_probability': 0.67
            }
        ],
        'game2': [
            {
                'timestamp': '2023-03-13T19:45:00',
                'current_quarter': 1,
                'current_home_score': 29,
                'current_away_score': 32,
                'predicted_home_score': 115.5,
                'predicted_away_score': 119.8,
                'win_probability': 0.38
            },
            {
                'timestamp': '2023-03-13T20:15:00',
                'current_quarter': 2,
                'current_home_score': 58,
                'current_away_score': 61,
                'predicted_home_score': 118.3,
                'predicted_away_score': 121.7,
                'win_probability': 0.42
            }
        ]
    }
    
    # Create dashboard
    dashboard = PredictionDashboard(use_plotly=True)
    
    # Display dashboard
    print("Creating dashboard visualization...")
    dashboard.create_live_dashboard(sample_games, sample_history)
    
    # Generate HTML version
    html_dashboard = dashboard.get_html_dashboard(sample_games, sample_history)
    print(f"\nGenerated HTML dashboard ({len(html_dashboard)} characters)")
    
    # Show HTML in notebook
    from IPython.display import HTML
    display(HTML("<h3>HTML Dashboard Preview:</h3>"))
    display(HTML(html_dashboard))
    
    return {
        'dashboard': dashboard,
        'sample_games': sample_games,
        'sample_history': sample_history
    }

# Run the demonstration if this cell is executed directly
if __name__ == '__main__':
    demo_results = demo_prediction_dashboard()

In [ ]:
# Cell 15: Enhanced Monitoring Function with Correct Team Matching

from models.dynamic_recommendation import generate_recommendations
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import traceback
import pytz
import os
import joblib
from supabase import create_client

# Make the official schedule globally available
OFFICIAL_SCHEDULE = None

def normalize_team_name(name):
    """Normalize team names for consistent matching"""
    if not name:
        return ""

    # Convert to string and lowercase
    name = str(name).lower().strip()

    # Standard team name mappings
    mappings = {
        'lakers': 'los angeles lakers',
        'la lakers': 'los angeles lakers',
        'clippers': 'los angeles clippers',
        'la clippers': 'los angeles clippers',
        'blazers': 'portland trail blazers',
        'sixers': 'philadelphia 76ers',
        'philly': 'philadelphia 76ers',
        'knicks': 'new york knicks',
        'ny knicks': 'new york knicks',
        'nets': 'brooklyn nets',
        'mavs': 'dallas mavericks',
        'cavs': 'cleveland cavaliers',
        'wolves': 'minnesota timberwolves',
        't-wolves': 'minnesota timberwolves',
        'celts': 'boston celtics',
        'pels': 'new orleans pelicans',
        'warriors': 'golden state warriors',
        'gsw': 'golden state warriors',
        'heat': 'miami heat',
        'bulls': 'chicago bulls',
        'hawks': 'atlanta hawks',
        'suns': 'phoenix suns',
        'bucks': 'milwaukee bucks',
        'jazz': 'utah jazz',
        'nuggets': 'denver nuggets',
        'rockets': 'houston rockets',
        'pacers': 'indiana pacers',
        'spurs': 'san antonio spurs',
        'kings': 'sacramento kings',
        'magic': 'orlando magic',
        'wizards': 'washington wizards',
        'raptors': 'toronto raptors',
        'thunder': 'oklahoma city thunder',
        'okc': 'oklahoma city thunder',
        'pistons': 'detroit pistons',
        'hornets': 'charlotte hornets',
        'grizzlies': 'memphis grizzlies',
        'grizz': 'memphis grizzlies'
    }

    # Check if the name is in mappings
    for key, value in mappings.items():
        if key in name:
            return value

    return name

def load_and_cache_official_schedule():
    """Loads and caches the official NBA schedule for today"""
    # Get today's date in Pacific Time (NBA standard timezone)
    pacific_tz = pytz.timezone('America/Los_Angeles')
    today = datetime.now(pacific_tz).strftime('%Y-%m-%d')

    try:
        # Try to fetch from database
        response = supabase.table("nba_game_schedule").select("*").eq("game_date", today).execute()

        if response.data:
            schedule_df = pd.DataFrame(response.data)
            print(f"Loaded {len(schedule_df)} games from official schedule for {today}")

            # Add normalized team names for better matching
            schedule_df['home_team_norm'] = schedule_df['home_team'].apply(normalize_team_name)
            schedule_df['away_team_norm'] = schedule_df['away_team'].apply(normalize_team_name)

            global OFFICIAL_SCHEDULE
            OFFICIAL_SCHEDULE = schedule_df
            
            # Print the matchups
            for _, game in schedule_df.iterrows():
                print(f"  - {game['home_team']} vs {game['away_team']}")
            
            return schedule_df
        else:
            print(f"No scheduled games found for today ({today})")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error loading official schedule: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def fetch_live_games_with_schedule_matching():
    """Fetch live games and match against the official schedule"""
    # First load the official schedule
    global OFFICIAL_SCHEDULE
    if OFFICIAL_SCHEDULE is None or OFFICIAL_SCHEDULE.empty:
        OFFICIAL_SCHEDULE = load_and_cache_official_schedule()

    try:
        # Get current date in Pacific Time
        pacific_tz = pytz.timezone('America/Los_Angeles')
        today_pacific = datetime.now(pacific_tz).strftime('%Y-%m-%d')

        print(f"Fetching live games for {today_pacific} (Pacific Time)")

        response = supabase.table("nba_live_game_stats").select("*").execute()
        
        if not response.data:
            print(f"No live games found. Checking for scheduled games...")
            
            # If there are scheduled games but no live games, convert schedule to live format
            if not OFFICIAL_SCHEDULE.empty:
                print("No live games found, but scheduled games exist. Creating pre-game entries.")
                live_games_df = convert_scheduled_to_live_format(OFFICIAL_SCHEDULE)
                return live_games_df
            else:
                print("No live or scheduled games found.")
                return pd.DataFrame()

        live_df = pd.DataFrame(response.data)
        
        # Filter for today's games if we have a date column
        if 'game_date' in live_df.columns:
            # Convert to datetime to handle different date formats
            live_df['game_date'] = pd.to_datetime(live_df['game_date'])
            today_dt = pd.to_datetime(today_pacific)
            live_df = live_df[live_df['game_date'].dt.date == today_dt.date()]
        
        print(f"Found {len(live_df)} live game records")

        # Validate and clean game data
        live_df = validate_game_data(live_df)

        # Match against the schedule if available
        if not OFFICIAL_SCHEDULE.empty:
            matched_df = match_teams_to_schedule(live_df, OFFICIAL_SCHEDULE)
            print(f"Matched {matched_df['matched_to_schedule'].sum()} of {len(matched_df)} games to schedule")
            return matched_df
        else:
            # If no schedule, just return the validated live data
            return live_df

    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()

        # If live data fails but we have the schedule, use that
        if not OFFICIAL_SCHEDULE.empty:
            print("Using schedule data as fallback")
            return convert_scheduled_to_live_format(OFFICIAL_SCHEDULE)

        return pd.DataFrame()

def find_recent_games_for_testing():
    """Find recent completed games to use for testing the prediction model"""
    print("No live games found. Fetching recent completed games for testing...")

    # Get dates to try, in order of preference
    try:
        dates_to_try = []
        today = datetime.now()

        # Try yesterday first, then previous days
        for i in range(1, 5):  # Try up to 4 previous days
            date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
            dates_to_try.append(date)

        # Try each date in sequence
        for test_date in dates_to_try:
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .eq("game_date", test_date)\
                .limit(5).execute()

            historical_games = response.data
            if historical_games:
                print(f"Found {len(historical_games)} games from {test_date} for testing.")

                # Simulate these as 'live' games by setting them to random quarters
                import random

                live_games = []
                for game in historical_games:
                    # Randomly select a quarter for simulation (1-4)
                    sim_quarter = random.randint(1, 4)

                    # Create a simulated live game where we only know scores up to the simulated quarter
                    sim_game = {
                        'game_id': game['game_id'],
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': sim_quarter,
                        'simulated_quarter': sim_quarter,  # Mark which quarter we're simulating
                        'matched_to_schedule': True  # Pretend it's matched
                    }

                    # Add quarter scores up to the simulated quarter
                    for q in range(1, 5):
                        q_field_home = f'home_q{q}'
                        q_field_away = f'away_q{q}'

                        if q <= sim_quarter and q_field_home in game and q_field_away in game:
                            sim_game[q_field_home] = game[q_field_home]
                            sim_game[q_field_away] = game[q_field_away]
                        else:
                            sim_game[q_field_home] = 0
                            sim_game[q_field_away] = 0

                    # Calculate the current score based on quarters we "know"
                    sim_game['home_score'] = sum([
                        float(sim_game.get(f'home_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                    ])
                    sim_game['away_score'] = sum([
                        float(sim_game.get(f'away_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                    ])

                    # Save the actual final scores for validation
                    sim_game['actual_final_home'] = game['home_score']
                    sim_game['actual_final_away'] = game['away_score']

                    live_games.append(sim_game)

                return pd.DataFrame(live_games)

        print("No recent games found for testing.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error finding recent games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def get_team_rolling_averages(days_lookback=60):
    """
    Retrieves the rolling scoring average for each team from historical data.
    
    Args:
        days_lookback: Number of days to look back for calculating the average (default: 60)
        
    Returns:
        Dictionary mapping team names to their rolling scoring average
    """
    try:
        # Calculate the date threshold
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        
        # Fetch recent historical game data
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        historical_data = response.data
        
        if not historical_data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return {}
        
        df = pd.DataFrame(historical_data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        df = df.sort_values('game_date')
        
        # Initialize dictionary for team averages
        team_avgs = {}
        
        # Get unique teams
        all_teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
        
        for team in all_teams:
            # Get home games where team is home
            home_games = df[df['home_team'] == team][['game_date', 'home_score']].rename(
                columns={'home_score': 'score'})
            
            # Get away games where team is away
            away_games = df[df['away_team'] == team][['game_date', 'away_score']].rename(
                columns={'away_score': 'score'})
            
            # Combine all games
            team_games = pd.concat([home_games, away_games]).sort_values('game_date')
            
            if not team_games.empty:
                # Calculate recent average (last 5 games if available)
                recent_games = team_games.tail(5)
                team_avgs[team] = recent_games['score'].mean()
            else:
                # Fallback to a reasonable default
                team_avgs[team] = 105.0  # NBA average is approximately 100-110 points per game
        
        return team_avgs
    except Exception as e:
        print(f"Error getting team rolling averages: {e}")
        return {}  # Return empty dict on error

def get_rest_data(team_name, game_date):
    """
    Calculates rest days for a team before a specific game date.
    
    Args:
        team_name: Name of the team
        game_date: Date of the current game
        
    Returns:
        Dictionary with rest days and back-to-back status
    """
    try:
        # Ensure game_date is datetime
        if isinstance(game_date, str):
            game_date = pd.to_datetime(game_date)
        
        # Try to find previous game with a simple query approach
        try:
            # First attempt - check home games
            home_response = supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("home_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
                
            # Second attempt - check away games
            away_response = supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("away_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
            
            # Combine results and find most recent
            all_games = home_response.data + away_response.data
            if not all_games:
                # If no previous game found, return default
                return {
                    'rest_days': 3,  # Default if no history
                    'is_back_to_back': False
                }
                
            # Sort by date (most recent first)
            all_games.sort(key=lambda x: x['game_date'], reverse=True)
            prev_game_date = pd.to_datetime(all_games[0]['game_date'])
            
        except Exception as e:
            print(f"Error in first query approach: {e}")
            # Try with normalized team name as fallback
            norm_team = normalize_team_name(team_name)
            if norm_team != team_name:
                try:
                    # Look for the team's previous game with normalized name
                    home_response = supabase.table("nba_historical_game_stats")\
                        .select("game_date")\
                        .eq("home_team", norm_team)\
                        .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                        .order("game_date", desc=True)\
                        .limit(1)\
                        .execute()
                        
                    away_response = supabase.table("nba_historical_game_stats")\
                        .select("game_date")\
                        .eq("away_team", norm_team)\
                        .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                        .order("game_date", desc=True)\
                        .limit(1)\
                        .execute()
                    
                    all_games = home_response.data + away_response.data
                    if not all_games:
                        return {
                            'rest_days': 3,
                            'is_back_to_back': False
                        }
                        
                    all_games.sort(key=lambda x: x['game_date'], reverse=True)
                    prev_game_date = pd.to_datetime(all_games[0]['game_date'])
                except Exception as e2:
                    print(f"Error in fallback query: {e2}")
                    return {
                        'rest_days': 2,  # Default if all queries fail
                        'is_back_to_back': False
                    }
            else:
                return {
                    'rest_days': 2,
                    'is_back_to_back': False
                }
        
        # Calculate days between games
        rest_days = (game_date - prev_game_date).days
        
        # Cap to reasonable values
        rest_days = max(min(rest_days, 10), 0)
        
        return {
            'rest_days': rest_days,
            'is_back_to_back': rest_days <= 1
        }
    except Exception as e:
        print(f"Error getting rest data for {team_name}: {e}")
        # Return default values on error
        return {
            'rest_days': 2,  # Reasonable default
            'is_back_to_back': False
        }

def get_previous_matchup_diff(home_team, away_team, max_lookback=5):
    """Gets the point differential from previous matchups between two teams."""
    try:
        # Use separate queries for home and away configurations to avoid syntax issues
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        
        # Combine results
        home_matchups = response_home.data
        away_matchups = response_away.data
        matchups = home_matchups + away_matchups
        
        # Sort by date (most recent first)
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
        
        if not matchups:
            return 0
        
        # Calculate point differential from home team perspective
        differentials = []
        for game in matchups:
            if game['home_team'] == home_team and game['away_team'] == away_team:
                diff = game['home_score'] - game['away_score']
            elif game['home_team'] == away_team and game['away_team'] == home_team:
                diff = game['away_score'] - game['home_score']
            else:
                continue
            differentials.append(diff)
        
        # Cap extreme values
        avg_diff = sum(differentials) / len(differentials) if differentials else 0
        return max(min(avg_diff, 15), -15)  # Cap at +/- 15 points
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        return 0

def convert_scheduled_to_live_format(schedule_df):
    # If you actually need to convert scheduled games to a live format, re-implement
    # For now, return an empty or minimal DataFrame
    return pd.DataFrame()
    
    for _, game in schedule_df.iterrows():
        # Create a basic live game structure
        live_game = {
            'game_id': game['game_id'],
            'home_team': game['home_team'],
            'away_team': game['away_team'],
            'game_date': game['game_date'],
            'matched_to_schedule': True,
            'current_quarter': 0,  # Pre-game
            'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
            'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
            'home_score': 0,
            'away_score': 0
        }
        live_format.append(live_game)
    
    return pd.DataFrame(live_format)

def validate_game_data(df):
    """Validate and clean game data"""
    # Make a copy to avoid modifying the original
    df = df.copy()
    
    # Convert date fields to datetime
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
    
    # Ensure quarter fields are numeric
    for q in range(1, 5):
        home_q = f'home_q{q}'
        away_q = f'away_q{q}'
        
        if home_q in df.columns:
            df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
        if away_q in df.columns:
            df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
    
    # Calculate current quarter based on non-zero quarter scores
    df['current_quarter'] = 0
    for idx, row in df.iterrows():
        max_quarter = 0
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if ((home_q in row and pd.notnull(row[home_q]) and row[home_q] > 0) or 
                (away_q in row and pd.notnull(row[away_q]) and row[away_q] > 0)):
                max_quarter = q
        
        df.at[idx, 'current_quarter'] = max_quarter
    
    # Ensure team names are strings
    if 'home_team' in df.columns:
        df['home_team'] = df['home_team'].astype(str)
    if 'away_team' in df.columns:
        df['away_team'] = df['away_team'].astype(str)
    
    # Calculate current score as sum of quarters
    if 'home_score' not in df.columns:
        df['home_score'] = 0
    if 'away_score' not in df.columns:
        df['away_score'] = 0
        
    for idx, row in df.iterrows():
        current_q = int(row.get('current_quarter', 0) or 0)
        
        home_score = 0
        away_score = 0
        for q in range(1, current_q + 1):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if home_q in row:
                home_score += float(row[home_q] or 0)
            if away_q in row:
                away_score += float(row[away_q] or 0)
        
        df.at[idx, 'home_score'] = home_score
        df.at[idx, 'away_score'] = away_score
    
    # Calculate score ratio
    df['score_ratio'] = 0.5  # Default to even
    for idx, row in df.iterrows():
        home_score = float(row.get('home_score', 0) or 0)
        away_score = float(row.get('away_score', 0) or 0)
        total = home_score + away_score
        
        if total > 0:
            df.at[idx, 'score_ratio'] = home_score / total
    
    # Initialize matched_to_schedule column if not present
    if 'matched_to_schedule' not in df.columns:
        df['matched_to_schedule'] = False
    
    return df

def match_teams_to_schedule(live_df, schedule_df):
    # If you actually need to match team names, re-implement the real logic
    # For now, just return live_df as-is or with a 'verified' column
    live_df['verified'] = False
    return live_df
    # Copy to avoid modifying the original
    live_df = live_df.copy()
    
    # Add normalized team names for better matching
    live_df['home_team_norm'] = live_df['home_team'].apply(normalize_team_name)
    live_df['away_team_norm'] = live_df['away_team'].apply(normalize_team_name)
    
    # Make sure schedule has normalized teams too
    if 'home_team_norm' not in schedule_df.columns:
        schedule_df = schedule_df.copy()
        schedule_df['home_team_norm'] = schedule_df['home_team'].apply(normalize_team_name)
        schedule_df['away_team_norm'] = schedule_df['away_team'].apply(normalize_team_name)
    
    # Try to match by exact normalized team names
    for live_idx, live_game in live_df.iterrows():
        # Skip already matched games
        if live_game['matched_to_schedule']:
            continue
            
        home_norm = live_game['home_team_norm']
        away_norm = live_game['away_team_norm']
        
        # Try both home/away combinations
        home_away_match = schedule_df[
            (schedule_df['home_team_norm'] == home_norm) &
            (schedule_df['away_team_norm'] == away_norm)
        ]
        
        away_home_match = schedule_df[
            (schedule_df['home_team_norm'] == away_norm) &
            (schedule_df['away_team_norm'] == home_norm)
        ]
        
        if not home_away_match.empty:
            # Found exact match
            match = home_away_match.iloc[0]
            live_df.at[live_idx, 'matched_to_schedule'] = True
            # Copy over the official game_id if needed
            if 'game_id' in match and match['game_id'] != live_game['game_id']:
                live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                live_df.at[live_idx, 'game_id'] = match['game_id']
                
        elif not away_home_match.empty:
            # Found match with teams reversed - this indicates a data issue
            print(f"Warning: Found teams in reverse order for game {live_game['game_id']}")
            match = away_home_match.iloc[0]
            live_df.at[live_idx, 'matched_to_schedule'] = True
            live_df.at[live_idx, 'teams_reversed'] = True
            # Copy over the official game_id if needed
            if 'game_id' in match and match['game_id'] != live_game['game_id']:
                live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                live_df.at[live_idx, 'game_id'] = match['game_id']
    
    # For unmatched games, try looser matching
    if not live_df[~live_df['matched_to_schedule']].empty:
        for live_idx, live_game in live_df[~live_df['matched_to_schedule']].iterrows():
            best_match = None
            best_score = 0
            is_reversed = False
            
            for _, sched_game in schedule_df.iterrows():
                # Try both directions
                direct_match = (live_game['home_team_norm'] in sched_game['home_team_norm'] or 
                               sched_game['home_team_norm'] in live_game['home_team_norm']) and \
                              (live_game['away_team_norm'] in sched_game['away_team_norm'] or 
                               sched_game['away_team_norm'] in live_game['away_team_norm'])
                               
                reverse_match = (live_game['home_team_norm'] in sched_game['away_team_norm'] or 
                                sched_game['away_team_norm'] in live_game['home_team_norm']) and \
                               (live_game['away_team_norm'] in sched_game['home_team_norm'] or 
                                sched_game['home_team_norm'] in live_game['away_team_norm'])
                
                # Simple scoring - each match is worth 1 point
                score = direct_match + reverse_match * 0.9  # Slightly prefer direct matches
                
                if score > best_score:
                    best_score = score
                    best_match = sched_game
                    is_reversed = reverse_match > direct_match
            
            if best_match is not None and best_score > 0:
                live_df.at[live_idx, 'matched_to_schedule'] = True
                live_df.at[live_idx, 'match_confidence'] = best_score
                live_df.at[live_idx, 'teams_reversed'] = is_reversed
                
                # Copy over the official game_id if needed
                if 'game_id' in best_match and best_match['game_id'] != live_game['game_id']:
                    live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                    live_df.at[live_idx, 'game_id'] = best_match['game_id']
    
    return live_df

def run_enhanced_prediction_pipeline():
    """
    Run the full prediction pipeline with all improvements
    
    1. Ensure official schedule is loaded
    2. Fetch live games with schedule matching
    3. If no live games, use historical games for testing
    4. Calculate enhanced features
    5. Generate predictions using the appropriate model
    6. Calculate confidence metrics and win probabilities
    7. Return predictions with validation metrics when possible
    """
    # 1. Load official schedule
    global OFFICIAL_SCHEDULE
    if OFFICIAL_SCHEDULE is None:
        OFFICIAL_SCHEDULE = load_and_cache_official_schedule()

    # 2. Fetch live games
    live_games_df = fetch_live_games_with_schedule_matching()

    # 3. Fall back to historical games if needed
    if live_games_df.empty:
        live_games_df = find_recent_games_for_testing()

        if live_games_df.empty:
            print("No games available for prediction")
            return pd.DataFrame()

    # 4. Calculate enhanced features
    try:
        # Get team rolling averages
        team_avgs = get_team_rolling_averages()

        # Calculate rest features
        for idx, game in live_games_df.iterrows():
            try:
                home_team = game['home_team']
                away_team = game['away_team']
                game_date = pd.to_datetime(game['game_date'])

                # Get rest data for both teams
                home_rest = get_rest_data(home_team, game_date)
                away_rest = get_rest_data(away_team, game_date)

                # Update DataFrame
                live_games_df.at[idx, 'rest_days_home'] = home_rest['rest_days']
                live_games_df.at[idx, 'rest_days_away'] = away_rest['rest_days']
                live_games_df.at[idx, 'is_back_to_back_home'] = int(home_rest['is_back_to_back'])
                live_games_df.at[idx, 'is_back_to_back_away'] = int(away_rest['is_back_to_back'])
                live_games_df.at[idx, 'rest_advantage'] = home_rest['rest_days'] - away_rest['rest_days']

                # Get previous matchup difference
                live_games_df.at[idx, 'prev_matchup_diff'] = get_previous_matchup_diff(home_team, away_team)
            except Exception as e:
                print(f"Error calculating rest features for game {game.get('game_id')}: {e}")
                # Set default values
                live_games_df.at[idx, 'rest_days_home'] = 2
                live_games_df.at[idx, 'rest_days_away'] = 2
                live_games_df.at[idx, 'is_back_to_back_home'] = 0
                live_games_df.at[idx, 'is_back_to_back_away'] = 0
                live_games_df.at[idx, 'rest_advantage'] = 0
                live_games_df.at[idx, 'prev_matchup_diff'] = 0

        # Calculate momentum features
        for idx, game in live_games_df.iterrows():
            try:
                current_quarter = int(game.get('current_quarter', 0) or 0)

                # Initialize momentum features
                live_games_df.at[idx, 'q1_to_q2_momentum'] = 0
                live_games_df.at[idx, 'q2_to_q3_momentum'] = 0
                live_games_df.at[idx, 'q3_to_q4_momentum'] = 0
                live_games_df.at[idx, 'cumulative_momentum'] = 0

                # Calculate quarter-to-quarter momentum
                if current_quarter >= 2:
                    # Q1 to Q2
                    home_q1 = float(game.get('home_q1', 0) or 0)
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    away_q1 = float(game.get('away_q1', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)

                    q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
                    # Cap to prevent extreme values
                    q1_to_q2 = max(min(q1_to_q2, 15), -15)
                    live_games_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2

                if current_quarter >= 3:
                    # Q2 to Q3
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)

                    q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
                    q2_to_q3 = max(min(q2_to_q3, 15), -15)
                    live_games_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3

                if current_quarter >= 4:
                    # Q3 to Q4
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    home_q4 = float(game.get('home_q4', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)
                    away_q4 = float(game.get('away_q4', 0) or 0)

                    q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
                    q3_to_q4 = max(min(q3_to_q4, 15), -15)
                    live_games_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4

                # Calculate weighted momentum
                weights = [0.2, 0.3, 0.5]  # Weights for each momentum period

                if current_quarter == 2:
                    momentum = live_games_df.at[idx, 'q1_to_q2_momentum']
                elif current_quarter == 3:
                    momentum = (
                        live_games_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        live_games_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
                    ) / (weights[0] + weights[1])
                elif current_quarter >= 4:
                    momentum = (
                        live_games_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        live_games_df.at[idx, 'q2_to_q3_momentum'] * weights[1] +
                        live_games_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
                    ) / sum(weights)
                else:
                    momentum = 0

                # Normalize to [-1, 1]
                live_games_df.at[idx, 'cumulative_momentum'] = max(min(momentum / 15.0, 1.0), -1.0)

            except Exception as e:
                print(f"Error calculating momentum for game {game.get('game_id')}: {e}")
                # Set defaults
                live_games_df.at[idx, 'q1_to_q2_momentum'] = 0
                live_games_df.at[idx, 'q2_to_q3_momentum'] = 0
                live_games_df.at[idx, 'q3_to_q4_momentum'] = 0
                live_games_df.at[idx, 'cumulative_momentum'] = 0

    except Exception as e:
        print(f"Error calculating enhanced features: {e}")
        traceback.print_exc()
        # Continue with basic features

    # 5. Make predictions
    try:
        # Load model from global scope or load it if not present
        model_to_use = None
        if 'model' in globals() and globals()['model'] is not None:
            model_to_use = globals()['model']
            print("Using 'model' from global scope")
        else:
            # Try to load model from various locations
            try:
                model_path = config.MODEL_PATH
                if os.path.exists(model_path):
                    model_to_use = joblib.load(model_path)
                    print(f"Loaded model from {model_path}")
            except Exception as e:
                print(f"Failed to load model: {e}")

        if model_to_use is None:
            print("No model available for prediction")
            return live_games_df  # Return with features but no predictions

        # Determine feature list based on model
        if hasattr(model_to_use, 'feature_importances_'):
            feature_count = len(model_to_use.feature_importances_)
            if feature_count > 8:
                # Enhanced model
                features_to_use = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
                print("Using enhanced feature set for prediction")
            else:
                # Original model
                features_to_use = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]

                # Add rolling scores if needed
                if 'rolling_home_score' not in live_games_df.columns:
                    live_games_df['rolling_home_score'] = live_games_df['home_team'].apply(
                        lambda t: team_avgs.get(t, 105.0))
                    live_games_df['rolling_away_score'] = live_games_df['away_team'].apply(
                        lambda t: team_avgs.get(t, 105.0))

                print("Using original feature set for prediction")
        else:
            # Default to enhanced features
            features_to_use = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4',
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
                'rest_days_home', 'rest_days_away', 'rest_advantage',
                'is_back_to_back_home', 'is_back_to_back_away',
                'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
            ]
            print("Using default enhanced feature set")

        # Make sure all required features exist and have reasonable values
        for feature in features_to_use:
            if feature not in live_games_df.columns:
                print(f"Adding missing feature: {feature}")
                live_games_df[feature] = 0
            
            # Convert to numeric and handle NaN values
            live_games_df[feature] = pd.to_numeric(live_games_df[feature], errors='coerce').fillna(0)

        # Prepare feature matrix
        X_pred = live_games_df[features_to_use].copy()

        # Generate predictions
        predictions = model_to_use.predict(X_pred)
        live_games_df['predicted_home_score'] = predictions

        # Calculate additional metrics
        for idx, row in live_games_df.iterrows():
            # Estimate away score (if model only predicts home score)
            # Use simple home court advantage (avg ~3.5 points) and previous matchup differential
            home_advantage = 3.5
            matchup_diff = row.get('prev_matchup_diff', 0)
            live_games_df.at[idx, 'predicted_away_score'] = row['predicted_home_score'] - home_advantage - matchup_diff

            # Calculate remaining quarters
            current_quarter = int(row.get('current_quarter', 0) or 0)
            if current_quarter > 0 and current_quarter < 4:
                # Get current score
                current_home_score = sum([float(row.get(f'home_q{q}', 0) or 0) for q in range(1, current_quarter + 1)])
                current_away_score = sum([float(row.get(f'away_q{q}', 0) or 0) for q in range(1, current_quarter + 1)])
                
                # Calculate remaining score
                remaining_home = row['predicted_home_score'] - current_home_score
                remaining_away = row['predicted_away_score'] - current_away_score
                
                # Store remaining scores
                live_games_df.at[idx, 'remaining_home_score'] = max(0, remaining_home)
                live_games_df.at[idx, 'remaining_away_score'] = max(0, remaining_away)
            
            # Calculate win probability
            score_diff = row['predicted_home_score'] - row['predicted_away_score']
            # Simple logistic function for win probability
            win_prob = 1 / (1 + np.exp(-0.1 * score_diff))
            live_games_df.at[idx, 'win_probability'] = win_prob
            
            # Calculate confidence based on quarter
            if current_quarter == 0:
                confidence = 0.5  # Pre-game
            elif current_quarter == 1:
                confidence = 0.6  # 1st quarter
            elif current_quarter == 2:
                confidence = 0.7  # 2nd quarter
            elif current_quarter == 3:
                confidence = 0.85  # 3rd quarter
            else:
                confidence = 0.95  # 4th quarter
            
            live_games_df.at[idx, 'prediction_confidence'] = confidence
            
            # If we have actual final scores (for testing), calculate errors
            if 'actual_final_home' in row and 'actual_final_away' in row:
                live_games_df.at[idx, 'home_score_error'] = row['predicted_home_score'] - row['actual_final_home']
                live_games_df.at[idx, 'away_score_error'] = row['predicted_away_score'] - row['actual_final_away']
                
                # Calculate percentage error
                if row['actual_final_home'] > 0:
                    live_games_df.at[idx, 'home_error_pct'] = abs(row['home_score_error'] / row['actual_final_home']) * 100
                if row['actual_final_away'] > 0:
                    live_games_df.at[idx, 'away_error_pct'] = abs(row['away_score_error'] / row['actual_final_away']) * 100

        # 6. Generate recommendations
        for idx, row in live_games_df.iterrows():
            try:
                model_outputs = {
                    "win_probability": row['win_probability'],
                    "momentum_shift": row.get('cumulative_momentum', 0),
                    "projected_margin": row['predicted_home_score'] - row['predicted_away_score'],
                    "total_projected_score": row['predicted_home_score'] + row['predicted_away_score'],
                    "quarter": row.get('current_quarter', 0),
                    "time_remaining": None  # Would need to be provided by data source
                }
                
                recommendations = generate_recommendations(model_outputs)
                
                # Store top recommendations
                for rec_key, rec_value in recommendations.items():
                    live_games_df.at[idx, f'rec_{rec_key}'] = rec_value
            except Exception as e:
                print(f"Error generating recommendations for game {row.get('game_id')}: {e}")

    except Exception as e:
        print(f"Error making predictions: {e}")
        traceback.print_exc()

    # Return the dataframe with predictions
    return live_games_df

In [ ]:
# Cell 16 - Fetch official schedule from database

import numpy as np
import pandas as pd  # Added import for pandas
from datetime import datetime, timedelta
import time
import traceback
import pytz
import os
import joblib
import json
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from pathlib import Path
import logging
from typing import Dict, List, Optional, Tuple, Union

# Setup structured logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('nba_monitor')

# Import Supabase client if available, otherwise create a placeholder for demo purposes
try:
    from supabase import create_client, Client
    
    # Try to get environment variables
    import os
    SUPABASE_URL = os.getenv("SUPABASE_URL")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY")
    
    # Create a supabase client if credentials are available
    if SUPABASE_URL and SUPABASE_KEY:
        supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
        logger.info("Supabase client initialized with credentials.")
    else:
        logger.warning("Supabase credentials not found. Using mock client for demo purposes.")
        supabase = None
except ImportError:
    logger.warning("Supabase package not installed. Using mock client for demo purposes.")
    supabase = None

class MockSupabaseClient:
    """A mock Supabase client for testing when real client isn't available"""
    
    def table(self, table_name):
        return MockSupabaseQuery(table_name)
    
class MockSupabaseQuery:
    """A mock query builder"""
    
    def __init__(self, table_name):
        self.table_name = table_name
        self.filters = []
        
    def select(self, columns):
        return self
        
    def eq(self, column, value):
        self.filters.append((column, "=", value))
        return self
        
    def lt(self, column, value):
        self.filters.append((column, "<", value))
        return self
        
    def gt(self, column, value):
        self.filters.append((column, ">", value))
        return self
        
    def gte(self, column, value):
        self.filters.append((column, ">=", value))
        return self
        
    def order(self, column, desc=False):
        return self
        
    def limit(self, num):
        return self
        
    def execute(self):
        # Return empty mock data
        return MockSupabaseResponse([])
        
class MockSupabaseResponse:
    """A mock response object"""
    
    def __init__(self, data):
        self.data = data


class DataFetcher:
    """
    Handles all data fetching operations from the database.
    """
    
    def __init__(self, supabase_client):
        """Initialize with Supabase client."""
        self.supabase = supabase_client if supabase_client else MockSupabaseClient()

    def get_official_schedule(self, date=None):
        """Load the official NBA schedule for a specific date (defaults to today)."""
        # Get today's date in Pacific Time if not specified
        if date is None:
            pacific_tz = pytz.timezone('America/Los_Angeles')
            date = datetime.now(pacific_tz).strftime('%Y-%m-%d')
            
        try:
            # Fetch from database
            response = self.supabase.table("nba_game_schedule").select("*").eq("game_date", date).execute()

            if response.data:
                schedule_df = pd.DataFrame(response.data)
                logger.info(f"Loaded {len(schedule_df)} games from official schedule for {date}")
                return schedule_df
            else:
                logger.warning(f"No scheduled games found for {date}")
                return pd.DataFrame()
        except Exception as e:
            logger.error(f"Error loading official schedule: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def get_live_games(self):
        """Fetch current live games."""
        try:
            # Get current date in Pacific Time
            pacific_tz = pytz.timezone('America/Los_Angeles')
            today_pacific = datetime.now(pacific_tz).strftime('%Y-%m-%d')

            logger.info(f"Fetching live games for {today_pacific} (Pacific Time)")
            response = self.supabase.table("nba_live_game_stats").select("*").execute()
            
            if not response.data:
                logger.info("No live games found.")
                return pd.DataFrame()

            return pd.DataFrame(response.data)
            
        except Exception as e:
            logger.error(f"Error fetching live games: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    
    def get_team_rolling_averages(self, days_lookback=60):
        """Get team rolling scoring averages from historical data."""
        try:
            # Calculate the date threshold
            threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
            
            # Fetch recent historical game data
            response = self.supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
            historical_data = response.data
            
            if not historical_data:
                logger.warning(f"No historical game data available from the last {days_lookback} days.")
                return {}
            
            df = pd.DataFrame(historical_data)
            df['game_date'] = pd.to_datetime(df['game_date'])
            df = df.sort_values('game_date')
            
            # Initialize dictionary for team averages
            team_avgs = {}
            
            # Get unique teams
            all_teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
            
            for team in all_teams:
                # Get home games where team is home
                home_games = df[df['home_team'] == team][['game_date', 'home_score']].rename(
                    columns={'home_score': 'score'})
                
                # Get away games where team is away
                away_games = df[df['away_team'] == team][['game_date', 'away_score']].rename(
                    columns={'away_score': 'score'})
                
                # Combine all games
                team_games = pd.concat([home_games, away_games]).sort_values('game_date')
                
                if not team_games.empty:
                    # Calculate recent average (last 5 games if available)
                    recent_games = team_games.tail(5)
                    team_avgs[team] = recent_games['score'].mean()
                else:
                    # Fallback to a reasonable default
                    team_avgs[team] = 105.0  # NBA average is approximately 100-110 points per game
            
            return team_avgs
        except Exception as e:
            logger.error(f"Error getting team rolling averages: {e}")
            traceback.print_exc()
            return {}
    
    def get_rest_data(self, team_name, game_date):
        """Calculate rest days for a team before a specific game."""
        try:
            # Ensure game_date is datetime
            if isinstance(game_date, str):
                game_date = pd.to_datetime(game_date)
            
            # Try to find previous game
            home_response = self.supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("home_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
                
            away_response = self.supabase.table("nba_historical_game_stats")\
                .select("game_date")\
                .eq("away_team", team_name)\
                .lt("game_date", game_date.strftime('%Y-%m-%d'))\
                .order("game_date", desc=True)\
                .limit(1)\
                .execute()
            
            # Combine results and find most recent
            all_games = home_response.data + away_response.data
            if not all_games:
                # If no previous game found, return default
                return {
                    'rest_days': 3,  # Default if no history
                    'is_back_to_back': False
                }
                
            # Sort by date (most recent first)
            all_games.sort(key=lambda x: x['game_date'], reverse=True)
            prev_game_date = pd.to_datetime(all_games[0]['game_date'])
            
            # Calculate days between games
            rest_days = (game_date - prev_game_date).days
            
            # Cap to reasonable values
            rest_days = max(min(rest_days, 10), 0)
            
            return {
                'rest_days': rest_days,
                'is_back_to_back': rest_days <= 1
            }
        except Exception as e:
            logger.error(f"Error getting rest data for {team_name}: {e}")
            traceback.print_exc()
            # Return default values on error
            return {
                'rest_days': 2,  # Reasonable default
                'is_back_to_back': False
            }
    
    def get_previous_matchup_diff(self, home_team, away_team, max_lookback=5):
        """Get point differential from previous matchups between two teams."""
        try:
            # Use separate queries for home and away configurations
            response_home = self.supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", home_team)\
                .eq("away_team", away_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
                
            response_away = self.supabase.table("nba_historical_game_stats").select("*")\
                .eq("home_team", away_team)\
                .eq("away_team", home_team)\
                .order('game_date', desc=True)\
                .limit(max_lookback).execute()
            
            # Combine results
            home_matchups = response_home.data
            away_matchups = response_away.data
            matchups = home_matchups + away_matchups
            
            # Sort by date (most recent first)
            if matchups:
                matchups.sort(key=lambda x: x['game_date'], reverse=True)
                matchups = matchups[:max_lookback]
            
            if not matchups:
                return 0
            
            # Calculate point differential from home team perspective
            differentials = []
            for game in matchups:
                if game['home_team'] == home_team and game['away_team'] == away_team:
                    diff = game['home_score'] - game['away_score']
                elif game['home_team'] == away_team and game['away_team'] == home_team:
                    diff = game['away_score'] - game['home_score']
                else:
                    continue
                differentials.append(diff)
            
            # Cap extreme values
            avg_diff = sum(differentials) / len(differentials) if differentials else 0
            return max(min(avg_diff, 15), -15)  # Cap at +/- 15 points
        except Exception as e:
            logger.warning(f"Error getting previous matchups: {e}")
            return 0
    
    def find_recent_games_for_testing(self, days_to_look_back=5, limit=5):
        """Find recent completed games to use for testing the prediction model."""
        logger.info("Fetching recent completed games for testing...")

        # Get dates to try, in order of preference
        try:
            dates_to_try = []
            today = datetime.now()

            # Try previous days
            for i in range(1, days_to_look_back+1):
                date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
                dates_to_try.append(date)

            # Try each date in sequence
            for test_date in dates_to_try:
                response = self.supabase.table("nba_historical_game_stats")\
                    .select("*")\
                    .eq("game_date", test_date)\
                    .limit(limit).execute()

                historical_games = response.data
                if historical_games:
                    logger.info(f"Found {len(historical_games)} games from {test_date} for testing.")

                    # Simulate these as 'live' games by setting them to random quarters
                    import random

                    live_games = []
                    for game in historical_games:
                        # Randomly select a quarter for simulation (1-4)
                        sim_quarter = random.randint(1, 4)

                        # Create a simulated live game where we only know scores up to the simulated quarter
                        sim_game = {
                            'game_id': game['game_id'],
                            'home_team': game['home_team'],
                            'away_team': game['away_team'],
                            'game_date': game['game_date'],
                            'current_quarter': sim_quarter,
                            'simulated_quarter': sim_quarter,  # Mark which quarter we're simulating
                            'matched_to_schedule': True,  # Pretend it's matched
                            'game_status': 'LIVE'
                        }

                        # Add quarter scores up to the simulated quarter
                        for q in range(1, 5):
                            q_field_home = f'home_q{q}'
                            q_field_away = f'away_q{q}'

                            if q <= sim_quarter and q_field_home in game and q_field_away in game:
                                sim_game[q_field_home] = game[q_field_home]
                                sim_game[q_field_away] = game[q_field_away]
                            else:
                                sim_game[q_field_home] = 0
                                sim_game[q_field_away] = 0

                        # Calculate the current score based on quarters we "know"
                        sim_game['home_score'] = sum([
                            float(sim_game.get(f'home_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                        ])
                        sim_game['away_score'] = sum([
                            float(sim_game.get(f'away_q{q}', 0) or 0) for q in range(1, sim_quarter+1)
                        ])

                        # Save the actual final scores for validation
                        sim_game['actual_home_final'] = game['home_score']
                        sim_game['actual_away_final'] = game['away_score']

                        live_games.append(sim_game)

                    return pd.DataFrame(live_games)

            logger.warning("No recent games found for testing.")
            return pd.DataFrame()
        except Exception as e:
            logger.error(f"Error finding recent games: {e}")
            traceback.print_exc()
            return pd.DataFrame()


class DataProcessor:
    """
    Handles data processing, cleaning, and feature engineering.
    """
    
    @staticmethod
    def normalize_team_name(name):
        """Normalize team names for consistent matching."""
        if not name:
            return ""

        # Convert to string and lowercase
        name = str(name).lower().strip()

        # Standard team name mappings
        mappings = {
            'lakers': 'los angeles lakers',
            'la lakers': 'los angeles lakers',
            'clippers': 'los angeles clippers',
            'la clippers': 'los angeles clippers',
            'blazers': 'portland trail blazers',
            'sixers': 'philadelphia 76ers',
            'philly': 'philadelphia 76ers',
            'knicks': 'new york knicks',
            'ny knicks': 'new york knicks',
            'nets': 'brooklyn nets',
            'mavs': 'dallas mavericks',
            'cavs': 'cleveland cavaliers',
            'wolves': 'minnesota timberwolves',
            't-wolves': 'minnesota timberwolves',
            'celts': 'boston celtics',
            'pels': 'new orleans pelicans',
            'warriors': 'golden state warriors',
            'gsw': 'golden state warriors',
            'heat': 'miami heat',
            'bulls': 'chicago bulls',
            'hawks': 'atlanta hawks',
            'suns': 'phoenix suns',
            'bucks': 'milwaukee bucks',
            'jazz': 'utah jazz',
            'nuggets': 'denver nuggets',
            'rockets': 'houston rockets',
            'pacers': 'indiana pacers',
            'spurs': 'san antonio spurs',
            'kings': 'sacramento kings',
            'magic': 'orlando magic',
            'wizards': 'washington wizards',
            'raptors': 'toronto raptors',
            'thunder': 'oklahoma city thunder',
            'okc': 'oklahoma city thunder',
            'pistons': 'detroit pistons',
            'hornets': 'charlotte hornets',
            'grizzlies': 'memphis grizzlies',
            'grizz': 'memphis grizzlies'
        }

        # Check if the name is in mappings
        for key, value in mappings.items():
            if key in name:
                return value

        return name
    
    def prepare_schedule_data(self, schedule_df):
        """Add normalized team names to schedule data."""
        if schedule_df.empty:
            return schedule_df
            
        # Add normalized team names for better matching
        result_df = schedule_df.copy()
        result_df['home_team_norm'] = result_df['home_team'].apply(self.normalize_team_name)
        result_df['away_team_norm'] = result_df['away_team'].apply(self.normalize_team_name)
        
        return result_df
    
    def validate_game_data(self, df):
        """Validate and clean game data."""
        if df.empty:
            return df
            
        # Make a copy to avoid modifying the original
        df = df.copy()
        
        # Convert date fields to datetime
        if 'game_date' in df.columns:
            df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce')
        
        # Ensure quarter fields are numeric
        for q in range(1, 5):
            home_q = f'home_q{q}'
            away_q = f'away_q{q}'
            
            if home_q in df.columns:
                df[home_q] = pd.to_numeric(df[home_q], errors='coerce').fillna(0)
            else:
                df[home_q] = 0
                
            if away_q in df.columns:
                df[away_q] = pd.to_numeric(df[away_q], errors='coerce').fillna(0)
            else:
                df[away_q] = 0
        
        # Calculate current quarter based on non-zero quarter scores
        df['current_quarter'] = 0
        for idx, row in df.iterrows():
            max_quarter = 0
            for q in range(1, 5):
                home_q = f'home_q{q}'
                away_q = f'away_q{q}'
                
                if ((home_q in row and pd.notnull(row[home_q]) and row[home_q] > 0) or 
                    (away_q in row and pd.notnull(row[away_q]) and row[away_q] > 0)):
                    max_quarter = q
            
            # Only update if detected quarter is higher than existing value
            existing_q = row.get('current_quarter', 0)
            if pd.isna(existing_q) or existing_q < max_quarter:
                df.at[idx, 'current_quarter'] = max_quarter
        
        # Ensure team names are strings
        if 'home_team' in df.columns:
            df['home_team'] = df['home_team'].astype(str)
        if 'away_team' in df.columns:
            df['away_team'] = df['away_team'].astype(str)
        
        # Calculate current score as sum of quarters
        if 'home_score' not in df.columns:
            df['home_score'] = 0
        if 'away_score' not in df.columns:
            df['away_score'] = 0
            
        for idx, row in df.iterrows():
            current_q = int(row.get('current_quarter', 0) or 0)
            
            home_score = 0
            away_score = 0
            for q in range(1, current_q + 1):
                home_q = f'home_q{q}'
                away_q = f'away_q{q}'
                
                if home_q in row and pd.notnull(row[home_q]):
                    home_score += float(row[home_q] or 0)
                if away_q in row and pd.notnull(row[away_q]):
                    away_score += float(row[away_q] or 0)
            
            # Only update if calculated scores are different from existing values
            if abs(df.at[idx, 'home_score'] - home_score) > 0.1 or pd.isna(df.at[idx, 'home_score']):
                df.at[idx, 'home_score'] = home_score
            if abs(df.at[idx, 'away_score'] - away_score) > 0.1 or pd.isna(df.at[idx, 'away_score']):
                df.at[idx, 'away_score'] = away_score
        
        # Calculate score ratio
        df['score_ratio'] = 0.5  # Default to even
        for idx, row in df.iterrows():
            home_score = float(row.get('home_score', 0) or 0)
            away_score = float(row.get('away_score', 0) or 0)
            total = home_score + away_score
            
            if total > 0:
                df.at[idx, 'score_ratio'] = home_score / total
        
        # Initialize matched_to_schedule column if not present
        if 'matched_to_schedule' not in df.columns:
            df['matched_to_schedule'] = False
        
        # Add game status column
        if 'game_status' not in df.columns:
            df['game_status'] = 'UNKNOWN'
            for idx, row in df.iterrows():
                status, quarter, time_remaining = self._determine_game_status(row)
                df.at[idx, 'game_status'] = status
                df.at[idx, 'time_remaining_mins'] = time_remaining
        
        # Add normalized team names
        df['home_team_norm'] = df['home_team'].apply(self.normalize_team_name)
        df['away_team_norm'] = df['away_team'].apply(self.normalize_team_name)
        
        return df
    
    def _determine_game_status(self, game_row):
        """Determine game status based on quarter and score data."""
        # Default values
        status = 'UNKNOWN'
        quarter = int(game_row.get('current_quarter', 0) or 0)
        time_remaining = 48  # Full game
        
        # Check if game is marked as finished
        if game_row.get('is_finished', False):
            return 'FINAL', quarter, 0
        
        # Game has started (at least one quarter with points)
        if quarter > 0:
            # All quarters have data - likely finished
            if quarter == 4:
                home_q4 = float(game_row.get('home_q4', 0) or 0)
                away_q4 = float(game_row.get('away_q4', 0) or 0)
                if home_q4 > 0 and away_q4 > 0:
                    return 'FINAL', 4, 0
            
            # In progress
            status = 'LIVE'
            time_remaining = (4 - quarter) * 12
            if quarter == 4:
                # Assume we're midway through the 4th
                time_remaining = 6
        else:
            # No quarters with data - scheduled or delayed
            # If we have a game date, check if it's in the past
            if 'game_date' in game_row:
                game_date = pd.to_datetime(game_row['game_date'])
                now = datetime.now(tz=game_date.tzinfo) if game_date.tzinfo else datetime.now()
                
                if game_date < now:
                    # Game should have started but no data - likely delayed
                    status = 'DELAYED'
                else:
                    # Game is in the future
                    status = 'SCHEDULED'
                    # Add time until game starts to the 48 minutes
                    mins_to_start = (game_date - now).total_seconds() / 60
                    time_remaining = 48 + max(0, mins_to_start)
            else:
                # No date info - assume scheduled
                status = 'SCHEDULED'
        
        return status, quarter, time_remaining
    
    def match_teams_to_schedule(self, live_df, schedule_df):
        """Match live games to the official schedule using team names."""
        if live_df.empty or schedule_df.empty:
            return live_df
            
        # Copy to avoid modifying the original
        live_df = live_df.copy()
        
        # Make sure both dataframes have normalized team names
        live_df = self.validate_game_data(live_df)
        schedule_df = self.prepare_schedule_data(schedule_df)
        
        # Try to match by exact normalized team names
        for live_idx, live_game in live_df.iterrows():
            # Skip already matched games
            if live_game.get('matched_to_schedule', False):
                continue
                
            home_norm = live_game['home_team_norm']
            away_norm = live_game['away_team_norm']
            
            # Try both home/away combinations
            home_away_match = schedule_df[
                (schedule_df['home_team_norm'] == home_norm) &
                (schedule_df['away_team_norm'] == away_norm)
            ]
            
            away_home_match = schedule_df[
                (schedule_df['home_team_norm'] == away_norm) &
                (schedule_df['away_team_norm'] == home_norm)
            ]
            
            if not home_away_match.empty:
                # Found exact match
                match = home_away_match.iloc[0]
                live_df.at[live_idx, 'matched_to_schedule'] = True
                # Copy over the official game_id if needed
                if 'game_id' in match and match['game_id'] != live_game['game_id']:
                    live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                    live_df.at[live_idx, 'game_id'] = match['game_id']
                    
            elif not away_home_match.empty:
                # Found match with teams reversed - this indicates a data issue
                logger.warning(f"Found teams in reverse order for game {live_game['game_id']}")
                match = away_home_match.iloc[0]
                live_df.at[live_idx, 'matched_to_schedule'] = True
                live_df.at[live_idx, 'teams_reversed'] = True
                # Copy over the official game_id if needed
                if 'game_id' in match and match['game_id'] != live_game['game_id']:
                    live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                    live_df.at[live_idx, 'game_id'] = match['game_id']
        
        # For unmatched games, try looser matching
        if not live_df[~live_df['matched_to_schedule']].empty:
            for live_idx, live_game in live_df[~live_df['matched_to_schedule']].iterrows():
                best_match = None
                best_score = 0
                is_reversed = False
                
                for _, sched_game in schedule_df.iterrows():
                    # Try both directions
                    direct_match = (live_game['home_team_norm'] in sched_game['home_team_norm'] or 
                                   sched_game['home_team_norm'] in live_game['home_team_norm']) and \
                                  (live_game['away_team_norm'] in sched_game['away_team_norm'] or 
                                   sched_game['away_team_norm'] in live_game['away_team_norm'])
                                   
                    reverse_match = (live_game['home_team_norm'] in sched_game['away_team_norm'] or 
                                    sched_game['away_team_norm'] in live_game['home_team_norm']) and \
                                   (live_game['away_team_norm'] in sched_game['home_team_norm'] or 
                                    sched_game['home_team_norm'] in live_game['away_team_norm'])
                    
                    # Simple scoring - each match is worth 1 point
                    score = direct_match + reverse_match * 0.9  # Slightly prefer direct matches
                    
                    if score > best_score:
                        best_score = score
                        best_match = sched_game
                        is_reversed = reverse_match > direct_match
                
                if best_match is not None and best_score > 0:
                    live_df.at[live_idx, 'matched_to_schedule'] = True
                    live_df.at[live_idx, 'match_confidence'] = best_score
                    live_df.at[live_idx, 'teams_reversed'] = is_reversed
                    
                    # Copy over the official game_id if needed
                    if 'game_id' in best_match and best_match['game_id'] != live_game['game_id']:
                        live_df.at[live_idx, 'original_game_id'] = live_game['game_id']
                        live_df.at[live_idx, 'game_id'] = best_match['game_id']
        
        return live_df
    
    def convert_scheduled_to_live_format(self, schedule_df):
        """Convert scheduled games to a live data format."""
        if schedule_df.empty:
            return pd.DataFrame()
            
        live_format = []
        
        for _, game in schedule_df.iterrows():
            # Create a basic live game structure
            live_game = {
                'game_id': game['game_id'],
                'home_team': game['home_team'],
                'away_team': game['away_team'],
                'game_date': game['game_date'],
                'matched_to_schedule': True,
                'current_quarter': 0,  # Pre-game
                'home_q1': 0, 'home_q2': 0, 'home_q3': 0, 'home_q4': 0,
                'away_q1': 0, 'away_q2': 0, 'away_q3': 0, 'away_q4': 0,
                'home_score': 0,
                'away_score': 0,
                'game_status': 'SCHEDULED'}

            # Calculate time until game
            try:
                game_dt = pd.to_datetime(game['game_date'])
                now = datetime.now(tz=game_dt.tzinfo) if game_dt.tzinfo else datetime.now()
                time_until_game = max(0, (game_dt - now).total_seconds() / 60)
                live_game['time_remaining_mins'] = 48 + time_until_game
            except:
                live_game['time_remaining_mins'] = 48
                
            live_format.append(live_game)
        
        return pd.DataFrame(live_format)
    
    def calculate_enhanced_features(self, games_df, team_avgs, data_fetcher):
        """Calculate enhanced features for prediction."""
        if games_df.empty:
            return pd.DataFrame()
            
        # Make a copy to avoid modifying the original
        enhanced_df = games_df.copy()
        
        # Add rolling averages
        enhanced_df['rolling_home_score'] = enhanced_df['home_team'].apply(
            lambda t: team_avgs.get(t, 105.0))
        enhanced_df['rolling_away_score'] = enhanced_df['away_team'].apply(
            lambda t: team_avgs.get(t, 105.0))
            
        # Calculate rest features
        for idx, game in enhanced_df.iterrows():
            try:
                home_team = game['home_team']
                away_team = game['away_team']
                game_date = pd.to_datetime(game['game_date'])

                # Get rest data for both teams
                home_rest = data_fetcher.get_rest_data(home_team, game_date)
                away_rest = data_fetcher.get_rest_data(away_team, game_date)

                # Update DataFrame
                enhanced_df.at[idx, 'rest_days_home'] = home_rest['rest_days']
                enhanced_df.at[idx, 'rest_days_away'] = away_rest['rest_days']
                enhanced_df.at[idx, 'is_back_to_back_home'] = int(home_rest['is_back_to_back'])
                enhanced_df.at[idx, 'is_back_to_back_away'] = int(away_rest['is_back_to_back'])
                enhanced_df.at[idx, 'rest_advantage'] = home_rest['rest_days'] - away_rest['rest_days']

                # Get previous matchup difference
                enhanced_df.at[idx, 'prev_matchup_diff'] = data_fetcher.get_previous_matchup_diff(home_team, away_team)
            except Exception as e:
                logger.warning(f"Error calculating rest features for game {game.get('game_id')}: {e}")
                # Set default values
                enhanced_df.at[idx, 'rest_days_home'] = 2
                enhanced_df.at[idx, 'rest_days_away'] = 2
                enhanced_df.at[idx, 'is_back_to_back_home'] = 0
                enhanced_df.at[idx, 'is_back_to_back_away'] = 0
                enhanced_df.at[idx, 'rest_advantage'] = 0
                enhanced_df.at[idx, 'prev_matchup_diff'] = 0
        
        # Calculate momentum features
        for idx, game in enhanced_df.iterrows():
            try:
                current_quarter = int(game.get('current_quarter', 0) or 0)

                # Initialize momentum features
                enhanced_df.at[idx, 'q1_to_q2_momentum'] = 0
                enhanced_df.at[idx, 'q2_to_q3_momentum'] = 0
                enhanced_df.at[idx, 'q3_to_q4_momentum'] = 0
                enhanced_df.at[idx, 'cumulative_momentum'] = 0

                # Calculate quarter-to-quarter momentum
                if current_quarter >= 2:
                    # Q1 to Q2
                    home_q1 = float(game.get('home_q1', 0) or 0)
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    away_q1 = float(game.get('away_q1', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)

                    q1_to_q2 = (home_q2 - home_q1) - (away_q2 - away_q1)
                    # Cap to prevent extreme values
                    q1_to_q2 = max(min(q1_to_q2, 15), -15)
                    enhanced_df.at[idx, 'q1_to_q2_momentum'] = q1_to_q2

                if current_quarter >= 3:
                    # Q2 to Q3
                    home_q2 = float(game.get('home_q2', 0) or 0)
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    away_q2 = float(game.get('away_q2', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)

                    q2_to_q3 = (home_q3 - home_q2) - (away_q3 - away_q2)
                    q2_to_q3 = max(min(q2_to_q3, 15), -15)
                    enhanced_df.at[idx, 'q2_to_q3_momentum'] = q2_to_q3

                if current_quarter >= 4:
                    # Q3 to Q4
                    home_q3 = float(game.get('home_q3', 0) or 0)
                    home_q4 = float(game.get('home_q4', 0) or 0)
                    away_q3 = float(game.get('away_q3', 0) or 0)
                    away_q4 = float(game.get('away_q4', 0) or 0)

                    q3_to_q4 = (home_q4 - home_q3) - (away_q4 - away_q3)
                    q3_to_q4 = max(min(q3_to_q4, 15), -15)
                    enhanced_df.at[idx, 'q3_to_q4_momentum'] = q3_to_q4

                # Calculate weighted momentum
                weights = [0.2, 0.3, 0.5]  # Weights for each momentum period

                if current_quarter == 2:
                    momentum = enhanced_df.at[idx, 'q1_to_q2_momentum']
                elif current_quarter == 3:
                    momentum = (
                        enhanced_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        enhanced_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
                    ) / (weights[0] + weights[1])
                elif current_quarter >= 4:
                    momentum = (
                        enhanced_df.at[idx, 'q1_to_q2_momentum'] * weights[0] +
                        enhanced_df.at[idx, 'q2_to_q3_momentum'] * weights[1] +
                        enhanced_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
                    ) / sum(weights)
                else:
                    momentum = 0

                # Normalize to [-1, 1]
                enhanced_df.at[idx, 'cumulative_momentum'] = max(min(momentum / 15.0, 1.0), -1.0)

            except Exception as e:
                logger.warning(f"Error calculating momentum for game {game.get('game_id')}: {e}")
                # Set defaults
                enhanced_df.at[idx, 'q1_to_q2_momentum'] = 0
                enhanced_df.at[idx, 'q2_to_q3_momentum'] = 0
                enhanced_df.at[idx, 'q3_to_q4_momentum'] = 0
                enhanced_df.at[idx, 'cumulative_momentum'] = 0
                
        return enhanced_df


class ModelManager:
    """
    Handles model loading and feature extraction.
    """
    
    def __init__(self):
        """Initialize the model manager."""
        self.model = None
        
    def load_model(self, model_path=None):
        """Load a prediction model from disk."""
        try:
            if model_path is None:
                # Try to use config.MODEL_PATH if available
                try:
                    import config
                    model_path = config.MODEL_PATH
                except (ImportError, AttributeError):
                    model_path = os.path.join(os.getcwd(), 'models', 'score_prediction_model.pkl')
            
            if os.path.exists(model_path):
                self.model = joblib.load(model_path)
                logger.info(f"Model loaded from {model_path}")
                return True
            else:
                logger.error(f"Model file not found at {model_path}")
                return False
        except Exception as e:
            logger.error(f"Error loading model: {e}")
            traceback.print_exc()
            return False
    
    def get_feature_list(self):
        """Determine which features to use based on model type."""
        if self.model is None:
            logger.error("No model loaded")
            return []
            
        # If model has feature_importances_, it's a tree-based model
        if hasattr(self.model, 'feature_importances_'):
            feature_count = len(self.model.feature_importances_)
            if feature_count > 8:
                # Enhanced model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
            else:
                # Original model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
        # If model has coefficients, it's a linear model
        elif hasattr(self.model, 'coef_'):
            feature_count = len(self.model.coef_)
            if feature_count > 8:
                # Enhanced model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
            else:
                # Original model
                return [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4',
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
        # Default to comprehensive feature list
        return [
            'home_q1', 'home_q2', 'home_q3', 'home_q4',
            'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff',
            'rest_days_home', 'rest_days_away', 'rest_advantage',
            'is_back_to_back_home', 'is_back_to_back_away',
            'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
        ]


class PredictionEngine:
    """
    Handles the actual prediction process.
    """
    
    def __init__(self, model_manager):
        """Initialize with a model manager."""
        self.model_manager = model_manager
        self.prediction_history = {}
        self.season_adjustment = 1.0
    
    def run_predictions(self, games_df):
        """
        Run predictions for all games in the DataFrame.
        
        Args:
            games_df: DataFrame with game data and enhanced features
            
        Returns:
            DataFrame with predictions and additional metrics
        """
        if games_df.empty:
            logger.info("No games available for prediction")
            return pd.DataFrame()
        
        # Ensure model is loaded
        if self.model_manager.model is None:
            logger.error("No model available for prediction")
            return games_df
        
        # Identify features to use
        feature_list = self.model_manager.get_feature_list()
        
        # Ensure all features exist
        enhanced_df = games_df.copy()
        for feature in feature_list:
            if feature not in enhanced_df.columns:
                logger.info(f"Adding missing feature: {feature}")
                enhanced_df[feature] = 0
                
            # Convert to numeric
            enhanced_df[feature] = pd.to_numeric(enhanced_df[feature], errors='coerce').fillna(0)
        
        # Extract feature matrix
        X_pred = enhanced_df[feature_list]
        
        # Generate predictions
        try:
            predictions = self.model_manager.model.predict(X_pred)
            enhanced_df['predicted_home_score'] = predictions
            
            # Calculate away score predictions and other metrics
            for idx, row in enhanced_df.iterrows():
                # Adjust for home court advantage and matchup history
                home_advantage = 3.5
                matchup_diff = row.get('prev_matchup_diff', 0)
                enhanced_df.at[idx, 'predicted_away_score'] = row['predicted_home_score'] - home_advantage - matchup_diff
                
                # Ensure predictions are at least current scores
                current_home = float(row.get('home_score', 0) or 0)
                current_away = float(row.get('away_score', 0) or 0)
                
                enhanced_df.at[idx, 'predicted_home_score'] = max(enhanced_df.at[idx, 'predicted_home_score'], current_home)
                enhanced_df.at[idx, 'predicted_away_score'] = max(enhanced_df.at[idx, 'predicted_away_score'], current_away)
                
                # Calculate score differential
                score_diff = enhanced_df.at[idx, 'predicted_home_score'] - enhanced_df.at[idx, 'predicted_away_score']
                
                # Calculate win probability using logistic function
                # Higher differential = higher probability, with steeper curve in later quarters
                quarter = int(row.get('current_quarter', 0) or 0)
                k_factor = 0.05 + (quarter * 0.025)  # Steeper curve in later quarters
                win_prob = 1.0 / (1.0 + np.exp(-k_factor * score_diff))
                enhanced_df.at[idx, 'win_probability'] = win_prob
                
                # Calculate remaining score
                if quarter > 0:
                    enhanced_df.at[idx, 'remaining_home_score'] = enhanced_df.at[idx, 'predicted_home_score'] - current_home
                    enhanced_df.at[idx, 'remaining_away_score'] = enhanced_df.at[idx, 'predicted_away_score'] - current_away
                
                # Calculate prediction confidence (increases by quarter)
                confidence_by_quarter = {0: 0.5, 1: 0.6, 2: 0.7, 3: 0.85, 4: 0.95}
                enhanced_df.at[idx, 'prediction_confidence'] = confidence_by_quarter.get(quarter, 0.5)
                
                # Calculate total projected score
                enhanced_df.at[idx, 'total_projected_score'] = enhanced_df.at[idx, 'predicted_home_score'] + enhanced_df.at[idx, 'predicted_away_score']
                
                # If we have actual finals, calculate errors
                if 'actual_home_final' in row and 'actual_away_final' in row:
                    enhanced_df.at[idx, 'home_score_error'] = enhanced_df.at[idx, 'predicted_home_score'] - row['actual_home_final']
                    enhanced_df.at[idx, 'away_score_error'] = enhanced_df.at[idx, 'predicted_away_score'] - row['actual_away_final']
                    enhanced_df.at[idx, 'home_error_pct'] = abs(enhanced_df.at[idx, 'home_score_error'] / row['actual_home_final']) * 100
                    enhanced_df.at[idx, 'away_error_pct'] = abs(enhanced_df.at[idx, 'away_score_error'] / row['actual_away_final']) * 100
            
        except Exception as e:
            logger.error(f"Error making predictions: {e}")
            traceback.print_exc()
        
        # Generate betting recommendations
        try:
            from models.dynamic_recommendation import generate_recommendations
            
            for idx, row in enhanced_df.iterrows():
                model_outputs = {
                    "win_probability": row.get('win_probability', 0.5),
                    "momentum_shift": row.get('cumulative_momentum', 0),
                    "projected_margin": row.get('predicted_home_score', 0) - row.get('predicted_away_score', 0),
                    "total_projected_score": row.get('predicted_home_score', 0) + row.get('predicted_away_score', 0),
                    "quarter": row.get('current_quarter', 0),
                    "time_remaining": row.get('time_remaining_mins', None)
                }
                
                try:
                    recommendations = generate_recommendations(model_outputs)
                    
                    # Store top recommendations
                    for rec_key, rec_value in recommendations.items():
                        enhanced_df.at[idx, f'rec_{rec_key}'] = rec_value
                except Exception as rec_error:
                    logger.warning(f"Error generating recommendations for game {row.get('game_id')}: {rec_error}")
        except ImportError:
            logger.warning("Recommendation module not available, skipping recommendations.")
        except Exception as e:
            logger.error(f"Error in recommendation generation: {e}")
        
        return enhanced_df
    
    def update_prediction_history(self, predictions_df):
        """Update the prediction history with new predictions."""
        if predictions_df.empty:
            return
            
        current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        for _, game in predictions_df.iterrows():
            game_key = f"{game['home_team']} vs {game['away_team']}"
            
            if game_key not in self.prediction_history:
                self.prediction_history[game_key] = []
            
            # Add current prediction
            self.prediction_history[game_key].append({
                'timestamp': current_time,
                'current_quarter': game['current_quarter'],
                'current_home_score': game.get('home_score', 0),
                'current_away_score': game.get('away_score', 0),
                'predicted_home_score': game['predicted_home_score'],
                'predicted_away_score': game['predicted_away_score'],
                'win_probability': game.get('win_probability', 0.5)
            })


class Visualizer:
    """
    Handles display and visualization of predictions.
    """
    
    def display_predictions(self, predictions_df, show_recommendations=True):
        """Display current predictions in a clean format."""
        if predictions_df.empty:
            print("\nNo predictions to display.")
            return
            
        print("\n===== PREDICTION RESULTS =====")
        
        for _, game in predictions_df.iterrows():
            current_q = game.get('current_quarter', 0)
            q_str = f"Quarter {current_q}" if current_q > 0 else "Pre-game"
            
            print(f"\n{game['home_team']} vs {game['away_team']} - {q_str}")
            
            if current_q > 0:
                print(f"Current: {game['home_team']} {game.get('home_score', 0):.0f} - " 
                      f"{game['away_team']} {game.get('away_score', 0):.0f}")
            
            print(f"Predicted: {game['home_team']} {game['predicted_home_score']:.1f} - " 
                  f"{game['away_team']} {game['predicted_away_score']:.1f}")
            
            if 'win_probability' in game:
                win_pct = game['win_probability'] * 100
                print(f"Win Probability: {win_pct:.1f}%")
            
            # If we have error metrics available, show them
            if 'home_score_error' in game:
                print(f"Validation - Prediction Error: {game['home_score_error']:.1f} points")
        
        # Display recommendations if requested
        if show_recommendations:
            print("\n===== GAME RECOMMENDATIONS =====")
            
            for _, game in predictions_df.iterrows():
                game_name = f"{game['home_team']} vs {game['away_team']}"
                print(f"\n{game_name}:")
                
                # Get all recommendation fields
                recs = {}
                for col in game.index:
                    if col.startswith('rec_'):
                        rec_key = col[4:]  # Remove 'rec_' prefix
                        if pd.notna(game[col]) and game[col]:
                            recs[rec_key] = game[col]
                
                if recs:
                    for rec_type, recommendation in recs.items():
                        print(f"  • {rec_type}: {recommendation}")
                else:
                    # Simple default recommendations if none available
                    score_diff = game.get('predicted_home_score', 0) - game.get('predicted_away_score', 0)
                    win_prob = game.get('win_probability', 0.5)
                    
                    if win_prob > 0.7:
                        print(f"  • betting_tip: Home team strong favorite.")
                    elif win_prob < 0.3:
                        print(f"  • betting_tip: Home team significant underdog.")
                    else:
                        print(f"  • betting_tip: Game appears competitive.")
                        
                    print(f"  • margin: Projected margin: {score_diff:.1f} points")
    
    def plot_prediction_history(self, prediction_history):
        """Plot the prediction history for all games."""
        if not prediction_history:
            print("No prediction history available to plot.")
            return
            
        # Check if we have at least one game with multiple predictions
        has_history = False
        for game_key, history in prediction_history.items():
            if len(history) > 1:
                has_history = True
                break
                
        if not has_history:
            print("Not enough history to plot trends.")
            return
            
        plt.figure(figsize=(12, 8))
        
        # For each game, plot prediction evolution
        for game_key, history in prediction_history.items():
            if len(history) > 1:
                times = range(len(history))
                home_preds = [p['predicted_home_score'] for p in history]
                away_preds = [p['predicted_away_score'] for p in history]
                
                # Extract team names
                teams = game_key.split(' vs ')
                home_team = teams[0]
                away_team = teams[1] if len(teams) > 1 else "Away"
                
                plt.plot(times, home_preds, 'o-', label=f"{home_team} (Home)")
                plt.plot(times, away_preds, 's-', label=f"{away_team} (Away)")
        
        plt.xlabel('Update Index')
        plt.ylabel('Predicted Final Score')
        plt.title('Prediction Evolution Over Time')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Add quarters as background
        quarters = None
        for game_key, history in prediction_history.items():
            if len(history) > 1:
                quarters = [h.get('current_quarter', 0) for h in history]
                break
                
        if quarters:
            last_q = -1
            colors = ['#f8f9fa', '#e9ecef', '#dee2e6', '#ced4da']
            
            for i, q in enumerate(quarters):
                if q != last_q:
                    plt.axvline(x=i, color='gray', linestyle='--', alpha=0.5)
                    last_q = q
        
        plt.tight_layout()
        plt.show()
        
        # Also plot win probability evolution if available
        has_win_prob = False
        for game_key, history in prediction_history.items():
            if len(history) > 1 and 'win_probability' in history[0]:
                has_win_prob = True
                break
                
        if has_win_prob:
            plt.figure(figsize=(12, 6))
            
            for game_key, history in prediction_history.items():
                if len(history) > 1 and 'win_probability' in history[0]:
                    times = range(len(history))
                    win_probs = [p.get('win_probability', 0.5) * 100 for p in history]
                    
                    # Extract team names
                    teams = game_key.split(' vs ')
                    home_team = teams[0]
                    away_team = teams[1] if len(teams) > 1 else "Away"
                    
                    plt.plot(times, win_probs, 'o-', label=f"{home_team} Win Probability")
            
            plt.xlabel('Update Index')
            plt.ylabel('Win Probability (%)')
            plt.title('Win Probability Evolution Over Time')
            plt.axhline(y=50, color='gray', linestyle='--', label='Even (50%)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.ylim(0, 100)
            plt.tight_layout()
            plt.show()
    
    def validate_predictions(self, predictions_df):
        """Validate predictions against actual results if available."""
        has_actuals = (
            not predictions_df.empty and 
            'actual_home_final' in predictions_df.columns and
            'actual_away_final' in predictions_df.columns
        )
        
        if not has_actuals:
            return
            
        print("\n===== VALIDATION RESULTS =====")
        
        # Calculate overall metrics
        home_errors = []
        away_errors = []
        
        for _, game in predictions_df.iterrows():
            if pd.isna(game['actual_home_final']) or pd.isna(game['actual_away_final']):
                continue
                
            home_error = abs(game['predicted_home_score'] - game['actual_home_final'])
            away_error = abs(game['predicted_away_score'] - game['actual_away_final'])
            
            home_errors.append(home_error)
            away_errors.append(away_error)
            
            # Print individual game results
            print(f"\n{game['home_team']} vs {game['away_team']} (Q{game['current_quarter']}):")
            print(f"  Predicted: {game['home_team']} {game['predicted_home_score']:.1f} - {game['away_team']} {game['predicted_away_score']:.1f}")
            print(f"  Actual:    {game['home_team']} {game['actual_home_final']:.1f} - {game['away_team']} {game['actual_away_final']:.1f}")
            print(f"  Error:     {home_error:.1f} pts ({home_error/game['actual_home_final']*100:.1f}%) / {away_error:.1f} pts ({away_error/game['actual_away_final']*100:.1f}%)")
        
        if home_errors and away_errors:
            # Calculate summary metrics
            avg_home_error = sum(home_errors) / len(home_errors)
            avg_away_error = sum(away_errors) / len(away_errors)
            avg_error = (avg_home_error + avg_away_error) / 2
            
            print("\nSummary:")
            print(f"  Average Home Error: {avg_home_error:.2f} points")
            print(f"  Average Away Error: {avg_away_error:.2f} points")
            print(f"  Overall Error:      {avg_error:.2f} points")


class DataStorage:
    """
    Handles saving and loading of results.
    """
    
    def __init__(self, output_dir=None):
        """Initialize with an output directory."""
        if output_dir is None:
            self.output_dir = Path("monitoring_output")
        else:
            self.output_dir = Path(output_dir)
            
        self.output_dir.mkdir(exist_ok=True)
    
    def save_results(self, predictions_df, prediction_history):
        """Save prediction results to disk."""
        if predictions_df.empty:
            return
            
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Save predictions to CSV
        predictions_file = self.output_dir / f"predictions_{timestamp}.csv"
        predictions_df.to_csv(predictions_file, index=False)
        logger.info(f"Saved predictions to {predictions_file}")
        
        # Save prediction history to JSON
        history_file = self.output_dir / f"prediction_history_{timestamp}.json"
        with open(history_file, 'w') as f:
            json.dump(prediction_history, f, indent=2)
        logger.info(f"Saved prediction history to {history_file}")


class GameMonitor:
    """
    Orchestrates the entire game monitoring pipeline.
    """
    
    def __init__(self, supabase_client=None, model_path=None):
        """Initialize the monitoring pipeline."""
        # Initialize components
        self.data_fetcher = DataFetcher(supabase_client)
        self.data_processor = DataProcessor()

In [ ]:
# Cell 17A – Integrated Live Dashboard (Dash-Based with Merged Features)

import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.graph_objs as go
import pandas as pd
import numpy as np
# Fix the datetime import by importing the classes directly
from datetime import datetime, timedelta
import pytz
import traceback

# Add the missing import for supabase
from caching.supabase_client import supabase

# ----- Timezone and Data Utilities -----
def get_current_time_pt():
    """Get current time in Pacific timezone."""
    pacific_tz = pytz.timezone("America/Los_Angeles")
    return datetime.now(pacific_tz)  # Now works with the fixed import

def fetch_live_games():
    """
    Fetch active live games from database.
    Returns DataFrame with live game data.
    """
    try:
        # Get current date in Pacific Time
        pacific_tz = pytz.timezone("America/Los_Angeles")
        now_pt = datetime.now(pacific_tz)  
        today_pt = now_pt.date()
        
        # Define start and end of today in PT
        start_pt = datetime.combine(today_pt, datetime.min.time()).replace(tzinfo=pacific_tz)
        end_pt = datetime.combine(today_pt, datetime.max.time()).replace(tzinfo=pacific_tz)
        
        # Convert to UTC for database query
        start_utc = start_pt.astimezone(pytz.UTC).isoformat()
        end_utc = end_pt.astimezone(pytz.UTC).isoformat()
        
        print(f"Fetching games for range: {start_utc} to {end_utc}")
        
        # Fetch all games for today
        response = supabase.table("nba_live_game_stats").select("*").gte("game_date", start_utc).lte("game_date", end_utc).execute()
        
        if not response.data:
            print("No games found for today")
            return pd.DataFrame()
            
        # Convert to DataFrame
        all_games = pd.DataFrame(response.data)
        
        # Process game data
        all_games = process_game_data(all_games)
        
        # Filter for active games
        active_games = all_games[all_games['game_status'] == 'live'].copy()
        
        if active_games.empty:
            print("No actively live games found")
            # If no active games, return all games
            return all_games
        
        print(f"Found {len(active_games)} active games")
        return active_games
        
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def process_game_data(df):
    """Process and determine game status for all games."""
    if df.empty:
        return df
    
    # Create a copy to avoid modifying original
    df = df.copy()
    
    # Convert date columns
    if 'game_date' in df.columns:
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce', utc=True)
        
    # Ensure numeric columns are numeric
    for col in ['home_score', 'away_score', 'current_quarter']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Make quarter columns numeric
    for q in range(1, 5):
        for team in ['home', 'away']:
            col = f'{team}_q{q}'
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Determine game status for each game
    df['game_status'] = 'unknown'  # Default status
    
    for idx, row in df.iterrows():
        # Check if game is marked as finished
        if row.get('is_finished', False):
            df.at[idx, 'game_status'] = 'finished'
            continue
            
        # Calculate highest quarter with data
        max_quarter = 0
        for q in range(1, 5):
            home_q = float(row.get(f'home_q{q}', 0) or 0)
            away_q = float(row.get(f'away_q{q}', 0) or 0)
            if home_q > 0 or away_q > 0:
                max_quarter = q
        
        # Update current_quarter if needed
        current_q = int(row.get('current_quarter', 0) or 0)
        if current_q < max_quarter:
            df.at[idx, 'current_quarter'] = max_quarter
            current_q = max_quarter
        
        # Determine status based on quarter data
        if current_q > 0:
            # If all quarters have data and it's Q4, likely finished
            if current_q == 4 and max_quarter == 4:
                df.at[idx, 'game_status'] = 'finished'
            else:
                df.at[idx, 'game_status'] = 'live'
        else:
            # No quarter data - scheduled game
            df.at[idx, 'game_status'] = 'upcoming'
        
        # Calculate scores from quarters if needed
        if current_q > 0:
            home_score = sum([float(row.get(f'home_q{q}', 0) or 0) for q in range(1, current_q + 1)])
            away_score = sum([float(row.get(f'away_q{q}', 0) or 0) for q in range(1, current_q + 1)])
            
            # Update scores if needed
            if row.get('home_score', 0) != home_score:
                df.at[idx, 'home_score'] = home_score
            if row.get('away_score', 0) != away_score:
                df.at[idx, 'away_score'] = away_score
    
    return df

# Custom prediction function
def predict_games_local(df):
    """Local prediction function for games."""
    print("Using local prediction implementation")
    
    if df.empty:
        return df
        
    result_df = df.copy()
    
    # Process each game
    for idx, row in result_df.iterrows():
        current_q = int(row.get('current_quarter', 0) or 0)
        home_score = float(row.get('home_score', 0) or 0)
        away_score = float(row.get('away_score', 0) or 0)
        game_status = row.get('game_status', '').lower()
        
        if game_status == 'finished':
            # For finished games, use the actual final score
            result_df.at[idx, 'predicted_home_score'] = home_score
            result_df.at[idx, 'predicted_away_score'] = away_score
            # Determine the winner after the fact
            result_df.at[idx, 'win_probability'] = 1.0 if home_score > away_score else 0.0
            
        elif game_status == 'live' and current_q > 0:
            # For live games, project based on quarters played
            quarters_played = current_q
            quarters_remaining = 4 - quarters_played
            
            # Calculate average points per quarter so far
            home_ppq = home_score / quarters_played if quarters_played > 0 else 0
            away_ppq = away_score / quarters_played if quarters_played > 0 else 0
            
            # Project remaining quarters (simple linear projection)
            projected_home = home_score + (home_ppq * quarters_remaining)
            projected_away = away_score + (away_ppq * quarters_remaining)
            
            # Adjust for typical 4th quarter scoring changes
            if current_q == 3:  # Going into 4th quarter
                # Teams typically score 3-5% differently in 4th quarters
                home_q4_factor = 1.03 if home_score > away_score else 0.97
                away_q4_factor = 1.03 if away_score > home_score else 0.97
                
                # Apply the adjustment
                projected_home = home_score + (home_ppq * quarters_remaining * home_q4_factor)
                projected_away = away_score + (away_ppq * quarters_remaining * away_q4_factor)
            
            # Save projections
            result_df.at[idx, 'predicted_home_score'] = round(projected_home, 1)
            result_df.at[idx, 'predicted_away_score'] = round(projected_away, 1)
            
            # Calculate win probability based on lead and time remaining
            score_diff = home_score - away_score
            time_played_pct = current_q / 4.0
            
            # Simple win probability model
            if score_diff == 0:
                win_prob = 0.5  # Tied game
            else:
                # Larger leads later in the game are more significant
                normalized_diff = score_diff / (10 * (1 - time_played_pct + 0.1))
                win_prob = 1 / (1 + np.exp(-normalized_diff))  # Sigmoid function
            
            result_df.at[idx, 'win_probability'] = win_prob
            
        else:
            # Pre-game - use league averages with slight home court advantage
            result_df.at[idx, 'predicted_home_score'] = 110.5  # Average home score
            result_df.at[idx, 'predicted_away_score'] = 105.8  # Average away score
            result_df.at[idx, 'win_probability'] = 0.6  # Home court advantage
    
    # Add confidence and recommendations
    for idx, row in result_df.iterrows():
        game_status = row.get('game_status', '').lower()
        
        # Add prediction confidence based on stage of game
        if game_status == 'finished':
            result_df.at[idx, 'prediction_confidence'] = 1.0  # 100% confidence for finished games
        elif game_status == 'live':
            # Confidence increases as game progresses
            current_q = int(row.get('current_quarter', 0) or 0)
            result_df.at[idx, 'prediction_confidence'] = min(0.5 + (current_q * 0.125), 0.95)
        else:
            result_df.at[idx, 'prediction_confidence'] = 0.65  # Base confidence for pre-game
    
    return result_df

def get_dashboard_data():
    """
    Get real dashboard data from the database.
    Returns a processed DataFrame with game information.
    """
    try:
        # Fetch live games
        df = fetch_live_games()
        
        if df.empty:
            print("No games available for dashboard")
            return create_dummy_data()
        
        # Add timestamp column
        df['timestamp'] = get_current_time_pt().strftime("%Y-%m-%d %H:%M:%S")
        
        # Ensure required columns
        required_cols = [
            'game_id', 'home_team', 'away_team', 'game_status', 
            'current_quarter', 'home_score', 'away_score'
        ]
        
        for col in required_cols:
            if col not in df.columns:
                if col in ['home_score', 'away_score', 'current_quarter']:
                    df[col] = 0
                else:
                    df[col] = "Unknown"
        
        # Always use the local prediction implementation
        return predict_games_local(df)
    except Exception as e:
        print(f"Error in get_dashboard_data: {e}")
        traceback.print_exc()
        # Make sure we always return something
        return create_dummy_data()

def create_dummy_data():
    """Create dummy data when no real data is available."""
    try:
        now = get_current_time_pt().strftime("%Y-%m-%d %H:%M:%S")
        data = {
            "game_id": ["game_1", "game_2", "game_3"],
            "home_team": ["Lakers", "Bulls", "Celtics"],
            "away_team": ["Warriors", "Heat", "Knicks"],
            "game_status": ["upcoming", "upcoming", "upcoming"],
            "current_quarter": [0, 0, 0],
            "home_score": [0, 0, 0],
            "away_score": [0, 0, 0],
            "predicted_home_score": [108, 105, 110],
            "predicted_away_score": [102, 103, 112],
            "win_probability": [0.55, 0.52, 0.45],
            "timestamp": [now, now, now]
        }
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error creating dummy data: {e}")
        # Absolute last resort - hardcoded dummy data with no time function calls
        data = {
            "game_id": ["emergency_fallback"],
            "home_team": ["Home Team"],
            "away_team": ["Away Team"],
            "game_status": ["upcoming"],
            "current_quarter": [0],
            "home_score": [0],
            "away_score": [0],
            "predicted_home_score": [100],
            "predicted_away_score": [100],
            "win_probability": [0.5],
            "timestamp": ["Emergency fallback data"]
        }
        return pd.DataFrame(data)

def get_prediction_history(game_id):
    """
    Get prediction history for a specific game.
    In production, this would fetch from a database.
    """
    try:
        # Try to fetch from history table or create realistic history
        # For now, using a simple simulation
        game_data = get_dashboard_data()
        game = game_data[game_data['game_id'] == game_id]
        
        if game.empty:
            return []
            
        game = game.iloc[0]
        current_quarter = int(game.get('current_quarter', 0))
        
        if current_quarter <= 0:
            return []  # No history for upcoming games
            
        # Create simulated history
        history = []
        
        # For each quarter up to current
        for q in range(1, current_quarter + 1):
            # Simulate predictions getting more accurate as game progresses
            accuracy_factor = q / 4  # Gets closer to 1 as game progresses
            noise_factor = 1 - accuracy_factor
            
            # Final predictions as reference
            final_home = float(game.get('predicted_home_score', 100))
            final_away = float(game.get('predicted_away_score', 100))
            final_wp = float(game.get('win_probability', 0.5))
            
            # Add some noise to early predictions
            home_noise = np.random.normal(0, 10 * noise_factor)
            away_noise = np.random.normal(0, 10 * noise_factor)
            wp_noise = np.random.normal(0, 0.15 * noise_factor)
            
            history.append({
                "current_quarter": q,
                "predicted_home_score": final_home + home_noise,
                "predicted_away_score": final_away + away_noise,
                "win_probability": max(0.01, min(0.99, final_wp + wp_noise)),
                "timestamp": (get_current_time_pt() - timedelta(minutes=15*(current_quarter-q))).strftime("%H:%M:%S")
            })
            
        return history
            
    except Exception as e:
        print(f"Error getting prediction history: {e}")
        traceback.print_exc()
        return []

# ----- Dash App Setup -----
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server  # For deployment purposes

# Get initial dashboard data
try:
    df_dashboard = get_dashboard_data()
    available_games = df_dashboard["game_id"].unique()
    initial_game = available_games[0] if len(available_games) > 0 else None
except Exception as e:
    print(f"Error getting initial data: {e}")
    df_dashboard = create_dummy_data()
    available_games = df_dashboard["game_id"].unique()
    initial_game = available_games[0] if len(available_games) > 0 else None

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H2("NBA Score Prediction Dashboard"), width=12)
    ]),
    dbc.Row([
        dbc.Col(
            dcc.Dropdown(
                id='game-dropdown',
                options=[{'label': f"{row['home_team']} vs {row['away_team']} ({row['game_status'].upper()})", 
                          'value': row['game_id']} for idx, row in df_dashboard.iterrows()],
                value=initial_game,
                clearable=False,
                style={"width": "100%"}
            ),
            width=6
        ),
        dbc.Col(
            html.Div(id='last-update', style={'textAlign': 'right', 'fontSize': '16px'}),
            width=6
        )
    ], className="my-2"),
    dbc.Row([
        dbc.Col(
            html.Div(id='game-status-indicator', className="alert alert-info"),
            width=12
        )
    ], className="my-2"),
    dbc.Row([
        dbc.Col(dcc.Graph(id='main-dashboard-graph'), width=12)
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id='history-graph'), width=12)
    ]),
    dcc.Interval(
        id='interval-component',
        interval=30*1000,  # Refresh every 30 seconds
        n_intervals=0
    )
], fluid=True)

# ----- Callback to Update Dashboard -----
@app.callback(
    [Output('main-dashboard-graph', 'figure'),
     Output('history-graph', 'figure'),
     Output('last-update', 'children'),
     Output('game-dropdown', 'options'),
     Output('game-status-indicator', 'children'),
     Output('game-status-indicator', 'className')],
    [Input('interval-component', 'n_intervals'),
     Input('game-dropdown', 'value')]
)
def update_dashboard(n_intervals, selected_game):
    try:
        # Fetch fresh dashboard data
        df = get_dashboard_data()
        
        # Update dropdown options
        dropdown_options = [
            {'label': f"{row['home_team']} vs {row['away_team']} ({row['game_status'].upper()})", 
            'value': row['game_id']} 
            for idx, row in df.iterrows()
        ]
        
        # Handle case when no games are available
        if df.empty or selected_game is None:
            empty_fig = go.Figure()
            empty_fig.update_layout(title="No game data available")
            return (empty_fig, empty_fig, "No data available", dropdown_options,
                    "No active games found", "alert alert-warning")
        
        # Filter for selected game
        game_data = df[df['game_id'] == selected_game]
        
        # If selected game is not in current data, take the first available game
        if game_data.empty and len(df) > 0:
            selected_game = df.iloc[0]['game_id']
            game_data = df[df['game_id'] == selected_game]
        
        if game_data.empty:
            empty_fig = go.Figure()
            empty_fig.update_layout(title="No game data available")
            return (empty_fig, empty_fig, "No data available", dropdown_options,
                    "Selected game not found", "alert alert-warning")
        
        game = game_data.iloc[0]
        timestamp = game.get('timestamp', get_current_time_pt().strftime("%Y-%m-%d %H:%M:%S"))
        
        # Create Main Graph: Bar Chart for Current vs Predicted Scores
        teams = [game['home_team'], game['away_team']]
        current_scores = [game['home_score'], game['away_score']]
        predicted_scores = [game['predicted_home_score'], game['predicted_away_score']]
        
        # Status indicator text and class
        game_status = game['game_status'].upper()
        current_q = int(game.get('current_quarter', 0))
        
        if game_status == 'LIVE':
            status_class = "alert alert-success"
            status_text = f"LIVE - Quarter {current_q}"
        elif game_status == 'FINISHED':
            status_class = "alert alert-secondary"
            status_text = "FINISHED"
        else:
            status_class = "alert alert-info"
            status_text = "UPCOMING"
        
        main_fig = go.Figure(data=[
            go.Bar(name='Current Score', x=teams, y=current_scores, marker_color='skyblue'),
            go.Bar(name='Predicted Final', x=teams, y=predicted_scores, marker_color='salmon')
        ])
        
        title = f"{game['home_team']} vs {game['away_team']} - {game_status}"
        if current_q > 0:
            title += f" (Q{current_q})"
            
        main_fig.update_layout(
            barmode='group', 
            title=title,
            xaxis_title="Team", 
            yaxis_title="Score"
        )
        
        # Add score labels to bars
        for i, (team, score, pred) in enumerate(zip(teams, current_scores, predicted_scores)):
            main_fig.add_annotation(
                x=team, y=score,
                text=f"{score:.0f}",
                showarrow=False,
                yshift=10
            )
            main_fig.add_annotation(
                x=team, y=pred,
                text=f"{pred:.1f}",
                showarrow=False,
                yshift=10
            )
        
        # Add Win Probability annotation if available
        if game_status.lower() == 'live' and 'win_probability' in game:
            win_prob = game['win_probability'] * 100
            prob_text = (f"{game['home_team']} Win: {win_prob:.1f}%" if win_prob > 50 
                        else f"{game['away_team']} Win: {100-win_prob:.1f}%")
            main_fig.add_annotation(
                x=0.5, y=0.05, 
                xref="paper", yref="paper",
                text=prob_text, 
                showarrow=False,
                font=dict(color="black", size=14),
                bgcolor="lightyellow", 
                opacity=0.8
            )
        
        # Create History Graph: Line Chart of Prediction Evolution
        history = get_prediction_history(selected_game)
        
        if history and len(history) >= 2:
            quarters = [pt["current_quarter"] for pt in history]
            home_preds = [pt["predicted_home_score"] for pt in history]
            away_preds = [pt["predicted_away_score"] for pt in history]
            win_probs = [pt["win_probability"] for pt in history]
            
            history_fig = go.Figure()
            history_fig.add_trace(go.Scatter(
                x=quarters, y=home_preds,
                mode='lines+markers',
                name=f"{game['home_team']} Prediction",
                line=dict(color='blue')
            ))
            history_fig.add_trace(go.Scatter(
                x=quarters, y=away_preds,
                mode='lines+markers',
                name=f"{game['away_team']} Prediction",
                line=dict(color='red')
            ))
            
            # Secondary axis for win probability
            history_fig.add_trace(go.Scatter(
                x=quarters, y=[wp * 100 for wp in win_probs],
                mode='lines+markers',
                name="Win Probability (%)",
                line=dict(color='green', dash='dash'),
                yaxis='y2'
            ))
            
            history_fig.update_layout(
                title="Prediction Evolution by Quarter",
                xaxis_title="Quarter",
                yaxis_title="Predicted Score",
                yaxis2=dict(
                    overlaying='y',
                    side='right',
                    title="Win Probability (%)",
                    range=[0, 100]
                )
            )
        else:
            history_fig = go.Figure()
            
            if game_status.lower() == 'upcoming':
                msg = "Game has not started - No prediction history available"
            else:
                msg = "No prediction history available"
                
            history_fig.update_layout(title=msg)
        
        last_update_text = f"Last Update: {timestamp}"
        
        return main_fig, history_fig, last_update_text, dropdown_options, status_text, status_class
    
    except Exception as e:
        print(f"Error in dashboard update: {e}")
        traceback.print_exc()
        # Create a fallback response
        empty_fig = go.Figure()
        empty_fig.update_layout(title="Error loading dashboard data")
        return (
            empty_fig, empty_fig, 
            f"Error: {str(e)}", 
            [{'label': 'Error', 'value': 'error'}],
            "Dashboard Error", 
            "alert alert-danger"
        )

# ----- To Run the Dashboard (for standalone execution) -----
if __name__ == '__main__':
    app.run_server(debug=True)

In [ ]:
# Cell 17B – Enhanced Visualization Components

import plotly.graph_objs as go
import numpy as np

def create_prediction_evolution_chart(data, actual_data=None, win_prob_data=None, confidence_intervals=None):
    """
    Create an enhanced prediction evolution chart.
    
    Parameters:
      - data: List/array of prediction values over time.
      - actual_data: (Optional) List/array of actual scores.
      - win_prob_data: (Optional) List/array of win probability values.
      - confidence_intervals: (Optional) Tuple (lower_bounds, upper_bounds) arrays for confidence intervals.
    
    Returns:
      - fig: A Plotly figure object.
    """
    time_points = list(range(len(data)))
    
    fig = go.Figure()
    
    # Plot prediction evolution.
    fig.add_trace(go.Scatter(
        x=time_points,
        y=data,
        mode='lines+markers',
        name='Predicted Score',
        line=dict(width=2)
    ))
    
    # Plot actual scores if provided.
    if actual_data is not None:
        fig.add_trace(go.Scatter(
            x=time_points,
            y=actual_data,
            mode='lines+markers',
            name='Actual Score',
            line=dict(width=2, dash='dash')
        ))
    
    # Plot win probability trend if provided.
    if win_prob_data is not None:
        fig.add_trace(go.Scatter(
            x=time_points,
            y=win_prob_data,
            mode='lines+markers',
            name='Win Probability',
            yaxis='y2',
            line=dict(width=2, color='green')
        ))
        # Set up a secondary y-axis.
        fig.update_layout(
            yaxis2=dict(
                title='Win Probability',
                overlaying='y',
                side='right',
                range=[0, 1]
            )
        )
    
    # Plot confidence intervals if provided.
    if confidence_intervals is not None:
        lower_bounds, upper_bounds = confidence_intervals
        fig.add_trace(go.Scatter(
            x=time_points + time_points[::-1],
            y=upper_bounds + lower_bounds[::-1],
            fill='toself',
            fillcolor='rgba(0,100,80,0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=True,
            name='Confidence Interval'
        ))
    
    # Add a dynamic confidence indicator annotation.
    if confidence_intervals is not None:
        last_confidence = upper_bounds[-1] - lower_bounds[-1]
        fig.add_annotation(
            x=time_points[-1],
            y=data[-1],
            text=f"Confidence: {last_confidence:.2f}",
            showarrow=True,
            arrowhead=1
        )
    
    fig.update_layout(
        title="Prediction Evolution Over Game Time",
        xaxis_title="Time Point (Interval/Quarter)",
        yaxis_title="Score",
        legend_title="Metrics",
        margin=dict(l=40, r=40, t=50, b=40)
    )
    
    return fig

# Example usage:
# predictions = [100, 102, 105, 107, 110]
# actual_scores = [98, 103, 106, 108, 111]
# win_probabilities = [0.55, 0.57, 0.60, 0.63, 0.65]
# lower_conf = [98, 100, 102, 104, 107]
# upper_conf = [102, 104, 108, 110, 113]
# fig = create_prediction_evolution_chart(predictions, actual_scores, win_probabilities, (lower_conf, upper_conf))
# fig.show()


In [ ]:
# Cell 19 - Visualize Feature Importance and Test Quarter-Specific Performance

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import os

print("Loading NBA game data...")

# Safely attempt to get an existing DataFrame from globals
working_df = None
for df_name in ['games_df', 'preprocessed_df', 'training_df', 'df']:
    if df_name in globals():
        working_df = globals()[df_name]
        print(f"Using existing {df_name} from previous cells")
        break

if working_df is None:
    # No existing DataFrame found, try to load from file
    data_path = "./data/nba_games.csv"  # Adjust path as needed
    if os.path.exists(data_path):
        working_df = pd.read_csv(data_path)
        print(f"Loaded data from {data_path}: {working_df.shape[0]} rows, {working_df.shape[1]} columns")
    else:
        # Create synthetic NBA game data for testing
        print("No data found in memory or file. Creating sample NBA game data for testing.")
        np.random.seed(42)
        n_samples = 1000
        working_df = pd.DataFrame({
            'game_id': range(1, n_samples + 1),
            'home_q1': np.random.randint(15, 35, n_samples),
            'home_q2': np.random.randint(15, 35, n_samples),
            'home_q3': np.random.randint(15, 35, n_samples),
            'home_q4': np.random.randint(15, 35, n_samples),
            'away_q1': np.random.randint(15, 35, n_samples),
            'away_q2': np.random.randint(15, 35, n_samples),
            'away_q3': np.random.randint(15, 35, n_samples),
            'away_q4': np.random.randint(15, 35, n_samples),
            'home_rest_days': np.random.randint(0, 5, n_samples),
            'away_rest_days': np.random.randint(0, 5, n_samples),
            'is_home_b2b': np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
            'is_away_b2b': np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
            'prev_matchup_diff': np.random.normal(0, 10, n_samples),
            # Add missing features that the model might expect
            'rolling_home_score': np.random.randint(80, 120, n_samples),
            'rolling_away_score': np.random.randint(80, 120, n_samples),
        })
        # Calculate additional features
        working_df['rest_advantage'] = working_df['home_rest_days'] - working_df['away_rest_days']
        working_df['home_total'] = working_df[['home_q1', 'home_q2', 'home_q3', 'home_q4']].sum(axis=1)
        working_df['away_total'] = working_df[['away_q1', 'away_q2', 'away_q3', 'away_q4']].sum(axis=1)
        working_df['final_score_diff'] = working_df['home_total'] - working_df['away_total']
        
        # Add momentum features
        working_df['q1_diff'] = working_df['home_q1'] - working_df['away_q1']
        working_df['q2_diff'] = working_df['home_q2'] - working_df['away_q2']
        working_df['q3_diff'] = working_df['home_q3'] - working_df['away_q3']
        working_df['q4_diff'] = working_df['home_q4'] - working_df['away_q4']
        working_df['q1_to_q2_momentum'] = working_df['q2_diff'] - working_df['q1_diff']
        working_df['q2_to_q3_momentum'] = working_df['q3_diff'] - working_df['q2_diff']
        working_df['q3_to_q4_momentum'] = working_df['q4_diff'] - working_df['q3_diff']
        working_df['cumulative_momentum'] = (working_df['q1_to_q2_momentum'] * 0.2 + 
                                             working_df['q2_to_q3_momentum'] * 0.3 + 
                                             working_df['q3_to_q4_momentum'] * 0.5)
        # Add game state features
        half_time_home = working_df['home_q1'] + working_df['home_q2']
        half_time_away = working_df['away_q1'] + working_df['away_q2']
        working_df['score_differential'] = half_time_home - half_time_away
        working_df['score_ratio'] = half_time_home / (half_time_home + half_time_away)
        print(f"Created sample data with {working_df.shape[0]} rows, {working_df.shape[1]} columns")

# Ensure we have the required columns for target calculation
# First, check if we can compute home_total and away_total if they don't exist
if 'home_total' not in working_df.columns and all(f'home_q{i}' in working_df.columns for i in range(1, 5)):
    working_df['home_total'] = working_df[['home_q1', 'home_q2', 'home_q3', 'home_q4']].sum(axis=1)
    print("Calculated home_total from quarter data")
    
if 'away_total' not in working_df.columns and all(f'away_q{i}' in working_df.columns for i in range(1, 5)):
    working_df['away_total'] = working_df[['away_q1', 'away_q2', 'away_q3', 'away_q4']].sum(axis=1)
    print("Calculated away_total from quarter data")

# Define target variable based on the NBA score prediction context
if 'final_score_diff' in working_df.columns:
    target_column = 'final_score_diff'
    print(f"Using existing column '{target_column}' as target")
elif 'score_differential' in working_df.columns:
    target_column = 'score_differential'
    print(f"Using existing column '{target_column}' as target")
elif 'home_total' in working_df.columns and 'away_total' in working_df.columns:
    working_df['final_score_diff'] = working_df['home_total'] - working_df['away_total']
    target_column = 'final_score_diff'
    print(f"Created and using '{target_column}' as target")
else:
    # As a last resort, if we have quarter data, create what we need
    if all(f'home_q{i}' in working_df.columns for i in range(1, 5)) and all(f'away_q{i}' in working_df.columns for i in range(1, 5)):
        working_df['home_total'] = working_df[['home_q1', 'home_q2', 'home_q3', 'home_q4']].sum(axis=1)
        working_df['away_total'] = working_df[['away_q1', 'away_q2', 'away_q3', 'away_q4']].sum(axis=1)
        working_df['final_score_diff'] = working_df['home_total'] - working_df['away_total']
        target_column = 'final_score_diff'
        print(f"Created and using '{target_column}' as target")
    else:
        # If we still can't create a target, show what columns we do have
        print("Available columns:", working_df.columns.tolist())
        raise KeyError("No valid target column found in the data and unable to create one from available columns.")

# Define features if not already defined
features_defined = 'features' in globals() and len(features) > 0

# Check if model exists and has feature_names_in_ attribute
model_has_features = 'model' in globals() and hasattr(model, 'feature_names_in_')

if model_has_features:
    print("Using features from the existing model")
    required_features = model.feature_names_in_.tolist()
    
    # Check if all required features exist in the DataFrame
    missing_features = [f for f in required_features if f not in working_df.columns]
    if missing_features:
        print(f"Warning: Missing features in data: {missing_features}")
        print("Creating missing features with default values")
        for feature in missing_features:
            # Use appropriate default values based on feature name
            if 'rolling' in feature:
                working_df[feature] = np.random.randint(80, 120, len(working_df))
            else:
                working_df[feature] = 0
    
    features = required_features
    print(f"Using {len(features)} features from the model")
    
elif not features_defined:
    # Use features relevant to NBA score prediction based on your project
    score_features = [col for col in working_df.columns if col.startswith('home_q') or col.startswith('away_q')]
    game_state_features = ['score_ratio', 'score_differential'] if 'score_ratio' in working_df.columns else []
    momentum_features = ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'] if 'q1_to_q2_momentum' in working_df.columns else []
    rest_features = ['home_rest_days', 'away_rest_days', 'rest_advantage', 'is_home_b2b', 'is_away_b2b'] if 'home_rest_days' in working_df.columns else []
    matchup_features = ['prev_matchup_diff'] if 'prev_matchup_diff' in working_df.columns else []
    rolling_features = ['rolling_home_score', 'rolling_away_score'] if 'rolling_home_score' in working_df.columns else []
    
    # Combine all feature groups
    features = score_features + game_state_features + momentum_features + rest_features + matchup_features + rolling_features
    # Ensure features exist in the DataFrame
    features = [f for f in features if f in working_df.columns]
    
    if not features:
        raise ValueError("No valid features found in the DataFrame. Check your column names.")
    
    print(f"Using {len(features)} automatically selected features for analysis.")

# Create train/test split if not already done
if 'X_test' not in globals() or 'y_test' not in globals():
    print("Creating train/test split...")
    X = working_df[features]
    y = working_df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # If no model exists yet, train one
    if 'model' not in globals():
        from sklearn.ensemble import GradientBoostingRegressor
        print("Training GradientBoostingRegressor model...")
        model = GradientBoostingRegressor(random_state=42)
        model.fit(X_train, y_train)
        
        # Display initial model performance
        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)
        train_mse = mean_squared_error(y_train, train_preds)
        test_mse = mean_squared_error(y_test, test_preds)
        print(f"Model trained. Train MSE: {train_mse:.4f}, Test MSE: {test_mse:.4f}, Test RMSE: {np.sqrt(test_mse):.4f}")

# Visualize feature importance if the model provides it
if 'model' in globals() and hasattr(model, 'feature_importances_'):
    # Diagnose feature length mismatch
    feature_len = len(features)
    importance_len = len(model.feature_importances_)
    print(f"Features length: {feature_len}, Feature importances length: {importance_len}")
    
    # Handle length mismatch
    if feature_len != importance_len:
        if hasattr(model, 'feature_names_in_'):
            print("Using model's feature_names_in_ attribute")
            features = model.feature_names_in_.tolist()
        elif feature_len > importance_len:
            print("Truncating features list to match feature_importances_")
            features = features[:importance_len]
        else:
            print("Adding placeholder features to match feature_importances_")
            features = features + [f"Unknown_{i}" for i in range(feature_len, importance_len)]
    
    # Create DataFrame column by column to avoid length issues
    feature_importances = pd.DataFrame()
    feature_importances['Feature'] = features
    feature_importances['Importance'] = model.feature_importances_
    feature_importances = feature_importances.sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importances.head(15))
    plt.title('Top 15 Features by Importance in Enhanced In-Game Model')
    plt.tight_layout()
    plt.show()
    
    # Group features by type for a pie chart visualization
    feature_groups = {
        'Quarter Scores': [col for col in features if col.startswith('home_q') or col.startswith('away_q')],
        'Game State': [col for col in features if col in ['score_ratio', 'score_differential']],
        'Matchup History': [col for col in features if col in ['prev_matchup_diff']],
        'Rest': [col for col in features if col in ['home_rest_days', 'away_rest_days', 'rest_advantage', 'is_home_b2b', 'is_away_b2b']],
        'Momentum': [col for col in features if col in ['q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum']],
        'Rolling Stats': [col for col in features if 'rolling' in col]
    }
    
    group_importance = {}
    for group, feats in feature_groups.items():
        valid_feats = [f for f in feats if f in features]
        if valid_feats:
            group_importance[group] = sum(
                model.feature_importances_[features.index(f)] for f in valid_feats
            )
    
    if group_importance:
        plt.figure(figsize=(10, 8))
        plt.pie(
            group_importance.values(), 
            labels=group_importance.keys(), 
            autopct='%1.1f%%', 
            shadow=True, 
            startangle=140
        )
        plt.axis('equal')
        plt.title('Feature Group Contribution to In-Game Predictions')
        plt.tight_layout()
        plt.show()

# Analyze model performance by quarter
print("\nAnalyzing performance by quarter...")

quarter_analysis = []
for current_quarter in range(1, 5):
    # Create a copy of the test data for this quarter analysis
    quarter_X_test = X_test.copy()
    
    # Get the expected feature names from the model
    if hasattr(model, 'feature_names_in_'):
        required_features = model.feature_names_in_
        
        # Create a new DataFrame with all the required features
        new_quarter_X_test = pd.DataFrame()
        for feature in required_features:
            if feature in quarter_X_test.columns:
                new_quarter_X_test[feature] = quarter_X_test[feature]
            else:
                # Feature is missing, add it with zeros
                print(f"Adding missing feature '{feature}' to quarter_X_test")
                new_quarter_X_test[feature] = 0
                
        quarter_X_test = new_quarter_X_test
    
    # Zero out data for future quarters
    for q in range(current_quarter + 1, 5):
        for col_prefix in ['home_q', 'away_q']:
            col_name = f"{col_prefix}{q}"
            if col_name in quarter_X_test.columns:
                quarter_X_test[col_name] = 0
    
    # Zero out momentum features not available yet
    if 'q1_to_q2_momentum' in quarter_X_test.columns and current_quarter < 2:
        quarter_X_test['q1_to_q2_momentum'] = 0
    if 'q2_to_q3_momentum' in quarter_X_test.columns and current_quarter < 3:
        quarter_X_test['q2_to_q3_momentum'] = 0
    if 'q3_to_q4_momentum' in quarter_X_test.columns and current_quarter < 4:
        quarter_X_test['q3_to_q4_momentum'] = 0
    
    # Adjust cumulative momentum based on available quarters
    if 'cumulative_momentum' in quarter_X_test.columns:
        if current_quarter < 2:
            quarter_X_test['cumulative_momentum'] = 0
        elif current_quarter < 3:
            if 'q1_to_q2_momentum' in quarter_X_test.columns:
                quarter_X_test['cumulative_momentum'] = quarter_X_test['q1_to_q2_momentum']
        elif current_quarter < 4:
            if 'q1_to_q2_momentum' in quarter_X_test.columns and 'q2_to_q3_momentum' in quarter_X_test.columns:
                quarter_X_test['cumulative_momentum'] = (
                    quarter_X_test['q1_to_q2_momentum'] * 0.2 + quarter_X_test['q2_to_q3_momentum'] * 0.3
                ) / 0.5
    
    # Ensure all feature columns are in the same order as expected by the model
    if hasattr(model, 'feature_names_in_'):
        quarter_X_test = quarter_X_test[model.feature_names_in_]
    
    # Generate predictions for the current quarter scenario
    try:
        quarter_preds = model.predict(quarter_X_test)
        quarter_mse = mean_squared_error(y_test, quarter_preds)
        quarter_mae = np.mean(np.abs(y_test - quarter_preds))
        
        quarter_analysis.append({
            'Quarter': current_quarter,
            'MSE': quarter_mse,
            'MAE': quarter_mae,
            'RMSE': np.sqrt(quarter_mse)
        })
    except Exception as e:
        print(f"Error predicting for quarter {current_quarter}: {e}")
        # Print detailed debugging information
        if hasattr(model, 'feature_names_in_'):
            print("Model expects these features:", model.feature_names_in_)
        print("Features in quarter_X_test:", quarter_X_test.columns.tolist())
        print("Shape of quarter_X_test:", quarter_X_test.shape)
        missing = [f for f in model.feature_names_in_ if f not in quarter_X_test.columns]
        if missing:
            print("Missing features:", missing)

if quarter_analysis:
    quarter_df = pd.DataFrame(quarter_analysis)
    print(quarter_df)
    
    plt.figure(figsize=(10, 6))
    plt.bar(quarter_df['Quarter'], quarter_df['RMSE'], color='salmon')
    plt.xlabel('Information Available Through Quarter')
    plt.ylabel('Root Mean Square Error')
    plt.title('Prediction Error by Available Quarter Information')
    plt.xticks([1, 2, 3, 4])
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No quarter analysis data was generated due to errors.")

In [ ]:
# Cell 19A – Anomaly Detection & Alerts Implementation

import smtplib
from email.mime.text import MIMEText
import logging

class AnomalyDetector:
    def __init__(self, swing_threshold=10, confidence_threshold=0.5):
        """
        Initialize the AnomalyDetector.
        :param swing_threshold: The minimum absolute difference between consecutive predictions that is considered anomalous.
        :param confidence_threshold: The minimum acceptable prediction confidence value.
        """
        self.swing_threshold = swing_threshold
        self.confidence_threshold = confidence_threshold
        self.previous_prediction = None

    def detect(self, current_prediction, current_confidence):
        """
        Detect anomalies based on unusual prediction swings or low confidence.
        :param current_prediction: Current prediction value (numeric).
        :param current_confidence: Current prediction confidence (numeric between 0 and 1).
        :return: List of alert messages (empty if no anomaly detected).
        """
        alerts = []
        if self.previous_prediction is not None:
            swing = abs(current_prediction - self.previous_prediction)
            if swing > self.swing_threshold:
                alerts.append(f"Unusual prediction swing detected: {swing} (Threshold: {self.swing_threshold})")
        if current_confidence < self.confidence_threshold:
            alerts.append(f"Low prediction confidence detected: {current_confidence} (Threshold: {self.confidence_threshold})")
        
        # Update the previous prediction for next detection
        self.previous_prediction = current_prediction
        return alerts

class Notifier:
    def __init__(self, smtp_server, smtp_port, username, password, from_addr, to_addrs):
        """
        Initialize the Notifier with SMTP settings.
        :param smtp_server: SMTP server address.
        :param smtp_port: SMTP server port.
        :param username: SMTP username.
        :param password: SMTP password.
        :param from_addr: Email address to send from.
        :param to_addrs: List of email addresses to notify.
        """
        self.smtp_server = smtp_server
        self.smtp_port = smtp_port
        self.username = username
        self.password = password
        self.from_addr = from_addr
        self.to_addrs = to_addrs

    def send_alert(self, subject, message):
        """
        Send an alert email.
        :param subject: Subject line for the email.
        :param message: Body message for the alert.
        """
        msg = MIMEText(message)
        msg['Subject'] = subject
        msg['From'] = self.from_addr
        msg['To'] = ", ".join(self.to_addrs)
        try:
            with smtplib.SMTP(self.smtp_server, self.smtp_port) as server:
                server.starttls()
                server.login(self.username, self.password)
                server.sendmail(self.from_addr, self.to_addrs, msg.as_string())
            logging.info("Alert email sent successfully.")
        except Exception as e:
            logging.error(f"Failed to send alert email: {e}")

# Example integration within your live monitoring loop:
if __name__ == "__main__":
    # Setup logging for anomaly detection
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    
    # Instantiate anomaly detector and notifier
    anomaly_detector = AnomalyDetector(swing_threshold=10, confidence_threshold=0.5)
    notifier = Notifier(
        smtp_server="smtp.example.com",  # Update with your SMTP server
        smtp_port=587,                   # Update with your SMTP port
        username="user@example.com",     # Update with your SMTP username
        password="password",             # Update with your SMTP password
        from_addr="user@example.com",    # Update with your from email address
        to_addrs=["alert@example.com"]   # Update with one or more recipient emails
    )
    
    # Simulated predictions and confidence values (replace with real-time data in production)
    simulated_predictions = [100, 105, 120, 115, 130]
    simulated_confidences = [0.9, 0.8, 0.4, 0.85, 0.95]
    
    for pred, conf in zip(simulated_predictions, simulated_confidences):
        alerts = anomaly_detector.detect(pred, conf)
        if alerts:
            alert_message = "\n".join(alerts)
            logging.warning(f"Anomaly Detected: {alert_message}")
            notifier.send_alert("Critical Anomaly Detected", alert_message)
        else:
            logging.info("No anomalies detected.")

In [ ]:
# Cell 20 -- Fetch scheduled games

import pandas as pd
import pytz
from datetime import datetime, timedelta
import os
import json
import random

# Initialize Supabase client
try:
    # Try to import from the project structure
    try:
        from backend.caching.supabase_client import supabase  # type: ignore
        print("Imported supabase client from project structure")
    except ImportError:
        # If that fails, try to import from the current directory
        try:
            from supabase_client import supabase  # type: ignore
            print("Imported supabase client from current directory")
        except ImportError:
            # If both imports fail, check if credentials exist as environment variables
            from supabase import create_client  # Make sure the 'supabase' package is installed
            supabase_url = os.environ.get("SUPABASE_URL")
            supabase_key = os.environ.get("SUPABASE_KEY")
            
            if supabase_url and supabase_key:
                supabase = create_client(supabase_url, supabase_key)
                print("Created supabase client from environment variables")
            else:
                # Look for a config file
                config_paths = [
                    "./config.json",
                    "./backend/config.json",
                    "../backend/config.json"
                ]
                
                config_found = False
                for path in config_paths:
                    if os.path.exists(path):
                        with open(path, 'r') as f:
                            config = json.load(f)
                            if 'SUPABASE_URL' in config and 'SUPABASE_KEY' in config:
                                supabase = create_client(config['SUPABASE_URL'], config['SUPABASE_KEY'])
                                print(f"Created supabase client from config file: {path}")
                                config_found = True
                                break
                
                if not config_found:
                    # Create a mock supabase client for testing
                    print("WARNING: No Supabase credentials found. Creating mock client for testing.")
                    
                    class MockSupabase:
                        def table(self, table_name):
                            self.current_table = table_name
                            return self
                        
                        def select(self, *args):
                            self.select_args = args
                            return self
                        
                        def eq(self, column, value):
                            self.eq_column = column
                            self.eq_value = value
                            return self
                        
                        def execute(self):
                            if self.current_table == "nba_game_schedule" and self.eq_column == "game_date":
                                teams = [
                                    "LAL", "LAC", "GSW", "PHX", "SAC", 
                                    "DEN", "UTA", "POR", "MEM", "DAL", 
                                    "HOU", "SAS", "OKC", "MIN", "NOP",
                                    "MIL", "CHI", "CLE", "DET", "IND", 
                                    "BOS", "NYK", "BKN", "PHI", "TOR",
                                    "MIA", "ORL", "ATL", "CHA", "WAS"
                                ]
                                seed = int(self.eq_value.replace('-', ''))
                                random.seed(seed)
                                num_games = random.randint(3, 5)
                                games = []
                                random.shuffle(teams)
                                for i in range(num_games):
                                    home_team = teams[i*2]
                                    away_team = teams[i*2+1]
                                    hour = random.randint(16, 20)
                                    minute = random.choice([0, 30])
                                    start_time = f"{hour:02d}:{minute:02d}:00"
                                    games.append({
                                        "id": f"mock-{self.eq_value}-{i+1}",
                                        "game_date": self.eq_value,
                                        "start_time": start_time,
                                        "home_team": home_team,
                                        "away_team": away_team,
                                        "status": "scheduled",
                                        "arena": f"{home_team} Arena",
                                        "city": "City",
                                        "state": "State",
                                        "country": "USA"
                                    })
                                return type('obj', (object,), {'data': games})
                            return type('obj', (object,), {'data': []})
                    
                    supabase = MockSupabase()
except Exception as e:
    print(f"Error initializing Supabase client: {e}")
    print("Creating mock Supabase client as fallback")
    
    class SimpleSupabase:
        def table(self, _): return self
        def select(self, *_): return self
        def eq(self, *_): return self
        def execute(self): return type('obj', (object,), {'data': []})
    
    supabase = SimpleSupabase()

def fetch_scheduled_games(date_str=None, timezone='America/Los_Angeles'):
    """
    Fetch scheduled games from the database for a specific date.
    
    Args:
        date_str: Date string in YYYY-MM-DD format (defaults to today in specified timezone)
        timezone: Timezone name (default: 'America/Los_Angeles' for Pacific Time)
        
    Returns:
        DataFrame with scheduled games or empty DataFrame if none found
    """
    try:
        if date_str is None:
            tz = pytz.timezone(timezone)
            date_str = datetime.now(tz).strftime('%Y-%m-%d')
        
        print(f"Fetching scheduled games for {date_str} ({timezone})")
        response = supabase.table("nba_game_schedule").select("*").eq("game_date", date_str).execute()
        
        if response and hasattr(response, 'data') and response.data:
            df = pd.DataFrame(response.data)
            print(f"Found {len(df)} scheduled games")
            return df
        else:
            print(f"No scheduled games found for {date_str}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"Error fetching scheduled games: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Test fetching today's schedule
today_schedule = fetch_scheduled_games()
from IPython.display import display
display(today_schedule)

# Test fetching tomorrow's schedule
tz = pytz.timezone('America/Los_Angeles')
tomorrow = (datetime.now(tz) + timedelta(days=1)).strftime('%Y-%m-%d')
tomorrow_schedule = fetch_scheduled_games(tomorrow)
display(tomorrow_schedule)


In [ ]:
# Cell 20A - Compute momentum features for prediction

def compute_momentum_features(df):
    """
    Compute momentum-based features for prediction.
    
    Args:
        df: DataFrame with game data including quarter scores
        
    Returns:
        DataFrame with momentum features added
    """
    import numpy as np
    
    try:
        # Make a copy to avoid modifying the original
        features_df = df.copy()
        
        # Initialize momentum columns
        features_df['q1_to_q2_momentum'] = 0
        features_df['q2_to_q3_momentum'] = 0
        features_df['q3_to_q4_momentum'] = 0
        features_df['cumulative_momentum'] = 0
        
        # Weights for quarter transitions (more weight to recent quarters)
        weights = [0.2, 0.3, 0.5]
        
        for idx, row in features_df.iterrows():
            try:
                # Get current quarter
                current_quarter = int(row.get('current_quarter', 0))
                
                # Extract quarter scores with error handling
                home_q1 = float(row.get('home_q1', 0) or 0)
                home_q2 = float(row.get('home_q2', 0) or 0)
                home_q3 = float(row.get('home_q3', 0) or 0)
                home_q4 = float(row.get('home_q4', 0) or 0)
                
                away_q1 = float(row.get('away_q1', 0) or 0)
                away_q2 = float(row.get('away_q2', 0) or 0)
                away_q3 = float(row.get('away_q3', 0) or 0)
                away_q4 = float(row.get('away_q4', 0) or 0)
                
                # Calculate quarter-to-quarter momentum
                if current_quarter >= 2:
                    features_df.at[idx, 'q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
                
                if current_quarter >= 3:
                    features_df.at[idx, 'q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
                    
                if current_quarter >= 4:
                    features_df.at[idx, 'q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
                
                # Calculate cumulative momentum based on available quarters
                if current_quarter == 2:
                    cum_momentum = features_df.at[idx, 'q1_to_q2_momentum']
                elif current_quarter == 3:
                    cum_momentum = (
                        features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                        features_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
                    ) / (weights[0] + weights[1])
                elif current_quarter >= 4:
                    cum_momentum = (
                        features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                        features_df.at[idx, 'q2_to_q3_momentum'] * weights[1] + 
                        features_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
                    ) / sum(weights)
                else:
                    cum_momentum = 0
                
                # Normalize momentum to range [-1, 1]
                features_df.at[idx, 'cumulative_momentum'] = np.clip(cum_momentum / 15.0, -1.0, 1.0)
                
            except Exception as e:
                print(f"Error processing row {idx}: {e}")
                # Keep default values for this row
        
        return features_df
        
    except Exception as e:
        print(f"Error computing momentum features: {e}")
        import traceback
        traceback.print_exc()
        return df  # Return original dataframe if processing fails

In [ ]:
# Cell 21: Enhanced feature engineering
def compute_momentum_features(df):
    """Compute momentum-based features for prediction."""
    features_df = df.copy()
    
    # Add momentum columns
    features_df['q1_to_q2_momentum'] = 0
    features_df['q2_to_q3_momentum'] = 0
    features_df['q3_to_q4_momentum'] = 0
    features_df['cumulative_momentum'] = 0
    
    for idx, row in features_df.iterrows():
        current_quarter = int(row.get('current_quarter', 0))
        
        # Quarter scores
        home_q1 = float(row.get('home_q1', 0))
        home_q2 = float(row.get('home_q2', 0))
        home_q3 = float(row.get('home_q3', 0))
        home_q4 = float(row.get('home_q4', 0))
        
        away_q1 = float(row.get('away_q1', 0))
        away_q2 = float(row.get('away_q2', 0))
        away_q3 = float(row.get('away_q3', 0))
        away_q4 = float(row.get('away_q4', 0))
        
        # Calculate quarter-to-quarter momentum
        if current_quarter >= 2:
            features_df.at[idx, 'q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
            
        if current_quarter >= 3:
            features_df.at[idx, 'q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
            
        if current_quarter >= 4:
            features_df.at[idx, 'q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
        
        # Weighted momentum calculation
        weights = [0.2, 0.3, 0.5]  # Weight recent quarters more heavily
        
        if current_quarter == 2:
            features_df.at[idx, 'cumulative_momentum'] = features_df.at[idx, 'q1_to_q2_momentum']
        elif current_quarter == 3:
            features_df.at[idx, 'cumulative_momentum'] = (
                features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                features_df.at[idx, 'q2_to_q3_momentum'] * weights[1]
            ) / (weights[0] + weights[1])
        elif current_quarter >= 4:
            features_df.at[idx, 'cumulative_momentum'] = (
                features_df.at[idx, 'q1_to_q2_momentum'] * weights[0] + 
                features_df.at[idx, 'q2_to_q3_momentum'] * weights[1] + 
                features_df.at[idx, 'q3_to_q4_momentum'] * weights[2]
            ) / sum(weights)
        
        # Normalize momentum to range [-1, 1]
        if features_df.at[idx, 'cumulative_momentum'] != 0:
            max_momentum = 15.0  # Typical max quarter differential
            features_df.at[idx, 'cumulative_momentum'] = max(min(
                features_df.at[idx, 'cumulative_momentum'] / max_momentum, 1.0
            ), -1.0)
    
    return features_df

In [ ]:
# Cell 22 - Fetch recent games for testing

def fetch_recent_games_for_testing(limit=5, days_lookback=5, simulate_live=True):
    """
    Find recent completed games to use for testing the prediction model.
    
    Args:
        limit: Maximum number of games to return
        days_lookback: Number of days to look back for recent games
        simulate_live: Whether to simulate games as in-progress 
        
    Returns:
        DataFrame with recent games (optionally simulated as in-progress)
    """
    try:
        # Look back up to specified days for recent games
        dates_to_try = []
        today = datetime.now()
        
        for i in range(1, days_lookback + 1):
            date = (today - timedelta(days=i)).strftime('%Y-%m-%d')
            dates_to_try.append(date)
        
        print(f"Searching for recent games in the last {days_lookback} days...")
        
        # Try each date until we find games
        for test_date in dates_to_try:
            response = supabase.table("nba_historical_game_stats")\
                .select("*")\
                .eq("game_date", test_date)\
                .limit(limit).execute()
            
            if response.data:
                print(f"Found {len(response.data)} historical games from {test_date}")
                
                # Convert to DataFrame
                historical_df = pd.DataFrame(response.data)
                
                if not simulate_live:
                    # Return historical games as-is
                    return historical_df
                
                # Simulate as in-progress games
                import random
                live_games = []
                
                for _, game in historical_df.iterrows():
                    # Randomly select a quarter (1-4) for simulation
                    sim_quarter = random.randint(1, 4)
                    
                    # Create simulated live game with data up to selected quarter
                    sim_game = {
                        'game_id': game['game_id'],
                        'home_team': game['home_team'],
                        'away_team': game['away_team'],
                        'game_date': game['game_date'],
                        'current_quarter': sim_quarter,
                        'simulated': True,  # Flag as simulated
                        'time_remaining': random.randint(1, 12),  # Random minutes remaining
                        'verified': True  # Mark as verified since from historical data
                    }
                    
                    # Add quarter scores up to simulated quarter
                    for q in range(1, 5):
                        sim_game[f'home_q{q}'] = game.get(f'home_q{q}', 0) if q <= sim_quarter else 0
                        sim_game[f'away_q{q}'] = game.get(f'away_q{q}', 0) if q <= sim_quarter else 0
                    
                    # Calculate current score based on visible quarters
                    sim_game['home_score'] = sum(
                        [sim_game.get(f'home_q{q}', 0) for q in range(1, sim_quarter+1)]
                    )
                    sim_game['away_score'] = sum(
                        [sim_game.get(f'away_q{q}', 0) for q in range(1, sim_quarter+1)]
                    )
                    
                    # Store actual final scores for validation
                    sim_game['actual_home_final'] = game['home_score']
                    sim_game['actual_away_final'] = game['away_score']
                    
                    live_games.append(sim_game)
                
                simulated_df = pd.DataFrame(live_games)
                print(f"Created {len(simulated_df)} simulated live games at various quarters")
                return simulated_df
                
        print("No recent games found for testing")
        return pd.DataFrame()
        
    except Exception as e:
        print(f"Error finding historical games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

In [ ]:
# Cell 24 -- Get rolling scoring averages for all teams

def get_team_rolling_averages(days_lookback=60, cache_key=None, refresh_cache=False):
    """
    Get rolling scoring averages for all teams.
    
    Args:
        days_lookback: Number of days to look back for statistics
        cache_key: Optional cache key for retrieving/storing results
        refresh_cache: Whether to force refresh the cache
        
    Returns:
        Dict with team averages
    """
    # Use caching to improve performance
    if cache_key and not refresh_cache:
        if cache_key in globals():
            print(f"Using cached team averages ({len(globals()[cache_key])} teams)")
            return globals()[cache_key]
    
    try:
        # Calculate threshold date
        threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
        print(f"Fetching team averages from {threshold_date} to present")
        
        # Fetch recent games for calculating averages
        response = supabase.table("nba_historical_game_stats").select("*").gte("game_date", threshold_date).execute()
        
        if not response.data:
            print("No historical data found for team averages")
            return {}
        
        # Calculate team averages
        team_avgs = {}
        df = pd.DataFrame(response.data)
        
        # Get unique teams
        home_teams = set(df['home_team'].unique())
        away_teams = set(df['away_team'].unique())
        all_teams = home_teams.union(away_teams)
        
        print(f"Calculating averages for {len(all_teams)} teams")
        
        # Constants for score calculations
        DEFAULT_AVG_SCORE = 105.0  # NBA league average if no data available
        MIN_GAMES_THRESHOLD = 3    # Minimum games needed for reliable average
        
        for team in all_teams:
            # Get home and away games
            home_games = df[df['home_team'] == team]['home_score']
            away_games = df[df['away_team'] == team]['away_score']
            
            # Combine and calculate average
            all_scores = pd.concat([home_games, away_games])
            if len(all_scores) >= MIN_GAMES_THRESHOLD:
                team_avgs[team] = all_scores.mean()
                
                # Calculate offensive and defensive ratings if needed later
                home_allowed = df[df['home_team'] == team]['away_score'].mean() if len(df[df['home_team'] == team]) > 0 else DEFAULT_AVG_SCORE
                away_allowed = df[df['away_team'] == team]['home_score'].mean() if len(df[df['away_team'] == team]) > 0 else DEFAULT_AVG_SCORE
                
                # Store these as additional metrics
                team_avgs[f"{team}_defensive"] = (home_allowed + away_allowed) / 2
            else:
                print(f"Warning: Insufficient data for {team}, using default average")
                team_avgs[team] = DEFAULT_AVG_SCORE
        
        # Cache results if requested
        if cache_key:
            globals()[cache_key] = team_avgs
            print(f"Cached team averages under key '{cache_key}'")
        
        return team_avgs
    
    except Exception as e:
        print(f"Error getting team averages: {e}")
        traceback.print_exc()
        return {}

def get_previous_matchup_diff(home_team, away_team, max_lookback=5):
    """
    Gets the point differential from previous matchups between two teams.
    
    Args:
        home_team: Home team name
        away_team: Away team name
        max_lookback: Maximum number of previous matchups to consider
        
    Returns:
        float: Average point differential from home team perspective
    """
    try:
        print(f"Looking for previous matchups between {home_team} and {away_team}")
        
        # Use separate queries for home and away configurations for reliable results
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_lookback).execute()
        
        # Combine results
        home_matchups = response_home.data if response_home.data else []
        away_matchups = response_away.data if response_away.data else []
        matchups = home_matchups + away_matchups
        
        # Sort by date (most recent first)
        if matchups:
            matchups.sort(key=lambda x: x['game_date'], reverse=True)
            matchups = matchups[:max_lookback]
            
            print(f"Found {len(matchups)} previous matchups")
        else:
            print("No previous matchups found")
            return 0
        
        # Calculate point differential from home team perspective
        differentials = []
        for game in matchups:
            if game['home_team'] == home_team and game['away_team'] == away_team:
                # Home team was home in this matchup
                diff = game['home_score'] - game['away_score']
            elif game['home_team'] == away_team and game['away_team'] == home_team:
                # Home team was away in this matchup (reverse diff)
                diff = game['away_score'] - game['home_score']
            else:
                continue
                
            differentials.append(diff)
            print(f"  {game['game_date']}: Differential {diff:+.1f} pts")
        
        if not differentials:
            return 0
            
        # Return average differential, capped to prevent extreme values
        avg_diff = sum(differentials) / len(differentials)
        # Cap extreme values for prediction stability
        MAX_DIFF = 15.0  # Maximum historical differential to consider
        capped_diff = max(min(avg_diff, MAX_DIFF), -MAX_DIFF)
        
        print(f"Average point differential: {avg_diff:.2f} (capped to {capped_diff:.2f})")
        return capped_diff
        
    except Exception as e:
        print(f"Error getting previous matchups: {e}")
        traceback.print_exc()
        return 0

def prepare_features(games_df, model=None):
    """
    Prepare features for prediction based on model type.
    
    Args:
        games_df: DataFrame with game data
        model: Prediction model (optional, used to determine feature requirements)
        
    Returns:
        DataFrame with features prepared for prediction
    """
    if games_df.empty:
        print("No games available for feature preparation")
        return pd.DataFrame()
    
    try:
        print(f"Preparing features for {len(games_df)} games")
        
        # Determine required features based on model
        is_enhanced_model = False
        expected_features = []
        
        if model is not None and hasattr(model, 'feature_importances_'):
            feature_count = len(model.feature_importances_)
            is_enhanced_model = (feature_count > 8)
            
            # Check if model has explicit feature names
            if hasattr(model, 'feature_names_in_'):
                expected_features = list(model.feature_names_in_)
                print(f"Using explicit model features: {expected_features}")
            else:
                # Infer features based on model type
                if is_enhanced_model:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                        'score_ratio', 'prev_matchup_diff',
                        'rest_days_home', 'rest_days_away', 'rest_advantage',
                        'is_back_to_back_home', 'is_back_to_back_away',
                        'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                    ]
                    print("Using enhanced feature set (15 features)")
                else:
                    expected_features = [
                        'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                        'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                    ]
                    print("Using standard feature set (8 features)")
        else:
            # Default to original features
            expected_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
            print("Using default feature set (no model provided)")
        
        # Get team averages
        team_avgs = get_team_rolling_averages(cache_key='team_averages')
        
        # Prepare feature DataFrame
        features = []
        
        for idx, game in games_df.iterrows():
            # Get basic game data
            game_id = game['game_id']
            home_team = game['home_team']
            away_team = game['away_team']
            current_quarter = int(game['current_quarter'])
            
            # Quarter scores with error handling
            home_q1 = float(game.get('home_q1', 0) or 0)
            home_q2 = float(game.get('home_q2', 0) or 0)
            home_q3 = float(game.get('home_q3', 0) or 0)
            home_q4 = float(game.get('home_q4', 0) or 0)
            
            away_q1 = float(game.get('away_q1', 0) or 0)
            away_q2 = float(game.get('away_q2', 0) or 0)
            away_q3 = float(game.get('away_q3', 0) or 0)
            away_q4 = float(game.get('away_q4', 0) or 0)
            
            # Calculate score ratio
            home_score = float(game.get('home_score', 0) or 0)
            away_score = float(game.get('away_score', 0) or 0)
            total_score = home_score + away_score
            score_ratio = home_score / total_score if total_score > 0 else 0.5
            
            # Get matchup history
            prev_matchup_diff = get_previous_matchup_diff(home_team, away_team)
            
            # Create base feature dictionary
            feature_dict = {
                'game_id': game_id,
                'home_team': home_team,
                'away_team': away_team,
                'home_q1': home_q1,
                'home_q2': home_q2,
                'home_q3': home_q3,
                'home_q4': home_q4,
                'score_ratio': score_ratio,
                'prev_matchup_diff': prev_matchup_diff,
                'current_quarter': current_quarter
            }
            
            # Add rolling averages
            if 'rolling_home_score' in expected_features:
                feature_dict['rolling_home_score'] = team_avgs.get(home_team, 105.0)
                feature_dict['rolling_away_score'] = team_avgs.get(away_team, 105.0)
            
            # Add enhanced features if needed
            if is_enhanced_model:
                # Default rest features (simplified for demonstration)
                feature_dict['rest_days_home'] = 2  # Default 2 days rest
                feature_dict['rest_days_away'] = 2
                feature_dict['rest_advantage'] = 0
                feature_dict['is_back_to_back_home'] = 0
                feature_dict['is_back_to_back_away'] = 0
                
                # Add momentum features if needed
                if ('q1_to_q2_momentum' in expected_features or 
                    'q2_to_q3_momentum' in expected_features or 
                    'q3_to_q4_momentum' in expected_features or 
                    'cumulative_momentum' in expected_features):
                    
                    # Calculate momentum features
                    feature_dict['q1_to_q2_momentum'] = 0
                    feature_dict['q2_to_q3_momentum'] = 0
                    feature_dict['q3_to_q4_momentum'] = 0
                    feature_dict['cumulative_momentum'] = 0
                    
                    if current_quarter >= 2:
                        feature_dict['q1_to_q2_momentum'] = (home_q2 - home_q1) - (away_q2 - away_q1)
                    
                    if current_quarter >= 3:
                        feature_dict['q2_to_q3_momentum'] = (home_q3 - home_q2) - (away_q3 - away_q2)
                    
                    if current_quarter >= 4:
                        feature_dict['q3_to_q4_momentum'] = (home_q4 - home_q3) - (away_q4 - away_q3)
                    
                    # Calculate cumulative momentum
                    weights = [0.2, 0.3, 0.5]  # Quarter momentum weights
                    
                    if current_quarter == 2:
                        feature_dict['cumulative_momentum'] = feature_dict['q1_to_q2_momentum']
                    elif current_quarter == 3:
                        feature_dict['cumulative_momentum'] = (
                            feature_dict['q1_to_q2_momentum'] * weights[0] + 
                            feature_dict['q2_to_q3_momentum'] * weights[1]
                        ) / (weights[0] + weights[1])
                    elif current_quarter >= 4:
                        feature_dict['cumulative_momentum'] = (
                            feature_dict['q1_to_q2_momentum'] * weights[0] + 
                            feature_dict['q2_to_q3_momentum'] * weights[1] + 
                            feature_dict['q3_to_q4_momentum'] * weights[2]
                        ) / sum(weights)
                    
                    # Normalize momentum to [-1, 1]
                    feature_dict['cumulative_momentum'] = max(
                        min(feature_dict['cumulative_momentum'] / 15.0, 1.0), -1.0
                    )
            
            features.append(feature_dict)
        
        # Create DataFrame
        features_df = pd.DataFrame(features)
        
        # Ensure all expected features exist with proper types
        for feature in expected_features:
            if feature not in features_df.columns:
                print(f"Adding missing feature: {feature}")
                features_df[feature] = 0
            
            # Ensure numeric
            features_df[feature] = pd.to_numeric(features_df[feature], errors='coerce').fillna(0)
        
        return features_df
        
    except Exception as e:
        print(f"Error preparing features: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def run_predictions(model, games_df):
    """
    Run predictions using the model and games data.
    
    Args:
        model: Prediction model
        games_df: DataFrame with games data
        
    Returns:
        DataFrame with prediction results
    """
    if games_df.empty:
        print("No games available for prediction")
        return pd.DataFrame()
    
    if model is None:
        print("No model provided for prediction")
        return pd.DataFrame()
    
    try:
        print(f"Running predictions for {len(games_df)} games")
        
        # Step 1: Prepare features
        features_df = prepare_features(games_df, model)
        if features_df.empty:
            print("Failed to prepare features")
            return pd.DataFrame()
        
        # Step 2: Get the right feature columns for prediction
        model_features = []
        
        # Check if model has explicit feature names
        if hasattr(model, 'feature_names_in_'):
            model_features = list(model.feature_names_in_)
            print(f"Using {len(model_features)} features from model definition")
        elif hasattr(model, 'feature_importances_'):
            # Infer features from importances length
            feature_count = len(model.feature_importances_)
            if feature_count > 8:
                model_features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                    'score_ratio', 'prev_matchup_diff',
                    'rest_days_home', 'rest_days_away', 'rest_advantage',
                    'is_back_to_back_home', 'is_back_to_back_away',
                    'q1_to_q2_momentum', 'q2_to_q3_momentum', 'q3_to_q4_momentum', 'cumulative_momentum'
                ]
                print(f"Using {len(model_features)} enhanced features based on model importances")
            else:
                model_features = [
                    'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                    'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
                ]
                print(f"Using {len(model_features)} standard features based on model importances")
        else:
            # Default to original features
            model_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
            print(f"Using {len(model_features)} default features (model type unknown)")
        
        # Make sure all required features exist
        for feature in model_features:
            if feature not in features_df.columns:
                print(f"Adding missing feature '{feature}' (required by model)")
                features_df[feature] = 0
        
        # Step 3: Make predictions
        X_pred = features_df[model_features]
        print(f"Making predictions using {X_pred.shape[1]} features")
        predictions = model.predict(X_pred)
        
        # Step 4: Create results DataFrame with predictions
        results = []
        
        for i, (idx, game) in enumerate(games_df.iterrows()):
            # Get game info
            game_id = game['game_id']
            home_team = game['home_team']
            away_team = game['away_team']
            current_quarter = int(game['current_quarter'])
            home_score = float(game['home_score'])
            away_score = float(game['away_score'])
            
            # Get prediction
            predicted_home_score = predictions[i]
            
            # Calculate away score prediction
            # Constants for prediction adjustments
            DIFF_WEIGHT_BASE = 0.3     # Base weight for matchup differential
            DIFF_WEIGHT_INCREMENT = 0.1  # Weight increment per quarter
            MOMENTUM_SCALE = 3.0       # Points impact of momentum
            
            # Get matchup differential
            prev_matchup_diff = features_df.loc[i, 'prev_matchup_diff'] if 'prev_matchup_diff' in features_df.columns else 0
            
            # Scale effect based on game progress
            diff_weight = min(DIFF_WEIGHT_BASE + (DIFF_WEIGHT_INCREMENT * current_quarter), 0.6)
            
            # Factor in momentum if available
            momentum_adj = 0
            if 'cumulative_momentum' in features_df.columns:
                momentum = features_df.loc[i, 'cumulative_momentum']
                momentum_adj = momentum * MOMENTUM_SCALE
            
            # Calculate predicted away score
            predicted_away_score = predicted_home_score - (prev_matchup_diff * diff_weight) - momentum_adj
            
            # Ensure predictions are at least current scores
            predicted_home_score = max(predicted_home_score, home_score)
            predicted_away_score = max(predicted_away_score, away_score)
            
            # Calculate win probability using logistic function
            score_diff = predicted_home_score - predicted_away_score
            game_progress = min(current_quarter / 4.0, 1.0)
            
            # Constants for win probability calculation
            K_BASE = 0.05      # Base confidence factor
            K_INCREMENT = 0.15  # Increment for game progress
            
            k_factor = K_BASE + (game_progress * K_INCREMENT)
            win_probability = 1.0 / (1.0 + np.exp(-k_factor * score_diff))
            
            # Calculate confidence based on quarter
            confidence_by_quarter = {
                0: 0.6,  # Pre-game
                1: 0.7,  # Q1
                2: 0.8,  # Q2
                3: 0.9,  # Q3
                4: 0.95  # Q4
            }
            prediction_confidence = confidence_by_quarter.get(current_quarter, 0.7)
            
            # Create result dictionary
            result = {
                'game_id': game_id,
                'home_team': home_team,
                'away_team': away_team,
                'current_quarter': current_quarter,
                'current_home_score': home_score,
                'current_away_score': away_score,
                'predicted_home_final': predicted_home_score,
                'predicted_away_final': predicted_away_score,
                'remaining_home_points': predicted_home_score - home_score,
                'remaining_away_points': predicted_away_score - away_score,
                'win_probability': win_probability,
                'prediction_confidence': prediction_confidence,
                'projected_margin': predicted_home_score - predicted_away_score,
                'total_projected_score': predicted_home_score + predicted_away_score,
                'prediction_timestamp': datetime.now().isoformat()
            }
            
            # Add actual finals if available (for historical testing)
            if 'actual_home_final' in game:
                result['actual_home_final'] = game['actual_home_final']
                result['actual_away_final'] = game['actual_away_final']
                
                # Calculate prediction error if actuals available
                if 'actual_home_final' in result:
                    home_error = abs(predicted_home_score - result['actual_home_final'])
                    away_error = abs(predicted_away_score - result['actual_away_final'])
                    result['avg_error'] = (home_error + away_error) / 2
                    result['winner_correct'] = ((predicted_home_score > predicted_away_score) == 
                                              (result['actual_home_final'] > result['actual_away_final']))
            
            results.append(result)
        
        return pd.DataFrame(results)
        
    except Exception as e:
        print(f"Error in prediction pipeline: {e}")
        traceback.print_exc()
        return pd.DataFrame()

In [ ]:
# Cell 26: Enhanced NBA Game Status Detection with Pacific Time Handling

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz, traceback, os
from sqlalchemy import create_engine
import config
from caching.supabase_client import supabase

def log_with_timestamp(message: str, level: str = "INFO"):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {level}: {message}")

def fetch_live_games_in_pacific_time():
    """
    Fetch all game data for today in Pacific Time with improved status detection.
    Returns:
        DataFrame with all games (pregame, live, and finished) with proper status flags.
    """
    try:
        log_with_timestamp("Fetching all games for today (Pacific Time)...")
        
        # Get current date in Pacific Time
        pacific_tz = pytz.timezone("America/Los_Angeles")
        now_pt = datetime.now(pacific_tz)
        today_pt = now_pt.date()
        
        # Define start and end of today in PT
        start_pt = datetime.combine(today_pt, datetime.min.time()).replace(tzinfo=pacific_tz)
        end_pt = datetime.combine(today_pt, datetime.max.time()).replace(tzinfo=pacific_tz)
        
        # Convert start and end to UTC ISO format strings
        start_utc = start_pt.astimezone(pytz.utc).isoformat()
        end_utc = end_pt.astimezone(pytz.utc).isoformat()
        
        log_with_timestamp(f"Fetching games from Supabase in range:\n  {start_utc} to {end_utc}")
        
        # Query Supabase for games whose 'game_date' is between start_utc and end_utc
        response = supabase.table("nba_live_game_stats").select("*")\
                          .gte("game_date", start_utc)\
                          .lte("game_date", end_utc)\
                          .execute()
        
        if not response.data:
            log_with_timestamp(f"No games found for today ({today_pt}).", "WARNING")
            return pd.DataFrame()
        
        df = pd.DataFrame(response.data)
        log_with_timestamp(f"Found {len(df)} rows for today in the database.")
        
        # Convert 'game_date' (stored in UTC) to a tz-aware datetime, then to Pacific Time
        df['game_date'] = pd.to_datetime(df['game_date'], errors='coerce', utc=True)
        df['game_date_pt'] = df['game_date'].dt.tz_convert(pacific_tz)
        df['date_only_pt'] = df['game_date_pt'].dt.date
        
        log_with_timestamp(f"Today's date in Pacific Time: {today_pt}")
        df_today = df[df['date_only_pt'] == today_pt].copy()
        log_with_timestamp(f"Rows matching today's PT date: {len(df_today)}")
        
        if df_today.empty:
            log_with_timestamp("No rows match today's date in Pacific Time.", "WARNING")
            return pd.DataFrame()
        
        # Ensure 'is_finished' is a proper boolean column
        if 'is_finished' not in df_today.columns:
            df_today['is_finished'] = False
        else:
            try:
                df_today['is_finished'] = df_today['is_finished'].astype(bool)
            except Exception:
                df_today['is_finished'] = df_today['is_finished'].apply(lambda x: bool(x) if pd.notna(x) else False)
        
        # Ensure 'current_quarter' exists and is numeric
        if 'current_quarter' not in df_today.columns:
            df_today['current_quarter'] = 0
        else:
            df_today['current_quarter'] = pd.to_numeric(df_today['current_quarter'], errors='coerce').fillna(0)
        
        # Set initial game status to 'unknown'
        df_today['game_status'] = 'unknown'
        
        # Process each game to determine its status
        for idx, row in df_today.iterrows():
            # First check if game is explicitly marked as finished
            if row.get('is_finished', False):
                df_today.at[idx, 'game_status'] = 'finished'
                continue
                
            # Check if all 4 quarters have data (likely finished but not marked)
            all_quarters_filled = True
            for q in range(1, 5):
                home_val = float(row.get(f'home_q{q}', 0) or 0)
                away_val = float(row.get(f'away_q{q}', 0) or 0)
                if home_val == 0 and away_val == 0:
                    all_quarters_filled = False
                    break
                    
            if all_quarters_filled:
                df_today.at[idx, 'game_status'] = 'finished'
                df_today.at[idx, 'current_quarter'] = 4
                continue
                
            # If any quarter has points, it's at least started
            quarters_played = 0
            for q in range(1, 5):
                home_val = float(row.get(f'home_q{q}', 0) or 0)
                away_val = float(row.get(f'away_q{q}', 0) or 0)
                if home_val > 0 or away_val > 0:
                    quarters_played = max(quarters_played, q)
            
            if quarters_played > 0:
                # Game has started
                df_today.at[idx, 'game_status'] = 'live'
                
                # Update current_quarter if needed
                if int(row.get('current_quarter', 0)) < quarters_played:
                    df_today.at[idx, 'current_quarter'] = quarters_played
            else:
                # No quarters played yet - it's pre-game
                df_today.at[idx, 'game_status'] = 'pregame'
                
            # Additional check: if home_score and away_score exist and are > 0,
            # but no quarter data, it's probably live but missing quarter breakdown
            if quarters_played == 0 and df_today.at[idx, 'game_status'] == 'pregame':
                home_score = float(row.get('home_score', 0) or 0)
                away_score = float(row.get('away_score', 0) or 0)
                if home_score > 0 or away_score > 0:
                    df_today.at[idx, 'game_status'] = 'live'
                    # Guess quarter based on score
                    total_score = home_score + away_score
                    if total_score > 180:  # Late game
                        df_today.at[idx, 'current_quarter'] = 4
                    elif total_score > 120:  # Mid game
                        df_today.at[idx, 'current_quarter'] = 3
                    elif total_score > 60:   # Early-mid game
                        df_today.at[idx, 'current_quarter'] = 2
                    else:                   # Early game
                        df_today.at[idx, 'current_quarter'] = 1
        
        # Get active live games
        active_live_games = df_today[df_today['game_status'] == 'live'].copy()
        log_with_timestamp(f"Found {len(active_live_games)} active live games")
        
        # Log overall game status breakdown
        log_with_timestamp("Game status breakdown:")
        status_counts = df_today['game_status'].value_counts()
        for status, count in status_counts.items():
            print(f"  {status.upper()}: {count} games")
        
        log_with_timestamp(f"Returning {len(df_today)} total games.")
        return df_today
        
    except Exception as e:
        log_with_timestamp(f"Error fetching games in Pacific Time: {str(e)}", "ERROR")
        traceback.print_exc()
        return pd.DataFrame()

def get_active_live_games():
    """
    Get only actively live games from all today's games.
    Returns:
        DataFrame with only live games.
    """
    all_games = fetch_live_games_in_pacific_time()
    if all_games.empty:
        log_with_timestamp("No games available for today", "WARNING")
        return pd.DataFrame()
    
    # Filter for only live games
    live_games = all_games[all_games['game_status'] == 'live'].copy()
    
    if live_games.empty:
        log_with_timestamp("No actively live games found", "WARNING")
        return pd.DataFrame()
    
    log_with_timestamp(f"Found {len(live_games)} actively live games")
    return live_games

# ---------------- Example Usage for Cell 26 ----------------
if __name__ == "__main__":
    # Get all games
    all_games_pt = fetch_live_games_in_pacific_time()
    if all_games_pt.empty:
        print("No games available in Pacific Time.")
    else:
        print("Games by status (Pacific Time):")
        status_counts = all_games_pt['game_status'].value_counts()
        for status, count in status_counts.items():
            print(f"  {status.upper()}: {count} games")
    
    # Get only active live games
    active_games = get_active_live_games()
    if active_games.empty:
        print("\nNo actively live games found.")
    else:
        print(f"\nFound {len(active_games)} actively live games:")
        for idx, game in active_games.iterrows():
            print(f"  {game['home_team']} vs {game['away_team']} - Q{int(game['current_quarter'])}")

In [ ]:
# Cell 27: Core Uncertainty Estimation Module

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union, Any
import json
from datetime import datetime

class UncertaintyEstimator:
    """
    Core class for estimating uncertainty in NBA score predictions.
    
    This class implements a context-aware uncertainty estimation approach that
    adjusts confidence intervals based on:
    1. Game quarter (uncertainty decreases as game progresses)
    2. Score differential (close games have higher uncertainty)
    3. Team momentum (high momentum can increase uncertainty)
    4. Historical calibration data
    
    Methodology:
    - Base uncertainty is derived from the quarter of play, with earlier quarters
      having wider intervals than later quarters
    - Context adjustments modify this base uncertainty based on game state
    - Intervals are centered around the predicted value
    """
    
    def __init__(self, confidence_level: float = 0.9):
        """
        Initialize the uncertainty estimator with default parameters.
        
        Args:
            confidence_level: Target confidence level (default: 0.9 for 90% coverage)
        """
        # Base width factors for different quarters
        # These control how wide the intervals are relative to the base uncertainty
        self.width_factors = {
            0: 3.0,  # Pre-game: widest intervals
            1: 2.5,  # 1st quarter
            2: 2.0,  # 2nd quarter 
            3: 1.5,  # 3rd quarter
            4: 1.0   # 4th quarter: narrowest intervals
        }
        
        # Confidence interval target (90% coverage by default)
        self.confidence_level = confidence_level
        
        # Calibration metrics dictionary to track interval performance
        self.calibration_data = {
            'total_predictions': 0,
            'in_interval_count': 0,
            'by_quarter': {q: {'count': 0, 'in_interval': 0} for q in range(5)}
        }
    
    def calculate_prediction_interval(
            self, 
            predicted_value: float, 
            quarter: int, 
            score_differential: float = 0, 
            momentum: float = 0,
            time_remaining: Optional[float] = None
        ) -> Tuple[float, float, float]:
        """
        Calculate confidence interval for a prediction with context awareness.
        
        Args:
            predicted_value: The predicted final score
            quarter: Current quarter (0-4, where 0 is pre-game)
            score_differential: Absolute score difference between teams
            momentum: Team momentum (-1 to 1, higher absolute value means more momentum)
            time_remaining: Minutes remaining in current quarter (optional)
            
        Returns:
            Tuple of (lower_bound, upper_bound, interval_width)
        """
        # Get base width factor for this quarter (or default to 2.0)
        base_factor = self.width_factors.get(quarter, 2.0)
        
        # Calculate base uncertainty (decreases as game progresses)
        # Formula: Maximum uncertainty at pre-game (quarter 0), decreases linearly 
        # to minimum at quarter 4. Scale of 10 points represents typical NBA scoring variation.
        base_uncertainty = 10.0 * (4.0 - min(quarter, 4)) / 4.0
        
        # Apply context adjustments
        context_adjustment = self._calculate_context_adjustment(
            quarter, score_differential, momentum, time_remaining)
        
        # Calculate adjusted interval width
        interval_width = base_uncertainty * base_factor * context_adjustment
        
        # Ensure interval width is reasonable (between min and max bounds)
        min_width = 2.0  # Minimum width to account for inherent randomness
        max_width = 30.0  # Maximum width to prevent extreme intervals
        interval_width = min(max(interval_width, min_width), max_width)
        
        # Calculate bounds centered on prediction
        lower_bound = predicted_value - interval_width / 2
        upper_bound = predicted_value + interval_width / 2
        
        return lower_bound, upper_bound, interval_width
    
    def _calculate_context_adjustment(
            self,
            quarter: int,
            score_differential: float,
            momentum: float = 0,
            time_remaining: Optional[float] = None
        ) -> float:
        """
        Calculate adjustment factor based on game context.
        
        Args:
            quarter: Current quarter (0-4)
            score_differential: Absolute score difference between teams
            momentum: Team momentum (-1 to 1)
            time_remaining: Minutes remaining in current quarter
            
        Returns:
            float: Context adjustment factor
        """
        context_adjustment = 1.0
        
        # 1. Adjust for close games vs blowouts
        if score_differential <= 5.0:
            # Close games have higher uncertainty
            context_adjustment *= 1.1
        elif score_differential >= 15.0:
            # Blowouts have lower uncertainty
            context_adjustment *= 0.9
        
        # 2. Adjust for momentum
        if abs(momentum) >= 0.5:
            # High momentum (either direction) increases uncertainty
            context_adjustment *= 1.15
        
        # 3. Final minutes adjustment
        if quarter == 4 and time_remaining is not None and time_remaining <= 5.0:
            # Late in close games, prediction becomes more certain
            context_adjustment *= 0.85
            
        # 4. Special case for pre-game predictions
        if quarter == 0:
            # Pre-game predictions rely on team averages which introduces more uncertainty
            context_adjustment *= 1.2
            
        return context_adjustment
    
    def update_calibration(self, 
                          predicted_value: float, 
                          actual_value: float,
                          quarter: int,
                          lower_bound: float,
                          upper_bound: float):
        """
        Update calibration metrics based on whether actual fell within interval.
        
        Args:
            predicted_value: The predicted score
            actual_value: The actual final score
            quarter: Quarter when prediction was made (0-4)
            lower_bound: Lower bound of confidence interval
            upper_bound: Upper bound of confidence interval
        """
        # Increment total predictions counter
        self.calibration_data['total_predictions'] += 1
        
        # Check if actual value is within interval
        in_interval = lower_bound <= actual_value <= upper_bound
        
        # Update overall metrics
        if in_interval:
            self.calibration_data['in_interval_count'] += 1
        
        # Update quarter-specific metrics
        if 0 <= quarter <= 4:
            self.calibration_data['by_quarter'][quarter]['count'] += 1
            if in_interval:
                self.calibration_data['by_quarter'][quarter]['in_interval'] += 1
    
    def get_calibration_report(self) -> Dict[str, Any]:
        """
        Get a report on uncertainty calibration.
        
        Returns:
            Dict with calibration metrics including overall coverage and
            quarter-specific coverage
        """
        report = {
            'total_predictions': self.calibration_data['total_predictions'],
            'overall_coverage': 0,
            'target_coverage': self.confidence_level,
            'by_quarter': {}
        }
        
        # Calculate overall coverage
        if report['total_predictions'] > 0:
            report['overall_coverage'] = (
                self.calibration_data['in_interval_count'] / 
                report['total_predictions']
            )
        
        # Calculate per-quarter coverage
        for quarter, data in self.calibration_data['by_quarter'].items():
            if data['count'] > 0:
                coverage = data['in_interval'] / data['count']
                report['by_quarter'][quarter] = {
                    'count': data['count'],
                    'coverage': coverage,
                    'width_factor': self.width_factors.get(quarter, 0)
                }
        
        # Add calibration quality assessment
        # Well-calibrated means the actual coverage is within 5% of the target
        report['is_well_calibrated'] = (
            abs(report['overall_coverage'] - self.confidence_level) < 0.05
            if report['total_predictions'] >= 30 else None
        )
        
        return report
    
    def apply_intervals_to_predictions(self, predictions_df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply prediction intervals to all predictions in a DataFrame.
        
        Args:
            predictions_df: DataFrame with prediction data
            
        Returns:
            DataFrame with confidence intervals added
        """
        # Create copy to avoid modifying original
        result_df = predictions_df.copy()
        
        # Add interval columns if they don't exist
        for col in ['home_lower_bound', 'home_upper_bound', 
                   'away_lower_bound', 'away_upper_bound', 
                   'home_interval_width', 'away_interval_width']:
            if col not in result_df.columns:
                result_df[col] = np.nan
        
        # Process each prediction
        for idx, row in result_df.iterrows():
            quarter = int(row.get('current_quarter', 0))
            
            # Get score differential
            home_score = float(row.get('current_home_score', row.get('home_score', 0)))
            away_score = float(row.get('current_away_score', row.get('away_score', 0)))
            score_diff = abs(home_score - away_score)
            
            # Get momentum if available
            momentum = float(row.get('momentum_shift', row.get('cumulative_momentum', 0)))
            
            # Get time remaining if available
            time_remaining = row.get('time_remaining')
            
            # Calculate intervals for home score
            home_pred = float(row.get('predicted_home_final', row.get('predicted_home_score', 0)))
            home_lower, home_upper, home_width = self.calculate_prediction_interval(
                home_pred, quarter, score_diff, momentum, time_remaining
            )
            
            # Calculate intervals for away score
            away_pred = float(row.get('predicted_away_final', row.get('predicted_away_score', 0)))
            away_lower, away_upper, away_width = self.calculate_prediction_interval(
                away_pred, quarter, score_diff, momentum, time_remaining
            )
            
            # Store in DataFrame
            result_df.loc[idx, 'home_lower_bound'] = home_lower
            result_df.loc[idx, 'home_upper_bound'] = home_upper
            result_df.loc[idx, 'home_interval_width'] = home_width
            
            result_df.loc[idx, 'away_lower_bound'] = away_lower
            result_df.loc[idx, 'away_upper_bound'] = away_upper
            result_df.loc[idx, 'away_interval_width'] = away_width
        
        return result_df


class ConfidenceVisualizer:
    """
    Specialized class for visualizing prediction confidence.
    
    This class provides methods to create various visualizations
    of prediction confidence and uncertainty.
    """
    
    def __init__(self, color_scheme: str = 'default'):
        """
        Initialize the visualizer with a color scheme.
        
        Args:
            color_scheme: Color scheme name ('default', 'dark', 'light')
        """
        # Color schemes
        self.color_schemes = {
            'default': {
                'background': '#f8f9fa',
                'text': '#212529',
                'home_team': '#0d6efd',
                'away_team': '#dc3545',
                'confidence_high': '#198754',
                'confidence_med': '#fd7e14',
                'confidence_low': '#dc3545',
                'grid': '#e9ecef',
                'axis': '#6c757d'
            },
            'dark': {
                'background': '#212529',
                'text': '#f8f9fa',
                'home_team': '#0dcaf0',
                'away_team': '#f85c70',
                'confidence_high': '#20c997',
                'confidence_med': '#fd7e14',
                'confidence_low': '#dc3545',
                'grid': '#495057',
                'axis': '#adb5bd'
            },
            'light': {
                'background': '#ffffff',
                'text': '#212529',
                'home_team': '#0d6efd',
                'away_team': '#dc3545',
                'confidence_high': '#198754',
                'confidence_med': '#fd7e14',
                'confidence_low': '#dc3545',
                'grid': '#f1f3f5',
                'axis': '#6c757d'
            }
        }
        
        # Set active color scheme
        self.colors = self.color_schemes.get(color_scheme, self.color_schemes['default'])
        
        # Quarter-specific colors for visual differentiation
        self.quarter_colors = {
            0: "#6c757d",  # Gray for pregame
            1: "#20c997",  # Teal for Q1
            2: "#0dcaf0",  # Cyan for Q2
            3: "#0d6efd",  # Blue for Q3
            4: "#6610f2"   # Purple for Q4
        }
    
    def visualize_confidence_intervals(self, predictions_df: pd.DataFrame) -> plt.Figure:
        """
        Create a figure visualizing prediction intervals by quarter.
        
        Args:
            predictions_df: DataFrame with prediction data including intervals
            
        Returns:
            matplotlib Figure object
        """
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Extract by quarter
        grouped = predictions_df.groupby('current_quarter')
        x_positions = []
        x_labels = []
        
        for i, (quarter, group) in enumerate(grouped):
            # Get mean interval widths
            if 'home_interval_width' in group.columns and 'away_interval_width' in group.columns:
                mean_width = (group['home_interval_width'].mean() + group['away_interval_width'].mean()) / 2
            else:
                # Estimate from upper/lower bounds
                home_width = (group['home_upper_bound'] - group['home_lower_bound']).mean()
                away_width = (group['away_upper_bound'] - group['away_lower_bound']).mean()
                mean_width = (home_width + away_width) / 2
            
            # Calculate error
            if 'actual_home_final' in group.columns and 'actual_away_final' in group.columns:
                home_error = abs(group['predicted_home_final'] - group['actual_home_final']).mean()
                away_error = abs(group['predicted_away_final'] - group['actual_away_final']).mean()
                mean_error = (home_error + away_error) / 2
            else:
                mean_error = None
            
            # Position on x-axis
            x_pos = i
            x_positions.append(x_pos)
            x_labels.append(f"Q{quarter}")
            
            # Plot interval width
            ax.bar(x_pos, mean_width, 
                  color=self.quarter_colors.get(quarter, self.colors['confidence_med']),
                  alpha=0.7)
            
            # Plot error if available
            if mean_error is not None:
                ax.plot(x_pos, mean_error, 'o', color='black', markersize=8)
            
            # Add interval size text
            ax.text(x_pos, mean_width + 1, f"{mean_width:.1f}", 
                   ha='center', va='bottom', fontsize=10)
        
        # Customize plot
        ax.set_ylabel("Points", fontsize=12)
        ax.set_title("Prediction Uncertainty by Quarter", fontsize=14)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add legend
        if any('actual_home_final' in group.columns for _, group in grouped):
            from matplotlib.lines import Line2D
            legend_elements = [
                Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
                      markersize=8, label='Mean Error'),
                plt.Rectangle((0,0), 1, 1, fc=self.colors['confidence_med'], 
                             alpha=0.7, label='Interval Width')
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            
        plt.tight_layout()
        return fig
    
    def create_confidence_indicator(self, prediction_data: Dict[str, Any]) -> str:
        """
        Create an SVG confidence indicator for a prediction.
        
        Args:
            prediction_data: Dict with prediction data including intervals
            
        Returns:
            String with SVG code for the indicator
        """
        # Extract key information
        home_team = prediction_data.get('home_team', 'Home')
        away_team = prediction_data.get('away_team', 'Away')
        quarter = int(prediction_data.get('current_quarter', 0))
        home_score = float(prediction_data.get('current_home_score', prediction_data.get('home_score', 0)))
        away_score = float(prediction_data.get('current_away_score', prediction_data.get('away_score', 0)))
        
        # Predicted scores
        home_pred = float(prediction_data.get('predicted_home_final', prediction_data.get('predicted_home_score', 0)))
        away_pred = float(prediction_data.get('predicted_away_final', prediction_data.get('predicted_away_score', 0)))
        
        # Intervals
        home_lower = float(prediction_data.get('home_lower_bound', home_pred - 10))
        home_upper = float(prediction_data.get('home_upper_bound', home_pred + 10))
        away_lower = float(prediction_data.get('away_lower_bound', away_pred - 10))
        away_upper = float(prediction_data.get('away_upper_bound', away_pred + 10))
        
        # Win probability
        win_prob = float(prediction_data.get('win_probability', 0.5))
        
        # Generate SVG using the template method
        svg = self._generate_confidence_svg(
            home_team, away_team, quarter, 
            home_score, away_score,
            home_pred, away_pred,
            home_lower, home_upper,
            away_lower, away_upper,
            win_prob
        )
        
        return svg
    
    def _generate_confidence_svg(self,
                               home_team: str,
                               away_team: str,
                               quarter: int,
                               home_score: float,
                               away_score: float,
                               home_pred: float,
                               away_pred: float,
                               home_lower: float,
                               home_upper: float,
                               away_lower: float,
                               away_upper: float,
                               win_prob: float) -> str:
        """
        Generate SVG code for confidence indicator.
        
        Args:
            home_team: Home team name
            away_team: Away team name
            quarter: Current quarter (0-4)
            home_score: Current home score
            away_score: Current away score
            home_pred: Predicted home score
            away_pred: Predicted away score
            home_lower: Home score lower bound
            home_upper: Home score upper bound
            away_lower: Away score lower bound
            away_upper: Away score upper bound
            win_prob: Win probability (0-1)
            
        Returns:
            String with SVG code
        """
        # SVG dimensions
        width = 360
        height = 220
        
        # Get quarter color
        quarter_color = self.quarter_colors.get(quarter, self.quarter_colors[0])
        
        # Get win probability color
        if win_prob > 0.65:
            win_color = self.colors['confidence_high']
        elif win_prob > 0.45:
            win_color = self.colors['confidence_med']
        else:
            win_color = self.colors['confidence_low']
            
        # Calculate vertical scale and positioning
        max_score = max(home_upper, away_upper, 130)
        y_scale = (height - 100) / max_score
        
        # Create SVG
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {width} {height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:500px;">
            <!-- Background -->
            <rect x="0" y="0" width="{width}" height="{height}" 
                  fill="{self.colors['background']}" rx="10" ry="10" />
            
            <!-- Header with Game Status -->
            <rect x="0" y="0" width="{width}" height="40" 
                  fill="{quarter_color}" rx="10" ry="10" />
            <text x="{width/2}" y="25" text-anchor="middle" 
                  font-family="system-ui" font-size="16" fill="white" font-weight="bold">
                {f"QUARTER {quarter}" if quarter > 0 else "PRE-GAME"} PREDICTION
            </text>
            
            <!-- Team Names and Current Score -->
            <text x="80" y="60" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['home_team']}">
                {home_team}
            </text>
            <text x="{width-80}" y="60" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['away_team']}">
                {away_team}
            </text>
            
            <text x="80" y="80" text-anchor="middle" 
                  font-family="system-ui" font-size="18" font-weight="bold" 
                  fill="{self.colors['text']}">
                {home_score:.0f}
            </text>
            <text x="{width-80}" y="80" text-anchor="middle" 
                  font-family="system-ui" font-size="18" font-weight="bold" 
                  fill="{self.colors['text']}">
                {away_score:.0f}
            </text>
            
            <!-- Confidence Intervals -->
            <rect x="60" y="{height - 70 - (home_upper * y_scale)}" 
                  width="40" height="{(home_upper - home_lower) * y_scale}" 
                  fill="{self.colors['home_team']}" fill-opacity="0.25" rx="5" ry="5" />
                  
            <rect x="{width-100}" y="{height - 70 - (away_upper * y_scale)}" 
                  width="40" height="{(away_upper - away_lower) * y_scale}" 
                  fill="{self.colors['away_team']}" fill-opacity="0.25" rx="5" ry="5" />
            
            <!-- Predicted Final Scores -->
            <line x1="40" x2="120" y1="{height - 70 - (home_pred * y_scale)}" 
                  y2="{height - 70 - (home_pred * y_scale)}" 
                  stroke="{self.colors['home_team']}" stroke-width="2.5" stroke-dasharray="2" />
                  
            <line x1="{width-120}" x2="{width-40}" y1="{height - 70 - (away_pred * y_scale)}" 
                  y2="{height - 70 - (away_pred * y_scale)}" 
                  stroke="{self.colors['away_team']}" stroke-width="2.5" stroke-dasharray="2" />
            
            <!-- Prediction Labels -->
            <text x="80" y="{height - 70 - (home_pred * y_scale) - 8}" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['home_team']}">
                {home_pred:.1f}
            </text>
            
            <text x="{width-80}" y="{height - 70 - (away_pred * y_scale) - 8}" text-anchor="middle" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{self.colors['away_team']}">
                {away_pred:.1f}
            </text>
            
            <!-- Win Probability Bar -->
            <text x="20" y="{height-45}" font-family="system-ui" font-size="12" 
                  fill="{self.colors['text']}">
                Win Probability:
            </text>
            
            <rect x="20" y="{height-35}" width="{width-40}" height="10" 
                  fill="#e9ecef" rx="5" ry="5" />
            <rect x="20" y="{height-35}" width="{(width-40) * win_prob}" height="10" 
                  fill="{win_color}" rx="5" ry="5" />
            
            <!-- Win Probability Value -->
            <text x="{width-20}" y="{height-45}" text-anchor="end" 
                  font-family="system-ui" font-size="14" font-weight="bold" 
                  fill="{win_color}">
                {win_prob*100:.1f}%
            </text>
            
            <!-- Timestamp -->
            <text x="{width-10}" y="{height-10}" text-anchor="end" 
                  font-family="system-ui" font-size="10" fill="{self.colors['axis']}">
                Updated: {datetime.now().strftime('%H:%M:%S')}
            </text>
        </svg>
        """
        
        return svg


class CalibrationEvaluator:
    """
    Evaluates and improves confidence interval calibration.
    
    This class implements methods to evaluate how well-calibrated
    uncertainty estimates are and adjust calibration factors.
    """
    
    def __init__(self, uncertainty_estimator=None):
        """
        Initialize the calibration evaluator.
        
        Args:
            uncertainty_estimator: UncertaintyEstimator instance to calibrate
        """
        self.uncertainty_estimator = uncertainty_estimator or UncertaintyEstimator()
        
        # Track calibration results
        self.calibration_history = []
    
    def evaluate_calibration(self, predictions_df: pd.DataFrame, 
                           actual_results_df: pd.DataFrame) -> Dict[str, Any]:
        """
        Evaluate how well-calibrated the prediction intervals are.
        
        Args:
            predictions_df: DataFrame with predictions and intervals
            actual_results_df: DataFrame with actual results
            
        Returns:
            Dict with calibration metrics
        """
        # Merge predictions with actuals
        merged = pd.merge(
            predictions_df,
            actual_results_df[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner',
            suffixes=('_pred', '_actual')
        )
        
        if merged.empty:
            return {'error': 'No matching games found'}
        
        # Initialize metrics
        results = {
            'total_evaluated': len(merged),
            'overall_coverage': 0,
            'by_quarter': {},
            'calibration_error': 0,
            'target_coverage': self.uncertainty_estimator.confidence_level
        }
        
        # Check if actual values fall within prediction intervals
        home_correct = ((merged['home_score'] >= merged['home_lower_bound']) & 
                      (merged['home_score'] <= merged['home_upper_bound']))
        away_correct = ((merged['away_score'] >= merged['away_lower_bound']) & 
                      (merged['away_score'] <= merged['away_upper_bound']))
        both_correct = home_correct & away_correct
        
        # Calculate overall coverage
        results['overall_coverage'] = both_correct.mean()
        results['home_coverage'] = home_correct.mean()
        results['away_coverage'] = away_correct.mean()
        
        # Calibration error (how far from target)
        results['calibration_error'] = abs(
            results['overall_coverage'] - results['target_coverage']
        )
        
        # Calculate per-quarter coverage
        for quarter in range(5):
            quarter_data = merged[merged['current_quarter'] == quarter]
            if not quarter_data.empty:
                q_home_correct = ((quarter_data['home_score'] >= quarter_data['home_lower_bound']) & 
                                (quarter_data['home_score'] <= quarter_data['home_upper_bound']))
                q_away_correct = ((quarter_data['away_score'] >= quarter_data['away_lower_bound']) & 
                                (quarter_data['away_score'] <= quarter_data['away_upper_bound']))
                q_both_correct = q_home_correct & q_away_correct
                
                results['by_quarter'][quarter] = {
                    'count': len(quarter_data),
                    'coverage': q_both_correct.mean(),
                    'width_factor': self.uncertainty_estimator.width_factors.get(quarter, 0)
                }
        
        # Calculate average interval width
        results['avg_home_width'] = (
            merged['home_upper_bound'] - merged['home_lower_bound']
        ).mean()
        results['avg_away_width'] = (
            merged['away_upper_bound'] - merged['away_lower_bound']
        ).mean()
        
        # Store calibration result
        self.calibration_history.append({
            'timestamp': datetime.now().isoformat(),
            'metrics': results,
            'width_factors': self.uncertainty_estimator.width_factors.copy()
        })
        
        return results
    
    def calibrate_width_factors(self, 
                              predictions_df: pd.DataFrame, 
                              actual_results_df: pd.DataFrame, 
                              adjustment_rate: float = 0.5) -> Dict[int, float]:
        """
        Calibrate width factors based on observed coverage rates.
        
        Args:
            predictions_df: DataFrame with predictions and intervals
            actual_results_df: DataFrame with actual results
            adjustment_rate: How much to adjust factors (0-1)
            
        Returns:
            Dict with updated width factors
        """
        # Evaluate current calibration
        calibration = self.evaluate_calibration(predictions_df, actual_results_df)
        
        # Initialize new factors with current values
        new_factors = self.uncertainty_estimator.width_factors.copy()
        
        # Adjust factors by quarter based on observed coverage
        for quarter, metrics in calibration.get('by_quarter', {}).items():
            if metrics['count'] >= 5:  # Only adjust with sufficient data
                current_coverage = metrics['coverage']
                target_coverage = self.uncertainty_estimator.confidence_level
                
                # Calculate adjustment ratio
                if current_coverage > 0:
                    adjustment_ratio = target_coverage / current_coverage
                    
                    # Apply adjustment with dampening
                    new_factors[quarter] = new_factors.get(quarter, 1.0) * (
                        1.0 + (adjustment_ratio - 1.0) * adjustment_rate
                    )
                    
                    # Ensure factor stays in reasonable range
                    new_factors[quarter] = min(max(new_factors[quarter], 0.5), 5.0)
        
        # Update estimator with new factors
        self.uncertainty_estimator.width_factors = new_factors
        
        # Record calibration update
        self.calibration_history.append({
            'timestamp': datetime.now().isoformat(),
            'action': 'calibrate',
            'before': calibration.get('width_factors', {}),
            'after': new_factors
        })
        
        return new_factors
    
    def visualize_calibration_history(self) -> plt.Figure:
        """
        Visualize the history of calibration metrics.
        
        Returns:
            matplotlib Figure object
        """
        if not self.calibration_history:
            return plt.figure()
        
        # Extract data from history
        timestamps = []
        overall_coverage = []
        quarter_coverage = {q: [] for q in range(5)}
        width_factors = {q: [] for q in range(5)}
        
        for entry in self.calibration_history:
            if 'metrics' in entry:
                metrics = entry['metrics']
                timestamps.append(entry.get('timestamp', ''))
                overall_coverage.append(metrics.get('overall_coverage', 0))
                
                for q, q_metrics in metrics.get('by_quarter', {}).items():
                    quarter_coverage[q].append(q_metrics.get('coverage', 0))
                    width_factors[q].append(entry.get('width_factors', {}).get(q, 0))
        
        # Create figure with subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Plot coverage rates
        ax1.axhline(y=self.uncertainty_estimator.confidence_level, color='black', 
                   linestyle='--', label=f'Target ({self.uncertainty_estimator.confidence_level:.0%})')
        
        ax1.plot(range(len(timestamps)), overall_coverage, 'o-', 
               label='Overall', linewidth=2, color='blue')
        
        # Plot quarter-specific coverage
        colors = ['gray', 'green', 'orange', 'red', 'purple']  # For quarters 0-4
        for q in range(5):
            if quarter_coverage[q]:
                ax1.plot(range(len(timestamps)), quarter_coverage[q], 's-', 
                       label=f'Q{q}', color=colors[q], alpha=0.7)
        
        ax1.set_ylabel('Coverage Rate')
        ax1.set_title('Confidence Interval Coverage Over Time')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Plot width factors
        for q in range(5):
            if width_factors[q]:
                ax2.plot(range(len(timestamps)), width_factors[q], 's-', 
                       label=f'Q{q} Factor', color=colors[q])
        
        ax2.set_xlabel('Calibration Iteration')
        ax2.set_ylabel('Width Factor')
        ax2.set_title('Confidence Interval Width Factors Over Time')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        return fig


# Integration functions

def add_uncertainty_to_predictions(predictions_df: pd.DataFrame, 
                                 uncertainty_estimator: Optional[UncertaintyEstimator] = None) -> pd.DataFrame:
    """
    Add uncertainty estimates to prediction results.
    
    Args:
        predictions_df: DataFrame with prediction results
        uncertainty_estimator: Optional custom uncertainty estimator
        
    Returns:
        DataFrame with uncertainty bounds added
    """
    # Create estimator if not provided
    if uncertainty_estimator is None:
        uncertainty_estimator = UncertaintyEstimator()
    
    # Apply intervals to predictions
    result_df = uncertainty_estimator.apply_intervals_to_predictions(predictions_df)
    
    # Add additional uncertainty-related fields
    for idx, row in result_df.iterrows():
        # Get prediction confidence based on interval width
        home_width = row['home_upper_bound'] - row['home_lower_bound']
        away_width = row['away_upper_bound'] - row['away_lower_bound']
        avg_width = (home_width + away_width) / 2
        
        # Add normalized prediction confidence (1.0 is highest confidence)
        # Formula maps interval width to confidence - wider intervals = lower confidence
        max_expected_width = 30.0  # Maximum reasonable interval width
        confidence = max(0.0, min(1.0, 1.0 - (avg_width / max_expected_width)))
        
        # Store confidence
        result_df.loc[idx, 'prediction_confidence'] = confidence
        
        # Add uncertainty ratio (ratio of uncertainty to predicted value)
        home_pred = row['predicted_home_final']
        away_pred = row['predicted_away_final']
        
        if home_pred > 0:
            result_df.loc[idx, 'home_uncertainty_ratio'] = home_width / home_pred
        if away_pred > 0:
            result_df.loc[idx, 'away_uncertainty_ratio'] = away_width / away_pred
    
    return result_df


# Function to create confidence visualizations for a game
def create_game_confidence_visualization(game_data: Dict[str, Any], 
                                       color_scheme: str = 'default') -> Dict[str, Any]:
    """
    Create confidence visualizations for a single game prediction.
    
    Args:
        game_data: Dict with game prediction data
        color_scheme: Color scheme for visualization
        
    Returns:
        Dict with visualization components (SVG indicator, interval data)
    """
    # Create visualizer
    visualizer = ConfidenceVisualizer(color_scheme=color_scheme)
    
    # Generate SVG indicator
    svg_indicator = visualizer.create_confidence_indicator(game_data)
    
    # Extract confidence interval data for API response
    confidence_data = {
        'home_team': game_data.get('home_team', 'Home'),
        'away_team': game_data.get('away_team', 'Away'),
        'home_score': {
            'current': game_data.get('home_score', 0),
            'predicted': game_data.get('predicted_home_final', 0),
            'lower_bound': game_data.get('home_lower_bound', 0),
            'upper_bound': game_data.get('home_upper_bound', 0),
            'interval_width': game_data.get('home_upper_bound', 0) - game_data.get('home_lower_bound', 0)
        },
        'away_score': {
            'current': game_data.get('away_score', 0),
            'predicted': game_data.get('predicted_away_final', 0),
            'lower_bound': game_data.get('away_lower_bound', 0),
            'upper_bound': game_data.get('away_upper_bound', 0),
            'interval_width': game_data.get('away_upper_bound', 0) - game_data.get('away_lower_bound', 0)
        },
        'win_probability': game_data.get('win_probability', 0.5),
        'quarter': game_data.get('current_quarter', 0),
        'visualization_svg': svg_indicator
    }
    
    return confidence_data


# Create instances for testing and demonstration
def demonstrate_uncertainty_estimation():
    """Demonstrate uncertainty estimation functionality."""
    # Create sample prediction data
    sample_data = [
        {
            'game_id': 1001,
            'home_team': 'Lakers',
            'away_team': 'Celtics',
            'current_quarter': 2,
            'home_score': 52,
            'away_score': 48,
            'predicted_home_final': 105.5,
            'predicted_away_final': 98.7,
            'win_probability': 0.65
        },
        {
            'game_id': 1002,
            'home_team': 'Warriors',
            'away_team': 'Bucks',
            'current_quarter': 4,
            'home_score': 98,
            'away_score': 92,
            'predicted_home_final': 106.2,
            'predicted_away_final': 100.8,
            'win_probability': 0.85
        }
    ]
    
    df = pd.DataFrame(sample_data)
    
    # Create instances
    uncertainty_estimator = UncertaintyEstimator()
    confidence_visualizer = ConfidenceVisualizer()
    calibration_evaluator = CalibrationEvaluator(uncertainty_estimator)
    
    # Add uncertainty to predictions
    predictions_with_uncertainty = add_uncertainty_to_predictions(df, uncertainty_estimator)
    
    print("Predictions with Uncertainty Estimates:")
    if 'get_ipython' in globals():
        from IPython.display import display
        display(predictions_with_uncertainty)
    else:
        print(predictions_with_uncertainty)
    
    # Create visualization for first game
    first_game = predictions_with_uncertainty.iloc[0].to_dict()
    svg = confidence_visualizer.create_confidence_indicator(first_game)
    
    # Display SVG
    if 'get_ipython' in globals():
        from IPython.display import HTML
        display(HTML(svg))
    else:
        print("SVG visualization created (length: {len(svg)} characters)")
    
    return predictions_with_uncertainty

# Run the demonstration if in notebook
if 'get_ipython' in globals():
    uncertainty_estimator = UncertaintyEstimator()
    confidence_visualizer = ConfidenceVisualizer()
    calibration_evaluator = CalibrationEvaluator(uncertainty_estimator)
    demonstrate_uncertainty_estimation()

In [ ]:
# Cell 27A: Auto-calibrating Confidence Intervals

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union, Any

def calibrate_confidence_intervals(validation_results: pd.DataFrame, 
                                 current_width_factors: Dict[int, float],
                                 target_coverage: float = 0.9,
                                 dampen_factor: float = 0.5) -> Dict[int, float]:
    """
    Adjust confidence interval widths based on historical accuracy.
    
    This function examines the historical coverage rates of prediction intervals
    and adjusts the width factors to try to achieve the target coverage rate.
    
    Args:
        validation_results: DataFrame with validation results containing 'quarter',
                           'actual_score', 'lower_bound', and 'upper_bound' columns
        current_width_factors: Dict mapping quarters (0-4) to current width factors
        target_coverage: Target coverage rate (default: 0.9 for 90% confidence)
        dampen_factor: Factor to dampen adjustments (0-1, lower = more conservative)
    
    Returns:
        Dict mapping quarters to adjusted width factors
    """
    coverage_by_quarter = {}
    
    # Calculate actual coverage rates
    for quarter in range(5):  # 0-4 for pre-game and quarters 1-4
        quarter_results = validation_results[validation_results['quarter'] == quarter]
        if not quarter_results.empty:
            in_interval = ((quarter_results['actual_score'] >= quarter_results['lower_bound']) & 
                          (quarter_results['actual_score'] <= quarter_results['upper_bound']))
            coverage_by_quarter[quarter] = in_interval.mean()
    
    # Adjust width factors
    adjusted_factors = {}
    for quarter, coverage in coverage_by_quarter.items():
        # Handle edge cases for coverage
        if coverage <= 0.01:  # Avoid division by zero or very small values
            # If coverage is near zero, significantly widen intervals
            adjusted_factors[quarter] = current_width_factors.get(quarter, 1.0) * 1.5
            continue
            
        # If coverage is too low, widen the intervals (ratio > 1)
        # If coverage is too high, narrow the intervals (ratio < 1)
        adjustment_ratio = target_coverage / coverage
        
        # Apply dampening to prevent overcorrection
        dampened_adjustment = 1.0 + (adjustment_ratio - 1.0) * dampen_factor
        
        # Calculate new factor with dampening
        new_factor = current_width_factors.get(quarter, 1.0) * dampened_adjustment
        
        # Ensure factors stay within reasonable bounds
        adjusted_factors[quarter] = max(min(new_factor, 5.0), 0.5)
    
    # Ensure all quarters have factors, using defaults for missing quarters
    for quarter in range(5):
        if quarter not in adjusted_factors:
            # Default mapping of quarter to width factor if not calculated
            default_factors = {0: 3.0, 1: 2.5, 2: 2.0, 3: 1.5, 4: 1.0}
            adjusted_factors[quarter] = current_width_factors.get(quarter, default_factors[quarter])
    
    return adjusted_factors


def validate_confidence_intervals(predictions_df: pd.DataFrame, 
                               actual_results_df: pd.DataFrame, 
                               confidence_level: float = 0.9) -> pd.DataFrame:
    """
    Validate that confidence intervals contain actual values at the expected rate.
    
    Args:
        predictions_df: DataFrame with predictions and confidence intervals containing
                      at minimum 'game_id', 'current_quarter', 'home_lower_bound',
                      'home_upper_bound', 'away_lower_bound', 'away_upper_bound'
        actual_results_df: DataFrame with actual final scores containing
                         at minimum 'game_id', 'home_score', 'away_score'
        confidence_level: Expected coverage rate (default: 0.9 for 90% confidence)
    
    Returns:
        DataFrame with validation metrics by quarter including coverage rates
        and interval widths
    """
    # Merge predictions with actual results
    merged = pd.merge(
        predictions_df,
        actual_results_df[['game_id', 'home_score', 'away_score']],
        on='game_id', 
        suffixes=('_pred', '_actual')
    )
    
    if merged.empty:
        print("Warning: No matching games found between predictions and actuals")
        return pd.DataFrame()
    
    # Calculate validation metrics
    results = []
    for quarter in range(5):
        quarter_data = merged[merged['current_quarter'] == quarter]
        if quarter_data.empty:
            continue
        
        # Check if actual scores fall within predicted intervals
        home_in_interval = ((quarter_data['home_score'] >= quarter_data['home_lower_bound']) & 
                           (quarter_data['home_score'] <= quarter_data['home_upper_bound']))
        
        away_in_interval = ((quarter_data['away_score'] >= quarter_data['away_lower_bound']) & 
                           (quarter_data['away_score'] <= quarter_data['away_upper_bound']))
        
        # Calculate coverage rates
        home_coverage = home_in_interval.mean()
        away_coverage = away_in_interval.mean()
        avg_coverage = (home_coverage + away_coverage) / 2
        
        # Calculate average interval widths
        home_width = (quarter_data['home_upper_bound'] - quarter_data['home_lower_bound']).mean()
        away_width = (quarter_data['away_upper_bound'] - quarter_data['away_lower_bound']).mean()
        
        results.append({
            'quarter': quarter,
            'home_coverage': home_coverage,
            'away_coverage': away_coverage,
            'avg_coverage': avg_coverage,
            'target_coverage': confidence_level,
            'coverage_error': avg_coverage - confidence_level,
            'home_interval_width': home_width,
            'away_interval_width': away_width,
            'avg_interval_width': (home_width + away_width) / 2,
            'sample_size': len(quarter_data)
        })
    
    return pd.DataFrame(results)


def generate_test_calibration_data(uncertainty_estimator, 
                                 num_samples: int = 50, 
                                 target_coverage: float = 0.9) -> pd.DataFrame:
    """
    Generate synthetic calibration data for testing calibration methods.
    
    Args:
        uncertainty_estimator: An instance of UncertaintyEstimator class
        num_samples: Number of synthetic samples to generate
        target_coverage: Target coverage rate to simulate
        
    Returns:
        DataFrame with synthetic validation data
    """
    np.random.seed(42)  # For reproducibility
    validation_data = []
    
    # Generate samples across quarters
    for i in range(num_samples):
        # Randomly select quarter
        quarter = np.random.randint(0, 5)
        
        # Generate predicted score (around 110 points with some variance)
        pred_score = np.random.normal(110, 5)
        
        # Sample score differential and momentum
        score_diff = np.random.uniform(0, 20)
        momentum = np.random.uniform(-0.8, 0.8)
        
        # Get interval from uncertainty estimator
        lower, upper, width = uncertainty_estimator.calculate_prediction_interval(
            pred_score, quarter, score_diff, momentum)
        
        # Generate actual score
        # Simulate a actual_within_interval with target_coverage probability
        if np.random.random() < target_coverage:
            # Generate score within interval
            actual_score = np.random.uniform(lower, upper)
        else:
            # Generate score outside interval
            if np.random.random() < 0.5:
                actual_score = np.random.uniform(upper, upper + width)
            else:
                actual_score = np.random.uniform(lower - width, lower)
        
        validation_data.append({
            'quarter': quarter,
            'predicted_score': pred_score,
            'lower_bound': lower,
            'upper_bound': upper,
            'interval_width': width,
            'actual_score': actual_score,
            'in_interval': (lower <= actual_score <= upper),
            'error': abs(pred_score - actual_score),
            'score_diff': score_diff,
            'momentum': momentum
        })
    
    return pd.DataFrame(validation_data)


def run_calibration_test(uncertainty_estimator) -> Dict[str, Any]:
    """
    Run a complete test of the calibration system.
    
    This function generates synthetic data, validates intervals, 
    calibrates the width factors, and visualizes the results.
    
    Args:
        uncertainty_estimator: An instance of UncertaintyEstimator
        
    Returns:
        Dict with test results including calibrated factors and metrics
    """
    # Step 1: Generate test validation data
    print("Generating synthetic validation data...")
    validation_data = generate_test_calibration_data(
        uncertainty_estimator, num_samples=200, target_coverage=0.9)
    
    # Calculate initial coverage
    initial_coverage = validation_data['in_interval'].mean()
    print(f"Initial overall coverage: {initial_coverage:.2%}")
    
    # Coverage by quarter
    quarter_coverage = validation_data.groupby('quarter')['in_interval'].mean()
    print("\nCoverage by quarter:")
    for quarter, coverage in quarter_coverage.items():
        print(f"Quarter {quarter}: {coverage:.2%}")
    
    # Step 2: Run calibration
    print("\nCalibrating width factors...")
    initial_factors = uncertainty_estimator.width_factors.copy()
    calibrated_factors = calibrate_confidence_intervals(
        validation_data, initial_factors)
    
    print("\nCalibration Results:")
    print("Quarter | Initial Factor | Calibrated Factor | Change")
    print("--------|----------------|-------------------|-------")
    for quarter in range(5):
        initial = initial_factors.get(quarter, 1.0)
        calibrated = calibrated_factors.get(quarter, 1.0)
        change = calibrated - initial
        print(f"   {quarter}    |      {initial:.2f}      |       {calibrated:.2f}        | {change:+.2f}")
    
    # Step 3: Visualize calibration
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot coverage by quarter
    ax.bar(
        [f"Q{q}" for q in range(5)],
        [quarter_coverage.get(q, 0) for q in range(5)],
        alpha=0.6,
        label="Initial Coverage"
    )
    
    # Add target line
    ax.axhline(y=0.9, color='red', linestyle='--', label="Target (90%)")
    
    # Customize plot
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Coverage Rate")
    ax.set_xlabel("Quarter")
    ax.set_title("Confidence Interval Coverage by Quarter")
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    
    return {
        'validation_data': validation_data,
        'initial_factors': initial_factors,
        'calibrated_factors': calibrated_factors,
        'initial_coverage': initial_coverage,
        'quarter_coverage': quarter_coverage,
        'visualization': fig
    }


def integrate_with_main_pipeline(game_predictions: pd.DataFrame,
                              actual_results: pd.DataFrame,
                              uncertainty_estimator,
                              apply_calibration: bool = True) -> Dict[str, Any]:
    """
    Integrate the calibration system with the main prediction pipeline.
    
    This function combines validation, calibration, and application of 
    uncertainty estimates within the main prediction workflow.
    
    Args:
        game_predictions: DataFrame with game predictions
        actual_results: DataFrame with actual game results
        uncertainty_estimator: Instance of UncertaintyEstimator
        apply_calibration: Whether to apply calibration updates (default: True)
        
    Returns:
        Dict with processing results and updated predictions
    """
    # Step 1: Apply current uncertainty estimates
    predictions_with_intervals = uncertainty_estimator.apply_intervals_to_predictions(
        game_predictions)
    
    # Step 2: Validate intervals against actuals
    validation_metrics = validate_confidence_intervals(
        predictions_with_intervals, actual_results)
    
    results = {
        'predictions': predictions_with_intervals,
        'validation_metrics': validation_metrics,
        'width_factors': uncertainty_estimator.width_factors.copy()
    }
    
    # Skip calibration if requested or if no validation data
    if not apply_calibration or validation_metrics.empty:
        return results
    
    # Step 3: Update calibration if we have actuals
    # Prepare data in format expected by calibrate_confidence_intervals
    validation_data = []
    
    # Merge actuals with predictions
    merged = pd.merge(
        predictions_with_intervals,
        actual_results[['game_id', 'home_score', 'away_score']],
        on='game_id'
    )
    
    if not merged.empty:
        # Process home scores
        home_data = merged[['current_quarter', 'home_lower_bound', 
                          'home_upper_bound', 'home_score']].copy()
        home_data.columns = ['quarter', 'lower_bound', 'upper_bound', 'actual_score']
        validation_data.append(home_data)
        
        # Process away scores
        away_data = merged[['current_quarter', 'away_lower_bound', 
                          'away_upper_bound', 'away_score']].copy()
        away_data.columns = ['quarter', 'lower_bound', 'upper_bound', 'actual_score']
        validation_data.append(away_data)
        
        # Combine and calibrate
        validation_df = pd.concat(validation_data, ignore_index=True)
        
        calibrated_factors = calibrate_confidence_intervals(
            validation_df, uncertainty_estimator.width_factors)
        
        # Update estimator with new factors
        if apply_calibration:
            uncertainty_estimator.width_factors = calibrated_factors
        
        # Record calibration changes
        results['calibrated_factors'] = calibrated_factors
        results['calibration_change'] = {
            q: calibrated_factors.get(q, 0) - results['width_factors'].get(q, 0)
            for q in range(5)
        }
        
        # Re-apply with calibrated factors if needed
        if apply_calibration:
            results['calibrated_predictions'] = uncertainty_estimator.apply_intervals_to_predictions(
                game_predictions)
    
    return results


# Example usage with the UncertaintyEstimator from Cell 27
if 'UncertaintyEstimator' in globals():
    # Create an estimator instance
    test_estimator = UncertaintyEstimator()
    
    # Run calibration test with synthetic data
    print("Running calibration test with synthetic data...")
    test_results = run_calibration_test(test_estimator)
    
    # Display results
    if 'get_ipython' in globals():
        # In Jupyter notebook, show the plot
        from IPython.display import display
        display(test_results['visualization'])
    
    print("\nSynthetic test complete. The calibration system is ready for integration.")


In [ ]:
# Cell 27B: Interactive Confidence Visualization for PWA Integration

import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional, Union, Any
from datetime import datetime

class InteractiveConfidenceDisplay:
    """
    Class for creating interactive visualizations of prediction confidence
    optimized for Progressive Web App (PWA) integration.
    
    This class provides methods to generate SVG-based visualizations
    in different styles for displaying NBA score predictions with
    confidence intervals.
    """
    
    def __init__(self):
        """Initialize the display generator with default settings."""
        # Quarter-specific colors for visual differentiation
        self.quarter_colors = {
            0: "#6c757d",  # Gray for pregame
            1: "#20c997",  # Teal for Q1
            2: "#0dcaf0",  # Cyan for Q2
            3: "#0d6efd",  # Blue for Q3
            4: "#6610f2"   # Purple for Q4
        }
        
        # Color schemes for teams and UI elements
        self.color_scheme = {
            'home_team': "#0066cc",
            'away_team': "#cc0000",
            'background': "#f8f9fa",
            'win_positive': "#198754",  # Green for positive win probability
            'win_negative': "#dc3545",  # Red for negative win probability
            'win_neutral': "#fd7e14",   # Orange for neutral/even win probability
            'grid': "#dddddd",
            'text': "#212529",
            'highlight': "#4CAF50"
        }
    
    def create_display(self, 
                      prediction_data: Dict[str, Any], 
                      design_style: str = 'pwa') -> str:
        """
        Create an interactive SVG visualization showing prediction confidence.
        
        Args:
            prediction_data: Dict with prediction details including team names,
                           scores, predictions, and confidence intervals
            design_style: Design style to use:
                        - 'minimal': Simple, clean visualization
                        - 'detailed': More detailed visualization with additional information
                        - 'pwa': Optimized for mobile Progressive Web App display
        
        Returns:
            str: SVG/HTML markup for the confidence display
        """
        # Validate inputs
        if not isinstance(prediction_data, dict):
            raise ValueError("Prediction data must be a dictionary")
        
        # Extract prediction data with fallbacks for missing values
        home_team = prediction_data.get('home_team', 'Home')
        away_team = prediction_data.get('away_team', 'Away')
        current_quarter = int(prediction_data.get('current_quarter', 0))
        home_score = float(prediction_data.get('home_score', 0))
        away_score = float(prediction_data.get('away_score', 0))
        
        # Get predicted scores
        predicted_home = float(prediction_data.get('predicted_home_score', 
                                                 prediction_data.get('predicted_home_final', 0)))
        predicted_away = float(prediction_data.get('predicted_away_score', 
                                                 prediction_data.get('predicted_away_final', 0)))
        
        # Get confidence interval data with reasonable defaults
        home_lower = float(prediction_data.get('home_lower_bound', predicted_home * 0.9))
        home_upper = float(prediction_data.get('home_upper_bound', predicted_home * 1.1))
        away_lower = float(prediction_data.get('away_lower_bound', predicted_away * 0.9))
        away_upper = float(prediction_data.get('away_upper_bound', predicted_away * 1.1))
        
        # Win probability
        win_prob = float(prediction_data.get('win_probability', 0.5))
        
        # Choose the appropriate visualization style
        if design_style == 'minimal':
            return self._create_minimal_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
        elif design_style == 'detailed':
            return self._create_detailed_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
        elif design_style == 'pwa':
            return self._create_pwa_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
        else:
            # Default to PWA style if unrecognized style is specified
            return self._create_pwa_display(
                home_team, away_team, current_quarter,
                home_score, away_score,
                predicted_home, predicted_away,
                home_lower, home_upper,
                away_lower, away_upper,
                win_prob
            )
    
    def _create_minimal_display(self,
                              home_team: str,
                              away_team: str,
                              current_quarter: int,
                              home_score: float,
                              away_score: float,
                              predicted_home: float,
                              predicted_away: float,
                              home_lower: float,
                              home_upper: float,
                              away_lower: float,
                              away_upper: float,
                              win_prob: float) -> str:
        """
        Create a clean, minimal SVG visualization with just the essentials.
        
        This version is optimized for simple embedding and smaller screen sizes.
        
        Args:
            Various parameters describing the game state and predictions
            
        Returns:
            str: SVG markup for the minimal visualization
        """
        # SVG dimensions
        svg_width = 400
        svg_height = 200
        
        # Calculate position and scaling - ensure we have room for all scores
        max_score = max(home_upper, away_upper, 130)  # Minimum bound of 130 points
        y_scale = (svg_height - 60) / max_score
        
        # Generate SVG
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {svg_width} {svg_height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:400px;" class="confidence-display minimal">
            <!-- Title -->
            <text x="{svg_width/2}" y="20" text-anchor="middle" font-family="Arial" font-size="16" font-weight="bold">
                {f"Quarter {current_quarter}" if current_quarter > 0 else "Pre-Game"} Prediction
            </text>
            
            <!-- Team names -->
            <text x="100" y="50" text-anchor="middle" font-family="Arial" font-size="14" fill="{self.color_scheme['home_team']}">
                {home_team}
            </text>
            <text x="{svg_width-100}" y="50" text-anchor="middle" font-family="Arial" font-size="14" fill="{self.color_scheme['away_team']}">
                {away_team}
            </text>
            
            <!-- Current scores -->
            <text x="100" y="75" text-anchor="middle" font-family="Arial" font-size="18" font-weight="bold">
                {home_score:.0f}
            </text>
            <text x="{svg_width-100}" y="75" text-anchor="middle" font-family="Arial" font-size="18" font-weight="bold">
                {away_score:.0f}
            </text>
            
            <!-- Confidence intervals -->
            <rect x="80" y="{svg_height - (home_upper * y_scale)}" width="40" height="{(home_upper - home_lower) * y_scale}" 
                  fill="{self.color_scheme['home_team']}" fill-opacity="0.3" />
            <rect x="{svg_width-120}" y="{svg_height - (away_upper * y_scale)}" width="40" height="{(away_upper - away_lower) * y_scale}" 
                  fill="{self.color_scheme['away_team']}" fill-opacity="0.3" />
            
            <!-- Predicted scores -->
            <line x1="80" x2="120" y1="{svg_height - (predicted_home * y_scale)}" y2="{svg_height - (predicted_home * y_scale)}" 
                  stroke="{self.color_scheme['home_team']}" stroke-width="2" />
            <line x1="{svg_width-120}" x2="{svg_width-80}" y1="{svg_height - (predicted_away * y_scale)}" y2="{svg_height - (predicted_away * y_scale)}" 
                  stroke="{self.color_scheme['away_team']}" stroke-width="2" />
            
            <!-- Predicted score labels -->
            <text x="100" y="{svg_height - (predicted_home * y_scale) - 5}" text-anchor="middle" font-family="Arial" font-size="12">
                {predicted_home:.1f}
            </text>
            <text x="{svg_width-100}" y="{svg_height - (predicted_away * y_scale) - 5}" text-anchor="middle" font-family="Arial" font-size="12">
                {predicted_away:.1f}
            </text>
            
            <!-- Win probability indicator -->
            <rect x="{svg_width/2 - 75}" y="{svg_height - 25}" width="150" height="15" fill="#eeeeee" rx="7" ry="7" />
            <rect x="{svg_width/2 - 75}" y="{svg_height - 25}" width="{150 * win_prob}" height="15" 
                  fill="{self._get_win_probability_color(win_prob)}" rx="7" ry="7" />
            <text x="{svg_width/2}" y="{svg_height - 12}" text-anchor="middle" font-family="Arial" font-size="10">
                {home_team} Win: {win_prob*100:.1f}%
            </text>
        </svg>
        """
        
        return svg
    
    def _create_detailed_display(self,
                               home_team: str,
                               away_team: str,
                               current_quarter: int,
                               home_score: float,
                               away_score: float,
                               predicted_home: float,
                               predicted_away: float,
                               home_lower: float,
                               home_upper: float,
                               away_lower: float,
                               away_upper: float,
                               win_prob: float) -> str:
        """
        Create a more detailed SVG with additional information and interactive elements.
        
        This version includes axis labels, range indicators, and detailed explanatory text.
        
        Args:
            Various parameters describing the game state and predictions
            
        Returns:
            str: SVG markup for the detailed visualization
        """
        # SVG dimensions
        svg_width = 500
        svg_height = 300
        
        # Calculate position and scaling
        max_score = max(home_upper, away_upper, 130)
        y_scale = (svg_height - 100) / max_score
        
        # Create tooltips for key elements
        home_tooltip = f"Predicted: {predicted_home:.1f} (range: {home_lower:.1f}-{home_upper:.1f})"
        away_tooltip = f"Predicted: {predicted_away:.1f} (range: {away_lower:.1f}-{away_upper:.1f})"
        
        # Calculate interval widths for display
        home_width = home_upper - home_lower
        away_width = away_upper - away_lower
        
        # Generate SVG with more details and tooltip elements
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {svg_width} {svg_height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:600px;" class="confidence-display detailed">
            <!-- Title and Game Info -->
            <text x="{svg_width/2}" y="25" text-anchor="middle" font-family="Arial" font-size="18" font-weight="bold">
                {f"Quarter {current_quarter}" if current_quarter > 0 else "Pre-Game"} Prediction
            </text>
            <text x="{svg_width/2}" y="45" text-anchor="middle" font-family="Arial" font-size="14">
                {home_team} vs {away_team}
            </text>
            
            <!-- Background grid lines -->
            <g stroke="{self.color_scheme['grid']}" stroke-width="1">
                <line x1="50" y1="80" x2="{svg_width-50}" y2="80" />
                <line x1="50" y1="130" x2="{svg_width-50}" y2="130" />
                <line x1="50" y1="180" x2="{svg_width-50}" y2="180" />
                <line x1="50" y1="230" x2="{svg_width-50}" y2="230" />
            </g>
            
            <!-- Score axis labels -->
            <text x="40" y="80" text-anchor="end" font-family="Arial" font-size="10">{int(svg_height/y_scale)}</text>
            <text x="40" y="130" text-anchor="end" font-family="Arial" font-size="10">{int((svg_height-50)/y_scale)}</text>
            <text x="40" y="180" text-anchor="end" font-family="Arial" font-size="10">{int((svg_height-100)/y_scale)}</text>
            <text x="40" y="230" text-anchor="end" font-family="Arial" font-size="10">{int((svg_height-150)/y_scale)}</text>
            
            <!-- Team columns with confidence intervals -->
            <!-- Home team -->
            <g class="home-team">
                <text x="150" y="70" text-anchor="middle" font-family="Arial" font-size="16" font-weight="bold" fill="{self.color_scheme['home_team']}">
                    {home_team}
                </text>
                <text x="150" y="90" text-anchor="middle" font-family="Arial" font-size="20" font-weight="bold">
                    {home_score:.0f}
                </text>
                
                <!-- Confidence interval -->
                <rect x="130" y="{svg_height - (home_upper * y_scale)}" width="40" height="{(home_upper - home_lower) * y_scale}" 
                      fill="{self.color_scheme['home_team']}" fill-opacity="0.3">
                    <title>{home_tooltip}</title>
                </rect>
                
                <!-- Predicted score marker -->
                <line x1="130" x2="170" y1="{svg_height - (predicted_home * y_scale)}" y2="{svg_height - (predicted_home * y_scale)}" 
                      stroke="{self.color_scheme['home_team']}" stroke-width="3" />
                
                <!-- Predicted score label -->
                <text x="150" y="{svg_height - (predicted_home * y_scale) - 10}" text-anchor="middle" 
                      font-family="Arial" font-size="14" font-weight="bold">
                    {predicted_home:.1f}
                </text>
                
                <!-- Range indicators -->
                <text x="180" y="{svg_height - (home_upper * y_scale)}" text-anchor="start" font-family="Arial" font-size="12">
                    ↑ {home_upper:.1f}
                </text>
                <text x="180" y="{svg_height - (home_lower * y_scale)}" text-anchor="start" font-family="Arial" font-size="12">
                    ↓ {home_lower:.1f}
                </text>
                
                <!-- Confidence width -->
                <text x="180" y="{svg_height - ((home_upper + home_lower) * y_scale/2)}" text-anchor="start" 
                      font-family="Arial" font-size="10" fill="#666666">
                    ±{home_width/2:.1f}
                </text>
            </g>
            
            <!-- Away team -->
            <g class="away-team">
                <text x="350" y="70" text-anchor="middle" font-family="Arial" font-size="16" font-weight="bold" fill="{self.color_scheme['away_team']}">
                    {away_team}
                </text>
                <text x="350" y="90" text-anchor="middle" font-family="Arial" font-size="20" font-weight="bold">
                    {away_score:.0f}
                </text>
                
                <!-- Confidence interval -->
                <rect x="330" y="{svg_height - (away_upper * y_scale)}" width="40" height="{(away_upper - away_lower) * y_scale}" 
                      fill="{self.color_scheme['away_team']}" fill-opacity="0.3">
                    <title>{away_tooltip}</title>
                </rect>
                
                <!-- Predicted score marker -->
                <line x1="330" x2="370" y1="{svg_height - (predicted_away * y_scale)}" y2="{svg_height - (predicted_away * y_scale)}" 
                      stroke="{self.color_scheme['away_team']}" stroke-width="3" />
                
                <!-- Predicted score label -->
                <text x="350" y="{svg_height - (predicted_away * y_scale) - 10}" text-anchor="middle" 
                      font-family="Arial" font-size="14" font-weight="bold">
                    {predicted_away:.1f}
                </text>
                
                <!-- Range indicators -->
                <text x="320" y="{svg_height - (away_upper * y_scale)}" text-anchor="end" font-family="Arial" font-size="12">
                    ↑ {away_upper:.1f}
                </text>
                <text x="320" y="{svg_height - (away_lower * y_scale)}" text-anchor="end" font-family="Arial" font-size="12">
                    ↓ {away_lower:.1f}
                </text>
                
                <!-- Confidence width -->
                <text x="320" y="{svg_height - ((away_upper + away_lower) * y_scale/2)}" text-anchor="end" 
                      font-family="Arial" font-size="10" fill="#666666">
                    ±{away_width/2:.1f}
                </text>
            </g>
            
            <!-- Win probability gauge -->
            <g class="win-probability">
                <text x="{svg_width/2}" y="{svg_height-50}" text-anchor="middle" font-family="Arial" font-size="14" font-weight="bold">
                    Win Probability
                </text>
                
                <!-- Probability bar -->
                <rect x="{svg_width/2 - 100}" y="{svg_height-40}" width="200" height="20" fill="#eeeeee" rx="10" ry="10" />
                <rect x="{svg_width/2 - 100}" y="{svg_height-40}" width="{200 * win_prob}" height="20" 
                      fill="{self._get_win_probability_color(win_prob)}" rx="10" ry="10">
                    <title>Win probability: {win_prob*100:.1f}%</title>
                </rect>
                
                <!-- Home team marker -->
                <text x="{svg_width/2 - 110}" y="{svg_height-30}" text-anchor="end" font-family="Arial" font-size="12">
                    {home_team}
                </text>
                
                <!-- Away team marker -->
                <text x="{svg_width/2 + 110}" y="{svg_height-30}" text-anchor="start" font-family="Arial" font-size="12">
                    {away_team}
                </text>
                
                <!-- Probability percentage -->
                <text x="{svg_width/2}" y="{svg_height-30}" text-anchor="middle" font-family="Arial" font-size="14" font-weight="bold">
                    {win_prob*100:.1f}%
                </text>
            </g>
            
            <!-- Confidence level explanation -->
            <text x="{svg_width/2}" y="{svg_height-10}" text-anchor="middle" font-family="Arial" font-size="10" fill="#666666">
                Shaded areas show 90% confidence intervals for final score predictions
            </text>
        </svg>
        """
        
        return svg
    
    def _create_pwa_display(self,
                          home_team: str,
                          away_team: str,
                          current_quarter: int,
                          home_score: float,
                          away_score: float,
                          predicted_home: float,
                          predicted_away: float,
                          home_lower: float,
                          home_upper: float,
                          away_lower: float,
                          away_upper: float,
                          win_prob: float) -> str:
        """
        Create a mobile-optimized visualization for Progressive Web App.
        
        This version is fully responsive, has touch-friendly elements, and 
        uses a modern design system optimized for mobile devices.
        
        Args:
            Various parameters describing the game state and predictions
            
        Returns:
            str: SVG markup for the PWA-optimized visualization
        """
        # SVG dimensions - mobile optimized
        svg_width = 360  # Standard mobile width
        svg_height = 240
        
        # Get quarter-specific color
        quarter_color = self.quarter_colors.get(current_quarter, self.quarter_colors[0])
        
        # Calculate position and scaling
        max_score = max(home_upper, away_upper, 130)
        y_scale = (svg_height - 80) / max_score
        
        # Calculate win probability styling
        win_color = self._get_win_probability_color(win_prob)
        
        # Determine status text based on win probability
        home_win_text = self._get_win_probability_text(win_prob)
        
        # Create detailed tooltips
        home_tooltip = f"Home prediction: {predicted_home:.1f} points\nConfidence range: {home_lower:.1f} to {home_upper:.1f}\nCurrent score: {home_score}"
        away_tooltip = f"Away prediction: {predicted_away:.1f} points\nConfidence range: {away_lower:.1f} to {away_upper:.1f}\nCurrent score: {away_score}"
        win_tooltip = f"Win probability: {win_prob*100:.1f}%\nBased on score projection and historical patterns"
        
        # Generate responsive SVG optimized for mobile PWA
        svg = f"""
        <svg width="100%" height="100%" viewBox="0 0 {svg_width} {svg_height}" 
             xmlns="http://www.w3.org/2000/svg" style="max-width:600px;" 
             class="prediction-chart" data-quarter="{current_quarter}">
            <!-- Main Container -->
            <rect x="0" y="0" width="{svg_width}" height="{svg_height}" fill="{self.color_scheme['background']}" rx="10" ry="10" />
            
            <!-- Header with Game Status -->
            <rect x="0" y="0" width="{svg_width}" height="40" fill="{quarter_color}" rx="10" ry="10" />
            <text x="{svg_width/2}" y="25" text-anchor="middle" font-family="system-ui" font-size="16" fill="white" font-weight="bold">
                {f"QUARTER {current_quarter}" if current_quarter > 0 else "PRE-GAME"} PREDICTION
            </text>
            
            <!-- Team Header -->
            <text x="80" y="60" text-anchor="middle" font-family="system-ui" font-size="14" font-weight="bold">
                {home_team}
            </text>
            <text x="{svg_width-80}" y="60" text-anchor="middle" font-family="system-ui" font-size="14" font-weight="bold">
                {away_team}
            </text>
            
            <!-- Current Score -->
            <text x="80" y="80" text-anchor="middle" font-family="system-ui" font-size="18" font-weight="bold">
                {home_score:.0f}
            </text>
            <text x="{svg_width-80}" y="80" text-anchor="middle" font-family="system-ui" font-size="18" font-weight="bold">
                {away_score:.0f}
            </text>
            
            <!-- VS divider -->
            <text x="{svg_width/2}" y="80" text-anchor="middle" font-family="system-ui" font-size="16">
                VS
            </text>
            
            <!-- Confidence Intervals with Modern Styling and Tooltips-->
            <g class="home-interval-group">
                <rect x="60" y="{svg_height - (home_upper * y_scale)}" width="40" height="{(home_upper - home_lower) * y_scale}" 
                      fill="{quarter_color}" fill-opacity="0.25" rx="5" ry="5" class="interval home-interval">
                    <title>{home_tooltip}</title>
                </rect>
                
                <!-- Interactive elements -->
                <rect x="50" y="{svg_height - (home_upper * y_scale) - 10}" width="60" height="{(home_upper - home_lower) * y_scale + 20}" 
                      fill="transparent" class="interval-hover home-hover" style="cursor:pointer" />
            </g>
            
            <g class="away-interval-group">
                <rect x="{svg_width-100}" y="{svg_height - (away_upper * y_scale)}" width="40" height="{(away_upper - away_lower) * y_scale}" 
                      fill="{quarter_color}" fill-opacity="0.25" rx="5" ry="5" class="interval away-interval">
                    <title>{away_tooltip}</title>
                </rect>
                
                <!-- Interactive elements -->
                <rect x="{svg_width-110}" y="{svg_height - (away_upper * y_scale) - 10}" width="60" height="{(away_upper - away_lower) * y_scale + 20}" 
                      fill="transparent" class="interval-hover away-hover" style="cursor:pointer" />
            </g>
            
            <!-- Predicted Final Scores -->
            <line x1="40" x2="120" y1="{svg_height - (predicted_home * y_scale)}" y2="{svg_height - (predicted_home * y_scale)}" 
                  stroke="{quarter_color}" stroke-width="2.5" stroke-dasharray="2" />
                  
            <line x1="{svg_width-120}" x2="{svg_width-40}" y1="{svg_height - (predicted_away * y_scale)}" y2="{svg_height - (predicted_away * y_scale)}" 
                  stroke="{quarter_color}" stroke-width="2.5" stroke-dasharray="2" />
            
            <!-- Prediction Labels -->
            <text x="80" y="{svg_height - (predicted_home * y_scale) - 8}" text-anchor="middle" font-family="system-ui" font-size="15" font-weight="bold">
                {predicted_home:.1f}
            </text>
            
            <text x="{svg_width-80}" y="{svg_height - (predicted_away * y_scale) - 8}" text-anchor="middle" font-family="system-ui" font-size="15" font-weight="bold">
                {predicted_away:.1f}
            </text>
            
            <!-- Divider line -->
            <line x1="20" y1="{svg_height-45}" x2="{svg_width-20}" y2="{svg_height-45}" stroke="#dee2e6" stroke-width="1" />
            
            <!-- Win Probability Section -->
            <text x="20" y="{svg_height-25}" text-anchor="start" font-family="system-ui" font-size="12">
                {home_team} WIN PROBABILITY: 
            </text>
            
            <!-- Modern Progress Bar with Tooltip -->
            <g class="win-probability-group">
                <rect x="20" y="{svg_height-15}" width="{svg_width-40}" height="8" fill="#e9ecef" rx="4" ry="4" />
                <rect x="20" y="{svg_height-15}" width="{(svg_width-40) * win_prob}" height="8" fill="{win_color}" rx="4" ry="4">
                    <title>{win_tooltip}</title>
                </rect>
                
                <!-- Interactive overlay -->
                <rect x="20" y="{svg_height-25}" width="{svg_width-40}" height="20" fill="transparent" style="cursor:pointer" />
            </g>
            
            <!-- Win Percentage -->
            <text x="{svg_width-20}" y="{svg_height-25}" text-anchor="end" font-family="system-ui" font-size="14" font-weight="bold" fill="{win_color}">
                {win_prob*100:.1f}% {home_win_text}
            </text>
            
            <!-- Add interactive JS capabilities -->
            <script type="text/javascript"><![CDATA[
            (function() {{
                // Add hover effects for intervals
                const intervalHovers = document.querySelectorAll('.interval-hover');
                intervalHovers.forEach(hover => {{
                    hover.addEventListener('mouseover', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.5');
                    }});
                    hover.addEventListener('mouseout', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.25');
                    }});
                    
                    // Mobile touch events
                    hover.addEventListener('touchstart', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.5');
                    }});
                    hover.addEventListener('touchend', function() {{
                        const interval = this.previousElementSibling;
                        interval.setAttribute('fill-opacity', '0.25');
                    }});
                }});
            }})();
            ]]></script>
        </svg>
        """
        
        return svg
    
    def _get_win_probability_color(self, win_prob: float) -> str:
        """Get the appropriate color for a win probability value."""
        if win_prob > 0.65:
            return self.color_scheme['win_positive']  # Strong favorite
        elif win_prob > 0.45:
            return self.color_scheme['win_neutral']   # Even match
        else:
            return self.color_scheme['win_negative']  # Underdog
    
    def _get_win_probability_text(self, win_prob: float) -> str:
        """Get descriptive text for win probability status."""
        if win_prob > 0.65:
            return "FAVORED"
        elif win_prob > 0.45:
            return "EVEN"
        else:
            return "UNDERDOG"


class PWAConfidenceDisplay:
    """
    Class for creating Progressive Web App (PWA) visualizations and components
    for displaying prediction confidence in responsive, mobile-friendly formats.
    
    This class builds on the InteractiveConfidenceDisplay to create complete
    PWA-ready components with responsive design and touch interactions.
    """
    
    def __init__(self):
        """Initialize the PWA display component generator."""
        self.confidence_display = InteractiveConfidenceDisplay()
    
    def create_game_card(self, game_data: Dict[str, Any]) -> str:
        """
        Create a complete game card component with predictions and confidence.
        
        Args:
            game_data: Dict with game information and predictions
            
        Returns:
            str: HTML component for the game card
        """
        # Extract basic game info
        game_id = game_data.get('game_id', '0')
        home_team = game_data.get('home_team', 'Home')
        away_team = game_data.get('away_team', 'Away')
        current_quarter = game_data.get('current_quarter', 0)
        game_status = game_data.get('game_status', 'SCHEDULED')
        
        # Generate the SVG visualization
        svg_visualization = self.confidence_display.create_display(game_data, 'pwa')
        
        # Create the complete card with responsive design
        card_html = f"""
        <div class="game-prediction-card" data-game-id="{game_id}" data-quarter="{current_quarter}">
            <style>
                .game-prediction-card {{
                    font-family: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue";
                    border-radius: 12px;
                    overflow: hidden;
                    box-shadow: 0 2px 8px rgba(0,0,0,0.1);
                    margin-bottom: 16px;
                    background-color: #fff;
                    width: 100%;
                    max-width: 600px;
                }}
                
                .game-header {{
                    display: flex;
                    justify-content: space-between;
                    padding: 12px 16px;
                    border-bottom: 1px solid #dee2e6;
                }}
                
                .game-teams {{
                    font-weight: bold;
                    font-size: 1.1rem;
                }}
                
                .game-status {{
                    font-size: 0.9rem;
                    padding: 4px 8px;
                    border-radius: 4px;
                    font-weight: 500;
                    background-color: #6c757d;
                    color: white;
                }}
                
                .status-live {{
                    background-color: #dc3545;
                }}
                
                .status-final {{
                    background-color: #198754;
                }}
                
                .game-visualization {{
                    padding: 8px;
                }}
                
                /* Responsive adjustments */
                @media (max-width: 480px) {{
                    .game-teams {{
                        font-size: 1rem;
                    }}
                    
                    .game-status {{
                        font-size: 0.8rem;
                    }}
                }}
            </style>
            
            <div class="game-header">
                <div class="game-teams">{home_team} vs {away_team}</div>
                <div class="game-status {self._get_status_class(game_status)}">
                    {self._format_game_status(game_status, current_quarter)}
                </div>
            </div>
            
            <div class="game-visualization">
                {svg_visualization}
            </div>
        </div>
        """
        
        return card_html
    
    def create_game_dashboard(self, games_data: List[Dict[str, Any]]) -> str:
        """
        Create a complete dashboard of game predictions for the PWA.
        
        Args:
            games_data: List of game data dictionaries
            
        Returns:
            str: Complete HTML for the dashboard
        """
        # Start the dashboard HTML
        dashboard_html = """
        <div class="prediction-dashboard">
            <style>
                .prediction-dashboard {
                    font-family: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue";
                    max-width: 1200px;
                    margin: 0 auto;
                    padding: 16px;
                }
                
                .dashboard-header {
                    text-align: center;
                    margin-bottom: 24px;
                }
                
                .dashboard-title {
                    font-size: 1.5rem;
                    font-weight: bold;
                    margin-bottom: 8px;
                }
                
                .dashboard-subtitle {
                    font-size: 1rem;
                    color: #6c757d;
                }
                
                .games-grid {
                    display: grid;
                    grid-template-columns: repeat(auto-fill, minmax(300px, 1fr));
                    gap: 16px;
                }
                
                /* Responsive adjustments */
                @media (max-width: 768px) {
                    .games-grid {
                        grid-template-columns: 1fr;
                    }
                }
            </style>
            
            <div class="dashboard-header">
                <div class="dashboard-title">NBA Score Predictions</div>
                <div class="dashboard-subtitle">Updated: """ + datetime.now().strftime("%Y-%m-%d %H:%M:%S") + """</div>
            </div>
            
            <div class="games-grid">
        """
        
        # Add game cards to dashboard
        for game_data in games_data:
            game_card = self.create_game_card(game_data)
            dashboard_html += game_card
        
        # Close the dashboard HTML
        dashboard_html += """
            </div>
        </div>
        """
        
        return dashboard_html
    
    def _get_status_class(self, status: str) -> str:
        """Get the CSS class for a game status."""
        status = status.upper()
        if status == 'LIVE':
            return 'status-live'
        elif status == 'FINAL':
            return 'status-final'
        else:
            return ''
    
    def _format_game_status(self, status: str, quarter: int) -> str:
        """Format the game status text."""
        status = status.upper()
        if status == 'LIVE':
            return f"LIVE Q{quarter}"
        elif status == 'FINAL':
            return "FINAL"
        else:
            return status


def create_interactive_confidence_display(prediction_data: Dict[str, Any], 
                                        design_style: str = 'pwa') -> str:
    """
    Create an interactive SVG visualization showing prediction confidence.
    
    This is a wrapper function that uses the InteractiveConfidenceDisplay class.
    
    Args:
        prediction_data: Dict with prediction details
        design_style: 'minimal', 'detailed', or 'pwa' for different visualization styles
        
    Returns:
        str: SVG/HTML markup for the confidence display
    """
    display_generator = InteractiveConfidenceDisplay()
    return display_generator.create_display(prediction_data, design_style)


def create_pwa_game_card(game_data: Dict[str, Any]) -> str:
    """
    Create a complete game card component for a PWA.
    
    This is a wrapper function that uses the PWAConfidenceDisplay class.
    
    Args:
        game_data: Dict with game information and predictions
        
    Returns:
        str: HTML component for the game card
    """
    pwa_display = PWAConfidenceDisplay()
    return pwa_display.create_game_card(game_data)


def generate_sample_prediction_data() -> Dict[str, Any]:
    """Generate sample prediction data for testing."""
    return {
        'game_id': 1001,
        'home_team': 'Lakers',
        'away_team': 'Celtics',
        'current_quarter': 2,
        'game_status': 'LIVE',
        'home_score': 54,
        'away_score': 51,
        'predicted_home_score': 112.5,
        'predicted_away_score': 104.8,
        'home_lower_bound': 105.2,
        'home_upper_bound': 119.8,
        'away_lower_bound': 97.5,
        'away_upper_bound': 112.1,
        'win_probability': 0.68
    }


# Example usage and testing
if __name__ == "__main__":
    # Generate sample prediction data
    sample_data = generate_sample_prediction_data()
    
    # Create display generator - renamed from 'display' to 'display_generator' to avoid naming conflicts
    display_generator = InteractiveConfidenceDisplay()
    
    # Test all styles
    for style in ['minimal', 'detailed', 'pwa']:
        print(f"\nTesting {style} style visualization:")
        svg = display_generator.create_display(sample_data, style)
        print(f"Generated {len(svg)} characters of SVG/HTML")
        
        # In a notebook, display the visualization
        if 'get_ipython' in globals():
            from IPython.display import HTML, display as ipython_display
            ipython_display(HTML(svg))
    
    # Test PWA components
    pwa_display = PWAConfidenceDisplay()
    game_card = pwa_display.create_game_card(sample_data)
    print(f"\nGenerated game card: {len(game_card)} characters")
    
    # Create a dashboard with multiple games
    games = [
        sample_data,
        {**sample_data, 'game_id': 1002, 'home_team': 'Warriors', 'away_team': 'Bucks', 
         'current_quarter': 4, 'win_probability': 0.51},
        {**sample_data, 'game_id': 1003, 'home_team': 'Heat', 'away_team': 'Nets', 
         'current_quarter': 0, 'game_status': 'SCHEDULED', 'win_probability': 0.45}
    ]
    
    dashboard = pwa_display.create_game_dashboard(games)
    print(f"\nGenerated dashboard: {len(dashboard)} characters")
    
    # In a notebook, display the dashboard
    if 'get_ipython' in globals():
        from IPython.display import HTML, display as ipython_display
        ipython_display(HTML(dashboard))

In [ ]:
# Cell 27C: Confidence Calibration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union, Any
from datetime import datetime
import json
import os

class CalibrationSystem:
    """
    A complete system for calibrating confidence intervals for NBA score predictions.
    
    This class handles the validation, calibration, and optimization of confidence
    interval width factors to ensure that interval coverage matches the desired
    target rate (e.g., 90% of actual values falling within predicted intervals).
    
    The system supports iterative calibration, visualization of results, and
    integration with the UncertaintyEstimator from Cell 27.
    """
    
    def __init__(self, 
                 initial_factors: Optional[Dict[int, float]] = None,
                 target_coverage: float = 0.9,
                 dampen_factor: float = 0.5,
                 min_factor: float = 0.5,
                 max_factor: float = 5.0):
        """
        Initialize the calibration system.
        
        Args:
            initial_factors: Starting width factors for different quarters. 
                          If None, uses default values.
            target_coverage: Target confidence interval coverage (default: 0.9 for 90%)
            dampen_factor: Factor to dampen adjustments (0-1, lower = more conservative)
            min_factor: Minimum allowed width factor
            max_factor: Maximum allowed width factor
        """
        # Default width factors if none provided
        if initial_factors is None:
            initial_factors = {
                0: 3.0,  # Pre-game: widest intervals
                1: 2.5,  # 1st quarter 
                2: 2.0,  # 2nd quarter
                3: 1.5,  # 3rd quarter
                4: 1.0   # 4th quarter: narrowest intervals
            }
        
        self.current_factors = initial_factors.copy()
        self.initial_factors = initial_factors.copy()
        self.target_coverage = target_coverage
        self.dampen_factor = dampen_factor
        self.min_factor = min_factor
        self.max_factor = max_factor
        
        # Track calibration history
        self.calibration_history = []
        
        # Track metrics
        self.metrics_history = []
    
    def calibrate_width_factors(self, 
                              validation_df: pd.DataFrame,
                              reset_to_initial: bool = False) -> Dict[int, float]:
        """
        Calibrate confidence interval width factors based on validation results.
        
        Args:
            validation_df: DataFrame with validation results containing:
                          - quarter: Game quarter
                          - in_interval: Boolean indicating if actual fell in prediction interval
            reset_to_initial: Whether to start from initial factors (default: False)
            
        Returns:
            Dict with updated width factors
        """
        # Reset to initial factors if requested
        if reset_to_initial:
            self.current_factors = self.initial_factors.copy()
        
        # Calculate actual coverage by quarter
        coverage_by_quarter = {}
        
        for quarter in range(5):  # 0-4
            quarter_data = validation_df[validation_df['quarter'] == quarter]
            if quarter_data.empty:
                continue
                
            # Calculate actual coverage rate
            if 'in_interval' in quarter_data.columns:
                coverage = quarter_data['in_interval'].mean()
            elif 'home_in_interval' in quarter_data.columns and 'away_in_interval' in quarter_data.columns:
                # If we have separate columns for home and away, calculate combined coverage
                both_in_interval = quarter_data['home_in_interval'] & quarter_data['away_in_interval']
                coverage = both_in_interval.mean()
            else:
                # Default to 0.5 if we can't determine coverage
                coverage = 0.5
                
            coverage_by_quarter[quarter] = coverage
        
        # Adjust factors
        updated_factors = self.current_factors.copy()
        adjustments = {}
        
        for quarter, coverage in coverage_by_quarter.items():
            if coverage < 0.01 or self.target_coverage == 0:
                # Avoid division by zero or very small values
                continue
                
            # Calculate adjustment ratio
            # If coverage is too low, increase factor; if too high, decrease factor
            adjustment_ratio = self.target_coverage / coverage
            
            # Apply adjustment with dampening to prevent overcorrection
            adjustment = (adjustment_ratio - 1.0) * self.dampen_factor
            updated_factors[quarter] = self.current_factors.get(quarter, 1.0) * (1.0 + adjustment)
            
            # Enforce bounds on width factors
            updated_factors[quarter] = max(min(updated_factors[quarter], self.max_factor), self.min_factor)
            
            # Record adjustment
            adjustments[quarter] = {
                'old_factor': self.current_factors.get(quarter, 1.0),
                'new_factor': updated_factors[quarter],
                'coverage': coverage,
                'adjustment_ratio': adjustment_ratio,
                'adjustment': adjustment
            }
        
        # Record calibration event
        self.calibration_history.append({
            'timestamp': datetime.now().isoformat(),
            'old_factors': self.current_factors.copy(),
            'new_factors': updated_factors.copy(),
            'coverage': coverage_by_quarter,
            'adjustments': adjustments,
            'target_coverage': self.target_coverage
        })
        
        # Update current factors
        self.current_factors = updated_factors.copy()
        
        return self.current_factors
    
    def validate_intervals(self, 
                         predictions_df: pd.DataFrame, 
                         actual_results_df: pd.DataFrame) -> pd.DataFrame:
        """
        Validate that confidence intervals contain actual values at the expected rate.
        
        Args:
            predictions_df: DataFrame with predictions and confidence intervals
            actual_results_df: DataFrame with actual final scores
        
        Returns:
            DataFrame with validation metrics
        """
        # Merge predictions with actual results
        merged = pd.merge(
            predictions_df,
            actual_results_df[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner'
        )
        
        if merged.empty:
            print("Warning: No matching games found between predictions and actuals")
            return pd.DataFrame()
        
        # Calculate validation metrics
        results = []
        for quarter in range(5):
            # Select data for this quarter
            quarter_data = merged[merged['current_quarter'] == quarter]
            if quarter_data.empty:
                continue
            
            # Check if actual scores fall within predicted intervals
            home_in_interval = ((quarter_data['home_score'] >= quarter_data['home_lower_bound']) & 
                              (quarter_data['home_score'] <= quarter_data['home_upper_bound']))
            
            away_in_interval = ((quarter_data['away_score'] >= quarter_data['away_lower_bound']) & 
                              (quarter_data['away_score'] <= quarter_data['away_upper_bound']))
            
            # Combined coverage (both home and away within intervals)
            both_in_interval = home_in_interval & away_in_interval
            
            # Calculate coverage rates
            home_coverage = home_in_interval.mean()
            away_coverage = away_in_interval.mean()
            combined_coverage = both_in_interval.mean()
            
            # Calculate interval widths
            home_width = (quarter_data['home_upper_bound'] - quarter_data['home_lower_bound']).mean()
            away_width = (quarter_data['away_upper_bound'] - quarter_data['away_lower_bound']).mean()
            avg_width = (home_width + away_width) / 2
            
            # Calculate errors
            if 'predicted_home_score' in quarter_data.columns and 'predicted_away_score' in quarter_data.columns:
                home_error = (quarter_data['predicted_home_score'] - quarter_data['home_score']).abs().mean()
                away_error = (quarter_data['predicted_away_score'] - quarter_data['away_score']).abs().mean()
                avg_error = (home_error + away_error) / 2
            else:
                home_error = away_error = avg_error = np.nan
            
            # Create result row
            results.append({
                'quarter': quarter,
                'home_coverage': home_coverage,
                'away_coverage': away_coverage,
                'combined_coverage': combined_coverage,
                'target_coverage': self.target_coverage,
                'coverage_error': combined_coverage - self.target_coverage,
                'home_interval_width': home_width,
                'away_interval_width': away_width,
                'avg_interval_width': avg_width,
                'home_error': home_error,
                'away_error': away_error,
                'avg_error': avg_error,
                'width_factor': self.current_factors.get(quarter, np.nan),
                'sample_size': len(quarter_data)
            })
        
        # Create DataFrame from results
        validation_df = pd.DataFrame(results)
        
        # Record metrics
        self.metrics_history.append({
            'timestamp': datetime.now().isoformat(),
            'validation_metrics': validation_df.to_dict(orient='records'),
            'current_factors': self.current_factors.copy()
        })
        
        return validation_df
    
    def prepare_validation_data(self, 
                              predictions_df: pd.DataFrame, 
                              actual_results_df: pd.DataFrame) -> pd.DataFrame:
        """
        Prepare validation data in the format needed for calibration.
        
        Args:
            predictions_df: DataFrame with predictions and intervals
            actual_results_df: DataFrame with actual results
            
        Returns:
            DataFrame with validation data ready for calibration
        """
        # Merge predictions with actuals
        merged = pd.merge(
            predictions_df,
            actual_results_df[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner'
        )
        
        if merged.empty:
            print("Warning: No matching games found for validation")
            return pd.DataFrame()
        
        # Process each row
        validation_rows = []
        
        for _, row in merged.iterrows():
            quarter = int(row.get('current_quarter', 0))
            
            # Check if home score is within interval
            home_in_interval = (row['home_score'] >= row['home_lower_bound'] and 
                              row['home_score'] <= row['home_upper_bound'])
            
            # Check if away score is within interval
            away_in_interval = (row['away_score'] >= row['away_lower_bound'] and 
                              row['away_score'] <= row['away_upper_bound'])
            
            # Add home row
            validation_rows.append({
                'game_id': row['game_id'],
                'quarter': quarter,
                'team_type': 'home',
                'predicted_score': row.get('predicted_home_score', np.nan),
                'actual_score': row['home_score'],
                'lower_bound': row['home_lower_bound'],
                'upper_bound': row['home_upper_bound'],
                'interval_width': row['home_upper_bound'] - row['home_lower_bound'],
                'in_interval': home_in_interval,
                'error': abs(row.get('predicted_home_score', row['home_score']) - row['home_score']),
                'home_in_interval': home_in_interval,
                'away_in_interval': away_in_interval,
                'both_in_interval': home_in_interval and away_in_interval
            })
            
            # Add away row
            validation_rows.append({
                'game_id': row['game_id'],
                'quarter': quarter,
                'team_type': 'away',
                'predicted_score': row.get('predicted_away_score', np.nan),
                'actual_score': row['away_score'],
                'lower_bound': row['away_lower_bound'],
                'upper_bound': row['away_upper_bound'],
                'interval_width': row['away_upper_bound'] - row['away_lower_bound'],
                'in_interval': away_in_interval,
                'error': abs(row.get('predicted_away_score', row['away_score']) - row['away_score']),
                'home_in_interval': home_in_interval,
                'away_in_interval': away_in_interval,
                'both_in_interval': home_in_interval and away_in_interval
            })
        
        return pd.DataFrame(validation_rows)
    
    def run_calibration_loop(self, 
                           predictions_history: List[pd.DataFrame], 
                           actual_results: pd.DataFrame,
                           iterations: int = 3,
                           reset_to_initial: bool = True,
                           apply_to_uncertainty_estimator: Optional[Any] = None) -> Dict[str, Any]:
        """
        Run an iterative calibration process to optimize confidence intervals.
        
        Args:
            predictions_history: List of DataFrames with historical predictions
            actual_results: DataFrame with actual game results
            iterations: Number of calibration iterations
            reset_to_initial: Whether to reset to initial factors before calibration
            apply_to_uncertainty_estimator: Optional UncertaintyEstimator to update
                                          with calibrated factors
            
        Returns:
            Dict with calibration results and metrics
        """
        # Reset to initial factors if requested
        if reset_to_initial:
            self.current_factors = self.initial_factors.copy()
            
        # Track iteration history
        iteration_history = []
        
        # Process iterations
        for iteration in range(iterations):
            print(f"Calibration iteration {iteration+1}/{iterations}")
            
            # Apply current factors to generate intervals
            all_validation_data = []
            
            for i, pred_df in enumerate(predictions_history):
                print(f"  Processing prediction set {i+1}/{len(predictions_history)}")
                
                # Apply current width factors to create intervals
                pred_with_intervals = self._apply_width_factors(pred_df)
                
                # Validate against actual results
                validation_data = self.prepare_validation_data(
                    pred_with_intervals, actual_results)
                
                # Add to collection
                if not validation_data.empty:
                    all_validation_data.append(validation_data)
            
            # Combine validation data
            if not all_validation_data:
                print("  Warning: No validation data available for calibration")
                continue
                
            combined_validation = pd.concat(all_validation_data, ignore_index=True)
            
            # Calculate metrics
            validation_metrics = self.validate_intervals(
                pd.concat([self._apply_width_factors(df) for df in predictions_history]),
                actual_results
            )
            
            # Print current metrics
            print("\nCurrent coverage by quarter:")
            for _, row in validation_metrics.iterrows():
                print(f"  Quarter {int(row['quarter'])}: {row['combined_coverage']:.1%} (target: {self.target_coverage:.1%})")
            
            # Calibrate factors
            new_factors = self.calibrate_width_factors(combined_validation)
            
            # Print updated factors
            print("\nUpdated width factors:")
            for quarter, factor in new_factors.items():
                print(f"  Quarter {quarter}: {factor:.2f}")
            
            # Record iteration
            iteration_history.append({
                'iteration': iteration + 1,
                'metrics': validation_metrics.to_dict(orient='records'),
                'factors': new_factors.copy()
            })
        
        # Apply to uncertainty estimator if provided
        if apply_to_uncertainty_estimator is not None:
            print("\nApplying calibrated factors to UncertaintyEstimator")
            apply_to_uncertainty_estimator.width_factors = self.current_factors.copy()
        
        # Create final results
        results = {
            'initial_factors': self.initial_factors,
            'final_factors': self.current_factors,
            'iteration_history': iteration_history,
            'validation_metrics': validation_metrics.to_dict(orient='records') if 'validation_metrics' in locals() else [],
            'calibration_history': self.calibration_history
        }
        
        # Visualize results
        self.visualize_calibration_results(iteration_history)
        
        return results
    
    def _apply_width_factors(self, predictions_df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply current width factors to generate confidence intervals.
        
        Args:
            predictions_df: DataFrame with game predictions
            
        Returns:
            DataFrame with confidence intervals added
        """
        # Make a copy to avoid modifying the original
        result_df = predictions_df.copy()
        
        # Add interval columns if they don't exist
        for col in ['home_lower_bound', 'home_upper_bound', 
                   'away_lower_bound', 'away_upper_bound']:
            if col not in result_df.columns:
                result_df[col] = np.nan
        
        # Apply width factors to each row
        for idx, row in result_df.iterrows():
            quarter = int(row.get('current_quarter', 0))
            width_factor = self.current_factors.get(quarter, 1.0)
            
            # Get predictions
            home_pred = float(row.get('predicted_home_score', row.get('predicted_home_final', 0)))
            away_pred = float(row.get('predicted_away_score', row.get('predicted_away_final', 0)))
            
            # Estimate uncertainty based on quarter
            base_uncertainty = 10.0 * (4.0 - min(quarter, 4)) / 4.0
            half_width = base_uncertainty * width_factor / 2
            
            # Create intervals
            result_df.loc[idx, 'home_lower_bound'] = home_pred - half_width
            result_df.loc[idx, 'home_upper_bound'] = home_pred + half_width
            result_df.loc[idx, 'away_lower_bound'] = away_pred - half_width
            result_df.loc[idx, 'away_upper_bound'] = away_pred + half_width
        
        return result_df
    
    def visualize_calibration_results(self, 
                                    iteration_history: List[Dict[str, Any]] = None,
                                    show_plot: bool = True) -> plt.Figure:
        """
        Visualize the calibration process and results.
        
        Args:
            iteration_history: List of iteration results (if None, uses self.calibration_history)
            show_plot: Whether to show the plot (default: True)
            
        Returns:
            matplotlib Figure object
        """
        # Use calibration history if no iteration history provided
        if iteration_history is None and self.calibration_history:
            # Convert calibration history to iteration format
            iteration_history = []
            for i, entry in enumerate(self.calibration_history):
                iteration_history.append({
                    'iteration': i + 1,
                    'factors': entry.get('new_factors', {}),
                    'coverage': entry.get('coverage', {})
                })
        
        if not iteration_history:
            print("No calibration history to visualize")
            return plt.figure()
        
        # Create figure
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Colors and markers for quarters
        markers = ['o', 's', '^', 'd', '*']
        colors = ['blue', 'green', 'red', 'purple', 'orange']
        
        # Data to plot
        iterations = [entry.get('iteration', i+1) for i, entry in enumerate(iteration_history)]
        
        # Plot coverages on the first axis
        ax1.set_title('Coverage Rate by Quarter During Calibration')
        ax1.axhline(y=self.target_coverage, color='black', linestyle='--', 
                   label=f'Target ({self.target_coverage:.0%})')
        
        # Track which quarters we've seen
        seen_quarters = set()
        
        # Plot coverage for each quarter
        for i, entry in enumerate(iteration_history):
            # Get coverage data - handle different formats
            if 'coverage' in entry:
                coverage_data = entry['coverage']
            elif 'metrics' in entry:
                # Convert metrics list to quarter coverage dict
                coverage_data = {}
                for metric in entry['metrics']:
                    if 'quarter' in metric and 'combined_coverage' in metric:
                        coverage_data[metric['quarter']] = metric['combined_coverage']
            else:
                continue
            
            # Plot each quarter's coverage
            for quarter, coverage in coverage_data.items():
                quarter = int(quarter)  # Ensure it's an integer
                
                # Add quarter to seen set
                seen_quarters.add(quarter)
                
                # Plot point for this iteration
                color_idx = quarter % len(colors)
                marker_idx = quarter % len(markers)
                
                # For first iteration, add to legend
                if i == 0:
                    ax1.scatter(iterations[i], coverage, 
                              marker=markers[marker_idx], color=colors[color_idx],
                              s=80, label=f'Quarter {quarter}')
                else:
                    ax1.scatter(iterations[i], coverage, 
                              marker=markers[marker_idx], color=colors[color_idx],
                              s=80)
        
        # Connect points for the same quarter
        for quarter in seen_quarters:
            quarter_coverage = []
            iteration_nums = []
            
            for i, entry in enumerate(iteration_history):
                # Extract coverage based on format
                if 'coverage' in entry and quarter in entry['coverage']:
                    quarter_coverage.append(entry['coverage'][quarter])
                    iteration_nums.append(iterations[i])
                elif 'metrics' in entry:
                    for metric in entry['metrics']:
                        if metric.get('quarter') == quarter and 'combined_coverage' in metric:
                            quarter_coverage.append(metric['combined_coverage'])
                            iteration_nums.append(iterations[i])
            
            # Plot connecting line if we have multiple points
            if len(quarter_coverage) > 1:
                color_idx = quarter % len(colors)
                ax1.plot(iteration_nums, quarter_coverage, 
                       color=colors[color_idx], alpha=0.7)
        
        # Configure first axis
        ax1.set_ylabel('Coverage Rate')
        ax1.set_ylim(0, 1.1)
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc='best')
        
        # Plot width factors on the second axis
        ax2.set_title('Width Factors by Quarter During Calibration')
        
        # Plot factors for each quarter
        for quarter in sorted(seen_quarters):
            quarter_factors = []
            iteration_nums = []
            
            for i, entry in enumerate(iteration_history):
                if 'factors' in entry and quarter in entry['factors']:
                    quarter_factors.append(entry['factors'][quarter])
                    iteration_nums.append(iterations[i])
            
            # Plot if we have data
            if quarter_factors:
                color_idx = quarter % len(colors)
                marker_idx = quarter % len(markers)
                
                ax2.plot(iteration_nums, quarter_factors, 
                       marker=markers[marker_idx], color=colors[color_idx],
                       label=f'Quarter {quarter}', linewidth=2)
        
        # Configure second axis
        ax2.set_xlabel('Calibration Iteration')
        ax2.set_ylabel('Width Factor')
        ax2.grid(True, alpha=0.3)
        ax2.legend(loc='best')
        
        # Final layout
        plt.tight_layout()
        
        if show_plot:
            plt.show()
        
        return fig
    
    def save_calibration_state(self, filepath: str) -> bool:
        """
        Save the current calibration state to a JSON file.
        
        Args:
            filepath: Path to save the calibration state
            
        Returns:
            bool: True if successful, False otherwise
        """
        try:
            state = {
                'current_factors': self.current_factors,
                'initial_factors': self.initial_factors,
                'target_coverage': self.target_coverage,
                'dampen_factor': self.dampen_factor,
                'min_factor': self.min_factor,
                'max_factor': self.max_factor,
                'calibration_history': self.calibration_history,
                'saved_at': datetime.now().isoformat()
            }
            
            with open(filepath, 'w') as f:
                json.dump(state, f, indent=2)
                
            print(f"Calibration state saved to {filepath}")
            return True
            
        except Exception as e:
            print(f"Error saving calibration state: {e}")
            return False
    
    def load_calibration_state(self, filepath: str) -> bool:
        """
        Load calibration state from a JSON file.
        
        Args:
            filepath: Path to the calibration state file
            
        Returns:
            bool: True if successful, False otherwise
        """
        try:
            if not os.path.exists(filepath):
                print(f"Calibration state file not found: {filepath}")
                return False
                
            with open(filepath, 'r') as f:
                state = json.load(f)
            
            # Load core parameters
            self.current_factors = state.get('current_factors', self.current_factors)
            self.initial_factors = state.get('initial_factors', self.initial_factors)
            self.target_coverage = state.get('target_coverage', self.target_coverage)
            self.dampen_factor = state.get('dampen_factor', self.dampen_factor)
            self.min_factor = state.get('min_factor', self.min_factor)
            self.max_factor = state.get('max_factor', self.max_factor)
            self.calibration_history = state.get('calibration_history', [])
            
            print(f"Calibration state loaded from {filepath}")
            print(f"Current factors: {self.current_factors}")
            return True
            
        except Exception as e:
            print(f"Error loading calibration state: {e}")
            return False


def calibrate_confidence_intervals(validation_df: pd.DataFrame, 
                                 current_factors: Dict[int, float],
                                 target_coverage: float = 0.9,
                                 dampen_factor: float = 0.5) -> Dict[int, float]:
    """
    Calibrate confidence interval width factors based on validation results.
    
    Standalone function version for backward compatibility.
    
    Args:
        validation_df: DataFrame with validation results
        current_factors: Dict mapping quarters to current width factors
        target_coverage: Target coverage rate (default: 0.9 for 90% intervals)
        dampen_factor: Factor to dampen adjustments (0-1, lower = more conservative)
        
    Returns:
        Dict with updated width factors
    """
    # Create calibration system and apply calibration
    calibration_system = CalibrationSystem(
        initial_factors=current_factors,
        target_coverage=target_coverage,
        dampen_factor=dampen_factor
    )
    
    return calibration_system.calibrate_width_factors(validation_df)


def validate_confidence_intervals(predictions_df: pd.DataFrame, 
                               actual_results_df: pd.DataFrame, 
                               confidence_level: float = 0.9) -> pd.DataFrame:
    """
    Validate that confidence intervals contain actual values at the expected rate.
    
    Standalone function version for backward compatibility.
    
    Args:
        predictions_df: DataFrame with predictions and confidence intervals
        actual_results_df: DataFrame with actual final scores
        confidence_level: Expected coverage rate (default: 0.9 for 90% confidence)
    
    Returns:
        DataFrame with validation metrics
    """
    # Create calibration system and run validation
    calibration_system = CalibrationSystem(target_coverage=confidence_level)
    
    return calibration_system.validate_intervals(predictions_df, actual_results_df)


def run_confidence_calibration_loop(predictions_history: List[pd.DataFrame], 
                                  actual_results: pd.DataFrame,
                                  initial_factors: Optional[Dict[int, float]] = None,
                                  iterations: int = 3) -> Tuple[Dict[int, float], List[Dict[str, Any]]]:
    """
    Run an iterative calibration process to optimize confidence intervals.
    
    Standalone function version for backward compatibility.
    
    Args:
        predictions_history: List of DataFrames with historical predictions
        actual_results: DataFrame with actual game results
        initial_factors: Starting width factors (default: {0:3.0, 1:2.5, 2:2.0, 3:1.5, 4:1.0})
        iterations: Number of calibration iterations
        
    Returns:
        tuple: (final_factors, calibration_history)
    """
    # Create calibration system
    calibration_system = CalibrationSystem(initial_factors=initial_factors)
    
    # Run calibration loop
    results = calibration_system.run_calibration_loop(
        predictions_history, actual_results, iterations)
    
    # Extract expected return values for backward compatibility
    return calibration_system.current_factors, results['iteration_history']


# Example usage and demo
def run_calibration_demo():
    """Run a demonstration of the calibration system with sample data."""
    print("Running calibration system demonstration...")
    
    # Create sample predictions
    sample_predictions = []
    for _ in range(3):  # Three sample prediction batches
        preds = pd.DataFrame([
            {'game_id': 1001, 'current_quarter': 1, 'predicted_home_score': 105, 'predicted_away_score': 100},
            {'game_id': 1002, 'current_quarter': 2, 'predicted_home_score': 110, 'predicted_away_score': 102},
            {'game_id': 1003, 'current_quarter': 3, 'predicted_home_score': 95, 'predicted_away_score': 90},
            {'game_id': 1004, 'current_quarter': 4, 'predicted_home_score': 118, 'predicted_away_score': 112}
        ])
        sample_predictions.append(preds)

    # Sample actual results
    sample_actuals = pd.DataFrame([
        {'game_id': 1001, 'home_score': 108, 'away_score': 97},
        {'game_id': 1002, 'home_score': 112, 'away_score': 105},
        {'game_id': 1003, 'home_score': 92, 'away_score': 96},
        {'game_id': 1004, 'home_score': 120, 'away_score': 110}
    ])
    
    # Create calibration system
    calibration_system = CalibrationSystem()
    
    # Run calibration loop
    results = calibration_system.run_calibration_loop(
        sample_predictions, sample_actuals, iterations=2)
    
    # Print final results
    print("\nFinal calibrated factors:")
    for quarter, factor in calibration_system.current_factors.items():
        print(f"Quarter {quarter}: {factor:.2f}")
    
    return results


# Run demonstration if executed directly
if __name__ == "__main__":
    demo_results = run_calibration_demo()

In [ ]:
# Cell 27D: Complete Context-Aware Confidence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import pickle
import os
import json
from sklearn.metrics import mean_absolute_error
from typing import Dict, Tuple, List, Optional, Union, Any


class ContextClassifier:
    """
    Classifies NBA game contexts to help determine appropriate confidence intervals.
    
    This class analyzes game state data to identify special situations like close games,
    blowouts, high-momentum scenarios, and high-leverage moments where prediction
    uncertainty may differ from standard patterns.
    """
    
    def __init__(self, thresholds: Optional[Dict[str, Any]] = None):
        """
        Initialize the context classifier with configurable thresholds.
        
        Args:
            thresholds: Dictionary of threshold values for different classifications.
                      If None, uses default thresholds.
        """
        # Default thresholds if none provided
        self.thresholds = thresholds or {
            'close_game': 5.0,       # Score differential at or below this is a close game
            'blowout': 15.0,         # Score differential at or above this is a blowout
            'high_momentum': 0.5,    # Absolute momentum at or above this is high momentum
            'final_minutes': 5.0,    # Minutes remaining at or below this is "final minutes"
            'win_prob_threshold': 0.45,  # Win prob between this and (1-this) is a toss-up
        }
    
    def is_close_game(self, score_diff: float) -> bool:
        """
        Determine if a game is considered "close" based on score differential.
        
        Args:
            score_diff: Absolute score difference between teams
            
        Returns:
            bool: True if the game is close
        """
        return score_diff <= self.thresholds['close_game']
    
    def is_blowout(self, score_diff: float) -> bool:
        """
        Determine if a game is considered a "blowout" based on score differential.
        
        Args:
            score_diff: Absolute score difference between teams
            
        Returns:
            bool: True if the game is a blowout
        """
        return score_diff >= self.thresholds['blowout']
    
    def is_high_momentum(self, momentum: float) -> bool:
        """
        Determine if a team has high momentum.
        
        Args:
            momentum: Team momentum value (-1 to 1)
            
        Returns:
            bool: True if momentum is high (in either direction)
        """
        return abs(momentum) >= self.thresholds['high_momentum']
    
    def is_final_minutes(self, quarter: int, time_remaining: Optional[float]) -> bool:
        """
        Determine if the game is in its final minutes.
        
        Args:
            quarter: Current quarter (0-4, where 0 is pre-game)
            time_remaining: Minutes remaining in the quarter
            
        Returns:
            bool: True if in final minutes of 4th quarter
        """
        return (quarter == 4 and 
                time_remaining is not None and 
                time_remaining <= self.thresholds['final_minutes'])
    
    def is_high_leverage_situation(self, 
                                 quarter: int, 
                                 score_diff: float, 
                                 time_remaining: Optional[float] = None,
                                 win_probability: Optional[float] = None) -> bool:
        """
        Determine if this is a high-leverage situation requiring specialized handling.
        
        High-leverage situations are critical moments that can significantly impact 
        the outcome of the game, such as close games in the final minutes.
        
        Args:
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference between teams
            time_remaining: Minutes remaining in the quarter
            win_probability: Current win probability (0-1)
            
        Returns:
            bool: True if this is a high-leverage situation
        """
        # Final minutes of 4th quarter in close game
        if self.is_final_minutes(quarter, time_remaining) and self.is_close_game(score_diff):
            return True
        
        # Close win probability regardless of quarter
        if win_probability is not None:
            threshold = self.thresholds['win_prob_threshold']
            is_toss_up = (win_probability >= threshold and 
                          win_probability <= (1 - threshold))
                          
            if is_toss_up and self.is_close_game(score_diff):
                return True
        
        return False
    
    def get_game_context(self, 
                        quarter: int,
                        score_diff: float,
                        momentum: float = 0.0,
                        time_remaining: Optional[float] = None,
                        win_probability: Optional[float] = None) -> Dict[str, bool]:
        """
        Get a complete analysis of the game context.
        
        Args:
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference
            momentum: Team momentum (-1 to 1)
            time_remaining: Minutes remaining in quarter
            win_probability: Win probability (0-1)
            
        Returns:
            Dict with context classifications
        """
        return {
            'close_game': self.is_close_game(score_diff),
            'blowout': self.is_blowout(score_diff),
            'high_momentum': self.is_high_momentum(momentum),
            'final_minutes': self.is_final_minutes(quarter, time_remaining),
            'high_leverage': self.is_high_leverage_situation(
                quarter, score_diff, time_remaining, win_probability)
        }


class ContextAwareConfidenceAdjuster:
    """
    Adjusts confidence intervals based on game context.
    
    This class implements context-specific adjustments to base confidence
    intervals to account for different game situations that may affect
    prediction certainty.
    """
    
    def __init__(self, adjustment_factors: Optional[Dict[str, float]] = None):
        """
        Initialize the context adjuster with configurable adjustment factors.
        
        Args:
            adjustment_factors: Dictionary mapping context types to adjustment
                              multipliers. If None, uses default values.
        """
        # Default adjustment factors if none provided
        self.adjustment_factors = adjustment_factors or {
            'close_game': 1.1,       # Increase width for close games (more uncertainty)
            'blowout': 0.9,          # Decrease width for blowouts (more certainty)
            'high_momentum': 1.15,   # Increase width for high momentum (more volatility)
            'final_minutes': 0.85,   # Decrease width in final minutes (more certainty)
            'overtime': 1.2,         # Wider intervals in overtime (more uncertainty)
            'high_leverage': 0.8,    # Special handling for high-leverage situations
        }
        
        # Context classifier for identifying game situations
        self.context_classifier = ContextClassifier()
    
    def calculate_adjustment(self, 
                          quarter: int, 
                          score_diff: float, 
                          momentum: float = 0.0,
                          time_remaining: Optional[float] = None,
                          win_probability: Optional[float] = None) -> float:
        """
        Calculate the context-specific adjustment factor for confidence intervals.
        
        Args:
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference
            momentum: Momentum value (-1 to 1)
            time_remaining: Minutes remaining in quarter
            win_probability: Current win probability
            
        Returns:
            float: Combined adjustment factor
        """
        # Start with base factor of 1.0
        adjustment = 1.0
        
        # Get game context
        context = self.context_classifier.get_game_context(
            quarter, score_diff, momentum, time_remaining, win_probability)
        
        # Apply adjustments based on context
        if context['close_game']:
            adjustment *= self.adjustment_factors['close_game']
        elif context['blowout']:
            adjustment *= self.adjustment_factors['blowout']
        else:
            # Linear interpolation between close and blowout
            close_threshold = self.context_classifier.thresholds['close_game']
            blowout_threshold = self.context_classifier.thresholds['blowout']
            
            # Avoid division by zero
            if blowout_threshold > close_threshold:
                blend = (score_diff - close_threshold) / (blowout_threshold - close_threshold)
                blend = max(0.0, min(1.0, blend))  # Clamp to [0, 1]
                
                adjustment *= ((1 - blend) * self.adjustment_factors['close_game'] + 
                              blend * self.adjustment_factors['blowout'])
        
        # Apply momentum adjustment
        if context['high_momentum']:
            adjustment *= self.adjustment_factors['high_momentum']
        
        # Apply final minutes adjustment
        if context['final_minutes']:
            adjustment *= self.adjustment_factors['final_minutes']
        
        # Apply high leverage adjustment
        if context['high_leverage']:
            # High leverage situations get specialized treatment, but never below 0.7
            adjustment = max(0.7, adjustment * self.adjustment_factors['high_leverage'])
        
        # Ensure adjustment is within reasonable bounds
        adjustment = max(0.5, min(2.0, adjustment))
        
        return adjustment
    
    def set_adjustment_factor(self, context_type: str, value: float) -> bool:
        """
        Update a specific adjustment factor.
        
        Args:
            context_type: Type of context to adjust (e.g., 'close_game')
            value: New adjustment factor value
            
        Returns:
            bool: True if update was successful
        """
        if context_type in self.adjustment_factors:
            self.adjustment_factors[context_type] = value
            return True
        return False
    
    def set_threshold(self, threshold_type: str, value: float) -> bool:
        """
        Update a threshold in the context classifier.
        
        Args:
            threshold_type: Type of threshold to update (e.g., 'close_game')
            value: New threshold value
            
        Returns:
            bool: True if update was successful
        """
        if threshold_type in self.context_classifier.thresholds:
            self.context_classifier.thresholds[threshold_type] = value
            return True
        return False


class ContextAwareConfidenceCalibration:
    """
    Advanced confidence calibration system that adapts to game context
    and provides specialized handling for high-leverage situations.
    
    This class combines base width factors with context-aware adjustments
    to create confidence intervals that account for different game situations.
    It also provides calibration functionality to improve interval accuracy
    over time.
    """
    
    def __init__(self, 
                 base_width_factors: Optional[Dict[int, float]] = None,
                 context_adjuster: Optional[ContextAwareConfidenceAdjuster] = None,
                 cache_path: str = "confidence_calibration_cache.pkl"):
        """
        Initialize the calibration system.
        
        Args:
            base_width_factors: Initial width factors by quarter (0-4).
                              If None, uses default values.
            context_adjuster: Custom context adjuster. If None, creates default.
            cache_path: Path to save/load calibration cache
        """
        # Default width factors by quarter if none provided
        self.width_factors = base_width_factors or {
            0: 3.0,  # Pre-game: wide intervals
            1: 2.5,  # 1st quarter
            2: 2.0,  # 2nd quarter
            3: 1.5,  # 3rd quarter
            4: 1.0,  # 4th quarter
        }
        
        # Context-aware adjustment system
        self.context_adjuster = context_adjuster or ContextAwareConfidenceAdjuster()
        
        # Cache for calibration history
        self.cache_path = cache_path
        self.calibration_history = []
        
        # Target coverage rate (default 90%)
        self.target_coverage = 0.9
        
        # Load cached calibration if available
        self._load_calibration_cache()
    
    def _load_calibration_cache(self) -> bool:
        """
        Load calibration cache from disk if available.
        
        Returns:
            bool: True if cache was loaded successfully
        """
        if os.path.exists(self.cache_path):
            try:
                with open(self.cache_path, 'rb') as f:
                    cached_data = pickle.load(f)
                    self.width_factors = cached_data.get('width_factors', self.width_factors)
                    
                    # Load context adjustments if available
                    if 'context_adjustments' in cached_data:
                        for key, value in cached_data['context_adjustments'].items():
                            self.context_adjuster.set_adjustment_factor(key, value)
                    
                    self.calibration_history = cached_data.get('calibration_history', [])
                    print(f"Loaded calibration cache from {self.cache_path}")
                return True
            except Exception as e:
                print(f"Error loading calibration cache: {e}")
        return False
    
    def _save_calibration_cache(self) -> bool:
        """
        Save current calibration to disk.
        
        Returns:
            bool: True if cache was saved successfully
        """
        try:
            with open(self.cache_path, 'wb') as f:
                pickle.dump({
                    'width_factors': self.width_factors,
                    'context_adjustments': self.context_adjuster.adjustment_factors,
                    'calibration_history': self.calibration_history,
                    'updated_at': datetime.now().isoformat()
                }, f)
            print(f"Saved calibration cache to {self.cache_path}")
            return True
        except Exception as e:
            print(f"Error saving calibration cache: {e}")
            return False
    
    def export_calibration_json(self, filepath: str) -> bool:
        """
        Export calibration data to a human-readable JSON file.
        
        Args:
            filepath: Path to save the JSON export
            
        Returns:
            bool: True if export was successful
        """
        try:
            export_data = {
                'width_factors': self.width_factors,
                'context_adjustments': self.context_adjuster.adjustment_factors,
                'context_thresholds': self.context_adjuster.context_classifier.thresholds,
                'target_coverage': self.target_coverage,
                'calibration_summary': {
                    'history_entries': len(self.calibration_history),
                    'last_updated': datetime.now().isoformat()
                }
            }
            
            with open(filepath, 'w') as f:
                json.dump(export_data, f, indent=2)
                
            print(f"Exported calibration data to {filepath}")
            return True
        except Exception as e:
            print(f"Error exporting calibration data: {e}")
            return False
    
    def calculate_prediction_interval(self, 
                                    predicted_value: float, 
                                    quarter: int, 
                                    score_diff: float, 
                                    momentum: float = 0.0,
                                    time_remaining: Optional[float] = None, 
                                    win_probability: Optional[float] = None) -> Tuple[float, float, float]:
        """
        Calculate confidence interval for a prediction with context awareness.
        
        Args:
            predicted_value: The predicted score
            quarter: Current quarter (0-4)
            score_diff: Absolute score difference
            momentum: Momentum value (-1 to 1)
            time_remaining: Minutes remaining in quarter
            win_probability: Win probability estimate
            
        Returns:
            Tuple[float, float, float]: (lower_bound, upper_bound, interval_width)
        """
        # Get base width factor for this quarter
        base_factor = self.width_factors.get(quarter, 2.0)
        
        # Calculate base uncertainty (decreases as game progresses)
        base_uncertainty = 10.0 * (4.0 - min(quarter, 4)) / 4.0
        
        # Get context adjustment
        context_adjustment = self.context_adjuster.calculate_adjustment(
            quarter, score_diff, momentum, time_remaining, win_probability)
        
        # Calculate interval width
        interval_width = base_uncertainty * base_factor * context_adjustment
        
        # Ensure interval width is reasonable
        min_width = 2.0  # Minimum width to account for inherent randomness
        max_width = 30.0  # Maximum width to prevent extreme intervals
        interval_width = min(max(interval_width, min_width), max_width)
        
        # Calculate bounds
        lower_bound = predicted_value - interval_width / 2
        upper_bound = predicted_value + interval_width / 2
        
        return lower_bound, upper_bound, interval_width
    
    def apply_prediction_intervals(self, predictions: pd.DataFrame) -> pd.DataFrame:
        """
        Apply prediction intervals to a DataFrame of predictions.
        
        Args:
            predictions: DataFrame with game predictions
            
        Returns:
            DataFrame with confidence intervals added
        """
        result = predictions.copy()
        
        # Add interval columns if they don't exist
        for col in ['home_lower_bound', 'home_upper_bound', 'away_lower_bound', 'away_upper_bound',
                  'home_interval_width', 'away_interval_width']:
            if col not in result.columns:
                result[col] = np.nan
        
        # Calculate intervals for each prediction
        for idx, row in result.iterrows():
            # Extract needed data with fallbacks for missing columns
            quarter = int(row.get('current_quarter', 0))
            
            # Get home and away predictions with fallbacks
            home_pred = float(row.get('predicted_home_score', 
                                   row.get('predicted_home_final', 0)))
            away_pred = float(row.get('predicted_away_score', 
                                   row.get('predicted_away_final', 0)))
            
            # Get game context information
            score_diff = abs(row.get('score_differential', 
                                  abs(home_pred - away_pred)))
            
            momentum = float(row.get('momentum_shift', 
                                  row.get('cumulative_momentum', 0)))
            
            time_remaining = row.get('time_remaining')
            win_prob = row.get('win_probability')
            
            # Calculate intervals for home team
            home_lower, home_upper, home_width = self.calculate_prediction_interval(
                home_pred, quarter, score_diff, momentum, time_remaining, win_prob)
            
            # Calculate intervals for away team
            away_lower, away_upper, away_width = self.calculate_prediction_interval(
                away_pred, quarter, score_diff, momentum, time_remaining, win_prob)
            
            # Update DataFrame
            result.loc[idx, 'home_lower_bound'] = home_lower
            result.loc[idx, 'home_upper_bound'] = home_upper
            result.loc[idx, 'home_interval_width'] = home_width
            
            result.loc[idx, 'away_lower_bound'] = away_lower
            result.loc[idx, 'away_upper_bound'] = away_upper
            result.loc[idx, 'away_interval_width'] = away_width
            
            # Add context information for reference
            result.loc[idx, 'context_adjustment'] = self.context_adjuster.calculate_adjustment(
                quarter, score_diff, momentum, time_remaining, win_prob)
            
            high_leverage = self.context_adjuster.context_classifier.is_high_leverage_situation(
                quarter, score_diff, time_remaining, win_prob)
            result.loc[idx, 'high_leverage'] = high_leverage
        
        return result
    
    def calculate_coverage(self, predictions: pd.DataFrame, actuals: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate coverage rates by comparing predictions with actual values.
        
        Args:
            predictions: DataFrame with predictions and intervals
            actuals: DataFrame with actual game results
        
        Returns:
            DataFrame with coverage analysis
        """
        # Merge predictions with actuals
        merged = pd.merge(
            predictions,
            actuals[['game_id', 'home_score', 'away_score']],
            on='game_id',
            how='inner',
            suffixes=('_pred', '')
        )
        
        if merged.empty:
            print("Warning: No matching games found between predictions and actuals")
            return pd.DataFrame()
        
        # Ensure quarter column exists (copy from current_quarter if needed)
        if 'quarter' not in merged.columns and 'current_quarter' in merged.columns:
            merged['quarter'] = merged['current_quarter']
        
        # Check if actual values fall within predicted intervals
        merged['home_in_interval'] = ((merged['home_score'] >= merged['home_lower_bound']) & 
                                    (merged['home_score'] <= merged['home_upper_bound']))
        merged['away_in_interval'] = ((merged['away_score'] >= merged['away_lower_bound']) & 
                                    (merged['away_score'] <= merged['away_upper_bound']))
        merged['in_interval'] = merged['home_in_interval'] & merged['away_in_interval']
        
        # Add error metrics
        if 'predicted_home_score' in merged.columns and 'predicted_away_score' in merged.columns:
            merged['home_error'] = (merged['predicted_home_score'] - merged['home_score']).abs()
            merged['away_error'] = (merged['predicted_away_score'] - merged['away_score']).abs()
            merged['total_error'] = merged['home_error'] + merged['away_error']
        
        return merged
    
    def update_width_factors(self, 
                           coverage_by_quarter: Dict[int, float], 
                           target_coverage: float = 0.9,
                           dampen_factor: float = 0.5) -> Dict[int, float]:
        """
        Update width factors based on observed coverage vs target.
        
        Args:
            coverage_by_quarter: Dict mapping quarters to observed coverage rates
            target_coverage: Target coverage rate (default: 0.9)
            dampen_factor: Factor to dampen adjustments (0-1)
            
        Returns:
            Dict with updated width factors
        """
        updated_factors = self.width_factors.copy()
        
        for quarter, coverage in coverage_by_quarter.items():
            if coverage < 0.01 or target_coverage < 0.01:
                # Skip very small values to avoid division issues
                continue
                
            # Calculate adjustment ratio
            adjustment_ratio = target_coverage / coverage
            
            # Apply adjustment with dampening to prevent overcorrection
            updated_factors[quarter] = updated_factors.get(quarter, 1.0) * (
                1.0 + (adjustment_ratio - 1.0) * dampen_factor
            )
            
            # Cap factors to reasonable bounds
            updated_factors[quarter] = max(0.5, min(5.0, updated_factors[quarter]))
        
        return updated_factors
    
    def run_calibration_iteration(self, 
                                predictions: pd.DataFrame, 
                                actuals: pd.DataFrame, 
                                target_coverage: float = 0.9) -> Dict[str, Any]:
        """
        Run a single calibration iteration to update width factors.
        
        Args:
            predictions: DataFrame with predictions and their metadata
            actuals: DataFrame with actual final scores
            target_coverage: Target coverage rate (default: 0.9 for 90% intervals)
            
        Returns:
            Dict with calibration results
        """
        print(f"Running calibration iteration (target coverage: {target_coverage:.1%})")
        
        # Store target coverage
        self.target_coverage = target_coverage
        
        # Apply current width factors to predictions
        augmented_predictions = self.apply_prediction_intervals(predictions)
        
        # Merge with actuals to calculate coverage
        validation_results = self.calculate_coverage(augmented_predictions, actuals)
        
        # Calculate per-quarter coverage rates
        coverage_by_quarter = {}
        for quarter in range(5):  # 0-4
            quarter_data = validation_results[validation_results['quarter'] == quarter]
            if not quarter_data.empty:
                coverage_by_quarter[quarter] = quarter_data['in_interval'].mean()
                print(f"Quarter {quarter}: Coverage = {coverage_by_quarter[quarter]:.1%} (target: {target_coverage:.1%})")
        
        # Update width factors
        old_factors = self.width_factors.copy()
        self.width_factors = self.update_width_factors(coverage_by_quarter, target_coverage)
        
        # Calculate factor changes
        factor_changes = {q: self.width_factors[q] - old_factors.get(q, 1.0) 
                        for q in self.width_factors}
        
        # Record calibration result
        result = {
            'timestamp': datetime.now().isoformat(),
            'old_factors': old_factors,
            'new_factors': self.width_factors.copy(),
            'coverage_by_quarter': coverage_by_quarter,
            'factor_changes': factor_changes,
            'target_coverage': target_coverage,
            'context_adjustments': self.context_adjuster.adjustment_factors.copy()
        }
        self.calibration_history.append(result)
        
        # Save updated calibration
        self._save_calibration_cache()
        
        return result
    
    def run_full_calibration(self, 
                           prediction_history: List[pd.DataFrame], 
                           actual_results: pd.DataFrame, 
                           iterations: int = 3,
                           target_coverage: float = 0.9) -> Dict[str, Any]:
        """
        Run a full calibration process with multiple iterations.
        
        Args:
            prediction_history: List of prediction DataFrames
            actual_results: DataFrame with actual results
            iterations: Number of calibration iterations
            target_coverage: Target coverage rate
            
        Returns:
            Dict with calibration summary
        """
        # Initialize results
        calibration_results = []
        
        # Store starting factors
        initial_factors = self.width_factors.copy()
        
        # Run iterations
        for i in range(iterations):
            print(f"\nCalibration iteration {i+1}/{iterations}")
            
            # Combine all predictions for this iteration
            try:
                all_predictions = pd.concat(prediction_history, ignore_index=True)
            except ValueError:
                if len(prediction_history) == 0:
                    print("Error: No prediction data provided")
                    return {'error': 'No prediction data provided'}
                else:
                    all_predictions = prediction_history[0]
            
            # Run calibration iteration
            result = self.run_calibration_iteration(all_predictions, actual_results, target_coverage)
            calibration_results.append(result)
            
            # Early stopping if we're close to target
            coverage_values = list(result['coverage_by_quarter'].values())
            if coverage_values:
                avg_coverage = sum(coverage_values) / len(coverage_values)
                coverage_error = abs(avg_coverage - target_coverage)
                
                if coverage_error < 0.02:  # Within 2% of target
                    print(f"Early stopping: Average coverage {avg_coverage:.1%} is within 2% of target {target_coverage:.1%}")
                    break
        
        # Calculate overall changes
        overall_changes = {}
        for quarter in self.width_factors.keys():
            if quarter in initial_factors:
                change = self.width_factors[quarter] - initial_factors[quarter]
                percent_change = (change / initial_factors[quarter]) * 100
                overall_changes[quarter] = {
                    'absolute': change,
                    'percent': percent_change
                }
        
        # Generate summary
        summary = {
            'iterations_run': len(calibration_results),
            'final_width_factors': self.width_factors,
            'initial_width_factors': initial_factors,
            'overall_changes': overall_changes,
            'final_coverage': calibration_results[-1]['coverage_by_quarter'] if calibration_results else None,
            'calibration_history': calibration_results
        }
        
        # Visualize calibration process
        self.visualize_calibration_results(summary)
        
        return summary
    
    def visualize_calibration_results(self, 
                                    calibration_summary: Dict[str, Any], 
                                    show_plot: bool = True) -> Optional[plt.Figure]:
        """
        Visualize calibration results.
        
        Args:
            calibration_summary: Dictionary with calibration results
            show_plot: Whether to display the plot immediately
        
        Returns:
            matplotlib Figure if created, None otherwise
        """
        # Skip if no calibration history
        if not calibration_summary.get('calibration_history'):
            print("No calibration history to visualize")
            return None
        
        # Create figure with two subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Extract data for plotting
        history = calibration_summary['calibration_history']
        iterations = list(range(1, len(history) + 1))
        
        # Use consistent quarters across all iterations
        all_quarters = set()
        for entry in history:
            all_quarters.update(entry['new_factors'].keys())
        quarters = sorted(all_quarters)
        
        # Colors for different quarters
        colors = ['blue', 'green', 'orange', 'red', 'purple']
        
        # Plot 1: Coverage by quarter over iterations
        ax1.set_title('Coverage by Quarter Over Calibration Iterations')
        
        for i, quarter in enumerate(quarters):
            # Extract coverage values for this quarter
            coverage_values = []
            for entry in history:
                coverage = entry['coverage_by_quarter'].get(quarter, np.nan)
                coverage_values.append(coverage)
            
            # Skip if no data for this quarter
            if all(np.isnan(x) for x in coverage_values):
                continue
                
            # Plot coverage trend
            ax1.plot(iterations, coverage_values, f'-o', color=colors[i % len(colors)], 
                    label=f'Quarter {quarter}')
        
        # Add target line
        target = history[0]['target_coverage']
        ax1.axhline(y=target, color='black', linestyle='--', label=f'Target ({target:.0%})')
        
        ax1.set_xlabel('Calibration Iteration')
        ax1.set_ylabel('Coverage Rate')
        ax1.set_ylim(0, 1.1)
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Plot 2: Width factors by quarter over iterations
        ax2.set_title('Width Factors by Quarter Over Calibration Iterations')
        
        for i, quarter in enumerate(quarters):
            # Extract factor values for this quarter
            factor_values = []
            for entry in history:
                factor = entry['new_factors'].get(quarter, np.nan)
                factor_values.append(factor)
            
            # Skip if no data for this quarter
            if all(np.isnan(x) for x in factor_values):
                continue
                
            # Plot factor trend
            ax2.plot(iterations, factor_values, f'-s', color=colors[i % len(colors)], 
                    label=f'Quarter {quarter}')
        
        ax2.set_xlabel('Calibration Iteration')
        ax2.set_ylabel('Width Factor')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        
        if show_plot:
            plt.show()
        
        return fig
    
    def analyze_high_leverage_performance(self, predictions: pd.DataFrame, 
                                       actuals: pd.DataFrame) -> Dict[str, Any]:
        """
        Analyze prediction performance specifically in high-leverage situations.
        
        Args:
            predictions: DataFrame with predictions
            actuals: DataFrame with actual results
            
        Returns:
            Dict with analysis results
        """
        # Apply intervals and identify high-leverage situations
        predictions_with_intervals = self.apply_prediction_intervals(predictions)
        
        # Calculate coverage
        coverage_data = self.calculate_coverage(predictions_with_intervals, actuals)
        
        if coverage_data.empty:
            return {'error': 'No matching data found'}
        
        # Split by high leverage vs normal
        high_leverage = coverage_data[coverage_data['high_leverage'] == True]
        normal = coverage_data[coverage_data['high_leverage'] == False]
        
        # Prepare analysis results
        results = {
            'summary': {
                'total_games': len(coverage_data),
                'high_leverage_games': len(high_leverage),
                'normal_games': len(normal)
            }
        }
        
        # Calculate coverage rates
        if not high_leverage.empty:
            results['high_leverage'] = {
                'coverage': high_leverage['in_interval'].mean(),
                'home_coverage': high_leverage['home_in_interval'].mean(),
                'away_coverage': high_leverage['away_in_interval'].mean()
            }
            
            # Add error metrics if available
            if 'home_error' in high_leverage.columns:
                results['high_leverage'].update({
                    'mean_home_error': high_leverage['home_error'].mean(),
                    'mean_away_error': high_leverage['away_error'].mean(),
                    'mean_total_error': high_leverage['total_error'].mean()
                })
        
        if not normal.empty:
            results['normal'] = {
                'coverage': normal['in_interval'].mean(),
                'home_coverage': normal['home_in_interval'].mean(),
                'away_coverage': normal['away_in_interval'].mean()
            }
            
            # Add error metrics if available
            if 'home_error' in normal.columns:
                results['normal'].update({
                    'mean_home_error': normal['home_error'].mean(),
                    'mean_away_error': normal['away_error'].mean(),
                    'mean_total_error': normal['total_error'].mean()
                })
        
        # Calculate comparison between high-leverage and normal situations
        if 'high_leverage' in results and 'normal' in results:
            results['comparison'] = {
                'coverage_diff': results['high_leverage']['coverage'] - results['normal']['coverage'],
                'home_coverage_diff': results['high_leverage']['home_coverage'] - results['normal']['home_coverage'],
                'away_coverage_diff': results['high_leverage']['away_coverage'] - results['normal']['away_coverage']
            }
            
            # Add error comparison if available
            if 'mean_total_error' in results['high_leverage'] and 'mean_total_error' in results['normal']:
                results['comparison']['error_ratio'] = (
                    results['high_leverage']['mean_total_error'] / 
                    results['normal']['mean_total_error'] if results['normal']['mean_total_error'] > 0 else np.nan
                )
        
        return results


def demo_confidence_calibration(display_details: bool = True) -> Dict[str, Any]:
    """
    Demonstrate the calibration system with sample data.
    
    Args:
        display_details: Whether to print detailed information
        
    Returns:
        Dict with demonstration results
    """
    # Create a calibration system
    calibration_system = ContextAwareConfidenceCalibration()
    
    # Create sample predictions with different game contexts
    predictions = pd.DataFrame([
        # Pre-game predictions
        {'game_id': 1001, 'current_quarter': 0, 'predicted_home_score': 105, 'predicted_away_score': 98, 
         'momentum_shift': 0, 'win_probability': 0.65},
        # Close game in Q2
        {'game_id': 1002, 'current_quarter': 2, 'predicted_home_score': 110, 'predicted_away_score': 107, 
         'momentum_shift': 0.3, 'win_probability': 0.55, 'time_remaining': 6},
        # Blowout in Q3
        {'game_id': 1003, 'current_quarter': 3, 'predicted_home_score': 95, 'predicted_away_score': 75, 
         'momentum_shift': -0.1, 'win_probability': 0.92, 'time_remaining': 8},
        # High leverage final minutes
        {'game_id': 1004, 'current_quarter': 4, 'predicted_home_score': 108, 'predicted_away_score': 106, 
         'momentum_shift': 0.6, 'win_probability': 0.52, 'time_remaining': 2},
    ])
    
    # Sample actual results
    actuals = pd.DataFrame([
        {'game_id': 1001, 'home_score': 108, 'away_score': 101},
        {'game_id': 1002, 'home_score': 112, 'away_score': 109},
        {'game_id': 1003, 'home_score': 98, 'away_score': 78},
        {'game_id': 1004, 'home_score': 110, 'away_score': 107},
    ])
    
    # Apply intervals and check coverage
    predictions_with_intervals = calibration_system.apply_prediction_intervals(predictions)
    coverage_results = calibration_system.calculate_coverage(predictions_with_intervals, actuals)
    
    if display_details:
        print("\nSample Predictions with Intervals:")
        for idx, row in predictions_with_intervals.iterrows():
            game_id = row['game_id']
            quarter = row['current_quarter']
            score_diff = abs(row['predicted_home_score'] - row['predicted_away_score'])
            context = "close game" if score_diff <= 5.0 else "blowout"
            
            if quarter == 4 and row.get('time_remaining', 12) <= 5.0:
                context = "final minutes"
            
            high_leverage = row.get('high_leverage', False)
            
            print(f"\nGame {game_id} (Q{quarter}, {context}, {'high leverage' if high_leverage else 'normal'}):")
            print(f"  Home: {row['predicted_home_score']:.1f} [{row['home_lower_bound']:.1f} - {row['home_upper_bound']:.1f}]")
            print(f"  Away: {row['predicted_away_score']:.1f} [{row['away_lower_bound']:.1f} - {row['away_upper_bound']:.1f}]")
            
            # Check if actuals fall within intervals
            actual = actuals[actuals['game_id'] == game_id]
            if not actual.empty:
                actual = actual.iloc[0]
                home_in = actual['home_score'] >= row['home_lower_bound'] and actual['home_score'] <= row['home_upper_bound']
                away_in = actual['away_score'] >= row['away_lower_bound'] and actual['away_score'] <= row['away_upper_bound']
                
                print(f"  Actual Home: {actual['home_score']} ({'within' if home_in else 'outside'} interval)")
                print(f"  Actual Away: {actual['away_score']} ({'within' if away_in else 'outside'} interval)")
        
        # Print coverage summary
        print("\nCoverage Summary:")
        for quarter in range(5):
            quarter_data = coverage_results[coverage_results['quarter'] == quarter]
            if not quarter_data.empty:
                home_coverage = quarter_data['home_in_interval'].mean()
                away_coverage = quarter_data['away_in_interval'].mean()
                full_coverage = quarter_data['in_interval'].mean()
                
                print(f"  Quarter {quarter}: Home {home_coverage:.0%}, Away {away_coverage:.0%}, Overall {full_coverage:.0%}")
    
    # Run a calibration iteration
    if display_details:
        print("\nRunning calibration iteration...")
    result = calibration_system.run_calibration_iteration(predictions, actuals)
    
    # Analyze high-leverage performance
    high_leverage_analysis = calibration_system.analyze_high_leverage_performance(
        predictions, actuals)
    
    return {
        'predictions_with_intervals': predictions_with_intervals,
        'coverage_results': coverage_results,
        'calibration_result': result,
        'high_leverage_analysis': high_leverage_analysis
    }


# Integration with previous modules
def integrate_with_uncertainty_estimator(uncertainty_estimator: Any) -> ContextAwareConfidenceCalibration:
    """
    Integrate the context-aware calibration system with the UncertaintyEstimator from Cell 27.
    
    Args:
        uncertainty_estimator: Instance of UncertaintyEstimator from Cell 27
        
    Returns:
        ContextAwareConfidenceCalibration instance initialized with the estimator's width factors
    """
    # Extract width factors from the uncertainty estimator
    width_factors = getattr(uncertainty_estimator, 'width_factors', None)
    
    # Create a context-aware calibration system with those factors
    calibration_system = ContextAwareConfidenceCalibration(
        base_width_factors=width_factors
    )
    
    return calibration_system


# Run demo if this cell is executed directly
if __name__ == "__main__":
    demo_results = demo_confidence_calibration()

In [ ]:
# Cell 27E: Comprehensive Performance Profiling and Optimization with Enhanced Feature Calculator

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from typing import Dict, List, Tuple, Optional, Union, Any
import json
from datetime import datetime, timedelta
import pytz
import os

# ------------------ NEW HELPER: Fetch Live Games from Supabase ------------------

from caching.supabase_client import supabase

def fetch_live_games_from_supabase_27E() -> pd.DataFrame:
    """
    Pull all rows from 'nba_live_game_stats' for today's date in Pacific Time,
    filter for rows whose 'status' (uppercased) is in {Q1, Q2, Q3, Q4, OT, BT, HT},
    and set 'game_status' to 'LIVE' for those rows.
    This ensures we only see truly live games, matching Cell 29's fix.
    """
    try:
        # Fetch all rows from Supabase
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if not response.data:
            print("No game data found in 'nba_live_game_stats'.")
            return pd.DataFrame()

        df = pd.DataFrame(response.data)

        # Convert game_date to datetime, localize to Pacific Time, filter for today
        if "game_date" in df.columns:
            df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce", utc=True)
            pacific_tz = pytz.timezone("America/Los_Angeles")
            df["game_date_pt"] = df["game_date"].dt.tz_convert(pacific_tz)
            df["date_only_pt"] = df["game_date_pt"].dt.date
            now_pt = datetime.now(pacific_tz)
            today_pt = now_pt.date()
            df = df[df["date_only_pt"] == today_pt].copy()
            print(f"Found {len(df)} games scheduled for today in PT (Cell 27E).")
        else:
            print("Column 'game_date' not found in the Supabase data!")
            return pd.DataFrame()

        # Normalize 'status' column and filter for live statuses
        if "status" in df.columns:
            df["status"] = df["status"].astype(str).str.upper()
        else:
            df["status"] = ""

        live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
        active_df = df[df["status"].isin(live_statuses)].copy()

        # Mark these as "LIVE" for consistent usage
        active_df["game_status"] = "LIVE"

        print(f"Fetched {len(active_df)} active live games (Cell 27E).")
        return active_df

    except Exception as e:
        print(f"Error in fetch_live_games_from_supabase_27E(): {e}")
        return pd.DataFrame()

# ------------------ BELOW: The classes from your original Cell 27E remain unchanged ------------------

class ColorManager:
    """
    Manages color schemes for NBA visualization components.
    ...
    (unchanged)
    """
    def __init__(self, base_scheme: str = "default"):
        # ... unchanged ...
        pass

    # ... rest of ColorManager code ...

class ConfidenceIndicator:
    """
    Creates compact, reusable visualization components for showing prediction confidence.
    ...
    (unchanged)
    """
    def __init__(self, color_manager: Optional[ColorManager] = None):
        # ... unchanged ...
        pass

    # ... rest of ConfidenceIndicator code ...

class GameVisualizer:
    """
    Creates visualizations for NBA game predictions with confidence intervals.
    ...
    (unchanged)
    """
    def __init__(self, color_manager: Optional[ColorManager] = None):
        # ... unchanged ...
        pass

    # ... rest of GameVisualizer code ...

class DashboardGenerator:
    """
    Generates complete dashboards for displaying NBA prediction confidence.
    ...
    (unchanged)
    """
    def __init__(self, 
                 color_manager: Optional[ColorManager] = None,
                 game_visualizer: Optional[GameVisualizer] = None):
        # ... unchanged ...
        pass

    # ... rest of DashboardGenerator code ...

class ConfidenceVisualizationSystem:
    """
    Complete system for generating NBA prediction confidence visualizations.
    ...
    (unchanged)
    """
    def __init__(self, color_scheme: str = 'default'):
        # ... unchanged ...
        pass

    # ... rest of ConfidenceVisualizationSystem code ...

def create_interactive_confidence_display(prediction_data: Dict[str, Any], 
                                        design_style: str = 'pwa') -> str:
    """
    Create an interactive SVG visualization showing prediction confidence.
    ...
    (unchanged)
    """
    # ... unchanged ...
    pass

# =============== REPLACED DEMO FUNCTION: fetch from supabase instead of sample data ===============

def demo_enhanced_visualization_27E():
    """
    Demonstrate the enhanced visualization using real-time data from Supabase,
    mirroring the fix from Cell 29 so we don't see old Lakers/Celtics, etc.
    """
    # 1) Fetch live games from Supabase
    live_games_df = fetch_live_games_from_supabase_27E()
    if live_games_df.empty:
        print("No active live games found for Cell 27E.")
        return

    # 2) Convert your DataFrame columns to match any needed naming. For example,
    #    if you rely on 'predicted_home_score' or 'predicted_away_score', ensure
    #    they exist or that your code can handle them. If your model is run earlier,
    #    you might already have columns. Otherwise, do a minimal placeholder:
    if 'predicted_home_score' not in live_games_df.columns:
        live_games_df['predicted_home_score'] = live_games_df['home_score'] + 10.0
    if 'predicted_away_score' not in live_games_df.columns:
        live_games_df['predicted_away_score'] = live_games_df['away_score'] + 8.0
    # For demonstration, set a basic 'win_probability' if missing
    if 'win_probability' not in live_games_df.columns:
        live_games_df['win_probability'] = 0.5

    # 3) Create a ConfidenceVisualizationSystem
    vis_system = ConfidenceVisualizationSystem(color_scheme="default")

    # 4) Generate a full dashboard for the live games
    dashboard_html = vis_system.create_dashboard(live_games_df, 
                                                 dashboard_id="confidence-dashboard-27E",
                                                 title="NBA Live Games (Cell 27E)")

    # 5) Display in a notebook if available
    try:
        from IPython.display import display, HTML
        display(HTML(dashboard_html))
    except:
        print(dashboard_html)

# If you want this code to run automatically, uncomment:
# if __name__ == "__main__":
#     demo_enhanced_visualization_27E()


In [ ]:
# Cell 28: Comprehensive Performance Profiling and Optimization with Enhanced Feature Calculator

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import traceback
import os
import json
import hashlib
import pickle
from functools import wraps, lru_cache
from datetime import datetime, timedelta
import psutil
import gc
from typing import Dict, List, Tuple, Callable, Optional, Union, Any

# Define placeholders for imported functions to avoid missing module errors
def fetch_recent_games_for_testing(limit=5):
    """Placeholder for fetch_recent_games_for_testing function."""
    print("Missing fetch_recent_games_for_testing module, using fallback")
    return None

def prepare_features(df, model=None):
    """Placeholder for prepare_features function."""
    print("Missing prepare_features module, using fallback")
    return df.copy()

def run_predictions(model, features_df):
    """Placeholder for run_predictions function."""
    print("Missing run_predictions module, using fallback")
    return features_df.copy()

def add_uncertainty_to_predictions(predictions_df):
    """Placeholder for add_uncertainty_to_predictions function."""
    print("Missing add_uncertainty_to_predictions module, using fallback")
    return predictions_df.copy()

def load_model():
    """Placeholder for load_model function."""
    print("Missing load_model module, using fallback")
    return None

class PerformanceProfiler:
    """
    A comprehensive profiler for NBA prediction pipeline performance measurement.
    
    This class provides methods to:
    1. Measure real-world performance using actual NBA data
    2. Profile complete end-to-end pipelines, not just components
    3. Track both time and memory usage
    4. Generate visualizations and optimization recommendations
    """
    
    def __init__(self, profile_dir="./performance_profiles"):
        """
        Initialize the performance profiler.
        
        Args:
            profile_dir: Directory to store performance profiles
        """
        self.profile_dir = profile_dir
        os.makedirs(profile_dir, exist_ok=True)
        
        # Tracking data
        self.timing_data = {}
        self.memory_data = {}
        self.function_calls = {}
        self.bottlenecks = []
        
        # Store previous runs for comparison
        self.previous_profiles = self._load_previous_profiles()
    
    def _load_previous_profiles(self):
        """Load previous profiles from storage."""
        profiles = []
        try:
            profile_files = [f for f in os.listdir(self.profile_dir) 
                            if f.endswith('.json')]
            
            for filename in profile_files:
                try:
                    with open(os.path.join(self.profile_dir, filename), 'r') as f:
                        profile = json.load(f)
                        profiles.append(profile)
                except Exception as e:
                    print(f"Error loading profile {filename}: {e}")
                    
            print(f"Loaded {len(profiles)} previous performance profiles")
            return profiles
        except Exception as e:
            print(f"Error loading profiles: {e}")
            return []
    
    def profile_function(self, func=None, label=None, track_memory=True):
        """
        Decorator to profile function execution time and optional memory usage.
        
        Args:
            func: Function to profile
            label: Custom label for the function (defaults to function name)
            track_memory: Whether to track memory usage
            
        Returns:
            Decorated function
        """
        def decorator(f):
            @wraps(f)
            def wrapper(*args, **kwargs):
                # Start tracking
                func_name = label or f.__name__
                start_time = time.time()
                
                # Track memory before
                if track_memory:
                    gc.collect()  # Force garbage collection
                    before_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
                
                # Execute function
                try:
                    result = f(*args, **kwargs)
                    success = True
                except Exception as e:
                    success = False
                    print(f"Error in {func_name}: {e}")
                    traceback.print_exc()
                    raise
                finally:
                    # Record execution time
                    end_time = time.time()
                    execution_time = end_time - start_time
                    
                    # Record in timing data
                    if func_name not in self.timing_data:
                        self.timing_data[func_name] = []
                    self.timing_data[func_name].append(execution_time)
                    
                    # Track function calls
                    if func_name not in self.function_calls:
                        self.function_calls[func_name] = 0
                    self.function_calls[func_name] += 1
                    
                    # Track memory after
                    if track_memory:
                        gc.collect()  # Force garbage collection
                        after_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
                        memory_diff = after_memory - before_memory
                        
                        if func_name not in self.memory_data:
                            self.memory_data[func_name] = []
                        self.memory_data[func_name].append(memory_diff)
                        
                        # Print memory usage for long-running functions
                        if execution_time > 1.0:
                            print(f"{func_name} memory change: {memory_diff:.2f} MB")
                
                # Print timing for long-running functions
                if execution_time > 0.5:
                    print(f"{func_name} execution time: {execution_time:.3f} seconds")
                
                if success:
                    return result
            
            return wrapper
        
        # Handle direct decoration without arguments
        if func is not None:
            return decorator(func)
        return decorator
    
    def profile_pipeline(self, pipeline_func, label, *pipeline_args, **pipeline_kwargs):
        """
        Profile an entire pipeline end-to-end.
        
        Args:
            pipeline_func: Function that runs the entire pipeline
            label: Label for this pipeline run
            *pipeline_args, **pipeline_kwargs: Arguments to pass to pipeline function
            
        Returns:
            Pipeline result and performance data
        """
        print(f"\n{'='*50}\nProfiling pipeline: {label}\n{'='*50}")
        
        # Start tracking
        pipeline_start = time.time()
        gc.collect()  # Force garbage collection
        start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        
        # Track component-level metrics with a context manager
        original_timing_data = self.timing_data.copy()
        original_memory_data = self.memory_data.copy()
        original_function_calls = self.function_calls.copy()
        
        # Run the pipeline
        try:
            result = pipeline_func(*pipeline_args, **pipeline_kwargs)
            success = True
        except Exception as e:
            success = False
            print(f"Pipeline error: {e}")
            traceback.print_exc()
            result = None
        finally:
            # Record execution time
            pipeline_end = time.time()
            execution_time = pipeline_end - pipeline_start
            
            # Record memory usage
            gc.collect()  # Force garbage collection
            end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
            memory_diff = end_memory - start_memory
            
            # Create profile data
            profile = {
                'label': label,
                'timestamp': datetime.now().isoformat(),
                'success': success,
                'execution_time': execution_time,
                'memory_diff_mb': memory_diff,
                'component_metrics': self._get_component_metrics(
                    original_timing_data, original_memory_data, original_function_calls),
                'bottlenecks': self._identify_bottlenecks(),
                'optimization_recommendations': []
            }
            
            # Add optimization recommendations
            profile['optimization_recommendations'] = self._generate_optimization_recommendations(profile)
            
            # Store profile
            profile_filename = f"profile_{label.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(os.path.join(self.profile_dir, profile_filename), 'w') as f:
                json.dump(profile, f, indent=2)
            
            print(f"\nPipeline {label} completed in {execution_time:.3f} seconds")
            print(f"Memory change: {memory_diff:.2f} MB")
            
            # Return both the pipeline result and profile data
            return result, profile
    
    def _get_component_metrics(self, original_timing, original_memory, original_calls):
        """
        Calculate component-specific metrics based on before/after data.
        
        Args:
            original_timing: Timing data before pipeline run
            original_memory: Memory data before pipeline run
            original_calls: Function call counts before pipeline run
            
        Returns:
            Dict with component metrics
        """
        component_metrics = {}
        
        # Process timing data
        for func_name, times in self.timing_data.items():
            original_times = original_timing.get(func_name, [])
            new_times = times[len(original_times):]
            
            if new_times:
                component_metrics[func_name] = {
                    'calls': len(new_times),
                    'total_time': sum(new_times),
                    'avg_time': sum(new_times) / len(new_times),
                    'min_time': min(new_times),
                    'max_time': max(new_times)
                }
                
                # Add memory metrics if available
                if func_name in self.memory_data:
                    original_mem = original_memory.get(func_name, [])
                    new_mem = self.memory_data[func_name][len(original_mem):]
                    
                    if new_mem:
                        component_metrics[func_name]['total_memory_mb'] = sum(new_mem)
                        component_metrics[func_name]['avg_memory_mb'] = sum(new_mem) / len(new_mem)
        
        return component_metrics
    
    def _identify_bottlenecks(self):
        """Identify performance bottlenecks in the pipeline."""
        bottlenecks = []
        
        # Time-based bottlenecks
        time_threshold = 0.5  # Functions taking > 0.5s on average
        
        for func_name, metrics in self._get_current_component_metrics().items():
            # Add to bottlenecks if average time exceeds threshold
            if metrics.get('avg_time', 0) > time_threshold:
                bottlenecks.append({
                    'component': func_name,
                    'type': 'execution_time',
                    'value': metrics['avg_time'],
                    'severity': 'high' if metrics['avg_time'] > 2.0 else 'medium',
                    'description': f"{func_name} takes {metrics['avg_time']:.2f}s on average"
                })
            
            # Memory-based bottlenecks (if function allocates > 50MB)
            if metrics.get('avg_memory_mb', 0) > 50:
                bottlenecks.append({
                    'component': func_name,
                    'type': 'memory_usage',
                    'value': metrics['avg_memory_mb'],
                    'severity': 'high' if metrics['avg_memory_mb'] > 200 else 'medium',
                    'description': f"{func_name} allocates {metrics['avg_memory_mb']:.2f}MB on average"
                })
                
            # Excessive function calls (if called more than 100 times)
            if metrics.get('calls', 0) > 100:
                bottlenecks.append({
                    'component': func_name,
                    'type': 'excessive_calls',
                    'value': metrics['calls'],
                    'severity': 'medium',
                    'description': f"{func_name} called {metrics['calls']} times"
                })
        
        # Sort by severity and value
        bottlenecks.sort(key=lambda x: (0 if x['severity'] == 'high' else 1, -x['value']))
        
        return bottlenecks
    
    def _get_current_component_metrics(self):
        """Get the most recent metrics for all components."""
        metrics = {}
        
        for func_name, times in self.timing_data.items():
            if times:
                metrics[func_name] = {
                    'calls': self.function_calls.get(func_name, 0),
                    'total_time': sum(times),
                    'avg_time': sum(times) / len(times),
                    'min_time': min(times),
                    'max_time': max(times)
                }
                
                # Add memory metrics if available
                if func_name in self.memory_data and self.memory_data[func_name]:
                    mem_values = self.memory_data[func_name]
                    metrics[func_name]['total_memory_mb'] = sum(mem_values)
                    metrics[func_name]['avg_memory_mb'] = sum(mem_values) / len(mem_values)
        
        return metrics
    
    def _generate_optimization_recommendations(self, profile):
        """Generate optimization recommendations based on performance profile."""
        recommendations = []
        
        # Process bottlenecks
        for bottleneck in profile.get('bottlenecks', []):
            component = bottleneck['component']
            bottleneck_type = bottleneck['type']
            
            if bottleneck_type == 'execution_time':
                if 'fetch' in component.lower() or 'get' in component.lower():
                    recommendations.append({
                        'component': component,
                        'recommendation': 'Implement caching for database or API requests',
                        'potential_impact': 'high',
                        'implementation_difficulty': 'medium'
                    })
                elif 'feature' in component.lower():
                    recommendations.append({
                        'component': component,
                        'recommendation': 'Optimize feature calculation with vectorized operations',
                        'potential_impact': 'high',
                        'implementation_difficulty': 'medium'
                    })
                elif 'model' in component.lower() or 'predict' in component.lower():
                    recommendations.append({
                        'component': component,
                        'recommendation': 'Consider model simplification or batch predictions',
                        'potential_impact': 'medium',
                        'implementation_difficulty': 'high'
                    })
            
            elif bottleneck_type == 'memory_usage':
                recommendations.append({
                    'component': component,
                    'recommendation': 'Reduce memory usage by processing data in chunks',
                    'potential_impact': 'medium',
                    'implementation_difficulty': 'medium'
                })
            
            elif bottleneck_type == 'excessive_calls':
                recommendations.append({
                    'component': component,
                    'recommendation': 'Reduce number of function calls through batching',
                    'potential_impact': 'high',
                    'implementation_difficulty': 'low'
                })
        
        # Add general recommendations if needed
        if profile.get('execution_time', 0) > 5.0:
            recommendations.append({
                'component': 'entire_pipeline',
                'recommendation': 'Implement parallel processing for independent tasks',
                'potential_impact': 'high',
                'implementation_difficulty': 'high'
            })
        
        return recommendations
    
    def visualize_profile(self, profile=None):
        """
        Create visualization of performance profile.
        
        Args:
            profile: Profile data dict (if None, visualizes most recent)
            
        Returns:
            Matplotlib figure
        """
        # Use the provided profile or the last run
        if profile is None:
            if not self.previous_profiles:
                print("No profiles available to visualize.")
                return None
            profile = self.previous_profiles[-1]
        
        # Create figure with subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Component timing chart (top)
        component_metrics = profile.get('component_metrics', {})
        components = []
        times = []
        colors = []
        
        for component, metrics in component_metrics.items():
            components.append(component)
            times.append(metrics.get('total_time', 0))
            
            # Determine color based on bottleneck severity
            is_bottleneck = False
            severity = 'low'
            
            for bottleneck in profile.get('bottlenecks', []):
                if bottleneck['component'] == component:
                    is_bottleneck = True
                    severity = bottleneck['severity']
                    break
            
            if is_bottleneck:
                if severity == 'high':
                    colors.append('#ff5252')  # Red for high severity
                else:
                    colors.append('#ffb142')  # Orange for medium severity
            else:
                colors.append('#2ecc71')  # Green for non-bottlenecks
        
        # Sort by execution time (descending)
        sorted_indices = np.argsort(times)[::-1]
        sorted_components = [components[i] for i in sorted_indices]
        sorted_times = [times[i] for i in sorted_indices]
        sorted_colors = [colors[i] for i in sorted_indices]
        
        # Plot component times
        ax1.barh(sorted_components, sorted_times, color=sorted_colors)
        ax1.set_title('Component Execution Times', fontsize=14)
        ax1.set_xlabel('Seconds', fontsize=12)
        ax1.set_ylabel('Component', fontsize=12)
        ax1.grid(axis='x', linestyle='--', alpha=0.7)
        
        # Add time labels
        for i, v in enumerate(sorted_times):
            if v >= 0.1:  # Only label significant times
                ax1.text(v + 0.05, i, f"{v:.2f}s", va='center')
        
        # Memory usage chart (bottom)
        memory_components = []
        memory_values = []
        memory_colors = []
        
        for component, metrics in component_metrics.items():
            if 'avg_memory_mb' in metrics:
                memory_components.append(component)
                memory_values.append(metrics['avg_memory_mb'])
                
                # Determine color based on memory usage
                if metrics['avg_memory_mb'] > 200:
                    memory_colors.append('#ff5252')  # Red for high memory
                elif metrics['avg_memory_mb'] > 50:
                    memory_colors.append('#ffb142')  # Orange for medium memory
                else:
                    memory_colors.append('#2ecc71')  # Green for low memory
        
        # Sort by memory usage (descending)
        if memory_components:
            mem_sorted_indices = np.argsort(memory_values)[::-1]
            mem_sorted_components = [memory_components[i] for i in mem_sorted_indices]
            mem_sorted_values = [memory_values[i] for i in mem_sorted_indices]
            mem_sorted_colors = [memory_colors[i] for i in mem_sorted_indices]
            
            # Plot memory usage
            ax2.barh(mem_sorted_components, mem_sorted_values, color=mem_sorted_colors)
            ax2.set_title('Component Memory Usage', fontsize=14)
            ax2.set_xlabel('Memory (MB)', fontsize=12)
            ax2.set_ylabel('Component', fontsize=12)
            ax2.grid(axis='x', linestyle='--', alpha=0.7)
            
            # Add memory labels
            for i, v in enumerate(mem_sorted_values):
                if v >= 1.0:  # Only label significant memory usage
                    ax2.text(v + 1, i, f"{v:.1f} MB", va='center')
        else:
            ax2.text(0.5, 0.5, 'No memory usage data available', 
                    ha='center', va='center', transform=ax2.transAxes, fontsize=14)
        
        # Add total metrics as a figure title
        plt.suptitle(
            f"Pipeline Performance Profile: {profile.get('label', 'Unnamed')}\n"
            f"Total Time: {profile.get('execution_time', 0):.2f}s, "
            f"Memory Change: {profile.get('memory_diff_mb', 0):.1f}MB",
            fontsize=16
        )
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        
        return fig
    
    def compare_profiles(self, label1, label2=None):
        """
        Compare two performance profiles.
        
        Args:
            label1: Label of first profile
            label2: Label of second profile (if None, uses most recent)
            
        Returns:
            Comparison data and visualization
        """
        # Find profiles by label
        profile1 = None
        profile2 = None
        
        for profile in self.previous_profiles:
            if profile.get('label') == label1:
                profile1 = profile
            elif label2 and profile.get('label') == label2:
                profile2 = profile
        
        # If label2 is None, use the most recent profile that isn't profile1
        if label2 is None and profile1 is not None:
            for profile in reversed(self.previous_profiles):
                if profile.get('label') != label1:
                    profile2 = profile
                    label2 = profile.get('label', 'Unnamed')
                    break
        
        if not profile1 or not profile2:
            print(f"Couldn't find profiles to compare.")
            return None
        
        print(f"Comparing profiles: '{label1}' vs '{label2}'")
        
        # Calculate differences
        comparison = {
            'profile1': label1,
            'profile2': label2,
            'total_time_diff': profile2['execution_time'] - profile1['execution_time'],
            'total_time_pct': ((profile2['execution_time'] / profile1['execution_time']) - 1) * 100 
                             if profile1['execution_time'] > 0 else float('inf'),
            'memory_diff': profile2['memory_diff_mb'] - profile1['memory_diff_mb'],
            'component_diffs': {}
        }
        
        # Compare common components
        components1 = profile1.get('component_metrics', {})
        components2 = profile2.get('component_metrics', {})
        
        all_components = set(components1.keys()) | set(components2.keys())
        
        for component in all_components:
            metrics1 = components1.get(component, {})
            metrics2 = components2.get(component, {})
            
            # Skip if component doesn't exist in one profile
            if not metrics1 or not metrics2:
                continue
            
            # Calculate time differences
            time1 = metrics1.get('total_time', 0)
            time2 = metrics2.get('total_time', 0)
            time_diff = time2 - time1
            time_pct = ((time2 / time1) - 1) * 100 if time1 > 0 else float('inf')
            
            # Calculate memory differences if available
            mem_diff = None
            mem_pct = None
            
            if 'total_memory_mb' in metrics1 and 'total_memory_mb' in metrics2:
                mem1 = metrics1['total_memory_mb']
                mem2 = metrics2['total_memory_mb']
                mem_diff = mem2 - mem1
                mem_pct = ((mem2 / mem1) - 1) * 100 if mem1 > 0 else float('inf')
            
            # Store component differences
            comparison['component_diffs'][component] = {
                'time_diff': time_diff,
                'time_pct': time_pct,
                'memory_diff': mem_diff,
                'memory_pct': mem_pct
            }
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Extract component data for plotting
        components = []
        time_pcts = []
        colors = []
        
        for component, diffs in comparison['component_diffs'].items():
            components.append(component)
            pct = diffs['time_pct']
            time_pcts.append(pct)
            
            # Color based on performance change
            if pct <= -10:  # Improved by 10%+
                colors.append('#2ecc71')  # Green
            elif pct >= 10:  # Worse by 10%+
                colors.append('#ff5252')  # Red
            else:
                colors.append('#3498db')  # Blue for minor changes
        
        # Sort by percentage change
        sorted_indices = np.argsort(time_pcts)
        sorted_components = [components[i] for i in sorted_indices]
        sorted_pcts = [time_pcts[i] for i in sorted_indices]
        sorted_colors = [colors[i] for i in sorted_indices]
        
        # Plot percentage changes
        bars = ax.barh(sorted_components, sorted_pcts, color=sorted_colors)
        
        # Add zero line
        ax.axvline(0, color='black', linestyle='-', alpha=0.7)
        
        # Add percentage labels
        for i, (bar, pct) in enumerate(zip(bars, sorted_pcts)):
            if abs(pct) >= 1.0:  # Only label significant changes
                text_color = 'white' if abs(pct) > 20 else 'black'
                x_pos = bar.get_width() if pct >= 0 else bar.get_width() - 5
                ax.text(x_pos, bar.get_y() + bar.get_height()/2, 
                       f"{pct:.1f}%", va='center', ha='right' if pct < 0 else 'left',
                       color=text_color, fontweight='bold')
        
        # Set chart title and labels
        ax.set_title(f"Performance Comparison: '{label2}' vs '{label1}'", fontsize=14)
        ax.set_xlabel('Percentage Change (%)', fontsize=12)
        ax.set_ylabel('Component', fontsize=12)
        
        # Add overall metrics as text
        time_change = comparison['total_time_pct']
        time_symbol = "↓" if time_change <= 0 else "↑"
        
        memory_pct = (comparison['memory_diff'] / profile1['memory_diff_mb']) * 100 if profile1['memory_diff_mb'] != 0 else float('inf')
        memory_symbol = "↓" if comparison['memory_diff'] <= 0 else "↑"
        
        overall_text = (
            f"Overall Time: {abs(time_change):.1f}% {time_symbol} " 
            f"({profile1['execution_time']:.2f}s → {profile2['execution_time']:.2f}s)\n"
            f"Overall Memory: {abs(memory_pct):.1f}% {memory_symbol} "
            f"({profile1['memory_diff_mb']:.1f}MB → {profile2['memory_diff_mb']:.1f}MB)"
        )
        
        plt.figtext(0.5, 0.01, overall_text, ha='center', fontsize=12, 
                   bbox=dict(facecolor='#f9f9f9', alpha=0.5, boxstyle='round,pad=0.5'))
        
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.subplots_adjust(bottom=0.1)
        
        return comparison, fig


class OptimizedFeatureCalculator:
    """
    Optimized feature calculation with caching and vectorized operations.
    
    This class implements optimized methods for calculating prediction features
    with advanced caching and performance tracking.
    """
    
    def __init__(self, cache_dir: str = "./feature_cache", 
                max_cache_entries: int = 100,
                cache_expiry_seconds: int = 3600):
        """
        Initialize the optimized feature calculator.
        
        Args:
            cache_dir: Directory to store cache files
            max_cache_entries: Maximum number of entries in memory cache
            cache_expiry_seconds: Time in seconds after which cache entries expire
        """
        # Create cache directory if it doesn't exist
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        
        # Cache settings
        self.max_cache_entries = max_cache_entries
        self.cache_expiry_seconds = cache_expiry_seconds
        
        # Initialize in-memory cache
        self.memory_cache = {}
        self.cache_timestamps = {}
        self.cache_hits = 0
        self.cache_misses = 0
        
        # Feature computation timing
        self.timing_stats = {
            'momentum_calculation': [],
            'rest_calculation': [],
            'matchup_calculation': [],
            'full_feature_generation': []
        }
    
    def _generate_cache_key(self, df: pd.DataFrame) -> str:
        """
        Generate a unique cache key for the DataFrame.
        
        Args:
            df: Input DataFrame
            
        Returns:
            str: Unique hash key for the DataFrame
        """
        # Extract core columns for key generation
        key_cols = ['game_id', 'current_quarter', 'home_score', 'away_score']
        key_cols = [col for col in key_cols if col in df.columns]
        
        if not key_cols:
            # Fallback to using all relevant columns
            all_cols = df.columns.tolist()
            possible_cols = ['game_id', 'home_team', 'away_team', 'current_quarter', 
                            'home_score', 'away_score', 'game_date']
            key_cols = [col for col in possible_cols if col in all_cols]
        
        # Create a string representation of key columns
        if key_cols:
            key_data = df[key_cols].to_json()
        else:
            # Last resort, use hash of the entire DataFrame
            key_data = str(hash(tuple(map(tuple, df.values))))
        
        # Generate hash
        return hashlib.md5(key_data.encode()).hexdigest()
    
    def _get_cache_path(self, cache_key: str) -> str:
        """Get cache file path from key."""
        return os.path.join(self.cache_dir, f"feature_cache_{cache_key}.pkl")
    
    def _load_from_cache(self, cache_key: str) -> Optional[pd.DataFrame]:
        """
        Load features from cache.
        
        Args:
            cache_key: Cache key
            
        Returns:
            Optional[pd.DataFrame]: Cached features or None if not found
        """
        # First check memory cache
        if cache_key in self.memory_cache:
            # Check if expired
            timestamp = self.cache_timestamps.get(cache_key, 0)
            if time.time() - timestamp <= self.cache_expiry_seconds:
                self.cache_hits += 1
                return self.memory_cache[cache_key]
            else:
                # Expired, remove from memory cache
                del self.memory_cache[cache_key]
                del self.cache_timestamps[cache_key]
        
        # Then check disk cache
        cache_path = self._get_cache_path(cache_key)
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'rb') as f:
                    cache_data = pickle.load(f)
                
                # Check timestamp
                timestamp = cache_data.get('timestamp', 0)
                if time.time() - timestamp <= self.cache_expiry_seconds:
                    # Add to memory cache
                    self.memory_cache[cache_key] = cache_data['features']
                    self.cache_timestamps[cache_key] = timestamp
                    
                    # Manage cache size
                    self._manage_cache_size()
                    
                    self.cache_hits += 1
                    return cache_data['features']
                else:
                    # Expired, remove file
                    os.remove(cache_path)
            except Exception as e:
                print(f"Error loading from cache: {str(e)}")
        
        self.cache_misses += 1
        return None
    
    def _save_to_cache(self, cache_key: str, features: pd.DataFrame):
        """
        Save features to cache.
        
        Args:
            cache_key: Cache key
            features: Feature DataFrame to cache
        """
        timestamp = time.time()
        
        # Save to memory cache
        self.memory_cache[cache_key] = features
        self.cache_timestamps[cache_key] = timestamp
        
        # Manage cache size
        self._manage_cache_size()
        
        # Save to disk cache
        cache_path = self._get_cache_path(cache_key)
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump({
                    'features': features,
                    'timestamp': timestamp
                }, f)
        except Exception as e:
            print(f"Error saving to cache: {str(e)}")
    
    def _manage_cache_size(self):
        """Manage memory cache size by removing oldest entries."""
        if len(self.memory_cache) > self.max_cache_entries:
            # Sort by timestamp and remove oldest entries
            sorted_keys = sorted(self.cache_timestamps.items(), key=lambda x: x[1])
            
            # Remove oldest entries to get back to max size
            num_to_remove = len(self.memory_cache) - self.max_cache_entries
            for i in range(num_to_remove):
                key_to_remove = sorted_keys[i][0]
                del self.memory_cache[key_to_remove]
                del self.cache_timestamps[key_to_remove]
    
    def calculate_features(self, df: pd.DataFrame, force_recalc: bool = False) -> pd.DataFrame:
        """
        Calculate features with caching for performance.
        
        Args:
            df: Input DataFrame
            force_recalc: Force recalculation even if cached
            
        Returns:
            pd.DataFrame: DataFrame with calculated features
        """
        # Skip caching for empty DataFrames
        if df.empty:
            return df.copy()
        
        # Generate cache key
        cache_key = self._generate_cache_key(df)
        
        # Try to load from cache if not forcing recalculation
        if not force_recalc:
            cached_features = self._load_from_cache(cache_key)
            if cached_features is not None:
                # Check for significant differences that would invalidate cache
                if self._is_cache_valid(df, cached_features):
                    return cached_features
        
        # No valid cache hit, calculate features
        start_time = time.time()
        
        # Calculate all features
        features_df = self._calculate_all_features(df)
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['full_feature_generation'].append(elapsed)
        
        # Save to cache
        self._save_to_cache(cache_key, features_df)
        
        return features_df
    
    def _is_cache_valid(self, original_df: pd.DataFrame, cached_df: pd.DataFrame) -> bool:
        """
        Check if cached features are still valid by comparing key fields.
        
        Args:
            original_df: Original input DataFrame
            cached_df: Cached feature DataFrame
            
        Returns:
            bool: True if cache is still valid
        """
        # Check if game IDs match
        if 'game_id' in original_df.columns and 'game_id' in cached_df.columns:
            orig_ids = set(original_df['game_id'])
            cached_ids = set(cached_df['game_id'])
            if orig_ids != cached_ids:
                return False
        
        # Check if quarters match
        if 'current_quarter' in original_df.columns and 'current_quarter' in cached_df.columns:
            for idx, row in original_df.iterrows():
                game_id = row.get('game_id')
                if game_id is not None:
                    orig_quarter = row.get('current_quarter')
                    cached_row = cached_df[cached_df['game_id'] == game_id]
                    
                    if not cached_row.empty:
                        cached_quarter = cached_row.iloc[0].get('current_quarter')
                        
                        # If quarter advanced, cache is invalid
                        if orig_quarter > cached_quarter:
                            return False
        
        return True
    
    def _calculate_all_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate all features using optimized methods.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with all calculated features
        """
        try:
            # Make a copy to avoid modifying the original
            features_df = df.copy()
            
            # Calculate features in order
            features_df = self._calculate_basic_features(features_df)
            features_df = self._calculate_momentum_features(features_df)
            features_df = self._calculate_rest_features(features_df)
            features_df = self._calculate_matchup_features(features_df)
            
            return features_df
            
        except Exception as e:
            print(f"Error calculating features: {str(e)}")
            traceback.print_exc()
            # Return original if calculation fails
            return df.copy()
    
    def _calculate_basic_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate basic features like score ratio."""
        # Calculate total score
        df['total_score'] = 0
        
        for idx, row in df.iterrows():
            home_score = float(row.get('home_score', 0) or 0)
            away_score = float(row.get('away_score', 0) or 0)
            df.at[idx, 'total_score'] = home_score + away_score
        
        # Calculate score ratio
        df['score_ratio'] = 0.5  # Default to even
        for idx, row in df.iterrows():
            home_score = float(row.get('home_score', 0) or 0)
            total = row.get('total_score', 0)
            
            if total > 0:
                df.at[idx, 'score_ratio'] = home_score / total
        
        return df
    
    def _calculate_momentum_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate momentum features using vectorized operations.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with momentum features
        """
        start_time = time.time()
        
        try:
            # Prepare arrays for vectorized operations
            n_rows = len(df)
            current_quarter = np.array(df['current_quarter'].fillna(0).astype(int))
            
            # Extract quarter scores as numpy arrays
            home_q1 = np.array([float(row.get('home_q1', 0) or 0) for _, row in df.iterrows()])
            home_q2 = np.array([float(row.get('home_q2', 0) or 0) for _, row in df.iterrows()])
            home_q3 = np.array([float(row.get('home_q3', 0) or 0) for _, row in df.iterrows()])
            home_q4 = np.array([float(row.get('home_q4', 0) or 0) for _, row in df.iterrows()])
            
            away_q1 = np.array([float(row.get('away_q1', 0) or 0) for _, row in df.iterrows()])
            away_q2 = np.array([float(row.get('away_q2', 0) or 0) for _, row in df.iterrows()])
            away_q3 = np.array([float(row.get('away_q3', 0) or 0) for _, row in df.iterrows()])
            away_q4 = np.array([float(row.get('away_q4', 0) or 0) for _, row in df.iterrows()])
            
            # Calculate quarter-to-quarter momentum shifts
            q1_to_q2_momentum = (home_q2 - home_q1) - (away_q2 - away_q1)
            q2_to_q3_momentum = (home_q3 - home_q2) - (away_q3 - away_q2)
            q3_to_q4_momentum = (home_q4 - home_q3) - (away_q4 - away_q3)
            
            # Initialize momentum arrays
            q1_to_q2_momentum_final = np.zeros(n_rows)
            q2_to_q3_momentum_final = np.zeros(n_rows)
            q3_to_q4_momentum_final = np.zeros(n_rows)
            cumulative_momentum = np.zeros(n_rows)
            
            # Apply masks based on current quarter
            q1_mask = current_quarter >= 2
            q1_to_q2_momentum_final[q1_mask] = q1_to_q2_momentum[q1_mask]
            
            q2_mask = current_quarter >= 3
            q2_to_q3_momentum_final[q2_mask] = q2_to_q3_momentum[q2_mask]
            
            q3_mask = current_quarter >= 4
            q3_to_q4_momentum_final[q3_mask] = q3_to_q4_momentum[q3_mask]
            
            # Calculate cumulative momentum based on quarter
            q2_only_mask = current_quarter == 2
            q3_only_mask = current_quarter == 3
            q4_only_mask = current_quarter >= 4
            
            # Weights for each quarter-to-quarter momentum
            weights = np.array([0.2, 0.3, 0.5])
            
            # Apply to each quarter scenario
            cumulative_momentum[q2_only_mask] = q1_to_q2_momentum_final[q2_only_mask]
            
            cumulative_momentum[q3_only_mask] = (
                q1_to_q2_momentum_final[q3_only_mask] * weights[0] +
                q2_to_q3_momentum_final[q3_only_mask] * weights[1]
            ) / (weights[0] + weights[1])
            
            cumulative_momentum[q4_only_mask] = (
                q1_to_q2_momentum_final[q4_only_mask] * weights[0] +
                q2_to_q3_momentum_final[q4_only_mask] * weights[1] +
                q3_to_q4_momentum_final[q4_only_mask] * weights[2]
            ) / np.sum(weights)
            
            # Normalize to [-1, 1]
            cumulative_momentum = np.clip(cumulative_momentum / 15.0, -1.0, 1.0)
            
            # Add results to DataFrame
            df['q1_to_q2_momentum'] = q1_to_q2_momentum_final
            df['q2_to_q3_momentum'] = q2_to_q3_momentum_final
            df['q3_to_q4_momentum'] = q3_to_q4_momentum_final
            df['cumulative_momentum'] = cumulative_momentum
            
        except Exception as e:
            print(f"Error in momentum calculation: {str(e)}")
            traceback.print_exc()
            
            # Initialize momentum features with zeros
            df['q1_to_q2_momentum'] = 0
            df['q2_to_q3_momentum'] = 0
            df['q3_to_q4_momentum'] = 0
            df['cumulative_momentum'] = 0
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['momentum_calculation'].append(elapsed)
        
        return df
    
    def _calculate_rest_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate rest-related features.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with rest features
        """
        start_time = time.time()
        
        try:
            # Set default values
            df['rest_days_home'] = 2
            df['rest_days_away'] = 2
            df['is_back_to_back_home'] = 0
            df['is_back_to_back_away'] = 0
            df['rest_advantage'] = 0
            
            # We'd need game schedule data to properly calculate rest
            # This would typically involve querying a database
            
            # For demonstration, we'll use default values
            # In a real implementation, you would look up previous games
            
            # Calculate rest advantage
            df['rest_advantage'] = df['rest_days_home'] - df['rest_days_away']
            
        except Exception as e:
            print(f"Error in rest calculation: {str(e)}")
            
            # Set defaults if calculation fails
            df['rest_days_home'] = 2
            df['rest_days_away'] = 2
            df['is_back_to_back_home'] = 0
            df['is_back_to_back_away'] = 0
            df['rest_advantage'] = 0
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['rest_calculation'].append(elapsed)
        
        return df
    
    def _calculate_matchup_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate matchup-related features.
        
        Args:
            df: Input DataFrame
            
        Returns:
            pd.DataFrame: DataFrame with matchup features
        """
        start_time = time.time()
        
        try:
            # Set default values
            df['prev_matchup_diff'] = 0
            
            # We'd need historical data to properly calculate matchup differentials
            # This would typically involve querying a database
            
            # For demonstration, we'll use default values
            # In a real implementation, you would look up previous matchups
            
        except Exception as e:
            print(f"Error in matchup calculation: {str(e)}")
            
            # Set defaults if calculation fails
            df['prev_matchup_diff'] = 0
        
        # Track timing
        elapsed = time.time() - start_time
        self.timing_stats['matchup_calculation'].append(elapsed)
        
        return df
    
    def get_performance_stats(self) -> Dict[str, Any]:
        """
        Get performance statistics for feature calculation.
        
        Returns:
            Dict with performance statistics
        """
        stats = {
            'cache_hits': self.cache_hits,
            'cache_misses': self.cache_misses,
            'cache_hit_rate': self.cache_hits / (self.cache_hits + self.cache_misses) if (self.cache_hits + self.cache_misses) > 0 else 0
        }
        
        # Add timing stats
        for feature, times in self.timing_stats.items():
            if times:
                stats[f'{feature}_avg_ms'] = np.mean(times) * 1000
                stats[f'{feature}_min_ms'] = np.min(times) * 1000
                stats[f'{feature}_max_ms'] = np.max(times) * 1000
        
        return stats
    
    def clear_cache(self, older_than_seconds: Optional[int] = None):
        """
        Clear the feature cache.
        
        Args:
            older_than_seconds: Only clear entries older than this many seconds
        """
        # Clear memory cache
        if older_than_seconds is None:
            # Clear all
            self.memory_cache = {}
            self.cache_timestamps = {}
        else:
            # Clear only old entries
            current_time = time.time()
            keys_to_remove = []
            
            for key, timestamp in self.cache_timestamps.items():
                if current_time - timestamp > older_than_seconds:
                    keys_to_remove.append(key)
            
            for key in keys_to_remove:
                del self.memory_cache[key]
                del self.cache_timestamps[key]
        
        # Clear disk cache
        for filename in os.listdir(self.cache_dir):
            if filename.startswith("feature_cache_") and filename.endswith(".pkl"):
                file_path = os.path.join(self.cache_dir, filename)
                
                if older_than_seconds is None:
                    # Remove all cache files
                    os.remove(file_path)
                else:
                    # Check timestamp
                    try:
                        with open(file_path, 'rb') as f:
                            cache_data = pickle.load(f)
                        
                        timestamp = cache_data.get('timestamp', 0)
                        if time.time() - timestamp > older_than_seconds:
                            os.remove(file_path)
                    except:
                        # Remove if can't read timestamp
                        os.remove(file_path)


class PredictionPipeline:
    """
    End-to-end prediction pipeline for benchmarking.
    
    This class implements a complete prediction pipeline that can be used
    to benchmark performance with various optimization strategies.
    """
    
    def __init__(self, use_optimization=False, model=None):
        """
        Initialize the prediction pipeline.
        
        Args:
            use_optimization: Whether to use optimized implementations
            model: Pre-loaded prediction model
        """
        self.use_optimization = use_optimization
        self.model = model
        
        # Create optimizer if needed
        if use_optimization:
            self.feature_calculator = OptimizedFeatureCalculator()
        
        # Initialize profiler for internal measurement
        self.profiler = PerformanceProfiler()
    
    @property
    def optimization_status(self):
        """Get the current optimization status string."""
        return "Optimized" if self.use_optimization else "Standard"
    
    def set_optimization(self, use_optimization):
        """Toggle optimization on/off."""
        self.use_optimization = use_optimization
        if use_optimization and not hasattr(self, 'feature_calculator'):
            self.feature_calculator = OptimizedFeatureCalculator()
    
    def load_test_data(self, sample_size=5):
        """
        Load test data for benchmarking.
        
        Args:
            sample_size: Number of games to load
            
        Returns:
            DataFrame with test data
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="load_test_data")
        def _load_data():
            # Use our pre-defined fallback function
            try:
                return fetch_recent_games_for_testing(limit=sample_size)
            except:
                # Fallback to generating synthetic data
                print("Using synthetic test data")
                return self._generate_synthetic_data(sample_size)
        
        return _load_data()
    
    def _generate_synthetic_data(self, sample_size):
        """Generate synthetic game data for testing."""
        import random
        
        synthetic_games = []
        
        for i in range(sample_size):
            # Generate a random quarter (1-4)
            quarter = random.randint(1, 4)
            
            # Generate quarter scores
            home_quarters = [random.randint(20, 35) for _ in range(quarter)]
            away_quarters = [random.randint(20, 35) for _ in range(quarter)]
            
            # Fill remaining quarters with zeros
            home_quarters.extend([0] * (4 - quarter))
            away_quarters.extend([0] * (4 - quarter))
            
            # Calculate current score
            home_score = sum(home_quarters)
            away_score = sum(away_quarters)
            
            # Create game dictionary
            game = {
                'game_id': 1000 + i,
                'home_team': f"Team{2*i}",
                'away_team': f"Team{2*i+1}",
                'current_quarter': quarter,
                'home_score': home_score,
                'away_score': away_score,
                'home_q1': home_quarters[0],
                'home_q2': home_quarters[1],
                'home_q3': home_quarters[2],
                'home_q4': home_quarters[3],
                'away_q1': away_quarters[0],
                'away_q2': away_quarters[1],
                'away_q3': away_quarters[2],
                'away_q4': away_quarters[3],
                'simulated': True
            }
            
            synthetic_games.append(game)
        
        return pd.DataFrame(synthetic_games)
    
    def prepare_features(self, games_df):
        """
        Prepare features for prediction.
        
        Args:
            games_df: DataFrame with game data
            
        Returns:
            DataFrame with features for prediction
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="prepare_features")
        def _prepare():
            if self.use_optimization:
                return self.feature_calculator.calculate_features(games_df)
            else:
                # Use our pre-defined fallback function
                try:
                    return prepare_features(games_df, self.model)
                except:
                    # Fallback to basic implementation
                    print("Using basic feature preparation")
                    return self._basic_feature_preparation(games_df)
        
        return _prepare()
    
    def _basic_feature_preparation(self, games_df):
        """Basic feature preparation implementation."""
        features_df = games_df.copy()
        
        # Ensure numeric types
        for col in games_df.columns:
            if col.startswith('home_q') or col.startswith('away_q') or col in ['home_score', 'away_score']:
                features_df[col] = pd.to_numeric(features_df[col], errors='coerce').fillna(0)
        
        # Calculate score ratio
        total_score = features_df['home_score'] + features_df['away_score']
        features_df['score_ratio'] = np.where(
            total_score > 0,
            features_df['home_score'] / total_score,
            0.5
        )
        
        # Add placeholder values for other features
        features_df['prev_matchup_diff'] = 0
        features_df['rolling_home_score'] = 105.0
        features_df['rolling_away_score'] = 105.0
        
        return features_df
    
    def make_predictions(self, features_df):
        """
        Make predictions using the model.
        
        Args:
            features_df: DataFrame with features
            
        Returns:
            DataFrame with predictions
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="make_predictions")
        def _predict():
            if self.model is None:
                return self._simulate_predictions(features_df)
            
            # Use our pre-defined fallback function
            try:
                return run_predictions(self.model, features_df)
            except Exception as e:
                print(f"Error using run_predictions: {e}")
                return self._basic_predictions(features_df)
        
        return _predict()
    
    def _simulate_predictions(self, features_df):
        """Simulate predictions when no model is available."""
        results = []
        
        for _, game in features_df.iterrows():
            # Extract game info
            game_id = game['game_id']
            home_team = game['home_team']
            away_team = game['away_team']
            current_quarter = int(game['current_quarter'])
            home_score = float(game['home_score'])
            away_score = float(game['away_score'])
            
            # Simple projection based on current score and quarter
            if current_quarter > 0:
                # Project final score based on current pace
                quarter_ratio = 4 / current_quarter
                predicted_home = home_score * quarter_ratio
                predicted_away = away_score * quarter_ratio
            else:
                # Pre-game prediction
                predicted_home = 105.0
                predicted_away = 102.0
            
            # Create result dictionary
            result = {
                'game_id': game_id,
                'home_team': home_team,
                'away_team': away_team,
                'current_quarter': current_quarter,
                'current_home_score': home_score,
                'current_away_score': away_score,
                'predicted_home_final': predicted_home,
                'predicted_away_final': predicted_away,
                'win_probability': 0.5 + (predicted_home - predicted_away) / 40.0  # Simple win probability
            }
            
            results.append(result)
        
        return pd.DataFrame(results)
    
    def _basic_predictions(self, features_df):
        """Basic prediction implementation when standard function isn't available."""
        if self.model is None:
            return self._simulate_predictions(features_df)
        
        # Get model features based on model type
        model_features = []
        
        if hasattr(self.model, 'feature_names_in_'):
            model_features = self.model.feature_names_in_
        elif hasattr(self.model, 'feature_importances_'):
            # Use the first 8 feature names as an approximation
            base_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio', 'rolling_home_score', 'rolling_away_score', 'prev_matchup_diff'
            ]
            model_features = base_features[:len(self.model.feature_importances_)]
        else:
            # Default basic features
            model_features = [
                'home_q1', 'home_q2', 'home_q3', 'home_q4', 
                'score_ratio'
            ]
        
        # Ensure all features exist
        for feature in model_features:
            if feature not in features_df.columns:
                features_df[feature] = 0
        
        # Make predictions
        X = features_df[model_features]
        predictions = self.model.predict(X)
        
        # Create results
        results_df = features_df.copy()
        results_df['predicted_home_final'] = predictions
        
        # Estimate away score based on home score and score ratio
        results_df['predicted_away_final'] = predictions * (1 - results_df['score_ratio']) / results_df['score_ratio']
        
        # Calculate win probability
        score_diff = results_df['predicted_home_final'] - results_df['predicted_away_final']
        results_df['win_probability'] = 1.0 / (1.0 + np.exp(-0.1 * score_diff))
        
        return results_df
    
    def post_process_predictions(self, predictions_df):
        """
        Apply post-processing to predictions.
        
        Args:
            predictions_df: DataFrame with raw predictions
            
        Returns:
            DataFrame with processed predictions
        """
        # Use decorator for profiling
        @self.profiler.profile_function(label="post_process_predictions")
        def _post_process():
            processed_df = predictions_df.copy()
            
            # Apply uncertainty estimation if optimized
            if self.use_optimization:
                return self._apply_optimized_post_processing(processed_df)
            else:
                return self._basic_post_processing(processed_df)
        
        return _post_process()
    
    def _apply_optimized_post_processing(self, predictions_df):
        """Apply optimized post-processing."""
        # Use our pre-defined fallback function
        try:
            return add_uncertainty_to_predictions(predictions_df)
        except:
            # Fallback to basic
            return self._basic_post_processing(predictions_df)
    
    def _basic_post_processing(self, predictions_df):
        """Apply basic post-processing to predictions."""
        # Ensure home prediction is at least current score
        predictions_df['predicted_home_final'] = np.maximum(
            predictions_df['predicted_home_final'],
            predictions_df['current_home_score']
        )
        
        # Ensure away prediction is at least current score
        if 'current_away_score' in predictions_df.columns:
            predictions_df['predicted_away_final'] = np.maximum(
                predictions_df['predicted_away_final'],
                predictions_df['current_away_score']
            )
        
        # Add simple confidence intervals
        quarter_width = {0: 30, 1: 25, 2: 20, 3: 15, 4: 10}
        
        predictions_df['home_lower_bound'] = predictions_df.apply(
            lambda row: row['predicted_home_final'] - quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        predictions_df['home_upper_bound'] = predictions_df.apply(
            lambda row: row['predicted_home_final'] + quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        predictions_df['away_lower_bound'] = predictions_df.apply(
            lambda row: row['predicted_away_final'] - quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        predictions_df['away_upper_bound'] = predictions_df.apply(
            lambda row: row['predicted_away_final'] + quarter_width.get(row['current_quarter'], 20) / 2,
            axis=1
        )
        
        return predictions_df
    
    def run_complete_pipeline(self, input_data=None, sample_size=5):
        """
        Run the complete prediction pipeline end-to-end.
        
        Args:
            input_data: Optional input data (loads test data if None)
            sample_size: Number of games to load if input_data is None
            
        Returns:
            DataFrame with complete prediction results
        """
        # Load data if not provided
        if input_data is None:
            games_df = self.load_test_data(sample_size)
        else:
            games_df = input_data
        
        if games_df is None or (isinstance(games_df, pd.DataFrame) and games_df.empty):
            print("No games available for processing")
            return pd.DataFrame()
        
        # Generate and process features
        features_df = self.prepare_features(games_df)
        
        # Make predictions
        predictions_df = self.make_predictions(features_df)
        
        # Post-process predictions
        final_predictions = self.post_process_predictions(predictions_df)
        
        return final_predictions


def benchmark_performance(compare_optimization=True, sample_sizes=[1, 5, 10], iterations=3):
    """
    Run comprehensive performance benchmarks.
    
    Args:
        compare_optimization: Whether to compare optimized vs. standard
        sample_sizes: List of sample sizes to test
        iterations: Number of iterations for each test
        
    Returns:
        Dictionary with benchmark results
    """
    print("\n" + "="*80)
    print(f"Running performance benchmarks ({iterations} iterations each)")
    print("="*80)
    
    # Initialize profiler
    profiler = PerformanceProfiler()
    
    # Try to load model - use our pre-defined fallback function
    model = None
    try:
        model = load_model()
        print("Successfully loaded prediction model")
    except Exception as e:
        print(f"Couldn't load model, will use simulated predictions: {e}")
    
    # Initialize pipelines
    standard_pipeline = PredictionPipeline(use_optimization=False, model=model)
    optimized_pipeline = PredictionPipeline(use_optimization=True, model=model)
    
    # Initialize results
    results = {
        'standard': {},
        'optimized': {},
        'comparison': {},
        'sample_sizes': sample_sizes,
        'iterations': iterations
    }
    
    # Run benchmarks for different sample sizes
    for sample_size in sample_sizes:
        print(f"\nTesting with {sample_size} games:")
        
        # Standard pipeline
        standard_times = []
        standard_memory = []
        
        for i in range(iterations):
            print(f"  Standard pipeline iteration {i+1}/{iterations}...")
            
            # Profile the complete pipeline
            _, profile = profiler.profile_pipeline(
                standard_pipeline.run_complete_pipeline,
                f"Standard_Size{sample_size}_Run{i+1}",
                None,
                sample_size
            )
            
            standard_times.append(profile['execution_time'])
            standard_memory.append(profile['memory_diff_mb'])
        
        # Store standard results
        results['standard'][sample_size] = {
            'avg_time': sum(standard_times) / len(standard_times),
            'min_time': min(standard_times),
            'max_time': max(standard_times),
            'avg_memory': sum(standard_memory) / len(standard_memory),
            'bottlenecks': profile.get('bottlenecks', [])
        }
        
        # Skip optimized if not comparing
        if not compare_optimization:
            continue
        
        # Optimized pipeline
        optimized_times = []
        optimized_memory = []
        
        for i in range(iterations):
            print(f"  Optimized pipeline iteration {i+1}/{iterations}...")
            
            # Profile the complete pipeline
            _, profile = profiler.profile_pipeline(
                optimized_pipeline.run_complete_pipeline,
                f"Optimized_Size{sample_size}_Run{i+1}",
                None,
                sample_size
            )
            
            optimized_times.append(profile['execution_time'])
            optimized_memory.append(profile['memory_diff_mb'])
        
        # Store optimized results
        results['optimized'][sample_size] = {
            'avg_time': sum(optimized_times) / len(optimized_times),
            'min_time': min(optimized_times),
            'max_time': max(optimized_times),
            'avg_memory': sum(optimized_memory) / len(optimized_memory),
            'bottlenecks': profile.get('bottlenecks', [])
        }
        
        # Calculate comparison metrics
        std_avg_time = results['standard'][sample_size]['avg_time']
        opt_avg_time = results['optimized'][sample_size]['avg_time']
        
        time_diff = std_avg_time - opt_avg_time
        time_pct = (time_diff / std_avg_time) * 100 if std_avg_time > 0 else 0
        
        results['comparison'][sample_size] = {
            'time_diff': time_diff,
            'time_pct_reduction': time_pct,
            'memory_diff': results['standard'][sample_size]['avg_memory'] - results['optimized'][sample_size]['avg_memory'],
            'speedup_factor': std_avg_time / opt_avg_time if opt_avg_time > 0 else 0
        }
        
        print(f"  Results for {sample_size} games:")
        print(f"    Standard: {std_avg_time:.3f}s")
        print(f"    Optimized: {opt_avg_time:.3f}s")
        print(f"    Improvement: {time_pct:.1f}% faster")
    
    # Visualize results
    if compare_optimization:
        fig = visualize_benchmark_results(results)
        plt.show()
    
    return results

def visualize_benchmark_results(results):
    """Create visualization of benchmark results."""
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Extract data for plotting
    sample_sizes = results['sample_sizes']
    
    std_times = [results['standard'][size]['avg_time'] for size in sample_sizes]
    std_err = [(results['standard'][size]['max_time'] - results['standard'][size]['min_time'])/2 
               for size in sample_sizes]
    
    if 'optimized' in results and results['optimized']:
        opt_times = [results['optimized'][size]['avg_time'] for size in sample_sizes]
        opt_err = [(results['optimized'][size]['max_time'] - results['optimized'][size]['min_time'])/2 
                  for size in sample_sizes]
        
        improvements = [results['comparison'][size]['time_pct_reduction'] for size in sample_sizes]
    
    # Plot execution times
    x = np.arange(len(sample_sizes))
    width = 0.35
    
    ax1.bar(x - width/2, std_times, width, label='Standard', color='#ff9999', 
           yerr=std_err, capsize=5)
    
    if 'optimized' in results and results['optimized']:
        ax1.bar(x + width/2, opt_times, width, label='Optimized', color='#66b3ff',
               yerr=opt_err, capsize=5)
    
    ax1.set_xlabel('Sample Size (number of games)')
    ax1.set_ylabel('Execution Time (seconds)')
    ax1.set_title('Pipeline Execution Time Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(sample_sizes)
    ax1.legend()
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add time labels
    for i, v in enumerate(std_times):
        ax1.text(i - width/2, v + 0.1, f"{v:.2f}s", ha='center')
    
    if 'optimized' in results and results['optimized']:
        for i, v in enumerate(opt_times):
            ax1.text(i + width/2, v + 0.1, f"{v:.2f}s", ha='center')
    
    # Plot improvement percentages if available
    if 'optimized' in results and results['optimized']:
        ax2.bar(sample_sizes, improvements, color='#99ff99')
        ax2.set_xlabel('Sample Size (number of games)')
        ax2.set_ylabel('Performance Improvement (%)')
        ax2.set_title('Optimization Improvement')
        ax2.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add percentage labels
        for i, v in enumerate(improvements):
            ax2.text(sample_sizes[i], v + 1, f"{v:.1f}%", ha='center')
    else:
        ax2.text(0.5, 0.5, 'No optimization comparison available', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    
    plt.tight_layout()
    return fig

# Run a benchmark demonstration
if __name__ == "__main__":
    # Create test data
    def create_test_data(num_games=5):
        """Create sample test data."""
        import random
        
        test_games = []
        for i in range(num_games):
            quarter = random.randint(1, 4)
            game = {
                'game_id': 1000 + i,
                'home_team': f"Team{i*2}",
                'away_team': f"Team{i*2+1}",
                'current_quarter': quarter,
                'home_q1': random.randint(20, 35) if quarter >= 1 else 0,
                'home_q2': random.randint(20, 35) if quarter >= 2 else 0,
                'home_q3': random.randint(20, 35) if quarter >= 3 else 0,
                'home_q4': random.randint(20, 35) if quarter >= 4 else 0,
                'away_q1': random.randint(20, 35) if quarter >= 1 else 0,
                'away_q2': random.randint(20, 35) if quarter >= 2 else 0,
                'away_q3': random.randint(20, 35) if quarter >= 3 else 0,
                'away_q4': random.randint(20, 35) if quarter >= 4 else 0
            }
            
            # Calculate current scores
            game['home_score'] = sum([game.get(f'home_q{q}', 0) for q in range(1, quarter+1)])
            game['away_score'] = sum([game.get(f'away_q{q}', 0) for q in range(1, quarter+1)])
            
            test_games.append(game)
        
        return pd.DataFrame(test_games)
    
    # Define a test pipeline
    def test_pipeline():
        """Run a simple test pipeline."""
        # Create test data
        test_data = create_test_data(num_games=5)
        
        # Create pipeline
        pipeline = PredictionPipeline(use_optimization=True)
        
        # Run quick benchmark
        profiler = PerformanceProfiler()
        _, profile = profiler.profile_pipeline(
            pipeline.run_complete_pipeline,
            "Sample_Benchmark",
            test_data
        )
        
        # Visualize profile
        fig = profiler.visualize_profile(profile)
        plt.show()
        
        # Run a more comprehensive benchmark
        print("\nRunning comprehensive benchmark...")
        benchmark_results = benchmark_performance(
            compare_optimization=True,
            sample_sizes=[1, 5, 10],
            iterations=2
        )
        
        print("\nBenchmark Summary:")
        for size in benchmark_results['sample_sizes']:
            if 'comparison' in benchmark_results and size in benchmark_results['comparison']:
                comparison = benchmark_results['comparison'][size]
                print(f"Sample size {size}: {comparison['time_pct_reduction']:.1f}% improvement, " 
                      f"{comparison['speedup_factor']:.2f}x speedup")
    
    test_pipeline()

In [ ]:
# Cell 29: Modular NBA Game Prediction Monitoring System

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import datetime
import time
import json
import os
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
import logging
from functools import wraps
import pytz
from datetime import datetime, timedelta

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("nba_monitoring")

# Import Supabase client (assumes correct configuration)
from caching.supabase_client import supabase

# --------------------------------------------------------------------
# HELPER: Fetch live game data from Supabase (mirroring Cells 1/3)
# --------------------------------------------------------------------

def fetch_live_games_from_supabase() -> pd.DataFrame:
    """
    Pull all rows from the 'nba_live_game_stats' table,
    convert game_date to Pacific Time, filter for today's games,
    and then select only rows where the stored status (uppercased)
    is in the live set {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}.
    Also adds a 'game_status' column set to 'live'.
    """
    try:
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if not response.data:
            print("No live game data available from Supabase.")
            return pd.DataFrame()
        
        df = pd.DataFrame(response.data)
        if "game_date" in df.columns:
            df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce", utc=True)
            pacific_tz = pytz.timezone("America/Los_Angeles")
            df["game_date_pt"] = df["game_date"].dt.tz_convert(pacific_tz)
            df["date_only_pt"] = df["game_date_pt"].dt.date
            now_pt = datetime.now(pacific_tz)
            today_pt = now_pt.date()
            df = df[df["date_only_pt"] == today_pt].copy()
            print(f"Found {len(df)} games scheduled for today in PT.")
        else:
            print("Column 'game_date' not found in Supabase data!")
        
        if "status" in df.columns:
            df["status"] = df["status"].astype(str).str.upper()
        else:
            df["status"] = ""
        
        live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
        active_df = df[df["status"].isin(live_statuses)].copy()
        # Add a new column 'game_status' set to 'live' for these filtered rows
        active_df["game_status"] = "live"
        print(f"Fetched {len(active_df)} active live games from Supabase.")
        return active_df
    except Exception as e:
        print(f"Error fetching live games from Supabase: {e}")
        return pd.DataFrame()

# ====================================================================
# DATA MANAGEMENT LAYER
# ====================================================================

class DataStore:
    """
    Manages storage and retrieval of prediction data and history.
    """
    def __init__(self, storage_dir: str = "./prediction_history"):
        self.storage_dir = storage_dir
        os.makedirs(storage_dir, exist_ok=True)
        self.recent_predictions = {}
        self.prediction_history = {}
        self._load_history()
        
    def _load_history(self):
        try:
            history_files = [f for f in os.listdir(self.storage_dir) if f.endswith('.json')]
            for file in history_files:
                try:
                    game_id = file.split('_')[1].split('.')[0]
                    file_path = os.path.join(self.storage_dir, file)
                    with open(file_path, 'r') as f:
                        game_history = json.load(f)
                    self.prediction_history[game_id] = game_history
                except Exception as e:
                    logger.warning(f"Error loading history file {file}: {str(e)}")
            logger.info(f"Loaded prediction history for {len(self.prediction_history)} games")
        except Exception as e:
            logger.error(f"Failed to load prediction history: {str(e)}")
    
    def save_prediction(self, prediction: Dict[str, Any]) -> bool:
        try:
            game_id = str(prediction.get('game_id'))
            if not game_id:
                logger.warning("Prediction missing game_id, cannot save")
                return False
            if 'timestamp' not in prediction:
                prediction['timestamp'] = datetime.now().isoformat()
            self.recent_predictions[game_id] = prediction
            if game_id not in self.prediction_history:
                self.prediction_history[game_id] = []
            self.prediction_history[game_id].append(prediction)
            self._save_game_history(game_id)
            return True
        except Exception as e:
            logger.error(f"Failed to save prediction: {str(e)}")
            return False
    
    def _save_game_history(self, game_id: str) -> bool:
        try:
            if game_id not in self.prediction_history:
                return False
            file_path = os.path.join(self.storage_dir, f"game_{game_id}.json")
            history = self.prediction_history[game_id]
            for pred in history:
                for key, value in pred.items():
                    if hasattr(value, "isoformat"):
                        pred[key] = value.isoformat()
            with open(file_path, 'w') as f:
                json.dump(history, f)
            return True
        except Exception as e:
            logger.error(f"Failed to save game history for {game_id}: {str(e)}")
            return False
    
    def get_game_history(self, game_id: str) -> List[Dict[str, Any]]:
        return self.prediction_history.get(game_id, [])
    
    def get_recent_prediction(self, game_id: str) -> Optional[Dict[str, Any]]:
        return self.recent_predictions.get(game_id)
    
    def get_all_recent_predictions(self) -> Dict[str, Dict[str, Any]]:
        return self.recent_predictions
    
    def clear_old_predictions(self, max_age_days: int = 7) -> int:
        cleared_count = 0
        cutoff_date = datetime.now() - timedelta(days=max_age_days)
        cutoff_str = cutoff_date.isoformat()
        for game_id in list(self.prediction_history.keys()):
            game_predictions = self.prediction_history[game_id]
            all_old = True
            for pred in game_predictions:
                if pred.get('timestamp', '') > cutoff_str:
                    all_old = False
                    break
            if all_old:
                del self.prediction_history[game_id]
                file_path = os.path.join(self.storage_dir, f"game_{game_id}.json")
                if os.path.exists(file_path):
                    os.remove(file_path)
                cleared_count += 1
        return cleared_count

# ====================================================================
# ALERT MANAGEMENT LAYER
# ====================================================================

class AlertManager:
    """
    Manages alerts and notifications for the monitoring system.
    """
    def __init__(self, data_store: DataStore, max_alerts: int = 100):
        self.data_store = data_store
        self.max_alerts = max_alerts
        self.alerts = []
        self.alert_thresholds = {
            'score_change': 15.0,
            'win_prob_change': 0.25,
            'unusual_momentum': 0.7,
            'prediction_error': 20.0,
            'confidence_mismatch': 0.3
        }
    
    def check_for_alerts(self, game_id: str) -> List[Dict[str, Any]]:
        """
        Check for alert conditions for a specific game.
        """
        game_history = self.data_store.get_game_history(game_id)
        if len(game_history) < 2:
            return []
        current = game_history[-1]
        previous = game_history[-2]
        new_alerts = []
        try:
            home_score_change = abs(
                current.get('predicted_home_score', current.get('predicted_home_final', 0)) - 
                previous.get('predicted_home_score', previous.get('predicted_home_final', 0))
            )
            away_score_change = abs(
                current.get('predicted_away_score', current.get('predicted_away_final', 0)) - 
                previous.get('predicted_away_score', previous.get('predicted_away_final', 0))
            )
            if home_score_change > self.alert_thresholds['score_change'] or away_score_change > self.alert_thresholds['score_change']:
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'game_id': game_id,
                    'alert_type': 'significant_score_change',
                    'message': f"Significant change in predicted score: Home {home_score_change:.1f} pts, Away {away_score_change:.1f} pts",
                    'severity': 'medium',
                    'data': {
                        'home_change': home_score_change,
                        'away_change': away_score_change,
                        'current': {
                            'home': current.get('predicted_home_score', current.get('predicted_home_final', 0)),
                            'away': current.get('predicted_away_score', current.get('predicted_away_final', 0))
                        },
                        'previous': {
                            'home': previous.get('predicted_home_score', previous.get('predicted_home_final', 0)),
                            'away': previous.get('predicted_away_score', previous.get('predicted_away_final', 0))
                        }
                    }
                }
                new_alerts.append(alert)
            
            win_prob_change = abs(
                current.get('win_probability', 0.5) - previous.get('win_probability', 0.5)
            )
            if win_prob_change > self.alert_thresholds['win_prob_change']:
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'game_id': game_id,
                    'alert_type': 'significant_win_prob_change',
                    'message': f"Win probability changed by {win_prob_change*100:.1f}% points",
                    'severity': 'high' if win_prob_change > 0.4 else 'medium',
                    'data': {
                        'win_prob_change': win_prob_change,
                        'current': current.get('win_probability', 0.5),
                        'previous': previous.get('win_probability', 0.5),
                        'direction': 'increased' if current.get('win_probability', 0.5) > previous.get('win_probability', 0.5) else 'decreased'
                    }
                }
                new_alerts.append(alert)
            
            momentum = abs(current.get('momentum_shift', current.get('cumulative_momentum', 0)))
            if momentum > self.alert_thresholds['unusual_momentum']:
                direction = "positive" if current.get('momentum_shift', current.get('cumulative_momentum', 0)) > 0 else "negative"
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'game_id': game_id,
                    'alert_type': 'unusual_momentum',
                    'message': f"Unusually high {direction} momentum: {momentum:.2f}",
                    'severity': 'medium',
                    'data': {
                        'momentum': current.get('momentum_shift', current.get('cumulative_momentum', 0)),
                        'threshold': self.alert_thresholds['unusual_momentum'],
                        'direction': direction
                    }
                }
                new_alerts.append(alert)
            
            if 'actual_home_score' in current and 'actual_away_score' in current:
                home_error = abs(
                    current.get('predicted_home_score', current.get('predicted_home_final', 0)) - current.get('actual_home_score', 0)
                )
                away_error = abs(
                    current.get('predicted_away_score', current.get('predicted_away_final', 0)) - current.get('actual_away_score', 0)
                )
                avg_error = (home_error + away_error) / 2
                if avg_error > self.alert_thresholds['prediction_error']:
                    alert = {
                        'timestamp': datetime.now().isoformat(),
                        'game_id': game_id,
                        'alert_type': 'high_prediction_error',
                        'message': f"High prediction error: {avg_error:.1f} points average",
                        'severity': 'high' if avg_error > 25 else 'medium',
                        'data': {
                            'home_error': home_error,
                            'away_error': away_error,
                            'avg_error': avg_error
                        }
                    }
                    new_alerts.append(alert)
            
            if 'prediction_confidence' in current and 'actual_home_score' in current and 'actual_away_score' in current:
                confidence = current.get('prediction_confidence', 0.8)
                actual_home = current.get('actual_home_score', 0)
                actual_away = current.get('actual_away_score', 0)
                predicted_home = current.get('predicted_home_score', current.get('predicted_home_final', 0))
                predicted_away = current.get('predicted_away_score', current.get('predicted_away_final', 0))
                actual_winner = 'home' if actual_home > actual_away else 'away'
                predicted_winner = 'home' if predicted_home > predicted_away else 'away'
                winner_correct = int(actual_winner == predicted_winner)
                confidence_mismatch = (confidence > 0.8 and not winner_correct) or (confidence < 0.5 and winner_correct)
                confidence_gap = confidence if not winner_correct else (1 - confidence) if winner_correct else 0
                if confidence_mismatch and confidence_gap > self.alert_thresholds['confidence_mismatch']:
                    alert = {
                        'timestamp': datetime.now().isoformat(),
                        'game_id': game_id,
                        'alert_type': 'confidence_mismatch',
                        'message': f"{'High confidence wrong prediction' if not winner_correct else 'Low confidence correct prediction'}",
                        'severity': 'medium',
                        'data': {
                            'confidence': confidence,
                            'winner_correct': winner_correct,
                            'confidence_gap': confidence_gap
                        }
                    }
                    new_alerts.append(alert)
            
            self.add_alerts(new_alerts)
        except Exception as e:
            logger.error(f"Error in check_for_alerts for game {game_id}: {e}")
        return new_alerts
    
    def add_alerts(self, alerts: List[Dict[str, Any]]):
        self.alerts.extend(alerts)
        if len(self.alerts) > self.max_alerts:
            self.alerts = self.alerts[-self.max_alerts:]
    
    def get_recent_alerts(self, limit: int = 10) -> List[Dict[str, Any]]:
        sorted_alerts = sorted(self.alerts, key=lambda x: x.get('timestamp', ''), reverse=True)
        return sorted_alerts[:limit]
    
    def get_alerts_by_game(self, game_id: str) -> List[Dict[str, Any]]:
        return [alert for alert in self.alerts if alert.get('game_id') == game_id]
    
    def set_alert_threshold(self, alert_type: str, value: float):
        if alert_type in self.alert_thresholds:
            self.alert_thresholds[alert_type] = value

# ====================================================================
# ANALYTICS LAYER
# ====================================================================

class PredictionAnalyzer:
    """
    Analyzes prediction data to generate insights and metrics.
    """
    def __init__(self, data_store: DataStore):
        self.data_store = data_store
        self.accuracy_metrics = {
            'by_quarter': {q: {'count': 0, 'mae': [], 'winner_accuracy': []} for q in range(5)},
            'by_context': {
                'close_game': {'count': 0, 'mae': [], 'winner_accuracy': []},
                'blowout': {'count': 0, 'mae': [], 'winner_accuracy': []},
                'high_momentum': {'count': 0, 'mae': [], 'winner_accuracy': []},
                'final_minutes': {'count': 0, 'mae': [], 'winner_accuracy': []}
            },
            'overall': {'count': 0, 'mae': [], 'winner_accuracy': []}
        }
    # (Other PredictionAnalyzer methods remain unchanged)
    def update_accuracy_metrics(self, game_id: str, actual_results: Dict[str, Any]) -> Dict[str, Any]:
        history = self.data_store.get_game_history(game_id)
        if not history:
            return {'status': 'error', 'message': 'No prediction history found'}
        update_summary = {
            'game_id': game_id,
            'predictions_processed': 0,
            'actual_home_score': actual_results.get('home_score', 0),
            'actual_away_score': actual_results.get('away_score', 0)
        }
        for prediction in history:
            quarter = prediction.get('current_quarter', 0)
            contexts = []
            home_score = prediction.get('home_score', prediction.get('current_home_score', 0))
            away_score = prediction.get('away_score', prediction.get('current_away_score', 0))
            score_diff = abs(home_score - away_score)
            if score_diff <= 5:
                contexts.append('close_game')
            if score_diff >= 20:
                contexts.append('blowout')
            momentum = abs(prediction.get('momentum_shift', prediction.get('cumulative_momentum', 0)))
            if momentum >= 0.6:
                contexts.append('high_momentum')
            if quarter == 4 and prediction.get('time_remaining', 12) <= 5:
                contexts.append('final_minutes')
            predicted_home = prediction.get('predicted_home_score', prediction.get('predicted_home_final', 0))
            predicted_away = prediction.get('predicted_away_score', prediction.get('predicted_away_final', 0))
            actual_home = actual_results.get('home_score', 0)
            actual_away = actual_results.get('away_score', 0)
            home_error = abs(predicted_home - actual_home)
            away_error = abs(predicted_away - actual_away)
            mae = (home_error + away_error) / 2
            actual_winner = 'home' if actual_home > actual_away else 'away'
            predicted_winner = 'home' if predicted_home > predicted_away else 'away'
            winner_correct = int(actual_winner == predicted_winner)
            if 0 <= quarter <= 4:
                self.accuracy_metrics['by_quarter'][quarter]['mae'].append(mae)
                self.accuracy_metrics['by_quarter'][quarter]['winner_accuracy'].append(winner_correct)
                self.accuracy_metrics['by_quarter'][quarter]['count'] += 1
            for context in contexts:
                if context in self.accuracy_metrics['by_context']:
                    self.accuracy_metrics['by_context'][context]['mae'].append(mae)
                    self.accuracy_metrics['by_context'][context]['winner_accuracy'].append(winner_correct)
                    self.accuracy_metrics['by_context'][context]['count'] += 1
            self.accuracy_metrics['overall']['mae'].append(mae)
            self.accuracy_metrics['overall']['winner_accuracy'].append(winner_correct)
            self.accuracy_metrics['overall']['count'] += 1
            update_summary['predictions_processed'] += 1
        if self.accuracy_metrics['overall']['mae']:
            update_summary['average_error'] = sum(self.accuracy_metrics['overall']['mae']) / len(self.accuracy_metrics['overall']['mae'])
        if self.accuracy_metrics['overall']['winner_accuracy']:
            update_summary['winner_accuracy'] = sum(self.accuracy_metrics['overall']['winner_accuracy']) / len(self.accuracy_metrics['overall']['winner_accuracy'])
        return update_summary
    
    def get_accuracy_summary(self) -> Dict[str, Any]:
        summary = {
            'by_quarter': {},
            'by_context': {},
            'overall': {},
            'count': self.accuracy_metrics['overall']['count']
        }
        for quarter, metrics in self.accuracy_metrics['by_quarter'].items():
            if metrics['mae']:
                summary['by_quarter'][quarter] = {
                    'mae': sum(metrics['mae']) / len(metrics['mae']),
                    'winner_accuracy': sum(metrics['winner_accuracy']) / len(metrics['winner_accuracy']),
                    'sample_size': len(metrics['mae']),
                    'count': metrics['count']
                }
        for context, metrics in self.accuracy_metrics['by_context'].items():
            if metrics['mae']:
                summary['by_context'][context] = {
                    'mae': sum(metrics['mae']) / len(metrics['mae']),
                    'winner_accuracy': sum(metrics['winner_accuracy']) / len(metrics['winner_accuracy']),
                    'sample_size': len(metrics['mae']),
                    'count': metrics['count']
                }
        overall = self.accuracy_metrics['overall']
        if overall['mae']:
            summary['overall'] = {
                'mae': sum(overall['mae']) / len(overall['mae']),
                'winner_accuracy': sum(overall['winner_accuracy']) / len(overall['winner_accuracy']),
                'sample_size': len(overall['mae']),
                'count': overall['count']
            }
        return summary
    
    def find_similar_games(self, prediction: Dict[str, Any], max_games: int = 5) -> List[Dict[str, Any]]:
        similar_games = []
        game_id = prediction.get('game_id', '')
        home_team = prediction.get('home_team', '')
        away_team = prediction.get('away_team', '')
        current_quarter = prediction.get('current_quarter', 0)
        score_diff = abs(
            prediction.get('home_score', prediction.get('current_home_score', 0)) - 
            prediction.get('away_score', prediction.get('current_away_score', 0))
        )
        momentum = abs(prediction.get('momentum_shift', prediction.get('cumulative_momentum', 0)))
        for hist_id, history in self.data_store.prediction_history.items():
            if hist_id == game_id:
                continue
            final_pred = history[-1] if history else None
            if not final_pred or 'actual_home_score' not in final_pred:
                continue
            similar_quarter_pred = None
            for pred in history:
                if pred.get('current_quarter', 0) == current_quarter:
                    similar_quarter_pred = pred
                    break
            if not similar_quarter_pred:
                continue
            similarity_factors = []
            teams_match = ((similar_quarter_pred.get('home_team', '') == home_team and 
                           similar_quarter_pred.get('away_team', '') == away_team) or
                          (similar_quarter_pred.get('home_team', '') == away_team and 
                           similar_quarter_pred.get('away_team', '') == home_team))
            if teams_match:
                similarity_factors.append(3.0)
            hist_score_diff = abs(
                similar_quarter_pred.get('home_score', similar_quarter_pred.get('current_home_score', 0)) - 
                similar_quarter_pred.get('away_score', similar_quarter_pred.get('current_away_score', 0))
            )
            score_diff_similarity = 1.0 - min(abs(score_diff - hist_score_diff) / 20.0, 1.0)
            similarity_factors.append(score_diff_similarity * 2.0)
            hist_momentum = abs(similar_quarter_pred.get('momentum_shift', similar_quarter_pred.get('cumulative_momentum', 0)))
            momentum_similarity = 1.0 - min(abs(momentum - hist_momentum) / 1.0, 1.0)
            similarity_factors.append(momentum_similarity)
            similarity = sum(similarity_factors) / len(similarity_factors) if similarity_factors else 0
            if similarity >= 0.6:
                actual_home = final_pred.get('actual_home_score', 0)
                actual_away = final_pred.get('actual_away_score', 0)
                predicted_home = similar_quarter_pred.get('predicted_home_score', similar_quarter_pred.get('predicted_home_final', 0))
                predicted_away = similar_quarter_pred.get('predicted_away_score', similar_quarter_pred.get('predicted_away_final', 0))
                home_error = predicted_home - actual_home
                away_error = predicted_away - actual_away
                similar_games.append({
                    'game_id': hist_id,
                    'similarity': similarity,
                    'home_team': similar_quarter_pred.get('home_team', ''),
                    'away_team': similar_quarter_pred.get('away_team', ''),
                    'quarter': similar_quarter_pred.get('current_quarter', 0),
                    'prediction': {
                        'home': predicted_home,
                        'away': predicted_away
                    },
                    'actual': {
                        'home': actual_home,
                        'away': actual_away
                    },
                    'error': {
                        'home': home_error,
                        'away': away_error,
                        'avg': (abs(home_error) + abs(away_error)) / 2
                    }
                })
        similar_games.sort(key=lambda x: x['similarity'], reverse=True)
        return similar_games[:max_games]
    
    def get_prediction_evolution(self, game_id: str) -> Dict[str, Any]:
        history = self.data_store.get_game_history(game_id)
        if not history:
            return {'game_id': game_id, 'status': 'error', 'message': 'No prediction history found'}
        timestamps = []
        quarters = []
        home_scores = []
        away_scores = []
        win_probs = []
        confidence_values = []
        for pred in history:
            ts = pred.get('timestamp', '')
            if isinstance(ts, str) and 'T' in ts:
                ts = ts.split('T')[1].split('.')[0]
            timestamps.append(ts)
            quarters.append(pred.get('current_quarter', 0))
            home_scores.append(pred.get('predicted_home_score', pred.get('predicted_home_final', 0)))
            away_scores.append(pred.get('predicted_away_score', pred.get('predicted_away_final', 0)))
            win_probs.append(pred.get('win_probability', 0.5))
            confidence = pred.get('prediction_confidence', None)
            confidence_values.append(confidence)
        actuals = None
        if history and 'actual_home_score' in history[-1]:
            actuals = {
                'home_score': history[-1].get('actual_home_score', 0),
                'away_score': history[-1].get('actual_away_score', 0)
            }
        return {
            'game_id': game_id,
            'home_team': history[0].get('home_team', '') if history else '',
            'away_team': history[0].get('away_team', '') if history else '',
            'timestamps': timestamps,
            'quarters': quarters,
            'home_scores': home_scores,
            'away_scores': away_scores,
            'win_probabilities': win_probs,
            'confidence_values': confidence_values,
            'count': len(history),
            'actual_results': actuals
        }

# ====================================================================
# VISUALIZATION LAYER
# ====================================================================

class DashboardTemplateEngine:
    """
    Handles HTML template management and rendering.
    """
    def __init__(self):
        self.global_css = """
            /* Global CSS styles */
            .dashboard { font-family: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; max-width: 1200px; margin: 0 auto; }
            .dashboard-header { text-align: center; padding: 1rem; margin-bottom: 1rem; background: #0d6efd; color: white; border-radius: 0.5rem; }
            .dashboard-section { margin-bottom: 2rem; }
            .section-header { border-bottom: 2px solid #dee2e6; padding-bottom: 0.5rem; margin-bottom: 1rem; display: flex; justify-content: space-between; align-items: center; }
            .section-header h2 { margin: 0; font-size: 1.5rem; }
            .game-cards { display: grid; grid-template-columns: repeat(auto-fill, minmax(350px, 1fr)); gap: 1rem; }
            .game-card { border: 1px solid #dee2e6; border-radius: 0.5rem; padding: 1rem; margin-bottom: 1rem; background: white; box-shadow: 0 0.125rem 0.25rem rgba(0,0,0,0.075); }
            .alert-list { border: 1px solid #dee2e6; border-radius: 0.5rem; padding: 1rem; }
            .alert-item { padding: 0.5rem; margin-bottom: 0.5rem; border-radius: 0.25rem; }
            .alert-high { background-color: #f8d7da; border-left: 4px solid #dc3545; }
            .alert-medium { background-color: #fff3cd; border-left: 4px solid #ffc107; }
            .alert-low { background-color: #d1e7dd; border-left: 4px solid #198754; }
        """
        self.dashboard_js = """
            function switchTab(tabId) {
                document.querySelectorAll('.tab-content').forEach(tab => { tab.classList.remove('active'); });
                document.querySelectorAll('.tab').forEach(tab => { tab.classList.remove('active'); });
                document.getElementById(tabId).classList.add('active');
                document.querySelectorAll('.tab').forEach(tab => {
                    if (tab.getAttribute('onclick').includes(tabId)) { tab.classList.add('active'); }
                });
            }
            document.addEventListener('DOMContentLoaded', function() { /* initialize charts if needed */ });
        """
    
    def render_header(self, title="NBA Prediction Monitoring Dashboard"):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        return f"""
        <div class="dashboard-header">
            <h1>{title}</h1>
            <p>Updated: {timestamp}</p>
        </div>
        """
    
    def render_active_games(self, games_df):
        if games_df.empty:
            return """
            <div class="dashboard-section">
                <div class="section-header">
                    <h2>Active Games</h2>
                </div>
                <p>No active games at this time.</p>
            </div>
            """
        html = """
        <div class="dashboard-section">
            <div class="section-header">
                <h2>Active Games</h2>
                <span class="update-status">Live updates</span>
            </div>
            <div class="game-cards">
        """
        for _, game in games_df.iterrows():
            game_id = game.get('game_id', '')
            home_team = game.get('home_team', '')
            away_team = game.get('away_team', '')
            current_quarter = game.get('current_quarter', 0)
            game_status = game.get('game_status', 'SCHEDULED')
            status_class = ""
            if game_status.upper() == "LIVE":
                status_class = "bg-danger text-white"
                status_text = f"LIVE Q{current_quarter}"
            elif game_status.upper() == "FINAL":
                status_class = "bg-success text-white"
                status_text = "FINAL"
            else:
                status_class = "bg-secondary text-white"
                status_text = "SCHEDULED"
            home_score = game.get('home_score', 0)
            away_score = game.get('away_score', 0)
            predicted_home = game.get('predicted_home_score', 0)
            predicted_away = game.get('predicted_away_score', 0)
            win_prob = game.get('win_probability', 0.5)
            win_prob_text = f"{win_prob*100:.1f}%"
            confidence = game.get('prediction_confidence', None)
            confidence_text = f"{confidence*100:.1f}%" if confidence is not None else "N/A"
            html += f"""
            <div class="game-card" data-game-id="{game_id}">
                <div class="d-flex justify-content-between align-items-center mb-2">
                    <h5 class="mb-0">{home_team} vs {away_team}</h5>
                    <span class="badge rounded-pill {status_class} px-3 py-2">{status_text}</span>
                </div>
                <div class="d-flex justify-content-between align-items-center mb-3">
                    <div class="text-center">
                        <div style="font-weight: bold; font-size: 1.5rem;">{home_score}</div>
                        <div style="color: #666;">{home_team}</div>
                    </div>
                    <div class="text-center">
                        <div style="font-weight: bold;">vs</div>
                    </div>
                    <div class="text-center">
                        <div style="font-weight: bold; font-size: 1.5rem;">{away_score}</div>
                        <div style="color: #666;">{away_team}</div>
                    </div>
                </div>
                <div class="mt-3">
                    <div class="d-flex justify-content-between text-muted">
                        <span>Predicted Final</span>
                        <span>Win Probability</span>
                    </div>
                    <div class="d-flex justify-content-between">
                        <strong>{predicted_home:.1f} - {predicted_away:.1f}</strong>
                        <strong>{win_prob_text}</strong>
                    </div>
                    <div class="progress mt-2" style="height: 8px;">
                        <div class="progress-bar" role="progressbar" style="width: {win_prob*100}%"></div>
                    </div>
                </div>
                <div class="mt-3 text-muted d-flex justify-content-between">
                    <small>Confidence: {confidence_text}</small>
                    <small>Game ID: {game_id}</small>
                </div>
            </div>
            """
        html += """
            </div>
        </div>
        """
        return html
    
    def render_alerts(self, alerts):
        if not alerts:
            return """
            <div class="dashboard-section">
                <div class="section-header">
                    <h2>Alerts & Notifications</h2>
                </div>
                <p>No alerts at this time.</p>
            </div>
            """
        html = """
        <div class="dashboard-section">
            <div class="section-header">
                <h2>Alerts & Notifications</h2>
            </div>
            <div class="alert-list">
        """
        for alert in alerts:
            severity = alert.get('severity', 'medium')
            alert_class = f"alert-{severity}"
            timestamp = alert.get('timestamp', '')
            if isinstance(timestamp, str) and 'T' in timestamp:
                timestamp = timestamp.split('T')[1].split('.')[0]
            html += f"""
            <div class="alert-item {alert_class}">
                <div>
                    <strong>{alert.get('alert_type', 'Alert').replace('_', ' ').title()}</strong> - {alert.get('message', '')}
                </div>
                <div style="font-size:0.8rem; color:#666">
                    Game: {alert.get('game_id', '')}, Time: {timestamp}
                </div>
            </div>
            """
        html += """
            </div>
        </div>
        """
        return html
    
    def render_historical_comparisons(self, active_games_df, prediction_analyzer):
        if active_games_df.empty:
            return """
            <div class="dashboard-section">
                <div class="section-header">
                    <h2>Historical Context & Performance</h2>
                </div>
                <p>No active games to compare.</p>
            </div>
            """
        html = """
        <div class="dashboard-section">
            <div class="section-header">
                <h2>Historical Context & Performance</h2>
            </div>
        """
        html += """<div class="tabs">"""
        for i, (_, game) in enumerate(active_games_df.iterrows()):
            game_id = game.get('game_id', i)
            home_team = game.get('home_team', '')
            away_team = game.get('away_team', '')
            active_class = "active" if i == 0 else ""
            html += f"""
            <div class="tab {active_class}" onclick="switchTab('game-tab-{game_id}')">
                {home_team} vs {away_team}
            </div>
            """
        html += """</div>"""
        for i, (_, game) in enumerate(active_games_df.iterrows()):
            game_id = str(game.get('game_id', i))
            active_class = "active" if i == 0 else ""
            html += f"""
            <div id="game-tab-{game_id}" class="tab-content {active_class}">
            """
            accuracy_summary = prediction_analyzer.get_accuracy_summary()
            html += """
            <div class="prediction-metrics">
            """
            quarter = int(game.get('current_quarter', 0))
            quarter_metrics = accuracy_summary.get('by_quarter', {}).get(quarter, {})
            quarter_mae = quarter_metrics.get('mae', 0)
            quarter_accuracy = quarter_metrics.get('winner_accuracy', 0)
            quarter_count = quarter_metrics.get('count', 0)
            overall_metrics = accuracy_summary.get('overall', {})
            overall_mae = overall_metrics.get('mae', 0)
            overall_accuracy = overall_metrics.get('winner_accuracy', 0)
            html += f"""
            <div class="metric-card">
                <div class="metric-label">Quarter {quarter} Error</div>
                <div class="metric-value">{quarter_mae:.1f}</div>
                <div class="metric-label">points (avg)</div>
                <div class="metric-label">Sample: {quarter_count}</div>
            </div>
            
            <div class="metric-card">
                <div class="metric-label">Quarter {quarter} Accuracy</div>
                <div class="metric-value">{quarter_accuracy*100:.1f}%</div>
                <div class="metric-label">winner prediction</div>
            </div>
            
            <div class="metric-card">
                <div class="metric-label">Overall Error</div>
                <div class="metric-value">{overall_mae:.1f}</div>
                <div class="metric-label">points (avg)</div>
            </div>
            
            <div class="metric-card">
                <div class="metric-label">Overall Accuracy</div>
                <div class="metric-value">{overall_accuracy*100:.1f}%</div>
                <div class="metric-label">winner prediction</div>
            </div>
            """
            html += """</div>"""
            similar_games = prediction_analyzer.find_similar_games(game.to_dict())
            if similar_games:
                html += """
                <div class="similar-games">
                    <h3>Similar Historical Games</h3>
                """
                for similar in similar_games:
                    similarity = similar.get('similarity', 0) * 100
                    html += f"""
                    <div class="similar-game-item">
                        <div>
                            <strong>{similar.get('home_team', '')} vs {similar.get('away_team', '')}</strong>
                            <span style="float:right; color:#666">Similarity: {similarity:.0f}%</span>
                        </div>
                        <div style="margin-top:0.5rem">
                            <div style="display:flex; justify-content:space-between">
                                <div>
                                    <span style="color:#666">Predicted:</span> 
                                    {similar.get('prediction', {}).get('home', 0):.1f} - {similar.get('prediction', {}).get('away', 0):.1f}
                                </div>
                                <div>
                                    <span style="color:#666">Actual:</span> 
                                    {similar.get('actual', {}).get('home', 0)} - {similar.get('actual', {}).get('away', 0)}
                                </div>
                            </div>
                            <div style="font-size:0.9rem; color:#666; text-align:right">
                                Error: {similar.get('error', {}).get('avg', 0):.1f} points average
                            </div>
                        </div>
                    </div>
                    """
                html += """</div>"""
            evolution_data = prediction_analyzer.get_prediction_evolution(game_id)
            if evolution_data and evolution_data.get('count', 0) > 1:
                chart_data = {
                    'labels': [f"Update {i+1}" for i in range(len(evolution_data.get('timestamps', [])))],
                    'datasets': [
                        {
                            'label': f"{evolution_data.get('home_team', 'Home')} Score",
                            'data': evolution_data.get('home_scores', []),
                            'borderColor': '#0d6efd',
                            'backgroundColor': 'rgba(13, 110, 253, 0.1)',
                        },
                        {
                            'label': f"{evolution_data.get('away_team', 'Away')} Score",
                            'data': evolution_data.get('away_scores', []),
                            'borderColor': '#dc3545',
                            'backgroundColor': 'rgba(220, 53, 69, 0.1)',
                        },
                        {
                            'label': 'Win Probability (%)',
                            'data': [wp * 100 for wp in evolution_data.get('win_probabilities', [])],
                            'borderColor': '#198754',
                            'backgroundColor': 'rgba(25, 135, 84, 0.1)',
                            'yAxisID': 'y1'
                        }
                    ],
                    'options': {
                        'scales': {
                            'y': {
                                'title': {'display': True, 'text': 'Score'}
                            },
                            'y1': {
                                'position': 'right',
                                'title': {'display': True, 'text': 'Win Probability (%)'},
                                'min': 0,
                                'max': 100,
                                'grid': {'drawOnChartArea': False}
                            }
                        }
                    }
                }
                if evolution_data.get('actual_results'):
                    actuals = evolution_data.get('actual_results')
                    chart_data['datasets'].append({
                        'label': 'Actual Home',
                        'data': [None] * (len(evolution_data.get('quarters', [])) - 1) + [actuals.get('home_score')],
                        'borderColor': '#0d6efd',
                        'backgroundColor': 'rgba(13, 110, 253, 0.5)',
                        'borderWidth': 2,
                        'pointRadius': 6,
                        'pointStyle': 'star'
                    })
                    chart_data['datasets'].append({
                        'label': 'Actual Away',
                        'data': [None] * (len(evolution_data.get('quarters', [])) - 1) + [actuals.get('away_score')],
                        'borderColor': '#dc3545',
                        'backgroundColor': 'rgba(220, 53, 69, 0.5)',
                        'borderWidth': 2,
                        'pointRadius': 6,
                        'pointStyle': 'star'
                    })
                html += f"""
                <div class="prediction-evolution mt-4">
                    <h3>Prediction Evolution</h3>
                    <div class="chart-container" style="position: relative; height: 300px;"></div>
                    <div class="evolution-data" style="display: none;">
                        {json.dumps(chart_data)}
                    </div>
                </div>
                """
            html += """</div>"""  # Close tab content
        html += """
        </div>
        """
        return html
    
    def render_dashboard(self, active_games_df, alerts, prediction_analyzer, container_id="monitoring-dashboard"):
        # IMPORTANT: Use only the active games that are marked as live.
        live_games_df = active_games_df[active_games_df['game_status'].str.lower() == 'live']
        html = f"""
        <div id="{container_id}" class="dashboard">
            <style>
                {self.global_css}
            </style>
            {self.render_header()}
            {self.render_active_games(live_games_df)}
            {self.render_alerts(alerts)}
            {self.render_historical_comparisons(live_games_df, prediction_analyzer)}
            <script>
                {self.dashboard_js}
            </script>
        </div>
        """
        return html

# ====================================================================
# CONTROLLER / COORDINATOR
# ====================================================================

class MonitoringDashboard:
    """
    Main dashboard controller that coordinates components.
    """
    def __init__(self, storage_dir: str = "./prediction_history"):
        self.data_store = DataStore(storage_dir)
        self.alert_manager = AlertManager(self.data_store)
        self.prediction_analyzer = PredictionAnalyzer(self.data_store)
        self.template_engine = DashboardTemplateEngine()
        self.active_games_df = pd.DataFrame()
        self.last_update = datetime.now()
    
    def add_prediction(self, prediction_data, check_for_alerts=True):
        if not isinstance(prediction_data, dict) or 'game_id' not in prediction_data:
            logger.warning("Invalid prediction data: missing game_id")
            return False
        success = self.data_store.save_prediction(prediction_data)
        if not success:
            return False
        if check_for_alerts:
            self.alert_manager.check_for_alerts(prediction_data['game_id'])
        return True
    
    def update_with_active_games(self):
        # Use our helper function to fetch live games from Supabase
        active_games_df = fetch_live_games_from_supabase()
        if active_games_df.empty:
            logger.info("No active games found.")
            self.active_games_df = active_games_df
            return False
        self.active_games_df = active_games_df
        for _, game in active_games_df.iterrows():
            self.add_prediction(game.to_dict(), check_for_alerts=True)
        self.last_update = datetime.now()
        return True
    
    def update_with_actual_results(self, game_id, actual_results):
        latest_prediction = self.data_store.get_recent_prediction(str(game_id))
        if not latest_prediction:
            return {'status': 'error', 'message': 'Game not found'}
        updated_prediction = latest_prediction.copy()
        updated_prediction['actual_home_score'] = actual_results.get('home_score', 0)
        updated_prediction['actual_away_score'] = actual_results.get('away_score', 0)
        updated_prediction['timestamp'] = datetime.now().isoformat()
        self.data_store.save_prediction(updated_prediction)
        update_summary = self.prediction_analyzer.update_accuracy_metrics(str(game_id), actual_results)
        return update_summary
    
    def render_dashboard(self, include_alerts=True, include_history=True):
        alerts = []
        if include_alerts:
            alerts = self.alert_manager.get_recent_alerts()
        html = self.template_engine.render_dashboard(self.active_games_df, alerts, self.prediction_analyzer)
        return html
    
    def display_in_notebook(self, include_alerts=True, include_history=True):
        from IPython.display import display, HTML, clear_output
        dashboard_html = self.render_dashboard(include_alerts=include_alerts, include_history=include_history)
        clear_output(wait=True)
        display(HTML(dashboard_html))
    
    def start_live_monitoring(self, fetch_games_func, update_interval=60, max_updates=None, is_notebook=True):
        update_count = 0
        try:
            while max_updates is None or update_count < max_updates:
                logger.info(f"Running monitoring update #{update_count+1}")
                try:
                    active_games = fetch_games_func()
                    if active_games is not None and not active_games.empty:
                        self.update_with_active_games()
                        if is_notebook:
                            self.display_in_notebook()
                        else:
                            print(f"\n=== Monitoring Update #{update_count+1} ===")
                            print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                            print(f"Active Games: {len(active_games)}")
                            print(f"Total Alerts: {len(self.alert_manager.alerts)}")
                            recent_alerts = self.alert_manager.get_recent_alerts(3)
                            if recent_alerts:
                                print("\nRecent Alerts:")
                                for alert in recent_alerts:
                                    print(f"- {alert.get('alert_type', 'Alert')}: {alert.get('message', '')}")
                    else:
                        logger.info("No active games found for monitoring")
                        if is_notebook:
                            self.display_in_notebook()
                        else:
                            print("No active games found for monitoring")
                except Exception as e:
                    logger.error(f"Error fetching games: {e}")
                update_count += 1
                if max_updates is None or update_count < max_updates:
                    time.sleep(update_interval)
        except KeyboardInterrupt:
            logger.info("Monitoring stopped by user")
        except Exception as e:
            logger.error(f"Error in monitoring: {e}")
            import traceback
            traceback.print_exc()
        finally:
            logger.info("Monitoring ended")
    
    def get_active_games(self):
        return self.active_games_df
    
    def get_alerts(self, limit=10):
        return self.alert_manager.get_recent_alerts(limit)
    
    def get_game_history(self, game_id):
        return self.data_store.get_game_history(str(game_id))
    
    def get_accuracy_summary(self):
        return self.prediction_analyzer.get_accuracy_summary()

# ====================================================================
# USAGE EXAMPLE
# ====================================================================

def demonstrate_dashboard():
    # For demonstration, fetch live data from Supabase.
    dashboard = MonitoringDashboard()
    dashboard.update_with_active_games()
    try:
        dashboard.display_in_notebook()
    except:
        print(dashboard.render_dashboard())
    return dashboard

if __name__ == "__main__":
    dashboard = demonstrate_dashboard()
